# Preparation: imports and data load

### imports

In [ ]:
# standard
import numpy as np
import matplotlib.pyplot as plt
import importlib
import os
import h5py
import json
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import time
import math
import gc


# custom

import axolotl_utils_ram
importlib.reload(axolotl_utils_ram)

from axolotl_utils_ram import extract_snippets_fast_ram, compute_and_plot_sta


import lighthouse_utils
importlib.reload(lighthouse_utils)

from lighthouse_utils import (
    find_valley_and_times,
    build_left_low_high_eis,
    filter_valley_spikes_by_batches,
    finalize_ei_with_cap,
    compute_prev_metrics, 
    plot_amp_vs_prev_metrics,
    plot_amp_prev_scatter_and_bar,
    lighthouse_morpho_pc_gmm
)


import plot_ei_waveforms
importlib.reload(plot_ei_waveforms)
import plot_ei_waveforms as pew

import collision_utils
importlib.reload(collision_utils)


import compute_sta_from_spikes
importlib.reload(compute_sta_from_spikes)


import joint_utils
importlib.reload(joint_utils)

from joint_utils import (
    cosine_two_eis
)

### load raw data and subtract baselines

In [ ]:
# --- Path and recording setup ---

# --- Path and recording setup ---
dat_path = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004.dat"
n_channels = 512
dtype = np.int16

# Sampling rate (your usual)
fs = 20_000

# What chunk to load
start_minutes   = 0
minutes_to_load = 30

# --- Get total number of samples in file ---
file_size_bytes = os.path.getsize(dat_path)
total_samples_in_file = file_size_bytes // (np.dtype(dtype).itemsize * n_channels)

# --- Convert desired time window -> samples ---
start_sample = int(start_minutes * 60 * fs)
n_samples    = int(minutes_to_load * 60 * fs)

# Clamp to file bounds
if start_sample >= total_samples_in_file:
    raise ValueError(f"start_sample={start_sample} beyond file length {total_samples_in_file}")

n_samples = min(n_samples, total_samples_in_file - start_sample)

# --- Read only that chunk ---
offset_bytes = start_sample * n_channels * np.dtype(dtype).itemsize
count_vals   = n_samples * n_channels

with open(dat_path, "rb") as f:
    f.seek(offset_bytes, os.SEEK_SET)
    raw_data = np.fromfile(f, dtype=dtype, count=count_vals)

raw_data = raw_data.reshape((n_samples, n_channels))  # [T, C]
print(f"Loaded raw_data {raw_data.shape} = {n_samples/fs/60:.1f} minutes")



# ONLY EI POSITIONS
h5_in_path = '/Volumes/Lab/Users/alexth/axolotl/201703151_kilosort_data001_spike_times.h5'  # from MATLAB export, to get EI positions

with h5py.File(h5_in_path, 'r') as f:
    # Load electrode positions
    ei_positions = f['/ei_positions'][:].T  # shape becomes [512 x 2]



baseline_path = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_baseline_derivative_20k_30min.json"

segment_len = 20_000
if os.path.exists(baseline_path):
    print(f"Loading baselines")
    with open(baseline_path, 'r') as f:
        data = json.load(f)
    baselines = np.array(data['baselines'], dtype=np.float32)
else:
    print(f"Computing baselines")
    baselines = axolotl_utils_ram.compute_baselines_int16_deriv_robust(raw_data, segment_len=segment_len, diff_thresh=10, trim_fraction=0.15) # shape (512, 360)

    with open(baseline_path, 'w') as f:
        json.dump({
            'baselines': baselines.tolist(),
        }, f)

print("subtracting baselines")

axolotl_utils_ram.subtract_segment_baselines_int16(raw_data=raw_data,
                                     baselines_f32=baselines,
                                     segment_len=segment_len) 




### copy original for easy recovery

In [ ]:

raw_orig = raw_data                      # keep a name for “do not touch”
raw_orig.setflags(write=False)           # any accidental write will error

raw_sub = raw_orig.copy()                # your working residual
raw_sub.setflags(write=True)

print("raw_orig dtype/shape:", raw_orig.dtype, raw_orig.shape, "GB:", raw_orig.nbytes/1e9)
print("raw_sub  dtype/shape:", raw_sub.dtype,  raw_sub.shape,  "GB:", raw_sub.nbytes/1e9)
print("shares_memory:", np.shares_memory(raw_orig, raw_sub))

In [ ]:
# Rename to the only two names we will use
raw_mod = raw_sub        # working copy you will subtract into
# raw_orig already exists and is the frozen original

# Kill the confusing aliases (optional but recommended)
del raw_sub
del raw_data             # raw_data was just another name for raw_orig

# Safety flags
raw_orig.setflags(write=False)  # original is read-only
raw_mod.setflags(write=True)    # working copy is writable

print("raw_orig is frozen, raw_mod is mutable.")
print("shares_memory:", np.shares_memory(raw_orig, raw_mod))

### restore from original

In [ ]:
raw_mod[:] = raw_orig

### load triggers

In [ ]:
path = '/Volumes/Lab/Users/alexth/axolotl/201711290/data004_triggers.h5'
with h5py.File(path, 'r') as h5:
    triggers_sec = h5['/triggers'][...]         # shape (11500, 1) or (11500,)
triggers_sec = np.ravel(triggers_sec).astype(np.float64)  # 1D float64

triggers_sec = triggers_sec[triggers_sec < 30*60]   # keep only triggers in first 30 minutes

len(triggers_sec)  # should be 11500 for full

### load onsets

In [ ]:
def read_onsets(path):
    with h5py.File(path, "r") as h5:
        on  = np.ravel(h5["/onsets_sec"][...]).astype(np.float64)
        dur = np.ravel(h5["/dur_sec"][...]).astype(np.float64)
        fr  = np.ravel(h5["/frame_idx"][...]).astype(np.int64)   # 0-based
        blk = np.ravel(h5["/block_idx"][...]).astype(np.int64)   # 0-based
        tri = np.ravel(h5["/trial_in_block"][...]).astype(np.int64)
    return on, dur, fr, blk, tri

# ---- UNIQUE ----
path_u = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_onset_unique.h5"
on_u, dur_u, fr_u, blk_u, tri_u = read_onsets(path_u)

# optional truncate to first 30 minutes (keeps alignment across arrays)
keep = np.isfinite(on_u) & (on_u < 30*60)
on_u30, dur_u30, fr_u30 = on_u[keep], dur_u[keep], fr_u[keep]
len(on_u), len(on_u30)

# ---- REPEATS ----
path_r = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_onset_repeat.h5"
on_r, dur_r, fr_r, blk_r, tri_r = read_onsets(path_r)

keep = np.isfinite(on_r) & (on_r < 30*60)
on_r30, dur_r30, fr_r30 = on_r[keep], dur_r[keep], fr_r[keep]
len(on_r), len(on_r30)

# ---- spike-window extraction helper (per onset) ----
# spikes_sec should be a sorted 1D float64 array for a cell
def spikes_in_trial(spikes_sec, onset_sec, dur_sec=0.5):
    # Use dur_sec from file: it already applies min(0.5, next_onset-onset)
    a = onset_sec
    b = onset_sec + dur_sec
    i0 = np.searchsorted(spikes_sec, a, side="left")
    i1 = np.searchsorted(spikes_sec, b, side="left")
    return spikes_sec[i0:i1] - onset_sec  # relative times in [0,dur)

# Example usage:
# rel = spikes_in_trial(spikes_sec, on_r[123], dur_r[123])  # repeat presentation 123

# ATTEMPT

In [ ]:
# === STEP 1 ONLY: valley selection + return LEFT / largest RIGHT spikes ===
# Assumes:
#   raw_mod : (T, C) baseline-subtracted array
#   find_valley_and_times imported from lighthouse_utils
#
# Returns:
#   - left_times, left_amps
#   - right_times, right_amps   (up to N_RIGHT_MAX largest right-of-threshold spikes)
#   - threshold_used
#   - if no accepted valley: returns up to N_RIGHT_MAX largest spikes overall in right_times/right_amps

import numpy as np
import matplotlib.pyplot as plt

from lighthouse_utils import find_valley_and_times

# -----------------------------
# Set me
# -----------------------------
# ch = 4

# -----------------------------
# Knobs
# -----------------------------
win = (-40, 80)
N_RIGHT_MAX = 5000
PLOT_HIST = True

VALLEY_KW = dict(
    window=win,
    start=0,
    stop=None,
    bin_width=10.0,
    valley_bins=5,
    min_valid_count=500,
    ratio_base=3,
    ratio_step=500,
    ratio_floor=2,
    ratio_cap=10,
)


def _pick_n_most_negative(times_1d, amps_1d, n_max):
    times_1d = np.asarray(times_1d, dtype=np.int64)
    amps_1d = np.asarray(amps_1d, dtype=np.float32)

    if times_1d.size == 0:
        return (
            np.array([], dtype=np.int64),
            np.array([], dtype=np.float32),
        )

    order = np.argsort(amps_1d)   # most negative first
    keep = order[:min(int(n_max), order.size)]
    return times_1d[keep], amps_1d[keep]


def _plot_amp_hist_with_valley(step1, ch):
    edges = np.asarray(step1["amp_hist_edges"], dtype=np.float64)
    counts = np.asarray(step1["amp_hist_counts"], dtype=np.float64)

    plt.figure(figsize=(12, 3))
    plt.bar(
        edges[:-1],
        counts,
        width=np.diff(edges),
        align="edge"
    )

    vlow = step1.get("valley_low", None)
    vhigh = step1.get("valley_high", None)
    if vlow is not None and vhigh is not None and np.isfinite(vlow) and np.isfinite(vhigh):
        plt.axvspan(vlow, vhigh, color="r", alpha=0.25)

    plt.title(f"Channel {ch}: amplitude histogram with valley")
    plt.xlabel("Amplitude on detection channel")
    plt.ylabel("Count")
    plt.ylim(0, 5000)
    plt.tight_layout()
    plt.show()


def valley_step1_only(ch, n_right_max=5000, plot_hist=True):
    C = raw_mod.shape[1]
    if not (0 <= ch < C):
        raise ValueError(f"ch={ch} out of range 0..{C-1}")

    step1 = find_valley_and_times(raw_mod, ch, **VALLEY_KW)

    left_times = np.asarray(step1.get("left_times", []), dtype=np.int64)
    left_amps  = np.asarray(step1.get("left_vals",  []), dtype=np.float32)

    all_times = np.asarray(step1.get("all_times", []), dtype=np.int64)
    all_amps  = np.asarray(step1.get("all_vals",  []), dtype=np.float32)

    valley_low = float(step1.get("valley_low", np.nan))
    valley_high = float(step1.get("valley_high", np.nan))
    accepted = bool(step1.get("accepted", False))

    print(
        f"\n[CH {ch}] valley [{valley_low:.1f}, {valley_high:.1f}) "
        f"Left={step1.get('left_count', -1)} Valley={step1.get('valley_count', -1)} "
        f"R*={step1.get('required_ratio', np.nan)}"
    )
    print(f"[CH {ch}] accepted = {accepted}")
    print(f"[CH {ch}] LEFT spikes = {left_times.size}")

    if plot_hist:
        _plot_amp_hist_with_valley(step1, ch)

    if accepted and np.isfinite(valley_low):
        right_mask = (all_amps >= valley_low)
        right_pool_times = all_times[right_mask]
        right_pool_amps  = all_amps[right_mask]

        right_times, right_amps = _pick_n_most_negative(
            right_pool_times,
            right_pool_amps,
            n_right_max,
        )

        print(
            f"[CH {ch}] RIGHT pool (amps >= valley_low): {right_pool_times.size} spikes | "
            f"returning {right_times.size} largest right spikes"
        )

        threshold_used = valley_low
        fallback_used = False

    else:
        right_times, right_amps = _pick_n_most_negative(
            all_times,
            all_amps,
            n_right_max,
        )

        print(
            f"[CH {ch}] no accepted valley -> returning {right_times.size} largest spikes overall"
        )

        threshold_used = np.nan
        fallback_used = True

    return dict(
        ch=int(ch),
        accepted=accepted,
        fallback_used=fallback_used,
        threshold_used=float(threshold_used) if np.isfinite(threshold_used) else np.nan,

        left_times=left_times,
        left_amps=left_amps,

        right_times=right_times,
        right_amps=right_amps,

        step1=step1,
    )

ch = 116
out1 = valley_step1_only(ch, n_right_max=N_RIGHT_MAX, plot_hist=PLOT_HIST)


# 395 - double LH? weird. Super LH and then probably a double cell hump..
# 469, 368, 505, 340   best pure
# 253, 405, 445 - good check. 445 - very good LH plus soup of 2. 405 - sus, many uncertain in the TR category
# 461 - main unit plus soup, UNRELIABLE
# 33 - LH plus A LOT OF soup. 
# 366 (plus 2 cells in soup), 445 - mild contam (~50 spikes), and 469 - pure (17spikes), 340 - the most pure (9-10spikes)
# 116, 279, 298 - complex channel with very good LH AND weird artifact/cell

# 53 - fake LH (2 units) but very good; 
# 353 - fake LH (2 units), not very good; MAY work after 357 first.. TEST FOR SOUP-like stuff
# 4 - bad LH (large valley), soup adds 2 good units
# 8 - interesting case. Small bump which is a unit plus another unit plus artifact. Plus more!

In [ ]:
# === STEP 2: mid-left subset -> EI -> significant channels -> kmeans(k=2) -> cluster EIs ===
# Added:
#   energy asymmetry report immediately after computing ei0 / ei1
#
# Assumes:
#   raw_mod
#   ei_positions
#   out1          : output from valley_step1_only(...)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from axolotl_utils_ram import extract_snippets_fast_ram
from collision_utils import median_ei_adaptive

try:
    pew
except NameError:
    import plot_ei_waveforms as pew

# -----------------------------
# Knobs
# -----------------------------
win = (-20, 40)

DISCOVERY_MAX = 5000
DISCOVERY_RNG_SEED = 123

LEFT_PCT_LOW = 30.0
LEFT_PCT_HIGH = 70.0

SIG_P2P_THR = 50.0
SIG_MAX_CHANNELS = 80
SIG_MIN_CHANNELS = 10

KMEANS_N_INIT = 50
KMEANS_RANDOM_STATE = 0

ENERGY_FRAC_DIFF_THR = 0.30   # "significantly bigger" = >30% bigger RMS


def _select_sig_channels_from_ei(
    ei,
    p2p_thr=30.0,
    max_channels=80,
    min_channels=10,
    force_include_main=True,
):
    ei = np.asarray(ei, dtype=np.float32)
    p2p = np.ptp(ei, axis=1)

    chans = np.flatnonzero(p2p >= float(p2p_thr)).astype(np.int32)
    if chans.size == 0:
        chans = np.argsort(p2p)[-int(min_channels):].astype(np.int32)

    chans = chans[np.argsort(p2p[chans])[::-1]]

    if chans.size > int(max_channels):
        chans = chans[:int(max_channels)]

    if chans.size < int(min_channels):
        top_more = np.argsort(p2p)[-int(min_channels):][::-1].astype(np.int32)
        chans = np.unique(np.concatenate([chans, top_more])).astype(np.int32)
        chans = chans[np.argsort(p2p[chans])[::-1]]

    if force_include_main:
        main_ch = int(np.argmin(np.min(ei, axis=1)))
        if main_ch not in chans:
            chans = np.unique(np.concatenate([chans, np.array([main_ch], dtype=np.int32)])).astype(np.int32)
            chans = chans[np.argsort(p2p[chans])[::-1]]

    return chans.astype(np.int32), p2p.astype(np.float32)

def _short_isi_pair_stats_by_cluster(times, labels, lo_samp=10, hi_samp=30):
    """
    times  : [N] spike times in samples
    labels : [N] cluster labels (0/1), aligned to times

    Counts adjacent pairs in time order with ISI in [lo_samp, hi_samp].
    Reports:
      - cross-cluster short-ISI pairs
      - within-cluster short-ISI pairs for cluster 0 and cluster 1
      - fractions normalized by total adjacent pairs of that type
    """
    times = np.asarray(times, dtype=np.int64)
    labels = np.asarray(labels, dtype=np.int8)

    if times.size != labels.size:
        raise ValueError("times and labels must have same length")

    if times.size < 2:
        return dict(
            n_pairs_total=0,
            n_short_total=0,
            n_cross_pairs_total=0,
            n_cross_short=0,
            frac_cross_short=np.nan,
            n_within0_pairs_total=0,
            n_within0_short=0,
            frac_within0_short=np.nan,
            n_within1_pairs_total=0,
            n_within1_short=0,
            frac_within1_short=np.nan,
        )

    order = np.argsort(times)
    t = times[order]
    lab = labels[order]

    dt = np.diff(t)
    short = (dt >= int(lo_samp)) & (dt <= int(hi_samp))

    left_lab = lab[:-1]
    right_lab = lab[1:]

    cross_mask = left_lab != right_lab
    within0_mask = (left_lab == 0) & (right_lab == 0)
    within1_mask = (left_lab == 1) & (right_lab == 1)

    n_pairs_total = int(dt.size)
    n_short_total = int(np.sum(short))

    n_cross_pairs_total = int(np.sum(cross_mask))
    n_cross_short = int(np.sum(short & cross_mask))
    frac_cross_short = (
        n_cross_short / n_cross_pairs_total if n_cross_pairs_total > 0 else np.nan
    )

    n_within0_pairs_total = int(np.sum(within0_mask))
    n_within0_short = int(np.sum(short & within0_mask))
    frac_within0_short = (
        n_within0_short / n_within0_pairs_total if n_within0_pairs_total > 0 else np.nan
    )

    n_within1_pairs_total = int(np.sum(within1_mask))
    n_within1_short = int(np.sum(short & within1_mask))
    frac_within1_short = (
        n_within1_short / n_within1_pairs_total if n_within1_pairs_total > 0 else np.nan
    )

    return dict(
        n_pairs_total=n_pairs_total,
        n_short_total=n_short_total,
        n_cross_pairs_total=n_cross_pairs_total,
        n_cross_short=n_cross_short,
        frac_cross_short=frac_cross_short,
        n_within0_pairs_total=n_within0_pairs_total,
        n_within0_short=n_within0_short,
        frac_within0_short=frac_within0_short,
        n_within1_pairs_total=n_within1_pairs_total,
        n_within1_short=n_within1_short,
        frac_within1_short=frac_within1_short,
    )

def _extract_all_channels(times, window):
    all_ch = np.arange(raw_mod.shape[1], dtype=np.int32)
    snips, valid_times = extract_snippets_fast_ram(
        raw_mod,
        np.asarray(times, dtype=np.int64),
        window=window,
        selected_channels=all_ch,
    )
    return snips.astype(np.float32, copy=False), np.asarray(valid_times, dtype=np.int64)


def _report_ei_energy_asymmetry(ei0, ei1, frac_thr=0.30, p2p_thr=30.0, min_channels=10):
    """
    Energy = RMS across time, computed per channel on the EI waveform.

    Only evaluates channels that have significant signal on AT LEAST ONE EI,
    using the same spirit as other channel-selection code.

    Reports:
      - channels where EI0 RMS > EI1 RMS by more than frac_thr
      - channels where EI1 RMS > EI0 RMS by more than frac_thr
      - mean signed fractional difference across the selected channels
    """
    ei0 = np.asarray(ei0, dtype=np.float32)
    ei1 = np.asarray(ei1, dtype=np.float32)

    rms0 = np.sqrt(np.mean(ei0**2, axis=1)).astype(np.float32)
    rms1 = np.sqrt(np.mean(ei1**2, axis=1)).astype(np.float32)

    p2p0 = np.ptp(ei0, axis=1).astype(np.float32)
    p2p1 = np.ptp(ei1, axis=1).astype(np.float32)

    # significant on at least one EI
    sel = np.flatnonzero((p2p0 >= float(p2p_thr)) | (p2p1 >= float(p2p_thr))).astype(np.int32)

    # fallback: if threshold is too strict, keep top channels by max p2p across the two EIs
    if sel.size < int(min_channels):
        score = np.maximum(p2p0, p2p1)
        sel = np.argsort(score)[-int(min_channels):].astype(np.int32)
        sel = sel[np.argsort(np.maximum(p2p0[sel], p2p1[sel]))[::-1]]

    rms0_sel = rms0[sel]
    rms1_sel = rms1[sel]

    denom = np.maximum(np.maximum(rms0_sel, rms1_sel), 1e-8)
    frac_signed_sel = (rms0_sel - rms1_sel) / denom   # positive => EI0 bigger, negative => EI1 bigger

    ei0_bigger_mask = frac_signed_sel > float(frac_thr)
    ei1_bigger_mask = frac_signed_sel < -float(frac_thr)

    ch0 = sel[ei0_bigger_mask].astype(np.int32)
    ch1 = sel[ei1_bigger_mask].astype(np.int32)

    n0 = int(ch0.size)
    n1 = int(ch1.size)

    mean_signed_all = float(np.mean(frac_signed_sel))
    mean_abs_all = float(np.mean(np.abs(frac_signed_sel)))

    mean_signed_ei0big = float(np.mean(frac_signed_sel[ei0_bigger_mask])) if n0 > 0 else np.nan
    mean_signed_ei1big = float(np.mean(frac_signed_sel[ei1_bigger_mask])) if n1 > 0 else np.nan

    print(f"[energy asymmetry] selected channels ({sel.size}): {sel.tolist()}")
    print(
        f"[energy asymmetry] EI0 > EI1 by >{100*frac_thr:.0f}% on {n0} selected channels | "
        f"EI1 > EI0 by >{100*frac_thr:.0f}% on {n1} selected channels"
    )
    print(
        f"[energy asymmetry] mean signed frac diff across selected channels = {mean_signed_all:.3f} "
        f"(positive => EI0 bigger)"
    )
    print(f"[energy asymmetry] mean abs frac diff across selected channels = {mean_abs_all:.3f}")

    if n0 > 0:
        print(
            f"[energy asymmetry] EI0-bigger channels: {ch0.tolist()} | "
            f"mean signed frac diff there = {mean_signed_ei0big:.3f}"
        )
    if n1 > 0:
        print(
            f"[energy asymmetry] EI1-bigger channels: {ch1.tolist()} | "
            f"mean signed frac diff there = {mean_signed_ei1big:.3f}"
        )

    return dict(
        selected_channels=sel,
        rms0=rms0,
        rms1=rms1,
        p2p0=p2p0,
        p2p1=p2p1,
        frac_signed_selected=frac_signed_sel,
        channels_ei0_bigger=ch0,
        channels_ei1_bigger=ch1,
        n_channels_ei0_bigger=n0,
        n_channels_ei1_bigger=n1,
        mean_signed_frac_diff_selected=mean_signed_all,
        mean_abs_frac_diff_selected=mean_abs_all,
        mean_signed_frac_diff_ei0_bigger=mean_signed_ei0big,
        mean_signed_frac_diff_ei1_bigger=mean_signed_ei1big,
    )


def midleft_kmeans_and_harm(step1_out):
    ch = int(step1_out["ch"])
    left_times = np.asarray(step1_out["left_times"], dtype=np.int64)
    left_amps = np.asarray(step1_out["left_amps"], dtype=np.float32)

    if left_times.size == 0:
        raise ValueError("No left spikes in step1_out.")
    if left_times.size != left_amps.size:
        raise ValueError("left_times and left_amps must have the same length.")

    # ---- 1) 30-70 percentile left spikes ----
    q_lo = float(np.percentile(left_amps, LEFT_PCT_LOW))
    q_hi = float(np.percentile(left_amps, LEFT_PCT_HIGH))

    mid_mask = (left_amps >= q_lo) & (left_amps <= q_hi)
    mid_times_all = left_times[mid_mask]

    print(
        f"\n[CH {ch}] LEFT amplitude percentiles: "
        f"p{LEFT_PCT_LOW:.0f}={q_lo:.1f}, p{LEFT_PCT_HIGH:.0f}={q_hi:.1f}"
    )
    print(f"[CH {ch}] mid-left pool size = {mid_times_all.size}")

    if mid_times_all.size == 0:
        raise ValueError("No left spikes in the requested percentile range.")

    rng_disc = np.random.default_rng(DISCOVERY_RNG_SEED)
    if mid_times_all.size > int(DISCOVERY_MAX):
        pick_idx = rng_disc.choice(mid_times_all.size, size=int(DISCOVERY_MAX), replace=False)
        disc_times = np.sort(mid_times_all[pick_idx]).astype(np.int64, copy=False)
    else:
        disc_times = np.sort(mid_times_all).astype(np.int64, copy=False)

    print(f"[CH {ch}] discovery set = {disc_times.size} spikes")

    # ---- 2) discovery snippets on all channels ----
    snips_disc, disc_valid_times = _extract_all_channels(disc_times, win)
    if snips_disc.shape[2] == 0:
        raise ValueError("No valid snippets in discovery set.")

    print(f"[CH {ch}] discovery valid snippets = {snips_disc.shape[2]}")

    # ---- 3) median EI from discovery set ----
    ei_mid = median_ei_adaptive(snips_disc).astype(np.float32, copy=False)

    # ---- 4) significant channels from discovery EI ----
    sig_chans_mid, _ = _select_sig_channels_from_ei(
        ei_mid,
        p2p_thr=SIG_P2P_THR,
        max_channels=SIG_MAX_CHANNELS,
        min_channels=SIG_MIN_CHANNELS,
        force_include_main=True,
    )

    print(f"[CH {ch}] discovery significant channels ({sig_chans_mid.size}): {sig_chans_mid.tolist()}")

    # ---- 5) k-means (k=2) on discovery spikes using significant channels only ----
    X = snips_disc[sig_chans_mid].transpose(2, 0, 1).reshape(snips_disc.shape[2], -1).astype(np.float32)

    km = KMeans(
        n_clusters=2,
        n_init=KMEANS_N_INIT,
        random_state=KMEANS_RANDOM_STATE,
    )
    labels = km.fit_predict(X)

    n0 = int(np.sum(labels == 0))
    n1 = int(np.sum(labels == 1))
    print(f"[CH {ch}] kmeans sizes: n0={n0}, n1={n1}")

    if n0 == 0 or n1 == 0:
        raise ValueError("KMeans produced an empty cluster.")
    
        # ---- 5b) short-ISI adjacency stats across / within KMeans clusters ----
    isi_diag = _short_isi_pair_stats_by_cluster(
        disc_valid_times,
        labels,
        lo_samp=10,
        hi_samp=30,
    )

    print(
        f"[CH {ch}] short-ISI adjacent pairs in [10,30] samples:"
    )
    print(
        f"  cross-cluster: {isi_diag['n_cross_short']}/{isi_diag['n_cross_pairs_total']} "
        f"({isi_diag['frac_cross_short']:.4f})"
        if np.isfinite(isi_diag['frac_cross_short']) else
        f"  cross-cluster: {isi_diag['n_cross_short']}/{isi_diag['n_cross_pairs_total']} (nan)"
    )
    print(
        f"  within cluster 0: {isi_diag['n_within0_short']}/{isi_diag['n_within0_pairs_total']} "
        f"({isi_diag['frac_within0_short']:.4f})"
        if np.isfinite(isi_diag['frac_within0_short']) else
        f"  within cluster 0: {isi_diag['n_within0_short']}/{isi_diag['n_within0_pairs_total']} (nan)"
    )
    print(
        f"  within cluster 1: {isi_diag['n_within1_short']}/{isi_diag['n_within1_pairs_total']} "
        f"({isi_diag['frac_within1_short']:.4f})"
        if np.isfinite(isi_diag['frac_within1_short']) else
        f"  within cluster 1: {isi_diag['n_within1_short']}/{isi_diag['n_within1_pairs_total']} (nan)"
    )

    # ---- 6) PCA scatter for visualization ----
    pca = PCA(n_components=2, svd_solver="randomized", random_state=0)
    Xpc = pca.fit_transform(X)
    vr = pca.explained_variance_ratio_

    plt.figure(figsize=(5, 5))
    plt.scatter(Xpc[labels == 0, 0], Xpc[labels == 0, 1], s=10, alpha=0.7, c="black", label=f"cluster 0 (n={n0})")
    plt.scatter(Xpc[labels == 1, 0], Xpc[labels == 1, 1], s=10, alpha=0.7, c="red",   label=f"cluster 1 (n={n1})")
    plt.xlabel(f"PC1 ({vr[0]*100:.1f}% var)")
    plt.ylabel(f"PC2 ({vr[1]*100:.1f}% var)")
    plt.title(f"CH {ch} | mid-left spikes on significant channels")
    plt.grid(alpha=0.25)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    # ---- 7) cluster EIs on ALL channels ----
    ei0 = median_ei_adaptive(snips_disc[:, :, labels == 0]).astype(np.float32, copy=False)
    ei1 = median_ei_adaptive(snips_disc[:, :, labels == 1]).astype(np.float32, copy=False)

    # ---- 7b) energy asymmetry report ----
    energy_diag = _report_ei_energy_asymmetry(
        ei0,
        ei1,
        frac_thr=ENERGY_FRAC_DIFF_THR,
        p2p_thr=SIG_P2P_THR,
        min_channels=SIG_MIN_CHANNELS,
    )

    # ---- 8) overlaid EI plot ----
    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111)
    pew.plot_ei_waveforms(
        [ei0, ei1],
        positions=ei_positions,
        ref_channel=ch,
        scale=70.0,
        ax=ax,
        colors=["black", "red"],
        alpha=[1.0, 1.0],
        linewidth=[1.0, 1.0],
        box_height=1.0,
        box_width=50.0,
        aspect=0.5,
    )
    ax.set_title(f"CH {ch} | KMeans cluster EIs from mid-left discovery set")
    plt.show()

    return dict(
        ch=ch,
        left_times=left_times.copy(),
        left_amps=left_amps.copy(),
        discovery_times=disc_valid_times,
        discovery_ei=ei_mid,
        discovery_sig_channels=sig_chans_mid,
        Xpc=Xpc.astype(np.float32, copy=False),
        kmeans_labels=labels.astype(np.int8, copy=False),
        ei0=ei0,
        ei1=ei1,
        energy_diag=energy_diag,
        isi_diag=isi_diag,
    )


out2 = midleft_kmeans_and_harm(out1)

In [ ]:
# === Shared-channel comparison: only channels where both EIs have decent signal
# === and similar amplitude. Then compute:
# ===   1) fixed-channel harm maps for EI0 and EI1
# ===   2) delta = harm0 - harm1
# ===   3) scatter of mean delta across selected channels
# ===   4) cosine between EI0 and EI1 on concatenated selected channels

import numpy as np
import matplotlib.pyplot as plt

from axolotl_utils_ram import extract_snippets_fast_ram

# -----------------------------
# Knobs
# -----------------------------
AMP_SIM_FRAC = 0.10      # within 10%
SHARED_P2P_THR = 50.0    # both EIs need at least this p2p on a channel
SHARED_MAX_CH = 40
SHARED_MIN_CH = 3

COMPARE_TEST_MAX = 5000
COMPARE_TEST_RNG_SEED = 456

HARM_LAG_RADIUS = 4
HARM_WEIGHT_BY_P2P = True
HARM_WEIGHT_BETA = 0.7

# win = (-20, 40)


def _roll_zero_2d(arr, shift):
    arr = np.asarray(arr)
    nch, T = arr.shape
    out = np.zeros_like(arr)
    if shift == 0:
        out[:] = arr
    elif shift > 0:
        out[:, shift:] = arr[:, :T-shift]
    else:
        s = -shift
        out[:, :T-s] = arr[:, s:]
    return out


def _pick_random_subset(times, n_max, rng, exclude_times=None):
    times = np.asarray(times, dtype=np.int64)

    if exclude_times is not None and len(exclude_times) > 0:
        exclude_times = np.asarray(exclude_times, dtype=np.int64)
        keep = ~np.isin(times, exclude_times)
        pool = times[keep]
    else:
        pool = times

    if pool.size == 0:
        pool = times.copy()

    if pool.size <= int(n_max):
        return np.sort(pool).astype(np.int64, copy=False)

    idx = rng.choice(pool.size, size=int(n_max), replace=False)
    return np.sort(pool[idx]).astype(np.int64, copy=False)


def compute_harm_map_fixed_channels_noamp(
    ei_sel,
    snips_sel,
    channel_labels=None,
    lag_radius=4,
    weight_by_p2p=True,
    weight_beta=0.7,
):
    ei_sel = np.asarray(ei_sel, dtype=np.float32)
    snips_sel = np.asarray(snips_sel, dtype=np.float32)

    if ei_sel.ndim != 2:
        raise ValueError("ei_sel must be [nch, T]")
    if snips_sel.ndim != 3:
        raise ValueError("snips_sel must be [nch, T, N]")
    if snips_sel.shape[:2] != ei_sel.shape:
        raise ValueError("snips_sel and ei_sel must match on [nch, T]")

    nch, T = ei_sel.shape
    N = snips_sel.shape[2]

    if channel_labels is None:
        channel_labels = np.arange(nch, dtype=np.int32)
    else:
        channel_labels = np.asarray(channel_labels, dtype=np.int32)

    ptp = np.ptp(ei_sel, axis=1).astype(np.float32)
    ch_main_local = int(np.argmin(np.min(ei_sel, axis=1)))
    t_neg = int(np.argmin(ei_sel[ch_main_local]))

    if bool(weight_by_p2p):
        w = np.power(np.maximum(ptp, 1e-6), float(weight_beta)).astype(np.float32)
    else:
        w = np.ones(nch, dtype=np.float32)
    w /= np.maximum(w.sum(), 1e-12)

    rms_raw = np.sqrt((snips_sel ** 2).mean(axis=1)).astype(np.float32)
    E_raw = np.maximum(rms_raw ** 2, 1e-12)

    lags = np.arange(-int(lag_radius), int(lag_radius) + 1, dtype=np.int32)
    deltas = np.empty((lags.size, nch, N), dtype=np.float32)
    ratios = np.empty((lags.size, nch, N), dtype=np.float32)

    for i, d in enumerate(lags):
        shifted = _roll_zero_2d(ei_sel, int(d))
        resid = snips_sel - shifted[:, :, None]
        E_res = (resid ** 2).mean(axis=1)
        rms_res = np.sqrt(E_res)
        deltas[i] = rms_res - rms_raw
        ratios[i] = E_res / E_raw

    mean_deltas = (deltas * w[None, :, None]).sum(axis=1)
    best_i = np.argmin(mean_deltas, axis=0)
    best_lags = lags[best_i]

    harm = np.take_along_axis(deltas, best_i[None, None, :], axis=0).squeeze(0)
    harm_ratio = np.take_along_axis(ratios, best_i[None, None, :], axis=0).squeeze(0)
    harm_r2 = 1.0 - harm_ratio

    out = {
        "selected_channels": channel_labels,
        "channel_ptp": ptp,
        "main_channel": int(channel_labels[ch_main_local]),
        "neg_peak_index": t_neg,
        "lags": lags,
        "best_lag_per_spike": best_lags,
        "harm_matrix": harm,
        "harm_matrix_r2": harm_r2,
        "mean_delta_unweighted": harm.mean(axis=0),
        "mean_delta_weighted": (harm * w[:, None]).sum(axis=0),
        "mean_r2_unweighted": harm_r2.mean(axis=0),
        "mean_r2_weighted": (harm_r2 * w[:, None]).sum(axis=0),
    }
    return out


def _best_lag_cosine_concat(ei0_sel, ei1_sel, max_lag=4, eps=1e-12):
    ei0_sel = np.asarray(ei0_sel, dtype=np.float32)
    ei1_sel = np.asarray(ei1_sel, dtype=np.float32)

    best_cos = -np.inf
    best_lag = 0

    x = ei0_sel.reshape(-1).astype(np.float32)
    x_norm = float(np.linalg.norm(x))

    for lag in range(-int(max_lag), int(max_lag) + 1):
        y2d = _roll_zero_2d(ei1_sel, lag)
        y = y2d.reshape(-1).astype(np.float32)
        y_norm = float(np.linalg.norm(y))
        cos = float(np.dot(x, y) / max(x_norm * y_norm, eps))
        if cos > best_cos:
            best_cos = cos
            best_lag = lag

    return best_cos, best_lag


def _select_shared_similar_channels(ei0, ei1,
                                    p2p_thr=20.0,
                                    amp_sim_frac=0.10,
                                    max_ch=40,
                                    min_ch=6):
    ei0 = np.asarray(ei0, dtype=np.float32)
    ei1 = np.asarray(ei1, dtype=np.float32)

    p2p0 = np.ptp(ei0, axis=1)
    p2p1 = np.ptp(ei1, axis=1)

    neg0 = np.abs(np.min(ei0, axis=1))
    neg1 = np.abs(np.min(ei1, axis=1))

    sig_mask = (p2p0 >= float(p2p_thr)) & (p2p1 >= float(p2p_thr))

    denom = np.maximum(np.maximum(neg0, neg1), 1e-6)
    frac_diff = np.abs(neg0 - neg1) / denom
    sim_mask = frac_diff <= float(amp_sim_frac)

    keep = np.flatnonzero(sig_mask & sim_mask).astype(np.int32)
    shared_score = np.minimum(p2p0, p2p1)

    if keep.size > 0:
        keep = keep[np.argsort(shared_score[keep])[::-1]]

    if keep.size > int(max_ch):
        keep = keep[:int(max_ch)]

    if keep.size < int(min_ch):
        fallback = np.flatnonzero(sig_mask).astype(np.int32)
        if fallback.size == 0:
            fallback = np.argsort(np.minimum(p2p0, p2p1))[-int(min_ch):].astype(np.int32)
        fallback = fallback[np.argsort(shared_score[fallback])[::-1]]
        keep = fallback[:max(int(min_ch), min(int(max_ch), fallback.size))]

    return keep.astype(np.int32), {
        "p2p0": p2p0.astype(np.float32),
        "p2p1": p2p1.astype(np.float32),
        "neg0": neg0.astype(np.float32),
        "neg1": neg1.astype(np.float32),
        "frac_diff": frac_diff.astype(np.float32),
        "shared_score": shared_score.astype(np.float32),
    }


def compare_eis_on_shared_channels(out2, raw_mod, window=(-40, 80)):
    ch = int(out2["ch"])
    ei0 = np.asarray(out2["ei0"], dtype=np.float32)
    ei1 = np.asarray(out2["ei1"], dtype=np.float32)
    left_times = np.asarray(out2["left_times"], dtype=np.int64)
    discovery_times = np.asarray(out2["discovery_times"], dtype=np.int64)

    rng = np.random.default_rng(COMPARE_TEST_RNG_SEED)
    test_times = _pick_random_subset(
        left_times,
        COMPARE_TEST_MAX,
        rng=rng,
        exclude_times=discovery_times,
    )

    shared_ch, diag = _select_shared_similar_channels(
        ei0, ei1,
        p2p_thr=SHARED_P2P_THR,
        amp_sim_frac=AMP_SIM_FRAC,
        max_ch=SHARED_MAX_CH,
        min_ch=SHARED_MIN_CH,
    )

    print(f"\n[CH {ch}] shared comparable channels ({shared_ch.size}): {shared_ch.tolist()}")
    print(f"[CH {ch}] comparison test set: {test_times.size} left spikes")

    print("[shared channels] ch | p2p0 p2p1 | neg0 neg1 | frac_diff")
    for cc in shared_ch:
        print(
            f"  {int(cc):3d} | "
            f"{diag['p2p0'][cc]:6.1f} {diag['p2p1'][cc]:6.1f} | "
            f"{diag['neg0'][cc]:6.1f} {diag['neg1'][cc]:6.1f} | "
            f"{diag['frac_diff'][cc]:.3f}"
        )

    snips_shared, valid_times = extract_snippets_fast_ram(
        raw_mod,
        test_times,
        window=window,
        selected_channels=shared_ch.astype(np.int32),
    )
    snips_shared = np.asarray(snips_shared, dtype=np.float32)

    ei0_shared = ei0[shared_ch].astype(np.float32, copy=False)
    ei1_shared = ei1[shared_ch].astype(np.float32, copy=False)

    harm0s = compute_harm_map_fixed_channels_noamp(
        ei_sel=ei0_shared,
        snips_sel=snips_shared,
        channel_labels=shared_ch,
        lag_radius=HARM_LAG_RADIUS,
        weight_by_p2p=HARM_WEIGHT_BY_P2P,
        weight_beta=HARM_WEIGHT_BETA,
    )

    harm1s = compute_harm_map_fixed_channels_noamp(
        ei_sel=ei1_shared,
        snips_sel=snips_shared,
        channel_labels=shared_ch,
        lag_radius=HARM_LAG_RADIUS,
        weight_by_p2p=HARM_WEIGHT_BY_P2P,
        weight_beta=HARM_WEIGHT_BETA,
    )

    H0 = np.asarray(harm0s["harm_matrix"], dtype=np.float32)
    H1 = np.asarray(harm1s["harm_matrix"], dtype=np.float32)
    Hdiff = H0 - H1

    m0 = H0.mean(axis=0)
    m1 = H1.mean(axis=0)
    mdiff = Hdiff.mean(axis=0)

    mw0 = np.asarray(harm0s["mean_delta_weighted"], dtype=np.float32)
    mw1 = np.asarray(harm1s["mean_delta_weighted"], dtype=np.float32)
    mwdiff = mw0 - mw1

    cos_shared, lag_shared = _best_lag_cosine_concat(
        ei0_shared,
        ei1_shared,
        max_lag=HARM_LAG_RADIUS,
    )
    print(f"[CH {ch}] concatenated cosine on shared comparable channels = {cos_shared:.4f} (best lag {lag_shared})")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].scatter(m0, m1, s=10, alpha=0.6)
    lo = float(min(np.min(m0), np.min(m1)))
    hi = float(max(np.max(m0), np.max(m1)))
    axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
    axes[0].set_xlabel("Mean harm0 across shared channels")
    axes[0].set_ylabel("Mean harm1 across shared channels")
    axes[0].set_title("Per-spike mean harm")
    axes[0].grid(alpha=0.25)

    order = np.argsort(mdiff)
    axes[1].scatter(np.arange(mdiff.size), mdiff[order], s=10, alpha=0.6, label="unweighted")
    axes[1].scatter(np.arange(mwdiff.size), mwdiff[order], s=10, alpha=0.6, label="weighted")
    axes[1].axhline(0.0, color="k", linestyle="--", lw=1)
    axes[1].set_xlabel("Spike rank (sorted by mean delta)")
    axes[1].set_ylabel("Mean(harm0 - harm1)")
    axes[1].set_title("Negative = EI0 better")
    axes[1].grid(alpha=0.25)
    axes[1].legend(frameon=False)

    axes[2].hist(mwdiff, bins=60, alpha=0.8)
    axes[2].axvline(0.0, color="k", linestyle="--", lw=1)
    axes[2].set_xlabel("Weighted mean(harm0 - harm1)")
    axes[2].set_ylabel("Count")
    axes[2].set_title(f"Shared-channel delta hist\ncos={cos_shared:.3f}, lag={lag_shared}")
    axes[2].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

    return dict(
        ch=ch,
        shared_channels=shared_ch,
        shared_diag=diag,
        test_times=valid_times,
        harm0_shared=harm0s,
        harm1_shared=harm1s,
        mean_harm0=m0,
        mean_harm1=m1,
        mean_delta=mdiff,
        mean_delta_weighted=mwdiff,
        cosine_shared=float(cos_shared),
        cosine_shared_lag=int(lag_shared),
    )


out3 = compare_eis_on_shared_channels(out2, raw_mod, window=win)

In [ ]:
# === STEP 4: classify ALL left spikes against EI0 / EI1 from out2, fast ===
# Uses:
#   - selected classification channels from union of significant channels on either EI
#   - best-lag cosine on concatenated snippets
#   - all-left-spike adjacent-pair 10-30 sample check after classification
#
# Assumes:
#   raw_mod
#   out2
#   win

import numpy as np
import matplotlib.pyplot as plt

from axolotl_utils_ram import extract_snippets_fast_ram

# -----------------------------
# Knobs
# -----------------------------
CLASS_P2P_THR = 20.0
CLASS_MAX_CH = 40
CLASS_MIN_CH = 12
CLASS_LAG_RADIUS = 4
COS_MARGIN = 0.0   # keep 0.0 for hard assignment; raise later if you want uncertainty
SHOW_SCATTER = True


def _roll_zero_2d(arr, shift):
    arr = np.asarray(arr)
    nch, T = arr.shape
    out = np.zeros_like(arr)
    if shift == 0:
        out[:] = arr
    elif shift > 0:
        out[:, shift:] = arr[:, :T-shift]
    else:
        s = -shift
        out[:, :T-s] = arr[:, s:]
    return out


def _pick_classification_channels(ei0, ei1, p2p_thr=20.0, max_ch=40, min_ch=12):
    """
    Broad/stable channel set for classifying all left spikes.
    Keep channels where either EI has decent signal, rank by max p2p across templates.
    """
    ei0 = np.asarray(ei0, dtype=np.float32)
    ei1 = np.asarray(ei1, dtype=np.float32)

    p2p0 = np.ptp(ei0, axis=1).astype(np.float32)
    p2p1 = np.ptp(ei1, axis=1).astype(np.float32)
    score = np.maximum(p2p0, p2p1)

    keep = np.flatnonzero((p2p0 >= float(p2p_thr)) | (p2p1 >= float(p2p_thr))).astype(np.int32)

    if keep.size == 0:
        keep = np.argsort(score)[-int(min_ch):].astype(np.int32)

    keep = keep[np.argsort(score[keep])[::-1]]

    if keep.size > int(max_ch):
        keep = keep[:int(max_ch)]

    if keep.size < int(min_ch):
        extra = np.argsort(score)[-int(min_ch):].astype(np.int32)
        keep = np.unique(np.concatenate([keep, extra])).astype(np.int32)
        keep = keep[np.argsort(score[keep])[::-1]]
        if keep.size > int(max_ch):
            keep = keep[:int(max_ch)]

    return keep.astype(np.int32), dict(
        p2p0=p2p0,
        p2p1=p2p1,
        score=score,
    )


def _build_lagged_template_bank(ei_sel, lag_radius=4, eps=1e-12):
    """
    ei_sel: [nch, T]
    Returns:
      bank: [n_lags, D] unit-normalized flattened lagged templates
      lags: [n_lags]
    """
    ei_sel = np.asarray(ei_sel, dtype=np.float32)
    lags = np.arange(-int(lag_radius), int(lag_radius) + 1, dtype=np.int32)

    bank = []
    for lag in lags:
        tmp = _roll_zero_2d(ei_sel, int(lag)).reshape(-1).astype(np.float32)
        norm = np.linalg.norm(tmp)
        bank.append(tmp / max(float(norm), eps))
    bank = np.stack(bank, axis=0).astype(np.float32)
    return bank, lags


def _best_lag_cosine_assign(snips_sel, ei0_sel, ei1_sel, lag_radius=4, eps=1e-12):
    """
    snips_sel: [nch, T, N]
    ei0_sel, ei1_sel: [nch, T]

    Vectorized best-lag cosine scoring.
    Returns per-spike:
      best_cos0, best_cos1, best_lag0, best_lag1, labels
    """
    snips_sel = np.asarray(snips_sel, dtype=np.float32)
    ei0_sel = np.asarray(ei0_sel, dtype=np.float32)
    ei1_sel = np.asarray(ei1_sel, dtype=np.float32)

    nch, T, N = snips_sel.shape
    D = nch * T

    X = snips_sel.transpose(2, 0, 1).reshape(N, D).astype(np.float32)  # [N, D]
    x_norm = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = X / np.maximum(x_norm, eps)

    bank0, lags = _build_lagged_template_bank(ei0_sel, lag_radius=lag_radius, eps=eps)  # [L, D]
    bank1, _ = _build_lagged_template_bank(ei1_sel, lag_radius=lag_radius, eps=eps)

    # cosine scores for all spikes against all lagged templates
    scores0 = Xn @ bank0.T   # [N, n_lags]
    scores1 = Xn @ bank1.T   # [N, n_lags]

    best_idx0 = np.argmax(scores0, axis=1)
    best_idx1 = np.argmax(scores1, axis=1)

    best_cos0 = scores0[np.arange(N), best_idx0].astype(np.float32)
    best_cos1 = scores1[np.arange(N), best_idx1].astype(np.float32)
    best_lag0 = lags[best_idx0].astype(np.int32)
    best_lag1 = lags[best_idx1].astype(np.int32)

    diff = best_cos0 - best_cos1
    labels = np.where(diff >= COS_MARGIN, 0, 1).astype(np.int8)

    return dict(
        best_cos0=best_cos0,
        best_cos1=best_cos1,
        best_lag0=best_lag0,
        best_lag1=best_lag1,
        cos_diff=diff.astype(np.float32),
        labels=labels,
        scores0=scores0.astype(np.float32),
        scores1=scores1.astype(np.float32),
    )


def _adjacent_short_isi_stats(times, labels, lo_samp=10, hi_samp=30):
    """
    Adjacent pairs only, in time order.
    """
    times = np.asarray(times, dtype=np.int64)
    labels = np.asarray(labels, dtype=np.int8)

    if times.size != labels.size:
        raise ValueError("times and labels must have same length")

    order = np.argsort(times)
    t = times[order]
    lab = labels[order]

    if t.size < 2:
        return dict(
            ordered_times=t,
            ordered_labels=lab,
            dt=np.array([], dtype=np.int64),
            short_mask=np.array([], dtype=bool),
            n_short_total=0,
            n_cross_short=0,
            n_within0_short=0,
            n_within1_short=0,
            frac_cross_of_short=np.nan,
            frac_within0_of_short=np.nan,
            frac_within1_of_short=np.nan,
        )

    dt = np.diff(t)
    short = (dt >= int(lo_samp)) & (dt <= int(hi_samp))

    left_lab = lab[:-1]
    right_lab = lab[1:]

    cross_short = short & (left_lab != right_lab)
    within0_short = short & (left_lab == 0) & (right_lab == 0)
    within1_short = short & (left_lab == 1) & (right_lab == 1)

    n_short_total = int(np.sum(short))
    n_cross_short = int(np.sum(cross_short))
    n_within0_short = int(np.sum(within0_short))
    n_within1_short = int(np.sum(within1_short))

    frac_cross_of_short = n_cross_short / n_short_total if n_short_total > 0 else np.nan
    frac_within0_of_short = n_within0_short / n_short_total if n_short_total > 0 else np.nan
    frac_within1_of_short = n_within1_short / n_short_total if n_short_total > 0 else np.nan

    return dict(
        ordered_times=t,
        ordered_labels=lab,
        dt=dt,
        short_mask=short,
        n_short_total=n_short_total,
        n_cross_short=n_cross_short,
        n_within0_short=n_within0_short,
        n_within1_short=n_within1_short,
        frac_cross_of_short=frac_cross_of_short,
        frac_within0_of_short=frac_within0_of_short,
        frac_within1_of_short=frac_within1_of_short,
    )


def classify_all_left_spikes_fast(out2, raw_mod, window=(-40, 80)):
    ch = int(out2["ch"])
    left_times = np.asarray(out2["left_times"], dtype=np.int64)
    ei0 = np.asarray(out2["ei0"], dtype=np.float32)
    ei1 = np.asarray(out2["ei1"], dtype=np.float32)

    if left_times.size == 0:
        raise ValueError("No left spikes available.")

    class_ch, class_diag = _pick_classification_channels(
        ei0,
        ei1,
        p2p_thr=CLASS_P2P_THR,
        max_ch=CLASS_MAX_CH,
        min_ch=CLASS_MIN_CH,
    )

    print(f"\n[CH {ch}] classification channels ({class_ch.size}): {class_ch.tolist()}")

    snips_left, valid_left_times = extract_snippets_fast_ram(
        raw_mod,
        left_times,
        window=window,
        selected_channels=class_ch.astype(np.int32),
    )
    snips_left = np.asarray(snips_left, dtype=np.float32)

    print(f"[CH {ch}] classifying {snips_left.shape[2]} valid left spikes")

    ei0_sel = ei0[class_ch].astype(np.float32, copy=False)
    ei1_sel = ei1[class_ch].astype(np.float32, copy=False)

    assign = _best_lag_cosine_assign(
        snips_left,
        ei0_sel,
        ei1_sel,
        lag_radius=CLASS_LAG_RADIUS,
    )

    labels_all = assign["labels"]
    best_cos0 = assign["best_cos0"]
    best_cos1 = assign["best_cos1"]
    cos_diff = assign["cos_diff"]

    n0 = int(np.sum(labels_all == 0))
    n1 = int(np.sum(labels_all == 1))
    print(f"[CH {ch}] all-left classification sizes: n0={n0}, n1={n1}")

    short_diag = _adjacent_short_isi_stats(
        valid_left_times,
        labels_all,
        lo_samp=10,
        hi_samp=30,
    )

    print(f"[CH {ch}] adjacent pairs with ISI in [10,30] samples:")
    print(f"  total short pairs: {short_diag['n_short_total']}")
    print(
        f"  cross-cluster: {short_diag['n_cross_short']} "
        f"({short_diag['frac_cross_of_short']:.4f})"
        if np.isfinite(short_diag["frac_cross_of_short"]) else
        f"  cross-cluster: {short_diag['n_cross_short']} (nan)"
    )
    print(
        f"  within cluster 0: {short_diag['n_within0_short']} "
        f"({short_diag['frac_within0_of_short']:.4f})"
        if np.isfinite(short_diag["frac_within0_of_short"]) else
        f"  within cluster 0: {short_diag['n_within0_short']} (nan)"
    )
    print(
        f"  within cluster 1: {short_diag['n_within1_short']} "
        f"({short_diag['frac_within1_of_short']:.4f})"
        if np.isfinite(short_diag["frac_within1_of_short"]) else
        f"  within cluster 1: {short_diag['n_within1_short']} (nan)"
    )

    if SHOW_SCATTER:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].scatter(best_cos0, best_cos1, c=labels_all, s=8, alpha=0.6)
        lo = float(min(np.min(best_cos0), np.min(best_cos1)))
        hi = float(max(np.max(best_cos0), np.max(best_cos1)))
        axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
        axes[0].set_xlabel("Best cosine to EI0")
        axes[0].set_ylabel("Best cosine to EI1")
        axes[0].set_title(f"CH {ch} | all left spikes")
        axes[0].grid(alpha=0.25)

        order = np.argsort(cos_diff)
        axes[1].scatter(np.arange(cos_diff.size), cos_diff[order], c=labels_all[order], s=8, alpha=0.6)
        axes[1].axhline(0.0, color="k", linestyle="--", lw=1)
        axes[1].set_xlabel("Spike rank (sorted by cos0 - cos1)")
        axes[1].set_ylabel("Best cosine EI0 - EI1")
        axes[1].set_title("Negative => EI1 better, Positive => EI0 better")
        axes[1].grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

    return dict(
        ch=ch,
        classification_channels=class_ch,
        classification_channel_diag=class_diag,
        valid_left_times=valid_left_times.astype(np.int64, copy=False),
        labels_all=labels_all.astype(np.int8, copy=False),
        best_cos0=best_cos0,
        best_cos1=best_cos1,
        best_lag0=assign["best_lag0"],
        best_lag1=assign["best_lag1"],
        cos_diff=cos_diff,
        short_isi_diag=short_diag,
    )


out4 = classify_all_left_spikes_fast(out2, raw_mod, window=win)

## Inspect one channel

In [ ]:
# Assumes these already exist from earlier cells:
#   raw_mod      : (T, C) int16 (baseline-subtracted)
#   triggers_sec  : 1D float array
#   ei_positions  : (512,2) float array (for EI plots)
# And these are imported earlier (from your imports cell):
#   find_valley_and_times, build_left_low_high_eis, filter_valley_spikes_by_batches, finalize_ei_with_cap
#   compute_prev_metrics, plot_amp_prev_scatter_and_bar (optional)
#   extract_snippets_fast_ram, cosine_two_eis
#   pew (plot_ei_waveforms module imported as pew) OR plot_ei_waveforms as pew

# --- set me ---
# ch = 0  # <- CHANGE THIS

# --- knobs (kept same spirit as lighthouse.ipynb loop) ---
win = (-40, 80)
min_cosine = 0.65

DO_KMEANS = True
DO_STA = False
DO_PSTH_RASTER = True
DO_ISI    = True
DO_PREV_METRICS = False

# left right comparison - number of spikes to take
LR_N_PER_SIDE = 2_000

# KMEANS params
SUBSAMPLE_MAX = 5_000
RMS_THRESH    = 5.0
N_PC          = 10
RNG           = np.random.RandomState(123)
KMEANS_ISO_SEP = 80

# ISI / rate params
sample_rate_hz = 20_000
bin_width_ms   = 0.5
max_ms         = 300.0
dt_ms          = 1000.0
sigma_ms       = 2500.0

# how many spikes to take for top/bottom groups
N_SP_FOR_GROUPS = 2000

N_PC_PER_CH    = 3

TL_FRAC_MIN    = 0.95
BL_FRAC_MIN    = 0.80
TR_FRAC_MAX    = 0.20

COS_MASK_ADC   = 10.0
COS_MASK_PAD   = 1

HARM_LAG_RADIUS = 4
HARM_WEIGHT_BETA = 0.7


CLOUD_TOPK      = 15
CLOUD_HIST_BINS = 50

# STA params (same as your loop)
STA_KW = dict(STA_DEPTH=30, STA_OFFSET=2, STA_CHUNK=1000, STA_REFRESH=2, SEED=22222, W=20, H=40, label="all",
              peak_frame=None, mode="bw")

# Try to ensure pew exists
try:
    pew
except NameError:
    import plot_ei_waveforms as pew


def _roll_zero_2d(arr, shift):
    nch, T = arr.shape
    out = np.zeros_like(arr)
    if shift == 0:
        out[:] = arr
    elif shift > 0:
        out[:, shift:] = arr[:, :T-shift]
    else:
        s = -shift
        out[:, :T-s] = arr[:, s:]
    return out


def _dilate_bool_mask(mask, pad=1):
    mask = np.asarray(mask, dtype=bool)
    pad = int(max(0, pad))
    if pad == 0 or mask.size == 0:
        return mask
    out = mask.copy()
    for k in range(1, pad + 1):
        out[:-k] |= mask[k:]
        out[k:]  |= mask[:-k]
    return out

def cloud_topk_cosine_scores_masked(
    ref_snips_2d,
    cand_snips_2d,
    mask_source_1d,
    amp_thr=10.0,
    pad=1,
    topk=15,
    leave_one_out_refpos=None,
    eps=1e-12,
):
    """
    ref_snips_2d  : [Nref, T] purified BL waveforms on one channel
    cand_snips_2d : [Ncand, T] candidate waveforms on same channel
    mask_source_1d: [T] waveform used to define signal mask (usually median purified BL)
    leave_one_out_refpos : [Ncand] int array, -1 if no self-match to remove,
                           otherwise index into ref_snips_2d to exclude for that candidate

    Returns:
      score : [Ncand]   median of top-k cosine similarities to BL cloud
      mask  : [T] bool  mask actually used
    """
    ref_snips_2d = np.asarray(ref_snips_2d, dtype=np.float32)
    cand_snips_2d = np.asarray(cand_snips_2d, dtype=np.float32)
    mask_source_1d = np.asarray(mask_source_1d, dtype=np.float32)

    assert ref_snips_2d.ndim == 2 and cand_snips_2d.ndim == 2
    assert ref_snips_2d.shape[1] == cand_snips_2d.shape[1] == mask_source_1d.size

    mask = np.abs(mask_source_1d) >= float(amp_thr)
    if int(mask.sum()) == 0:
        k_keep = int(min(5, mask_source_1d.size))
        keep = np.argsort(np.abs(mask_source_1d))[-k_keep:]
        mask[keep] = True
    mask = _dilate_bool_mask(mask, pad=pad)

    R = ref_snips_2d[:, mask]   # [Nref, M]
    X = cand_snips_2d[:, mask]  # [Ncand, M]

    Rn = np.linalg.norm(R, axis=1, keepdims=True)
    Xn = np.linalg.norm(X, axis=1, keepdims=True)

    R = R / np.maximum(Rn, eps)
    X = X / np.maximum(Xn, eps)

    sims = X @ R.T   # [Ncand, Nref]

    if leave_one_out_refpos is not None:
        leave_one_out_refpos = np.asarray(leave_one_out_refpos, dtype=np.int64)
        assert leave_one_out_refpos.shape[0] == sims.shape[0]
        valid = leave_one_out_refpos >= 0
        row_idx = np.flatnonzero(valid)
        col_idx = leave_one_out_refpos[valid]
        sims[row_idx, col_idx] = -np.inf

    nref = sims.shape[1]
    if nref == 0:
        raise ValueError("No reference BL spikes for cloud score.")

    if leave_one_out_refpos is None:
        k_eff = int(min(topk, nref))
    else:
        # if a row excludes itself, usable refs are at most nref-1
        k_eff = int(min(topk, max(1, nref - 1)))

    topk_vals = np.partition(sims, kth=nref - k_eff, axis=1)[:, -k_eff:]
    score = np.median(topk_vals, axis=1).astype(np.float32)
    return score, mask


def plot_cloud_score_histograms_by_channel(
    scores_bl,
    scores_tr,
    channel_labels,
    bins=50,
    title=None,
):
    """
    Separate BL and TR score histograms per channel, same x-axis.
    scores_bl : [nch, Nbl]
    scores_tr : [nch, Ntr]
    """
    scores_bl = np.asarray(scores_bl, dtype=np.float32)
    scores_tr = np.asarray(scores_tr, dtype=np.float32)
    channel_labels = np.asarray(channel_labels, dtype=np.int32)

    assert scores_bl.ndim == 2 and scores_tr.ndim == 2
    assert scores_bl.shape[0] == scores_tr.shape[0] == channel_labels.size

    nch = channel_labels.size
    fig, axes = plt.subplots(
        nch, 2,
        figsize=(12, 3.2 * nch),
        sharex=True,
        squeeze=False
    )

    bins_arr = np.linspace(-1.0, 1.0, int(bins) + 1)

    for i in range(nch):
        ax_bl = axes[i, 0]
        ax_tr = axes[i, 1]

        bl = scores_bl[i]
        tr = scores_tr[i]

        ax_bl.hist(bl[np.isfinite(bl)], bins=bins_arr, color="red", alpha=0.75)
        ax_tr.hist(tr[np.isfinite(tr)], bins=bins_arr, color="purple", alpha=0.75)

        bl_med = float(np.nanmedian(bl))
        tr_med = float(np.nanmedian(tr))
        ax_bl.axvline(bl_med, color="k", lw=1.2, alpha=0.8)
        ax_tr.axvline(tr_med, color="k", lw=1.2, alpha=0.8)

        ax_bl.set_xlim(-1.0, 1.0)
        ax_tr.set_xlim(-1.0, 1.0)

        ax_bl.grid(alpha=0.25)
        ax_tr.grid(alpha=0.25)

        ax_bl.set_ylabel(f"ch {int(channel_labels[i])}\ncount")
        ax_bl.set_title(f"BL | median={bl_med:.3f}")
        ax_tr.set_title(f"TR | median={tr_med:.3f}")

        if i == nch - 1:
            ax_bl.set_xlabel("BL-cloud score")
            ax_tr.set_xlabel("BL-cloud score")

    if title is not None:
        fig.suptitle(title, y=0.995)

    plt.tight_layout()
    plt.show()


def masked_cosine_batch(template_1d, snips_2d, amp_thr=10.0, pad=1, eps=1e-12):
    """
    template_1d : [T]
    snips_2d    : [N, T]
    Returns:
      cos  : [N]
      mask : [T] bool
    """
    template_1d = np.asarray(template_1d, dtype=np.float32)
    snips_2d = np.asarray(snips_2d, dtype=np.float32)
    assert snips_2d.ndim == 2 and snips_2d.shape[1] == template_1d.size

    mask = np.abs(template_1d) >= float(amp_thr)
    if int(mask.sum()) == 0:
        k = int(min(5, template_1d.size))
        if k > 0:
            keep = np.argsort(np.abs(template_1d))[-k:]
            mask[keep] = True

    mask = _dilate_bool_mask(mask, pad=pad)

    t = template_1d[mask]
    X = snips_2d[:, mask]

    t_norm = float(np.linalg.norm(t))
    x_norm = np.linalg.norm(X, axis=1)
    denom = np.maximum(t_norm * x_norm, eps)
    cos = (X @ t) / denom
    return cos.astype(np.float32), mask


def compute_harm_map_fixed_channels_noamp(
    ei_sel,
    snips_sel,
    channel_labels=None,
    lag_radius=4,
    weight_by_p2p=True,
    weight_beta=0.7,
):
    """
    Fixed-channel version of harm map:
      ei_sel    : [nch, T]
      snips_sel : [nch, T, N]
    """
    ei_sel = np.asarray(ei_sel, dtype=np.float32)
    snips_sel = np.asarray(snips_sel, dtype=np.float32)

    assert ei_sel.ndim == 2, "ei_sel must be [nch, T]"
    assert snips_sel.ndim == 3, "snips_sel must be [nch, T, N]"
    assert snips_sel.shape[:2] == ei_sel.shape, "snips_sel and ei_sel must match on [nch, T]"

    nch, T = ei_sel.shape
    N = snips_sel.shape[2]

    if channel_labels is None:
        channel_labels = np.arange(nch, dtype=np.int32)
    else:
        channel_labels = np.asarray(channel_labels, dtype=np.int32)
        assert channel_labels.size == nch

    ptp = np.ptp(ei_sel, axis=1).astype(np.float32)
    ch_main_local = int(np.argmin(np.min(ei_sel, axis=1)))
    t_neg = int(np.argmin(ei_sel[ch_main_local]))

    if bool(weight_by_p2p):
        w = np.power(np.maximum(ptp, 1e-6), float(weight_beta)).astype(np.float32)
    else:
        w = np.ones(nch, dtype=np.float32)
    w /= np.maximum(w.sum(), 1e-12)

    rms_raw = np.sqrt((snips_sel ** 2).mean(axis=1)).astype(np.float32)   # [nch, N]
    E_raw = np.maximum(rms_raw ** 2, 1e-12)

    lags = np.arange(-int(lag_radius), int(lag_radius) + 1, dtype=np.int32)
    deltas = np.empty((lags.size, nch, N), dtype=np.float32)
    ratios = np.empty((lags.size, nch, N), dtype=np.float32)

    for i, d in enumerate(lags):
        shifted = _roll_zero_2d(ei_sel, int(d))
        resid = snips_sel - shifted[:, :, None]
        E_res = (resid ** 2).mean(axis=1)
        rms_res = np.sqrt(E_res)
        deltas[i] = rms_res - rms_raw
        ratios[i] = E_res / E_raw

    mean_deltas = (deltas * w[None, :, None]).sum(axis=1)   # [n_lags, N]
    best_i = np.argmin(mean_deltas, axis=0)
    best_lags = lags[best_i]

    harm = np.take_along_axis(deltas, best_i[None, None, :], axis=0).squeeze(0)
    harm_ratio = np.take_along_axis(ratios, best_i[None, None, :], axis=0).squeeze(0)
    harm_r2 = 1.0 - harm_ratio

    out = {
        "selected_channels": channel_labels,
        "channel_ptp": ptp,
        "main_channel": int(channel_labels[ch_main_local]),
        "neg_peak_index": t_neg,
        "lags": lags,
        "best_lag_per_spike": best_lags,
        "harm_matrix": harm,
        "harm_matrix_r2": harm_r2,
        "mean_delta_unweighted": harm.mean(axis=0),
        "mean_delta_weighted": (harm * w[:, None]).sum(axis=0),
        "mean_r2_unweighted": harm_r2.mean(axis=0),
        "mean_r2_weighted": (harm_r2 * w[:, None]).sum(axis=0),
    }
    return out


def plot_harm_and_cosines_sharedx(
    harm_res,
    cosine_mat,
    channel_labels,
    spike_order=None,
    field="harm_matrix_r2",
    vline_at=None,
    title=None,
    vclip=None,
):
    H = np.asarray(harm_res[field], dtype=np.float32)
    C = np.asarray(cosine_mat, dtype=np.float32)
    channel_labels = np.asarray(channel_labels, dtype=np.int32)

    assert H.ndim == 2 and C.ndim == 2 and H.shape == C.shape

    if spike_order is None:
        order = np.arange(H.shape[1], dtype=np.int32)
    else:
        order = np.asarray(spike_order, dtype=np.int32)

    Hs = H[:, order]
    Cs = C[:, order]
    x = np.arange(order.size)

    fig, (ax0, ax1) = plt.subplots(
        2, 1, figsize=(20, 6), sharex=True,
        gridspec_kw={"height_ratios": [2.4, 1.2]}
    )

    if field == "harm_matrix_r2":
        vmax = 1.0 if vclip is None else float(vclip)
        im = ax0.imshow(Hs, aspect="auto", cmap="bwr", vmin=-abs(vmax), vmax=abs(vmax))
        cbar_label = "Harm map R²"
    else:
        vmax = float(np.percentile(np.abs(Hs), 98)) if vclip is None else float(vclip)
        im = ax0.imshow(Hs, aspect="auto", cmap="bwr", vmin=-vmax, vmax=vmax)
        cbar_label = "ΔRMS (res - raw)"

    fig.colorbar(im, ax=ax0, label=cbar_label)
    ax0.set_ylabel("Channel")
    ax0.set_yticks(np.arange(channel_labels.size))
    ax0.set_yticklabels([str(int(ch)) for ch in channel_labels])
    if title:
        ax0.set_title(title)

    for i, ch_lab in enumerate(channel_labels):
        ax1.plot(x, Cs[i], lw=1.2, label="ch %d" % int(ch_lab))

    ax1.set_ylabel("Cosine to purified BL")
    ax1.set_xlabel("Spike index")
    ax1.set_ylim(-1.05, 1.05)
    ax1.grid(alpha=0.25)
    ax1.legend(loc="upper right", frameon=False, ncol=min(4, channel_labels.size))

    if vline_at is not None:
        xpos = float(vline_at) - 0.5
        for ax in (ax0, ax1):
            ax.axvline(xpos, color="k", linestyle="--", linewidth=1.2, alpha=0.9)

    plt.tight_layout()
    plt.show()



def plot_cosine_vs_harm_by_channel(
    cosine_mat,
    harm_res,
    channel_labels,
    n_bl,
    field="harm_matrix_r2",
    title=None,
):
    """
    Scatter per discriminative channel:
      x = cosine to purified BL
      y = harm value on the same channel
    BL spikes are plotted first group, TR spikes second group.
    """
    C = np.asarray(cosine_mat, dtype=np.float32)
    H = np.asarray(harm_res[field], dtype=np.float32)
    channel_labels = np.asarray(channel_labels, dtype=np.int32)

    assert C.ndim == 2 and H.ndim == 2 and C.shape == H.shape
    assert C.shape[0] == channel_labels.size

    nch, N = C.shape
    n_bl = int(n_bl)
    ncols = int(min(3, nch))
    nrows = int(np.ceil(nch / float(ncols)))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5.5 * ncols, 4.5 * nrows),
        sharex=True, sharey=True
    )
    axes = np.atleast_1d(axes).ravel()

    xlim = (-1.05, 1.05)

    # robust shared y-limits across panels
    y_lo = float(np.nanpercentile(H, 1))
    y_hi = float(np.nanpercentile(H, 99))
    if np.isfinite(y_lo) and np.isfinite(y_hi) and (y_hi > y_lo):
        pad = 0.06 * (y_hi - y_lo)
        ylim = (y_lo - pad, y_hi + pad)
    else:
        ylim = None

    if field == "harm_matrix_r2":
        ylab = "Harm map R²"
    else:
        ylab = "Harm value"

    for i in range(nch):
        ax = axes[i]

        # BL group
        ax.scatter(
            C[i, :n_bl], H[i, :n_bl],
            c="red", s=10, alpha=0.45,
            label="BL" if i == 0 else None
        )

        # TR group
        ax.scatter(
            C[i, n_bl:], H[i, n_bl:],
            c="purple", s=10, alpha=0.45,
            label="TR" if i == 0 else None
        )

        ax.axvline(0.0, color="k", lw=0.6, alpha=0.25)
        ax.axhline(0.0, color="k", lw=0.6, alpha=0.25)
        ax.grid(alpha=0.25)

        ax.set_xlim(xlim)
        if ylim is not None:
            ax.set_ylim(ylim)

        bl_cos_med = float(np.nanmedian(C[i, :n_bl])) if n_bl > 0 else np.nan
        tr_cos_med = float(np.nanmedian(C[i, n_bl:])) if (N - n_bl) > 0 else np.nan
        bl_h_med = float(np.nanmedian(H[i, :n_bl])) if n_bl > 0 else np.nan
        tr_h_med = float(np.nanmedian(H[i, n_bl:])) if (N - n_bl) > 0 else np.nan

        ax.set_title(
            f"ch {int(channel_labels[i])} | "
            f"BL({bl_cos_med:.2f}, {bl_h_med:.2f}) "
            f"TR({tr_cos_med:.2f}, {tr_h_med:.2f})",
            fontsize=9
        )
        ax.set_xlabel("Cosine to purified BL")
        ax.set_ylabel(ylab)

    for j in range(nch, len(axes)):
        axes[j].axis("off")

    axes[0].legend(loc="best", frameon=False)

    if title is not None:
        fig.suptitle(title, y=0.98)

    plt.tight_layout()
    plt.show()


def inspect_lighthouse_channel(ch: int):
    t0 = time.time()
    spike_gallery_data = None
    cloud_diag = None
    C = raw_mod.shape[1]
    if not (0 <= ch < C):
        raise ValueError(f"ch={ch} out of range 0..{C-1}")

    # ---- STEP 1: valley detection ----
    step1 = find_valley_and_times(
        raw_mod, ch,
        window=win, start=0, stop=None,
        bin_width=10.0, valley_bins=5,
        min_valid_count=500,
        ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10
    )

    # Always expose these spike-time pools, even if the channel later fails.
    left_times_any = np.asarray(step1.get("left_times", []), dtype=np.int64)

    # EXTRA return-only pool: 5000 largest RIGHT spikes.
    # IMPORTANT: this is NOT used by the pipeline below.
    all_times_any = np.asarray(step1.get("all_times", []), dtype=np.int64)
    all_vals_any  = np.asarray(step1.get("all_vals", []), dtype=np.float32)
    valley_low_any = float(step1.get("valley_low", np.nan))

    if all_times_any.size > 0 and np.isfinite(valley_low_any):
        right_mask_any = (all_vals_any >= valley_low_any)
        right_times_pool_any = all_times_any[right_mask_any]
        right_vals_pool_any  = all_vals_any[right_mask_any]

        if right_times_pool_any.size > 0:
            right_order_any = np.argsort(right_vals_pool_any)   # most negative first
            n_right_pick_any = min(5000, right_order_any.size)
            right5000_times_any = np.asarray(
                right_times_pool_any[right_order_any[:n_right_pick_any]],
                dtype=np.int64
            )
        else:
            right5000_times_any = np.array([], dtype=np.int64)
    else:
        right5000_times_any = np.array([], dtype=np.int64)

    print(f"\n[CH {ch}] valley [{step1.get('valley_low', np.nan):.1f}, {step1.get('valley_high', np.nan):.1f}) "
          f"Left={step1.get('left_count', -1)} Valley={step1.get('valley_count', -1)} R*={step1.get('required_ratio', np.nan)}")
    print(f"[CH {ch}] Left spikes for filter: {len(step1.get('left_times', []))}, "
          f"right spikes: {len(step1.get('rightk_times_by_amp', []))}")

    # -----------------------------
    # 1) LEFT ISI 10..30 samples
    # -----------------------------
    left_times = np.asarray(step1.get("left_times", []), dtype=np.int64)
    left_times = np.sort(left_times)

    isi_pairs_10_30 = 0
    isi_spikes_involved = 0

    if left_times.size >= 2:
        d = np.diff(left_times)  # samples
        m = (d >= 10) & (d <= 30)
        isi_pairs_10_30 = int(np.sum(m))
        # spikes involved (unique), in case you prefer this number:
        idx = np.flatnonzero(m)
        involved = np.unique(np.concatenate([idx, idx + 1]))
        isi_spikes_involved = int(involved.size)

    print(f"[CH {ch}] LEFT short-ISI: {isi_pairs_10_30} adjacent pairs with ISI in [10,30] samples "
        f"(unique spikes involved: {isi_spikes_involved})")

    # -----------------------------
    # 2) Gaussian fit to RIGHT SIDE of LEFT bump
    #    Fit region: [mode_of_left_bump, valley_low)
    # -----------------------------
    edges = np.asarray(step1["amp_hist_edges"], dtype=np.float64)
    valley_low = float(step1.get("valley_low", np.nan))

    left_vals = np.asarray(step1.get("left_vals", []), dtype=np.float64)
    left_vals = left_vals[np.isfinite(left_vals)]

    gauss_gof = dict(ok=False, reason="")

    def _norm_cdf(x, mu, sig):
        # normal CDF using erf (no scipy)
        z = (x - mu) / (sig * math.sqrt(2.0))
        return 0.5 * (1.0 + math.erf(z))

    if left_vals.size >= 200 and np.isfinite(valley_low):
        # histogram of LEFT vals using same bin edges
        hL, _ = np.histogram(left_vals, bins=edges)
        centers = 0.5 * (edges[:-1] + edges[1:])

        # mode estimate from LEFT histogram peak
        imode = int(np.argmax(hL))
        mu = float(centers[imode])

        # right-side samples only (mu .. valley_low)
        xR = left_vals[(left_vals >= mu) & (left_vals < valley_low)]
        if xR.size >= 200:
            # sigma from right half; E[(X-mu)^2 | X>=mu] = sigma^2 for a symmetric normal
            sig = float(np.sqrt(np.mean((xR - mu) ** 2) + 1e-12))

            # choose bins on the right side [mu, valley_low)
            use = (centers >= mu) & (centers < valley_low)
            if np.sum(use) >= 6 and sig > 1e-6:
                obs = hL[use].astype(np.float64)

                # expected counts per bin via CDF differences
                exp = np.zeros_like(obs)
                e0 = edges[:-1][use]
                e1 = edges[1:][use]
                for i in range(obs.size):
                    p = _norm_cdf(e1[i], mu, sig) - _norm_cdf(e0[i], mu, sig)
                    exp[i] = xR.size * max(p, 0.0)

                # GOF: R^2 and reduced chi^2 (rough)
                ss_res = float(np.sum((obs - exp) ** 2))
                ss_tot = float(np.sum((obs - np.mean(obs)) ** 2) + 1e-12)
                r2 = 1.0 - ss_res / ss_tot

                chi2 = float(np.sum((obs - exp) ** 2 / (exp + 1e-6)))
                dof = max(int(obs.size - 1), 1)
                chi2_red = chi2 / dof

                gauss_gof = dict(ok=True, mu=mu, sigma=sig, r2=r2, chi2_red=chi2_red,
                                n_right=int(xR.size), n_bins=int(obs.size),
                                fit_centers=centers[use], fit_counts=exp)
            else:
                gauss_gof = dict(ok=False, reason="too_few_bins_or_sigma_bad")
        else:
            gauss_gof = dict(ok=False, reason=f"too_few_right_samples({xR.size})")
    else:
        gauss_gof = dict(ok=False, reason=f"too_few_left_vals({left_vals.size})_or_bad_valley_low")

    if gauss_gof["ok"]:
        print(f"[CH {ch}] LEFT bump Gaussian (right side) fit: mu={gauss_gof['mu']:.1f}, sigma={gauss_gof['sigma']:.1f}, "
            f"R2={gauss_gof['r2']:.3f}, chi2_red={gauss_gof['chi2_red']:.2f}, "
            f"Nright={gauss_gof['n_right']}, bins={gauss_gof['n_bins']}")
    else:
        print(f"[CH {ch}] LEFT bump Gaussian fit skipped: {gauss_gof.get('reason','unknown')}")

    # amp histogram + gaussian overlay (right side of left bump)
    try:
        plt.figure(figsize=(12, 3))
        plt.bar(step1['amp_hist_edges'][:-1],
                step1['amp_hist_counts'],
                width=np.diff(step1['amp_hist_edges']),
                align='edge')
        plt.axvspan(step1['valley_low'], step1['valley_high'], color='r', alpha=0.3)

        if gauss_gof.get("ok", False):
            plt.plot(gauss_gof["fit_centers"], gauss_gof["fit_counts"], lw=2)

        plt.title(f"Channel {ch}: amplitude histogram with valley"
                + (f" | GaussR2={gauss_gof['r2']:.3f}" if gauss_gof.get("ok", False) else ""))
        plt.ylim(0, 5000)
        plt.show()
    except Exception as e:
        print(f"[CH {ch}] amp histogram plot failed: {e}")

    if not step1.get("accepted", False):
        print(f"[CH {ch}] valley acceptance failed → returning spike-time pools only.")
        return dict(
            step1=step1,
            step2=None,
            step3=None,
            step4=None,
            kmeans=None,
            blind_split_diag=None,
            cloud_diag=None,
            spike_gallery_data=None,
            all_left_times=left_times_any,
            right5000_times=right5000_times_any,
        )


    # ---- OPTIONAL: diagnostic PCA/KMeans on smallest LEFT vs largest RIGHT(+valley) ----
    try:
        
        left_times_all = np.asarray(step1.get("left_times", []), dtype=np.int64)
        left_vals_all  = np.asarray(step1.get("left_vals",  []), dtype=np.float32)
        all_times_all  = np.asarray(step1.get("all_times",  []), dtype=np.int64)
        all_vals_all   = np.asarray(step1.get("all_vals",   []), dtype=np.float32)
        valley_low     = float(step1.get("valley_low", np.nan))

        if left_times_all.size == 0:
            raise ValueError("No LEFT spikes available.")
        if all_times_all.size == 0 or not np.isfinite(valley_low):
            raise ValueError("Need step1['all_times']/['all_vals'] and finite valley_low.")

        # smallest LEFT = least-negative LEFT spikes (closest to the valley)
        left_order = np.argsort(left_vals_all)[::-1]
        n_left_pick = min(LR_N_PER_SIDE, left_order.size)
        left_pick = left_times_all[left_order[:n_left_pick]]

        # largest RIGHT(+valley) = most-negative events among all spikes at/above valley_low
        right_mask = (all_vals_all >= valley_low)
        right_times_pool = all_times_all[right_mask]
        right_vals_pool  = all_vals_all[right_mask]
        if right_times_pool.size == 0:
            raise ValueError("No RIGHT/valley spikes available.")

        right_order = np.argsort(right_vals_pool)  # most negative first
        n_right_pick = min(LR_N_PER_SIDE, right_order.size)
        right_pick = right_times_pool[right_order[:n_right_pick]]

        left_amp_pick  = left_vals_all[left_order[:n_left_pick]]
        right_amp_pick = right_vals_pool[right_order[:n_right_pick]]

        pick_lr = np.concatenate([left_pick, right_pick]).astype(np.int64, copy=False)
        origin_lr = np.concatenate([
            np.zeros(left_pick.size,  dtype=np.int8),   # 0 = left
            np.ones(right_pick.size,  dtype=np.int8),   # 1 = right
        ])
        amp_pick_lr = np.concatenate([left_amp_pick, right_amp_pick]).astype(np.float32, copy=False)

        pre, post = int(win[0]), int(win[1])
        in_bounds = (pick_lr + pre >= 0) & (pick_lr + post < raw_mod.shape[0])

        snips_lr, valid_times_lr = extract_snippets_fast_ram(
            raw_mod, pick_lr, window=win,
            selected_channels=np.arange(C, dtype=np.int32)
        )
        origin_lr = origin_lr[in_bounds]
        amp_lr = amp_pick_lr[in_bounds]

        K_lr, L_lr, N_lr = snips_lr.shape
        print(f"[CH {ch}] left/right diag: snips {snips_lr.shape}; "
              f"picked LEFT={left_pick.size}, RIGHT={right_pick.size}, valid={N_lr}")

        if N_lr < 20:
            raise ValueError("Too few valid snippets for PCA/kmeans.")

        # same channel-selection logic as the existing KMeans block
        ei_mean_lr = snips_lr.mean(axis=2).astype(np.float32)
        rms_lr = np.sqrt(np.mean(ei_mean_lr**2, axis=1))
        sel_lr = np.flatnonzero(rms_lr > RMS_THRESH)
        if sel_lr.size == 0:
            sel_lr = np.argsort(rms_lr)[-16:]
            sel_lr.sort()
            print(f"[CH {ch}] left/right diag: no channels over RMS>{RMS_THRESH}; using top-16")

        X_lr = snips_lr[sel_lr, :, :].transpose(2, 0, 1).reshape(N_lr, sel_lr.size * L_lr).astype(np.float32)

        n_pc_lr = int(min(N_PC, X_lr.shape[0], X_lr.shape[1]))
        pca_lr = PCA(n_components=n_pc_lr, svd_solver='randomized', random_state=0)
        Xpc_lr = pca_lr.fit_transform(X_lr)
        vr_lr = pca_lr.explained_variance_ratio_

        km_lr = KMeans(n_clusters=2, n_init=50, random_state=0)
        lab_lr = km_lr.fit_predict(Xpc_lr)

        # cluster EIs
        ei_lr_c0 = snips_lr[:, :, lab_lr == 0].mean(axis=2).astype(np.float32)
        ei_lr_c1 = snips_lr[:, :, lab_lr == 1].mean(axis=2).astype(np.float32)
        lr_cos, lr_nch, lr_lag, lr_olen = cosine_two_eis(
            ei_lr_c0, ei_lr_c1, rms_gate=5.0, use_abs=True, best_align_lag=3
        )

        # contingency printout: how origin maps into clusters
        left_to_k0  = int(np.sum((origin_lr == 0) & (lab_lr == 0)))
        left_to_k1  = int(np.sum((origin_lr == 0) & (lab_lr == 1)))
        right_to_k0 = int(np.sum((origin_lr == 1) & (lab_lr == 0)))
        right_to_k1 = int(np.sum((origin_lr == 1) & (lab_lr == 1)))
        print(f"[CH {ch}] left/right diag contingency: "
              f"left→(k0={left_to_k0}, k1={left_to_k1}), "
              f"right→(k0={right_to_k0}, k1={right_to_k1})")

        # PCA scatter + amplitude-vs-PC1 scatter, same 4 colors
        pc1_lr = Xpc_lr[:, 0]
        pc2_lr = Xpc_lr[:, 1] if n_pc_lr >= 2 else np.zeros_like(pc1_lr)

        # use positive trough magnitude on x-axis so "bigger spike" is farther right
        amp_plot_lr = -amp_lr

        group_specs = [
            ("left-small, km0",  (origin_lr == 0) & (lab_lr == 0), "tab:blue"),
            ("left-small, km1",  (origin_lr == 0) & (lab_lr == 1), "tab:green"),
            ("right-large, km0", (origin_lr == 1) & (lab_lr == 0), "tab:orange"),
            ("right-large, km1", (origin_lr == 1) & (lab_lr == 1), "tab:red"),
        ]

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # left panel: PCA cloud
        for label, mask, color in group_specs:
            if np.any(mask):
                axes[0].scatter(
                    pc1_lr[mask], pc2_lr[mask],
                    c=color, s=16, alpha=0.65,
                    label=label
                )

        for km_id in [0, 1]:
            m = (lab_lr == km_id)
            if np.any(m):
                ctr = np.array([pc1_lr[m].mean(), pc2_lr[m].mean()])
                axes[0].scatter(
                    [ctr[0]], [ctr[1]],
                    c="k", marker="X", s=180, linewidths=1.0
                )

        axes[0].set_xlabel(f"PC1 ({vr_lr[0]*100:.1f}% var)")
        axes[0].set_ylabel(f"PC2 ({vr_lr[1]*100:.1f}% var)" if n_pc_lr >= 2 else "PC2")
        axes[0].set_title("PCA scatter")
        axes[0].grid(alpha=0.3)
        axes[0].legend(frameon=False, fontsize=9)

        # right panel: amplitude vs PC1
        for label, mask, color in group_specs:
            if np.any(mask):
                axes[1].scatter(
                    amp_plot_lr[mask], pc1_lr[mask],
                    c=color, s=16, alpha=0.65,
                    label=label
                )

        axes[1].set_xlabel(f"Amplitude on ch {ch} (-trough ADC)")
        axes[1].set_ylabel(f"PC1 ({vr_lr[0]*100:.1f}% var)")
        axes[1].set_title("Amplitude vs PC1")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        # EI overlay from the two KMeans clusters
        fig = plt.figure(figsize=(20, 12))
        ax = fig.add_subplot(111)
        pew.plot_ei_waveforms(
            [ei_lr_c0, ei_lr_c1],
            positions=ei_positions,
            ref_channel=ch,
            scale=70.0,
            ax=ax,
            colors=["tab:blue", "tab:orange"],
            alpha=[1.0, 1.0],
            linewidth=[1.0, 1.0],
            box_height=1.0,
            box_width=50.0,
            aspect=0.5,
        )
        ax.set_title(f"Pre-KMeans diag cluster EIs (small LEFT vs large RIGHT+valley) | cos={lr_cos:.3f}")
        plt.show()

        # EI from RIGHT(+valley) spikes assigned to the majority-LEFT cluster
        majority_left_cluster = 0 if left_to_k0 >= left_to_k1 else 1
        right_in_left_mask = (origin_lr == 1) & (lab_lr == majority_left_cluster)
        n_right_in_left = int(np.sum(right_in_left_mask))

        print(f"[CH {ch}] right spikes assigned to majority-LEFT cluster "
              f"(km{majority_left_cluster}): {n_right_in_left}")

        if n_right_in_left > 0:
            ei_right_in_left = snips_lr[:, :, right_in_left_mask].mean(axis=2).astype(np.float32)

            fig = plt.figure(figsize=(20, 12))
            ax = fig.add_subplot(111)
            pew.plot_ei_waveforms(
                ei_right_in_left,
                positions=ei_positions,
                ref_channel=ch,
                scale=70.0,
                ax=ax,
                colors=["tab:purple"],
                alpha=1.0,
                linewidth=1.0,
                box_height=1.0,
                box_width=50.0,
                aspect=0.5,
            )
            ax.set_title(
                f"EI from RIGHT(+valley) spikes assigned to majority-LEFT cluster "
                f"(km{majority_left_cluster}, n={n_right_in_left})"
            )
            plt.show()
        else:
            print(f"[CH {ch}] no RIGHT(+valley) spikes landed in majority-LEFT cluster; skipping EI plot.")

        # main-channel mean waveform overlays:
        # left panel = KMeans clusters
        # right panel = original LEFT vs RIGHT(+valley) subsets used in this comparison
        t_ms_lr = (np.arange(L_lr) + win[0]) / float(sample_rate_hz) * 1000.0

        left_mask_lr  = (origin_lr == 0)
        right_mask_lr = (origin_lr == 1)

        mean_left_origin  = snips_lr[:, :, left_mask_lr].mean(axis=2).astype(np.float32) if np.any(left_mask_lr) else None
        mean_right_origin = snips_lr[:, :, right_mask_lr].mean(axis=2).astype(np.float32) if np.any(right_mask_lr) else None

        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

        # left: cluster means on main channel
        axes[0].plot(
            t_ms_lr, ei_lr_c0[ch],
            color="tab:blue", lw=2,
            label=f"cluster 0 (n={int(np.sum(lab_lr == 0))})"
        )
        axes[0].plot(
            t_ms_lr, ei_lr_c1[ch],
            color="tab:orange", lw=2,
            label=f"cluster 1 (n={int(np.sum(lab_lr == 1))})"
        )
        axes[0].axvline(0.0, color="k", lw=0.75, alpha=0.3)
        axes[0].axhline(0.0, color="k", lw=0.75, alpha=0.15)
        axes[0].set_xlabel("time (ms)")
        axes[0].set_ylabel("ADC")
        axes[0].set_title(f"KMeans cluster means on ch {ch}")
        axes[0].legend(frameon=False)

        # right: original subset means on main channel
        if mean_left_origin is not None:
            axes[1].plot(
                t_ms_lr, mean_left_origin[ch],
                color="tab:green", lw=2,
                label=f"LEFT subset (n={int(np.sum(left_mask_lr))})"
            )
        if mean_right_origin is not None:
            axes[1].plot(
                t_ms_lr, mean_right_origin[ch],
                color="tab:red", lw=2,
                label=f"RIGHT(+valley) subset (n={int(np.sum(right_mask_lr))})"
            )
        axes[1].axvline(0.0, color="k", lw=0.75, alpha=0.3)
        axes[1].axhline(0.0, color="k", lw=0.75, alpha=0.15)
        axes[1].set_xlabel("time (ms)")
        axes[1].set_title(f"Original subset means on ch {ch}")
        axes[1].legend(frameon=False)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"[CH {ch}] left/right diag block failed: {type(e).__name__}: {e}")

    # ---- OPTIONAL: KMEANS sanity check on LEFT spikes ----
    km_info = None
    if DO_KMEANS:
        try:
            left_times = np.asarray(step1['left_times'], dtype=np.int64)
            n_total = left_times.size
            if n_total == 0:
                raise ValueError("No spikes in step1['left_times'].")

            # Prefer isolated LEFT events for KMeans to avoid AA doublets inside the EI window.
            # Isolation is defined w.r.t. ALL local minima on this channel (step1['all_times']).
            iso_sep = int(globals().get("KMEANS_ISO_SEP", 80))  # samples; 80 = 4 ms at 20 kHz
            left_for_km = left_times

            all_times = np.asarray(step1.get("all_times", []), dtype=np.int64)
            if all_times.size:
                # Map LEFT times into indices of all_times (both are sorted)
                idx = np.searchsorted(all_times, left_times)
                ok_map = (idx < all_times.size) & (all_times[idx] == left_times)
                idx_ok = idx[ok_map]
                lt_ok  = left_times[ok_map]

                dt_prev = np.full(lt_ok.size, np.iinfo(np.int64).max, dtype=np.int64)
                dt_next = np.full(lt_ok.size, np.iinfo(np.int64).max, dtype=np.int64)

                m = idx_ok > 0
                dt_prev[m] = lt_ok[m] - all_times[idx_ok[m] - 1]

                m = (idx_ok + 1) < all_times.size
                dt_next[m] = all_times[idx_ok[m] + 1] - lt_ok[m]

                iso = (dt_prev >= iso_sep) & (dt_next >= iso_sep)
                left_iso = lt_ok[iso]

                if left_iso.size >= 20:
                    left_for_km = left_iso
                    print(f"[CH {ch}] kmeans: using isolated LEFT events: {left_iso.size}/{left_times.size} (sep≥{iso_sep} samp)")
                else:
                    print(f"[CH {ch}] kmeans: isolated LEFT too small ({left_iso.size}); using full LEFT ({left_times.size})")
            else:
                print(f"[CH {ch}] kmeans: step1['all_times'] missing/empty; using full LEFT ({left_times.size})")

            n_total_km = left_for_km.size
            pick = left_for_km if n_total_km <= SUBSAMPLE_MAX else left_for_km[RNG.choice(n_total_km, SUBSAMPLE_MAX, replace=False)]

            snips, valid_times = extract_snippets_fast_ram(
                raw_mod, pick, window=win,
                selected_channels=np.arange(C, dtype=np.int32)
            )
            K, L, N = snips.shape
            print(f"[CH {ch}] kmeans: snips {snips.shape}; kept {N}/{pick.size} after bounds")

            if N < 10:
                raise ValueError("Too few valid snippets for PCA/kmeans.")

            ei_mean = snips.mean(axis=2).astype(np.float32)
            rms = np.sqrt(np.mean(ei_mean**2, axis=1))
            sel = np.flatnonzero(rms > RMS_THRESH)
            if sel.size == 0:
                sel = np.argsort(rms)[-16:]
                sel.sort()
                print(f"[CH {ch}] kmeans: no channels over RMS>{RMS_THRESH}; using top-16")

            X = snips[sel, :, :].transpose(2, 0, 1).reshape(N, sel.size * L).astype(np.float32)

            n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
            pca = PCA(n_components=n_pc, svd_solver='randomized', random_state=0)
            Xpc = pca.fit_transform(X)
            vr = pca.explained_variance_ratio_

            km = KMeans(n_clusters=2, n_init=50, random_state=0)
            lab = km.fit_predict(Xpc)

            ei_c0 = snips[:, :, lab == 0].mean(axis=2).astype(np.float32)
            ei_c1 = snips[:, :, lab == 1].mean(axis=2).astype(np.float32)
            base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
                ei_c0, ei_c1, rms_gate=5.0, use_abs=True, best_align_lag=3
            )

            plt.figure(figsize=(4, 4))
            plt.scatter(Xpc[:, 0], Xpc[:, 1], c=lab, s=10, alpha=0.85)
            c0 = Xpc[lab == 0, :2].mean(axis=0)
            c1 = Xpc[lab == 1, :2].mean(axis=0)
            plt.scatter([c0[0], c1[0]], [c0[1], c1[1]], marker='X', s=140, edgecolor='k', linewidths=1)
            plt.xlabel(f"PC1 ({vr[0]*100:.1f}% var)")
            plt.ylabel(f"PC2 ({vr[1]*100:.1f}% var)" if n_pc >= 2 else "PC2")
            plt.title(f"k=2 PCs (Ksel={sel.size}, L={L}, N={N}), cosine {base_cos:.2f}")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

            fig = plt.figure(figsize=(20, 8))
            ax = fig.add_subplot(111)
            pew.plot_ei_waveforms(
                [ei_c0, ei_c1], positions=ei_positions, scale=70.0, ax=ax,
                colors=["black", "red"], alpha=1.0, linewidth=1.0,
                box_height=1.0, box_width=50.0, aspect=0.5,
            )
            ax.set_title(f"KMeans cluster EIs (cos={base_cos:.3f})")
            plt.show()

            n0 = int(np.sum(lab == 0))
            n1 = int(np.sum(lab == 1))

            km_info = dict(
                # small scalars (safe to print)
                N_total=int(n_total), N_used=int(N),
                n0=n0, n1=n1,
                Ksel=int(sel.size), L=int(L),
                pc1_ev=float(vr[0]*100.0) if len(vr) > 0 else np.nan,
                pc2_ev=float(vr[1]*100.0) if len(vr) > 1 else np.nan,
                cosine=float(base_cos),
                lag=int(base_lag), nch=int(base_nch), overlap_len=int(base_olen),

                # payload for later analysis (DO NOT print)
                sel=np.asarray(sel, dtype=np.int32),
                valid_times=np.asarray(valid_times, dtype=np.int64),
                labels=np.asarray(lab, dtype=np.int8),
                ei_c0=ei_c0.astype(np.float32),
                ei_c1=ei_c1.astype(np.float32),
            )

            km_summary = {k: km_info[k] for k in ["N_total","N_used","n0","n1","Ksel","L","pc1_ev","pc2_ev","cosine","lag"]}
            print(f"[CH {ch}] kmeans summary: {km_summary}")

        except Exception as e:
            print(f"[CH {ch}] kmeans block failed: {type(e).__name__}: {e}")

    # ---- STEP 2/3/4: build EI the real lighthouse way ----
    step2 = build_left_low_high_eis(
        raw_mod, step1['left_times'], step1['left_vals'],
        window=win, left_cap=500, reducer='median', rms_gate=5.0
    )
    print(f"[CH {ch}] base cosine (low vs high) = {step2.get('base_cosine', np.nan):.3f} "
          f"(mid-high={step2.get('cos_mid_high', np.nan):.3f}, low-mid={step2.get('cos_low_mid', np.nan):.3f})")
    if step2.get("base_cosine", 0.0) < min_cosine:
        print(f"[CH {ch}] left_low vs left_high too different (<{min_cosine}); aborting channel.")
        return dict(
            step1=step1,
            step2=step2,
            step3=None,
            step4=None,
            kmeans=km_info,
            blind_split_diag=None,
            cloud_diag=cloud_diag,
            spike_gallery_data=spike_gallery_data,
            all_left_times=left_times_any,
            right5000_times=right5000_times_any,
        )

    step3 = filter_valley_spikes_by_batches(
        raw_mod,
        step1['rightk_times_by_amp'],
        step1['left_times'], step1['left_vals'],
        window=win,
        batch_size=100,
        reducer='median',
        tail_n=100,
        abs_cos_min=0.85,
        base_cos_cap=0.97,
        cos_slack=0.02,
        rms_gate=5.0,
        best_align_lag=3,
        use_abs=True
    )
    print(f"[CH {ch}] base tail cosine={step3.get('base_tail_cosine', np.nan):.3f} "
          f"dyn_floor={step3.get('dyn_floor', np.nan):.3f} "
          f"accepted_right={int(np.asarray(step3.get('accepted_right_times', [])).size)}")
    if "batch_cosines" in step3:
        print(f"[CH {ch}] batch cosines:", [f"{x:.2f}" for x in step3["batch_cosines"]])

    step4 = finalize_ei_with_cap(
        raw_mod,
        step1['left_times'],
        step3['accepted_right_times'],
        window=win,
        final_max_spikes=5_000,
        reducer='median',
        rng_seed=123,
        ref_ch=ch
    )
    print(f"[CH {ch}] Union size={step4.get('final_times', np.array([])).size} main_ch={step4.get('main_ch', None)} "
          f"times_used={step4.get('final_times_used', np.array([])).size}")

    # ---- PLOTS: final EI ----
    try:
        fig = plt.figure(figsize=(20, 8))
        ax = fig.add_subplot(111)
        pew.plot_ei_waveforms(
            step4['final_ei'], positions=ei_positions, scale=70.0, ax=ax,
            colors=["black", "red"], alpha=1.0, linewidth=1.0,
            box_height=1.0, box_width=50.0, aspect=0.5,
        )
        ax.set_title(f"Final EI (ch {ch})")
        plt.show()
    except Exception as e:
        print(f"[CH {ch}] final EI plot failed: {e}")

    spikes_in_samples = np.sort(np.asarray(step4['final_times'], dtype=np.int64))


    # ---- PSTH + repeats raster (fixed 0.500s window; uniform normalization by #trials) ----
    if 'DO_PSTH_RASTER' in globals() and DO_PSTH_RASTER:
        WIN_SEC = 0.5
        BIN_MS  = 1.0
        bin_s   = BIN_MS / 1000.0
        edges   = np.arange(0.0, WIN_SEC + bin_s, bin_s)
        centers = edges[:-1] + 0.5 * bin_s
        nbins   = len(centers)

        spikes_sec = spikes_in_samples / float(sample_rate_hz)

        def simple_psth(spikes_sec, onsets_sec):
            onsets_sec = np.asarray(onsets_sec, dtype=np.float64)
            onsets_sec = onsets_sec[np.isfinite(onsets_sec)]
            if onsets_sec.size == 0:
                return None, 0
            counts = np.zeros(nbins, dtype=np.float64)
            for onset in onsets_sec:
                i0 = np.searchsorted(spikes_sec, onset, side="left")
                i1 = np.searchsorted(spikes_sec, onset + WIN_SEC, side="left")
                rel = spikes_sec[i0:i1] - onset
                if rel.size == 0:
                    continue
                idx = (rel / bin_s).astype(np.int64)
                idx = idx[(idx >= 0) & (idx < nbins)]
                np.add.at(counts, idx, 1)
            ntr = int(onsets_sec.size)
            rate_hz = counts / (ntr * bin_s)  # uniform divide by N trials everywhere + constant bin width
            return rate_hz, ntr

        # pick the onset arrays you loaded (prefer truncated 30 min if present)
        try:
            on_u_use = on_u30
        except NameError:
            try:
                on_u_use = on_u
            except NameError:
                on_u_use = None

        try:
            on_r_use = on_r30
            blk_use  = blk_r30
            tri_use  = tri_r30
        except NameError:
            try:
                on_r_use = on_r
                blk_use  = blk_r
                tri_use  = tri_r
            except NameError:
                on_r_use = None

        # Unique PSTH
        if on_u_use is not None:
            psth_u, ntr_u = simple_psth(spikes_sec, on_u_use)
            plt.figure(figsize=(12, 3))
            plt.plot(centers * 1000.0, psth_u, lw=1.5)
            plt.xlabel("time from onset (ms)")
            plt.ylabel("Hz")
            plt.title(f"Unique PSTH (fixed 0.5s, N={ntr_u} trials)")
            plt.xlim(0, 500)
            plt.grid(alpha=0.25)
            plt.tight_layout()
            plt.show()
        else:
            print(f"[CH {ch}] No on_u30/on_u found; skipping unique PSTH.")

        # Repeat raster + Repeat PSTH
        if on_r_use is not None:
            # repeat raster (ticks), show only first 3 blocks present + zoom first 10s
            on_r_use = np.asarray(on_r_use, dtype=np.float64)
            blk_use  = np.asarray(blk_use,  dtype=np.int64)
            tri_use  = np.asarray(tri_use,  dtype=np.int64)

            # --- show only first 3 blocks that exist in this truncated window ---
            blocks_to_show = np.sort(np.unique(blk_use))[:3]
            # --- zoom to first 10 seconds on rectified axis ---
            TMAX_SEC = 10.0
            KMAX = int(np.floor(TMAX_SEC / WIN_SEC))  # 10s / 0.5s = 20 tiles

            mask = np.isfinite(on_r_use) & np.isin(blk_use, blocks_to_show) & (tri_use < KMAX)
            on_m  = on_r_use[mask]
            blk_m = blk_use[mask]
            tri_m = tri_use[mask]

            # collect spike x-positions per block, then draw as vertical ticks
            spikes_sec = np.asarray(spikes_sec, dtype=np.float64)
            spikes_sec = np.sort(spikes_sec)

            plt.figure(figsize=(20, 4))

            for b in blocks_to_show:
                xs = []
                sel = np.where(blk_m == b)[0]
                for onset, k in zip(on_m[sel], tri_m[sel]):
                    i0 = np.searchsorted(spikes_sec, onset, side="left")
                    i1 = np.searchsorted(spikes_sec, onset + WIN_SEC, side="left")
                    rel = spikes_sec[i0:i1] - onset
                    if rel.size:
                        xs.append(k * WIN_SEC + rel)
                if len(xs):
                    x = np.concatenate(xs)
                    plt.vlines(x, b - 0.45, b + 0.45, linewidth=0.8)  # full-height-ish ticks

            # light onset grid, only for what we show
            for k in range(KMAX):
                plt.axvline(k * WIN_SEC, color="r", lw=0.7, alpha=0.7)

            plt.xlabel("rectified time within repeats segment (s)")
            plt.ylabel("block")
            plt.title(f"Repeat raster (first 3 blocks, ticks, 0–{TMAX_SEC:.0f}s)")
            plt.ylim(blocks_to_show.min() - 0.5, blocks_to_show.max() + 0.5)
            plt.xlim(0, TMAX_SEC)
            plt.tight_layout()
            plt.show()

            psth_r, ntr_r = simple_psth(spikes_sec, on_r_use)
            plt.figure(figsize=(12, 3))
            plt.plot(centers * 1000.0, psth_r, lw=1.5)
            plt.xlabel("time from onset (ms)")
            plt.ylabel("Hz")
            plt.title(f"Repeat PSTH (fixed 0.5s, N={ntr_r} trials)")
            plt.xlim(0, 500)
            plt.grid(alpha=0.25)
            plt.tight_layout()
            plt.show()
        else:
            print(f"[CH {ch}] No on_r30/on_r found; skipping repeat raster/PSTH.")
    # ---- PLOTS: STA (optional, wrapped) ----
    if DO_STA:
        try:
            # compute_and_plot_sta is in your environment in the original notebook; if missing, this will throw.
            sta0, red_tc0, green_tc0, blue_tc0 = compute_and_plot_sta(spikes_in_samples, triggers_sec, **STA_KW)
        except Exception as e:
            print(f"[CH {ch}] STA block skipped/failed: {type(e).__name__}: {e}")
            

    # ---- PLOTS: ISI + smoothed firing rate ----
    if DO_ISI and spikes_in_samples.size > 2:
        try:
            t_ms = (spikes_in_samples / float(sample_rate_hz)) * 1000.0
            isi_ms = np.diff(t_ms)

            bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
            counts, edges = np.histogram(isi_ms, bins=bins)
            centers = edges[:-1] + 0.5 * bin_width_ms

            dt = dt_ms / 1000.0
            sigma_samples = sigma_ms / dt_ms
            spikes_sec = spikes_in_samples / float(sample_rate_hz)
            total_duration = float(spikes_sec.max() + 0.1) if spikes_sec.size > 0 else 1.0
            time_vector = np.arange(0.0, total_duration, dt)
            rate_counts, _ = np.histogram(spikes_sec, bins=np.append(time_vector, total_duration))
            rate = gaussian_filter1d(rate_counts / dt, sigma=sigma_samples)

            fig, axes = plt.subplots(1, 2, figsize=(20, 3))
            axes[0].plot(centers, counts, lw=1.5)
            axes[0].set_xlim(0, max_ms)
            axes[0].set_xlabel("ISI (ms)")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"ISI hist (bin={bin_width_ms} ms, max={max_ms} ms)\nN pairs={isi_ms.size}")

            axes[1].plot(time_vector, rate)
            axes[1].set_xlim(0, total_duration)
            axes[1].set_xlabel("Time (s)")
            axes[1].set_ylabel("Hz")
            axes[1].set_title("Smoothed firing rate")

            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"[CH {ch}] ISI/rate plot failed: {e}")

    # ---- PLOTS: top-left / bottom-left / top-right medians + per-channel blind split ----
    blind_split_diag = None
    try:
        n_sp_for_groups = int(globals().get("N_SP_FOR_GROUPS", 500))
        n_pc_per_ch = int(globals().get("N_PC_PER_CH", 3))

        # choose "main channel" from the final EI (fallback to ch)
        main_ch = int(step4.get('main_ch', ch))

        # final accepted LH spikes -> split into top-left / bottom-left by main-channel amplitude
        all_times = np.sort(np.asarray(step4.get('final_times', spikes_in_samples), dtype=np.int64))
        if all_times.size < 50:
            print(f"[CH {ch}] too few spikes ({all_times.size}) for top-left / bottom-left comparison; skipping.")
        else:
            sn_main, valid_times_main = extract_snippets_fast_ram(
                raw_mod, all_times, window=win,
                selected_channels=np.asarray([main_ch], dtype=np.int32)
            )  # (1, L, Nvalid)

            if sn_main.shape[2] < 50:
                print(f"[CH {ch}] too few valid snippets on main_ch for amplitude ranking; skipping.")
            else:
                amps = np.max(np.abs(sn_main[0, :, :]), axis=0).astype(np.float32)

                order = np.argsort(amps)  # ascending
                N = int(sn_main.shape[2])
                nsel_left = int(min(n_sp_for_groups, N // 2))

                low_idx = order[:nsel_left]
                high_idx = order[-nsel_left:]

                low_times = np.asarray(valid_times_main[low_idx], dtype=np.int64)
                high_times = np.asarray(valid_times_main[high_idx], dtype=np.int64)

                # top 12 channels from the final EI
                final_ei = np.asarray(step4['final_ei'], dtype=np.float32)  # (C, L)
                chan_amp = np.max(np.abs(final_ei), axis=1)
                top12 = np.argsort(chan_amp)[-12:][::-1].astype(np.int32)

                # extract only the capped groups on top12 channels
                sn_hi, _ = extract_snippets_fast_ram(raw_mod, high_times, window=win, selected_channels=top12)
                sn_lo, _ = extract_snippets_fast_ram(raw_mod, low_times, window=win, selected_channels=top12)

                med_hi = np.median(sn_hi, axis=2).astype(np.float32)  # top-left
                med_lo = np.median(sn_lo, axis=2).astype(np.float32)  # bottom-left

                # top-right = largest right(+valley) spikes, ranked directly from step1 amplitudes
                med_right = None
                sn_right = None
                nsel_right = 0

                all_step1_times = np.asarray(step1.get('all_times', []), dtype=np.int64)
                all_step1_vals = np.asarray(step1.get('all_vals', []), dtype=np.float32)
                valley_low = float(step1.get('valley_low', np.nan))

                if all_step1_times.size > 0 and all_step1_vals.size == all_step1_times.size and np.isfinite(valley_low):
                    right_mask = np.isfinite(all_step1_vals) & (all_step1_vals >= valley_low)
                    right_pool_times = np.asarray(all_step1_times[right_mask], dtype=np.int64)
                    right_pool_vals = np.asarray(all_step1_vals[right_mask], dtype=np.float32)

                    if right_pool_times.size > 0:
                        right_order = np.argsort(right_pool_vals)  # most negative first
                        nsel_right = int(min(n_sp_for_groups, right_order.size))
                        right_times = np.asarray(right_pool_times[right_order[:nsel_right]], dtype=np.int64)

                        if right_times.size > 0:
                            sn_right, _ = extract_snippets_fast_ram(
                                raw_mod, right_times, window=win, selected_channels=top12
                            )
                            if sn_right.shape[2] > 0:
                                med_right = np.median(sn_right, axis=2).astype(np.float32)
                                nsel_right = int(sn_right.shape[2])

                print(
                    f"[CH {ch}] group overlays: "
                    f"top-left={sn_hi.shape[2]}, bottom-left={sn_lo.shape[2]}, top-right={nsel_right}"
                )

                # waveform overlays
                L = med_hi.shape[1]
                t_ms = (np.arange(L) + win[0]) / float(sample_rate_hz) * 1000.0

                fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
                axes = axes.ravel()

                for i in range(min(12, top12.size)):
                    ax = axes[i]
                    ax.plot(t_ms, med_hi[i, :], color="black", lw=1.5, label="top-left" if i == 0 else None)
                    ax.plot(t_ms, med_lo[i, :], color="red", lw=1.5, label="bottom-left" if i == 0 else None)
                    if med_right is not None:
                        ax.plot(
                            t_ms, med_right[i, :],
                            color="purple", lw=1.5,
                            label="top-right(+valley)" if i == 0 else None
                        )

                    ax.axvline(0, color="k", lw=0.5, alpha=0.25)
                    ax.axhline(0, color="k", lw=0.5, alpha=0.15)
                    ax.set_title(f"ch {int(top12[i])} (EIamp={chan_amp[top12[i]]:.1f})", fontsize=10)

                for j in range(12, len(axes)):
                    axes[j].axis("off")

                axes[0].legend(loc="upper right", frameon=False)
                title = (
                    f"CH {ch} | main_ch={main_ch} | medians: "
                    f"top-left {sn_hi.shape[2]} vs bottom-left {sn_lo.shape[2]}"
                )
                if med_right is not None:
                    title += f" + top-right {nsel_right}"
                else:
                    title += " | no top-right overlay"
                fig.suptitle(title, y=0.98)
                plt.tight_layout()
                plt.show()

                # per-channel blind 2-way split diagnostics on the already-extracted waveforms
                if sn_right is None or sn_right.shape[2] < 10:
                    print(f"[CH {ch}] not enough top-right snippets for per-channel blind split; skipping.")
                else:
                    from matplotlib.lines import Line2D

                    blind_split_diag = []
                    blind_split_detail = []

                    fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=False, sharey=False)
                    axes = axes.ravel()

                    for i in range(min(12, top12.size)):
                        w_hi = sn_hi[i, :, :].T.astype(np.float32)     # (Nhi, L)
                        w_lo = sn_lo[i, :, :].T.astype(np.float32)     # (Nlo, L)
                        w_rt = sn_right[i, :, :].T.astype(np.float32)  # (Nrt, L)

                        n_hi = int(w_hi.shape[0])
                        n_lo = int(w_lo.shape[0])
                        n_rt = int(w_rt.shape[0])

                        X = np.vstack([w_hi, w_lo, w_rt])
                        grp = np.concatenate([
                            np.zeros(n_hi, dtype=np.int8),
                            np.ones(n_lo, dtype=np.int8),
                            np.full(n_rt, 2, dtype=np.int8),
                        ])

                        n_pc_use = int(max(1, min(n_pc_per_ch, X.shape[0], X.shape[1])))
                        pca = PCA(n_components=n_pc_use, svd_solver='randomized', random_state=0)
                        Xpc = pca.fit_transform(X)
                        vr = pca.explained_variance_ratio_

                        km = KMeans(n_clusters=2, n_init=50, random_state=0)
                        lab = km.fit_predict(Xpc)

                        hi_mask = (grp == 0)
                        lo_mask = (grp == 1)
                        rt_mask = (grp == 2)

                        # X = cluster containing the majority of top-left spikes
                        x_cluster = 0 if np.sum(lab[hi_mask] == 0) >= np.sum(lab[hi_mask] == 1) else 1

                        hi_in_x = (lab[hi_mask] == x_cluster)
                        lo_in_x = (lab[lo_mask] == x_cluster)
                        rt_in_x = (lab[rt_mask] == x_cluster)

                        frac_hi = float(np.mean(hi_in_x))
                        frac_lo = float(np.mean(lo_in_x))
                        frac_rt = float(np.mean(rt_in_x))
                        delta = float(frac_lo - frac_rt)

                        blind_split_diag.append(dict(
                            channel=int(top12[i]),
                            frac_top_left_in_X=frac_hi,
                            frac_bottom_left_in_X=frac_lo,
                            frac_top_right_in_X=frac_rt,
                            delta_bottom_minus_right=delta,
                            pc1_var=float(vr[0]) if vr.size >= 1 else np.nan,
                            pc2_var=float(vr[1]) if vr.size >= 2 else np.nan,
                            n_top_left=n_hi,
                            n_bottom_left=n_lo,
                            n_top_right=n_rt,
                            x_cluster=int(x_cluster),
                        ))

                        blind_split_detail.append(dict(
                            channel=int(top12[i]),
                            idx_in_top12=int(i),
                            frac_top_left_in_X=frac_hi,
                            frac_bottom_left_in_X=frac_lo,
                            frac_top_right_in_X=frac_rt,
                            delta_bottom_minus_right=delta,
                            bottom_left_in_X_mask=lo_in_x.copy(),
                            top_right_in_X_mask=rt_in_x.copy(),
                        ))

                        pc1 = Xpc[:, 0]
                        pc2 = Xpc[:, 1] if Xpc.shape[1] >= 2 else np.zeros_like(pc1)

                        ax = axes[i]
                        for group_code, color in [(0, "black"), (1, "red"), (2, "purple")]:
                            m_in = (grp == group_code) & (lab == x_cluster)
                            m_out = (grp == group_code) & (lab != x_cluster)

                            if np.any(m_in):
                                ax.scatter(pc1[m_in], pc2[m_in], c=color, s=8, alpha=0.55)
                            if np.any(m_out):
                                ax.scatter(
                                    pc1[m_out], pc2[m_out],
                                    facecolors='none', edgecolors=color,
                                    s=14, alpha=0.65, linewidths=0.6
                                )

                        for k in [0, 1]:
                            mk = (lab == k)
                            if np.any(mk):
                                ctr = np.array([pc1[mk].mean(), pc2[mk].mean()])
                                ax.scatter(
                                    [ctr[0]], [ctr[1]],
                                    c=("tab:orange" if k == x_cluster else "tab:cyan"),
                                    marker='X', s=70, linewidths=0.8
                                )

                        pc1_txt = f"{vr[0]*100:.1f}%" if vr.size >= 1 else "?"
                        pc2_txt = f"{vr[1]*100:.1f}%" if vr.size >= 2 else "?"
                        ax.set_title(
                            f"ch {int(top12[i])} | TL {frac_hi:.2f} BL {frac_lo:.2f} TR {frac_rt:.2f}",
                            fontsize=9
                        )
                        ax.set_xlabel(f"PC1 ({pc1_txt})", fontsize=8)
                        ax.set_ylabel(f"PC2 ({pc2_txt})", fontsize=8)
                        ax.grid(alpha=0.2)

                    for j in range(12, len(axes)):
                        axes[j].axis("off")

                    legend_handles = [
                        Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                               markeredgecolor='black', markersize=6, label='top-left, in X'),
                        Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
                               markeredgecolor='red', markersize=6, label='bottom-left, in X'),
                        Line2D([0], [0], marker='o', color='w', markerfacecolor='purple',
                               markeredgecolor='purple', markersize=6, label='top-right, in X'),
                        Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
                               markeredgecolor='black', markersize=6, label='not in X'),
                    ]
                    axes[0].legend(handles=legend_handles, loc='best', frameon=False, fontsize=8)

                    fig.suptitle(
                        f"CH {ch} | per-channel blind split on top-left / bottom-left / top-right waveforms",
                        y=0.98
                    )
                    plt.tight_layout()
                    plt.show()

                    blind_split_diag = sorted(
                        blind_split_diag,
                        key=lambda d: (d["delta_bottom_minus_right"], d["frac_top_left_in_X"]),
                        reverse=True
                    )

                    print(f"[CH {ch}] per-channel blind split summary "
                          f"(sorted by bottom-left minus top-right in X):")
                    for row in blind_split_diag:
                        print(
                            f"  ch {row['channel']:3d}: "
                            f"TL={row['frac_top_left_in_X']:.3f}, "
                            f"BL={row['frac_bottom_left_in_X']:.3f}, "
                            f"TR={row['frac_top_right_in_X']:.3f}, "
                            f"Δ={row['delta_bottom_minus_right']:.3f}"
                        )

                    # ---- select discriminative channels and purify BL ----
                    discrim_rows = [
                        row for row in blind_split_detail
                        if row["frac_top_left_in_X"] >= TL_FRAC_MIN
                        and row["frac_bottom_left_in_X"] >= BL_FRAC_MIN
                        and row["frac_top_right_in_X"] <= TR_FRAC_MAX
                    ]
                    discrim_rows = sorted(
                        discrim_rows,
                        key=lambda d: (d["delta_bottom_minus_right"], d["frac_top_left_in_X"]),
                        reverse=True
                    )

                    if len(discrim_rows) == 0:
                        print(f"[CH {ch}] no channels passed TL/BL/TR thresholds; skipping purified-BL cosine/harm diag.")
                    else:
                        discrim_idx = np.asarray([row["idx_in_top12"] for row in discrim_rows], dtype=np.int32)
                        discrim_channels = np.asarray([row["channel"] for row in discrim_rows], dtype=np.int32)

                        # purified BL = BL spikes that land in X on ALL discriminative channels
                        purified_bl_mask = np.ones(sn_lo.shape[2], dtype=bool)
                        for row in discrim_rows:
                            purified_bl_mask &= np.asarray(row["bottom_left_in_X_mask"], dtype=bool)

                        n_purified = int(np.sum(purified_bl_mask))
                        print(
                            f"[CH {ch}] discriminative channels: {discrim_channels.tolist()} | "
                            f"purified BL = {n_purified}/{int(sn_lo.shape[2])}"
                        )

                        if n_purified < 20:
                            print(f"[CH {ch}] purified BL set too small for cosine/harm plot; skipping.")
                        else:
                            # purified BL median (used only to define mask), on discriminative channels
                            bl_template = np.median(
                                sn_lo[discrim_idx][:, :, purified_bl_mask],
                                axis=2
                            ).astype(np.float32)   # [nch_good, L]

                            # references for BL cloud on each selected channel
                            ref_snips = sn_lo[discrim_idx][:, :, purified_bl_mask].astype(np.float32)   # [nch_good, L, Nref]
                            n_ref = int(ref_snips.shape[2])

                            # candidate spikes split by group
                            bl_cand = sn_lo[discrim_idx].astype(np.float32)      # [nch_good, L, Nbl]
                            tr_cand = sn_right[discrim_idx].astype(np.float32)   # [nch_good, L, Ntr]

                            n_bl_plot = int(bl_cand.shape[2])
                            n_tr_plot = int(tr_cand.shape[2])

                            # for BL leave-one-out:
                            purified_idx = np.flatnonzero(purified_bl_mask).astype(np.int64)
                            refpos_by_bl = -np.ones(n_bl_plot, dtype=np.int64)
                            refpos_by_bl[purified_idx] = np.arange(purified_idx.size, dtype=np.int64)

                            cloud_scores_bl = np.empty((discrim_channels.size, n_bl_plot), dtype=np.float32)
                            cloud_scores_tr = np.empty((discrim_channels.size, n_tr_plot), dtype=np.float32)
                            cloud_masks = []

                            for jj in range(discrim_channels.size):
                                ref_j = ref_snips[jj].T      # [Nref, L]
                                bl_j  = bl_cand[jj].T        # [Nbl, L]
                                tr_j  = tr_cand[jj].T        # [Ntr, L]
                                tpl_j = bl_template[jj]      # [L]

                                score_bl_j, mask_j = cloud_topk_cosine_scores_masked(
                                    ref_j,
                                    bl_j,
                                    tpl_j,
                                    amp_thr=COS_MASK_ADC,
                                    pad=COS_MASK_PAD,
                                    topk=CLOUD_TOPK,
                                    leave_one_out_refpos=refpos_by_bl,
                                )

                                score_tr_j, _ = cloud_topk_cosine_scores_masked(
                                    ref_j,
                                    tr_j,
                                    tpl_j,
                                    amp_thr=COS_MASK_ADC,
                                    pad=COS_MASK_PAD,
                                    topk=CLOUD_TOPK,
                                    leave_one_out_refpos=None,
                                )

                                cloud_scores_bl[jj] = score_bl_j
                                cloud_scores_tr[jj] = score_tr_j
                                cloud_masks.append(mask_j)

                            plot_cloud_score_histograms_by_channel(
                                cloud_scores_bl,
                                cloud_scores_tr,
                                discrim_channels,
                                bins=CLOUD_HIST_BINS,
                                title=(
                                    f"CH {ch} | BL-cloud score by discriminative channel "
                                    f"(top-k cosine, k={CLOUD_TOPK})"
                                ),
                            )

                            # ---- store data for follow-up manual spike inspection ----
                            discrim_weights = np.asarray(
                                [max(float(row["delta_bottom_minus_right"]), 1e-6) for row in discrim_rows],
                                dtype=np.float32
                            )
                            discrim_weights /= np.maximum(discrim_weights.sum(), 1e-12)

                            # aggregate BL-cloud score across discriminative channels
                            # only for ranking / browsing, not as a final classifier
                            agg_score_bl = (
                                discrim_weights[:, None] * cloud_scores_bl
                            ).sum(axis=0).astype(np.float32)

                            agg_score_tr = (
                                discrim_weights[:, None] * cloud_scores_tr
                            ).sum(axis=0).astype(np.float32)

                            order_bl_low = np.argsort(agg_score_bl).astype(np.int64)         # lowest first
                            order_tr_high = np.argsort(agg_score_tr)[::-1].astype(np.int64)  # highest first

                            t_ms_gallery = (
                                (np.arange(sn_hi.shape[1]) + win[0]) / float(sample_rate_hz) * 1000.0
                            ).astype(np.float32)

                            spike_gallery_data = dict(
                                source_ch=int(ch),
                                main_ch=int(main_ch),

                                top12=top12.copy(),
                                discrim_channels=discrim_channels.copy(),
                                discrim_idx=discrim_idx.copy(),
                                discrim_weights=discrim_weights.copy(),

                                t_ms=t_ms_gallery,

                                med_top_left=med_hi.copy(),
                                med_bottom_left=med_lo.copy(),
                                med_top_right=None if med_right is None else med_right.copy(),

                                sn_top_left=sn_hi.copy(),
                                sn_bottom_left=sn_lo.copy(),
                                sn_top_right=None if sn_right is None else sn_right.copy(),

                                purified_bl_mask=purified_bl_mask.copy(),

                                cloud_scores_bl=cloud_scores_bl.copy(),   # [n_good, Nbl]
                                cloud_scores_tr=cloud_scores_tr.copy(),   # [n_good, Ntr]
                                agg_score_bl=agg_score_bl.copy(),         # [Nbl]
                                agg_score_tr=agg_score_tr.copy(),         # [Ntr]

                                order_bl_low=order_bl_low.copy(),
                                order_tr_high=order_tr_high.copy(),

                                times_bottom_left=low_times.copy(),
                                times_top_right=None if med_right is None else right_times.copy(),
                                times_top_left=high_times.copy(),
                            )

                            print(f"[CH {ch}] BL-cloud score summary:")
                            for jj, ch_j in enumerate(discrim_channels):
                                bl_med = float(np.nanmedian(cloud_scores_bl[jj]))
                                tr_med = float(np.nanmedian(cloud_scores_tr[jj]))
                                bl_q05 = float(np.nanpercentile(cloud_scores_bl[jj], 5))
                                tr_q95 = float(np.nanpercentile(cloud_scores_tr[jj], 95))
                                print(
                                    f"  ch {int(ch_j):3d}: "
                                    f"BL med={bl_med:.3f}, BL q05={bl_q05:.3f} | "
                                    f"TR med={tr_med:.3f}, TR q95={tr_q95:.3f}"
                                )

                            cloud_diag = dict(
                                discriminative_channels=discrim_channels,
                                purified_bl_mask=purified_bl_mask,
                                bl_template=bl_template,
                                cloud_scores_bl=cloud_scores_bl,
                                cloud_scores_tr=cloud_scores_tr,
                                cloud_masks=np.asarray(cloud_masks, dtype=bool),
                                refpos_by_bl=refpos_by_bl,
                                cloud_topk=int(CLOUD_TOPK),
                            )

    except Exception as e:
        print(f"[CH {ch}] top/bottom/right overlay or blind-split plot failed: {type(e).__name__}: {e}")
    # ---- PLOTS: amplitude vs previous-spike metrics (optional) ----
    if DO_PREV_METRICS:
        try:
            if "final_amplitudes" in step4:
                amps = np.asarray(step4["final_amplitudes"])
                window_ms = 50.0
                _ = compute_prev_metrics(step4['final_times'], sample_rate_hz=sample_rate_hz, window_ms=window_ms)
                plot_amp_prev_scatter_and_bar(
                    step4['final_times'], amps,
                    sample_rate_hz=sample_rate_hz,
                    window_ms=window_ms,
                    max_k=20,
                    min_count=5,
                    error='std',
                    alpha=0.15,
                    title_suffix=f"ch {ch}, window={window_ms} ms"
                )
                plt.show()
            else:
                print(f"[CH {ch}] step4 has no 'final_amplitudes' → skipping prev-metrics plot.")
        except Exception as e:
            print(f"[CH {ch}] prev-metrics plot failed: {type(e).__name__}: {e}")

    print(f"[CH {ch}] done in {time.time()-t0:.1f}s")
    return dict(
        step1=step1,
        step2=step2,
        step3=step3,
        step4=step4,
        kmeans=km_info,
        blind_split_diag=blind_split_diag,
        cloud_diag=cloud_diag,
        spike_gallery_data=spike_gallery_data,
        all_left_times=left_times_any,
        right5000_times=right5000_times_any,
    )

ch = 461 #461, 309, 487 AB shard; 340 AB shard but tricky; 353 and 256 real 2; 395 doublets;  501 unclear;  475 unclear; 332 - maybe 2 cells?
out = inspect_lighthouse_channel(ch)

# 395 - double LH? weird. Super LH and then probably a double cell hump..
# 469, 368, 505, 340   best pure
# 253, 405, 445 - good check. 445 - very good LH plus soup of 2. 405 - sus, many uncertain in the TR category
# 461 - main unit plus soup, UNRELIABLE
# 33 - LH plus soup, good. 366 (plus 2 cells in soup), 445 - mild contam (~50 spikes), and 469 - pure (17spikes), 340 - the most pure (9-10spikes)
# 116, 279, 298 - complex channel with very good LH AND weird artifact/cell

# 53 - fake LH (2 units) but very good; 
# 353 - fake LH (2 units), not very good; MAY work after 357 first.. TEST FOR SOUP-like stuff
# 4 - bad LH (large valley), soup adds 2 good units
# 8 - interesting case. Small bump which is a unit plus another unit plus artifact. Plus more!


## ISLANDS

In [ ]:
# ============================================================
# K-sweep proposal generator from `out`
# - builds ONE pooled feature space from LEFT spikes
# - runs deterministic k-means for K = 2, 3, 4
# - for each cluster, builds CENTRAL-CORE templates at several fractions
# - checks EI stability across tightening levels
# - plots PCA scatter, EI overlays, pairwise EI cosine matrices,
#   and main-channel waveform stability
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from matplotlib import cm

# -------------------------
# USER KNOBS
# -------------------------
# proposal source
#   "left_only"         -> use only LEFT spikes
#   "left_plus_right5k" -> use LEFT spikes plus returned 5k strongest RIGHT spikes
PROPOSAL_SOURCE = "left_plus_right5k"

USE_ISOLATED_POOL = True
BALANCE_LR_FOR_KMEANS = True

ISO_SEP = 80                  # samples; same spirit as inspect_lighthouse_channel
SUBSAMPLE_MAX = 10000

RMS_THRESH = 5.0
MIN_FEATURE_CH = 16
MAX_FEATURE_CH = 80
N_PC = 2

K_LIST = [3]



# core fractions used for tightening/stability
CORE_FRACS = [0.50, 0.70]
PLOT_CORE_FRAC = 0.50         # the one used for pairwise EI comparison / overlay

# EI build
EI_CAP = 1000                 # cap spikes per EI for speed/memory
EI_REDUCER = "median"         # "median" or "mean"

# pairwise EI comparison
EI_RMS_GATE = 5.0
EI_ALIGN_LAG = 3

# plotting
SCATTER_ALPHA = 0.45
SCATTER_SIZE = 8
MAIN_WF_LW = 1.6

# AB-shard screening for small extra clusters
SMALL_CLUSTER_FRAC = 0.10     # <= this * largest cluster => test as possible shard
AB_MAX_LAG = 3
AB_SUPPORT_REL = 0.10
AB_SUPPORT_ABS = 30.0
AB_TIME_KEEP_REL = 0.10
AB_FRAC_IN_THR = 0.20
AB_OUT_IN_RATIO_THR = 2.0
AB_RESID_FRAC_MIN = 0.08
AB_SHARED_COS_THR = 0.95
AB_SHARED_ALPHA_THR = 0.95
DO_AB_PLOTS = True

# -------------------------
# CHECKS
# -------------------------
assert out is not None and "step1" in out and out["step1"] is not None, "Need out['step1']"
assert "raw_mod" in globals(), "Need raw_mod in notebook"
assert "ei_positions" in globals(), "Need ei_positions in notebook"
assert "extract_snippets_fast_ram" in globals(), "Need extract_snippets_fast_ram"
assert "cosine_two_eis" in globals(), "Need cosine_two_eis from joint_utils"
assert "pew" in globals(), "Need plot_ei_waveforms as pew"
assert "PCA" in globals() and "KMeans" in globals(), "Need sklearn PCA and KMeans imported"

win_use = globals().get("win", (-40, 80))
sample_rate_hz_use = float(globals().get("sample_rate_hz", 20_000))
rng = np.random.RandomState(0)

# -------------------------
# HELPERS
# -------------------------


def _safe_unique_sorted(x):
    x = np.asarray(x, dtype=np.int64).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x.astype(np.int64)
    return np.sort(np.unique(x.astype(np.int64)))

def _subsample_times(times_abs, max_n, rng):
    times_abs = _safe_unique_sorted(times_abs)
    if times_abs.size <= max_n:
        return times_abs
    idx = np.sort(rng.choice(times_abs.size, size=max_n, replace=False))
    return times_abs[idx]

def _median_or_mean(snips, reducer="median"):
    if reducer == "median":
        return np.median(snips, axis=2).astype(np.float32)
    elif reducer == "mean":
        return np.mean(snips, axis=2).astype(np.float32)
    else:
        raise ValueError("reducer must be 'median' or 'mean'")

def _extract_full_ei(raw, times_abs, window, reducer="median", max_spikes=1000):
    times_abs = _safe_unique_sorted(times_abs)
    if times_abs.size == 0:
        return None, np.array([], dtype=np.int64)
    times_use = _subsample_times(times_abs, max_spikes, rng)
    sn, tv = extract_snippets_fast_ram(
        raw,
        times_use,
        window=window,
        selected_channels=np.arange(raw.shape[1], dtype=np.int32)
    )  # [C, L, N]
    tv = np.asarray(tv, dtype=np.int64)
    if sn.shape[2] == 0:
        return None, tv
    ei = _median_or_mean(sn, reducer=reducer)
    return ei, tv


def _balanced_subsample_two_pools(times_a, times_b, max_n, rng):
    """
    Balanced subsample from two spike pools.
    Tries to split max_n evenly across pools.
    If one side has too few spikes, take all from that side and fill from the other.
    Returns sorted unique int64 times.
    """
    times_a = _safe_unique_sorted(times_a)
    times_b = _safe_unique_sorted(times_b)

    if max_n is None or max_n <= 0:
        return _safe_unique_sorted(np.r_[times_a, times_b])

    n_a = times_a.size
    n_b = times_b.size
    n_tot = n_a + n_b

    if n_tot <= max_n:
        return _safe_unique_sorted(np.r_[times_a, times_b])

    half = max_n // 2

    # initial target: half/half
    take_a = min(n_a, half)
    take_b = min(n_b, max_n - take_a)

    # if B was too small, refill from A
    rem = max_n - (take_a + take_b)
    if rem > 0:
        add_a = min(rem, n_a - take_a)
        take_a += add_a
        rem -= add_a

    # if A was too small, refill from B
    if rem > 0:
        add_b = min(rem, n_b - take_b)
        take_b += add_b
        rem -= add_b

    if take_a > 0:
        idx_a = np.sort(rng.choice(n_a, size=take_a, replace=False))
        sub_a = times_a[idx_a]
    else:
        sub_a = np.array([], dtype=np.int64)

    if take_b > 0:
        idx_b = np.sort(rng.choice(n_b, size=take_b, replace=False))
        sub_b = times_b[idx_b]
    else:
        sub_b = np.array([], dtype=np.int64)

    return _safe_unique_sorted(np.r_[sub_a, sub_b])

def _extract_main_channel_amp(raw, times_abs, window, ch):
    times_abs = _safe_unique_sorted(times_abs)
    if times_abs.size == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
    sn, tv = extract_snippets_fast_ram(
        raw,
        times_abs,
        window=window,
        selected_channels=np.asarray([ch], dtype=np.int32)
    )  # [1, L, N]
    tv = np.asarray(tv, dtype=np.int64)
    if sn.shape[2] == 0:
        return tv, np.array([], dtype=np.float32)
    amp = np.ptp(sn[0, :, :], axis=0).astype(np.float32)
    return tv, amp

def _mean_ei_rms_channels(raw, times_abs, window):
    if times_abs.size == 0:
        raise RuntimeError("No times provided for RMS channel selection")
    sn, tv = extract_snippets_fast_ram(
        raw,
        times_abs,
        window=window,
        selected_channels=np.arange(raw.shape[1], dtype=np.int32)
    )  # [C, L, N]
    if sn.shape[2] == 0:
        raise RuntimeError("No valid snippets for RMS channel selection")
    ei_mean = np.mean(sn, axis=2).astype(np.float32)
    rms = np.sqrt(np.mean(ei_mean ** 2, axis=1)).astype(np.float32)
    return rms

def _pairwise_cosine_matrix(eis, rms_gate=5.0, best_align_lag=3):
    n = len(eis)
    M = np.full((n, n), np.nan, dtype=np.float32)
    for i in range(n):
        M[i, i] = 1.0
        for j in range(i + 1, n):
            if (eis[i] is None) or (eis[j] is None):
                continue
            c, nch, lag, olen = cosine_two_eis(
                eis[i], eis[j],
                rms_gate=rms_gate,
                use_abs=True,
                best_align_lag=best_align_lag
            )
            M[i, j] = float(c)
            M[j, i] = float(c)
    return M

def _plot_cosine_matrix(M, labels, title):
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(M, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="EI cosine")
    plt.tight_layout()
    plt.show()


def _shift_ei_time(ei, lag):
    """Shift along time axis by lag samples. lag>0 moves waveform later."""
    if lag == 0:
        return ei
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :-lag]
    else:
        out[:, :lag] = ei[:, -lag:]
    return out

def _support_from_ei_containment(ei, support_rel=0.10, support_abs=30.0):
    p2p = np.ptp(ei, axis=1)
    thr = max(float(support_abs), float(support_rel) * float(np.max(p2p) + 1e-12))
    S = p2p >= thr
    return S, p2p, thr

def _best_lag_on_support(X, Y, S, max_lag=3, time_keep_rel=0.10):
    Xs = X[S, :]
    env = np.max(np.abs(Xs), axis=0) if Xs.size else np.max(np.abs(X), axis=0)
    tthr = float(time_keep_rel) * float(np.max(env) + 1e-12)
    T = env >= tthr
    if not np.any(T):
        T = np.ones(X.shape[1], dtype=bool)

    A = X[S, :][:, T].ravel().astype(np.float64)
    nA = np.linalg.norm(A) + 1e-12

    best = {"lag": 0, "cos": -np.inf, "T": T, "tthr": tthr}
    for lag in range(-max_lag, max_lag + 1):
        Ys = _shift_ei_time(Y, lag)
        B = Ys[S, :][:, T].ravel().astype(np.float64)
        nB = np.linalg.norm(B) + 1e-12
        cos = float((A @ B) / (nA * nB))
        if cos > best["cos"]:
            best = {"lag": int(lag), "cos": cos, "T": T, "tthr": tthr}
    return best

def _containment_metrics(X, Y,
                         max_lag=3,
                         support_rel=0.10,
                         support_abs=30.0,
                         time_keep_rel=0.10):
    """
    Tests whether X is contained in Y:
      X ~ shared core, Y ~ candidate bigger/additive EI
    """
    S, p2pX, thr = _support_from_ei_containment(X, support_rel=support_rel, support_abs=support_abs)

    best = _best_lag_on_support(X, Y, S, max_lag=max_lag, time_keep_rel=time_keep_rel)
    lag = best["lag"]
    Yal = _shift_ei_time(Y, lag)
    T = best["T"]

    A = X[S, :][:, T].ravel().astype(np.float64)
    B = Yal[S, :][:, T].ravel().astype(np.float64)
    denom = float(A @ A) + 1e-12
    alpha = float((A @ B) / denom)

    R = Yal - alpha * X

    Yin = Yal[S, :]
    Rin = R[S, :]
    Yout = Yal[~S, :]
    Rout = R[~S, :]

    Ein = float(np.linalg.norm(Rin))
    Eout = float(np.linalg.norm(Rout))
    frac_in = float(np.linalg.norm(Rin) / (np.linalg.norm(Yin) + 1e-12))
    frac_all = float(np.linalg.norm(R) / (np.linalg.norm(Yal) + 1e-12))
    out_in = float(Eout / (Ein + 1e-12))

    SR, p2pR, thrR = _support_from_ei_containment(R, support_rel=support_rel, support_abs=support_abs)
    jacc = float(np.sum(S & SR) / (np.sum(S | SR) + 1e-12))

    return dict(
        lag=int(lag),
        cos_on_support=float(best["cos"]),
        alpha=float(alpha),
        support_thr=float(thr),
        support_n=int(np.sum(S)),
        frac_in=float(frac_in),
        frac_all=float(frac_all),
        out_in=float(out_in),
        resid_support_thr=float(thrR),
        resid_support_n=int(np.sum(SR)),
        support_resid_jacc=float(jacc),
        X_support=S,
        resid_support=SR,
        Y_aligned=Yal,
        resid=R,
    )

def _is_contained(m, frac_in_thr=0.20, out_in_ratio_thr=2.0):
    return (m["frac_in"] <= frac_in_thr) and (m["out_in"] >= out_in_ratio_thr)

def _ab_pair_verdict(ei_a, ei_b,
                     max_lag=3,
                     support_rel=0.10,
                     support_abs=30.0,
                     time_keep_rel=0.10,
                     frac_in_thr=0.20,
                     out_in_ratio_thr=2.0,
                     resid_frac_min=0.08,
                     shared_cos_thr=0.95,
                     shared_alpha_thr=0.95):
    """
    Same logic as the LH notebook AB-shard cell, but generic for any EI pair.
    """
    m_ab = _containment_metrics(
        ei_a, ei_b,
        max_lag=max_lag,
        support_rel=support_rel,
        support_abs=support_abs,
        time_keep_rel=time_keep_rel
    )  # "a contained in b?"
    m_ba = _containment_metrics(
        ei_b, ei_a,
        max_lag=max_lag,
        support_rel=support_rel,
        support_abs=support_abs,
        time_keep_rel=time_keep_rel
    )  # "b contained in a?"

    c_ab = _is_contained(m_ab, frac_in_thr=frac_in_thr, out_in_ratio_thr=out_in_ratio_thr)
    c_ba = _is_contained(m_ba, frac_in_thr=frac_in_thr, out_in_ratio_thr=out_in_ratio_thr)

    shared_ab = (m_ab["cos_on_support"] >= shared_cos_thr) and (m_ab["alpha"] >= shared_alpha_thr)
    shared_ba = (m_ba["cos_on_support"] >= shared_cos_thr) and (m_ba["alpha"] >= shared_alpha_thr)
    shared_core = bool(shared_ab or shared_ba)

    if shared_core and c_ab and c_ba and (m_ab["frac_all"] < resid_frac_min) and (m_ba["frac_all"] < resid_frac_min):
        verdict = "SAME_UNIT_SPLIT"
    elif shared_core and ((c_ab and not c_ba) or (c_ba and not c_ab)):
        verdict = "AB_SHARD"
    elif shared_core:
        verdict = "SHARED_CORE_COMPLEX"
    elif (not c_ab) and (not c_ba):
        verdict = "TWO_UNITS"
    else:
        verdict = "AMBIGUOUS"

    return {
        "verdict": verdict,
        "shared_core": shared_core,
        "shared_dirs": {"a_to_b": bool(shared_ab), "b_to_a": bool(shared_ba)},
        "contained_dirs": {"a_in_b": bool(c_ab), "b_in_a": bool(c_ba)},
        "m_ab": m_ab,
        "m_ba": m_ba,
    }

def _core_times_by_fraction(times_cluster, dist_cluster, frac):
    n = int(times_cluster.size)
    n_keep = max(20, int(np.floor(frac * n)))
    n_keep = min(n_keep, n)
    order = np.argsort(dist_cluster)   # closest to centroid first
    return np.asarray(times_cluster[order[:n_keep]], dtype=np.int64), n_keep

# -------------------------
# SOURCE SPIKES = selectable pool
# -------------------------
left_times_all = _safe_unique_sorted(
    out.get("all_left_times", out["step1"].get("left_times", []))
)
if left_times_all.size == 0:
    raise RuntimeError("No LEFT spikes found in out")

right5k_times_all = _safe_unique_sorted(
    out.get("right5000_times", out["step1"].get("rightk_times_by_amp", []))
)

if PROPOSAL_SOURCE == "left_only":
    pool_times_all = left_times_all.copy()
    print(f"Proposal source: LEFT only ({pool_times_all.size} spikes)")
elif PROPOSAL_SOURCE == "left_plus_right5k":
    pool_times_all = _safe_unique_sorted(np.r_[left_times_all, right5k_times_all])
    print(
        f"Proposal source: LEFT + RIGHT5k | "
        f"left={left_times_all.size}, right5k={right5k_times_all.size}, union={pool_times_all.size}"
    )
else:
    raise ValueError("PROPOSAL_SOURCE must be 'left_only' or 'left_plus_right5k'")

# optional isolation filter on the chosen pool
pool_for_features = pool_times_all.copy()
if USE_ISOLATED_POOL:
    all_times = _safe_unique_sorted(out["step1"].get("all_times", []))
    if all_times.size > 0:
        idx = np.searchsorted(all_times, pool_times_all)
        ok_map = (idx < all_times.size) & (all_times[idx] == pool_times_all)
        idx_ok = idx[ok_map]
        pool_ok = pool_times_all[ok_map]

        dt_prev = np.full(pool_ok.size, np.iinfo(np.int64).max, dtype=np.int64)
        dt_next = np.full(pool_ok.size, np.iinfo(np.int64).max, dtype=np.int64)

        m = idx_ok > 0
        dt_prev[m] = pool_ok[m] - all_times[idx_ok[m] - 1]
        m = (idx_ok + 1) < all_times.size
        dt_next[m] = all_times[idx_ok[m] + 1] - pool_ok[m]

        iso = (dt_prev >= ISO_SEP) & (dt_next >= ISO_SEP)
        pool_iso = pool_ok[iso]

        if pool_iso.size >= 100:
            pool_for_features = pool_iso
            print(
                f"Using isolated proposal pool: {pool_iso.size}/{pool_times_all.size} "
                f"(sep≥{ISO_SEP} samples)"
            )
        else:
            print(
                f"Isolated proposal pool too small ({pool_iso.size}); "
                f"using full proposal pool ({pool_times_all.size})"
            )
    else:
        print("step1['all_times'] missing/empty; using full proposal pool.")
else:
    print("USE_ISOLATED_POOL=False; using full proposal pool.")

# subsample for feature-space construction
if (PROPOSAL_SOURCE == "left_plus_right5k") and BALANCE_LR_FOR_KMEANS:
    left_pool_feat = np.intersect1d(pool_for_features, left_times_all, assume_unique=False)
    right_pool_feat = np.intersect1d(pool_for_features, right5k_times_all, assume_unique=False)

    times_feat = _balanced_subsample_two_pools(
        left_pool_feat,
        right_pool_feat,
        SUBSAMPLE_MAX,
        rng
    )

    n_left_feat = np.intersect1d(times_feat, left_pool_feat).size
    n_right_feat = np.intersect1d(times_feat, right_pool_feat).size

    print(
        f"Feature-space spikes used (balanced LEFT/RIGHT): {times_feat.size} | "
        f"left={n_left_feat}, right={n_right_feat}"
    )
else:
    times_feat = _subsample_times(pool_for_features, SUBSAMPLE_MAX, rng)
    print(f"Feature-space spikes used: {times_feat.size}")

# -------------------------
# CHANNEL SELECTION
# -------------------------
rms = _mean_ei_rms_channels(raw_mod, times_feat, win_use)
sel = np.flatnonzero(rms > RMS_THRESH).astype(np.int32)

if sel.size > MAX_FEATURE_CH:
    sel = np.argsort(rms)[-MAX_FEATURE_CH:]
elif sel.size < MIN_FEATURE_CH:
    sel = np.argsort(rms)[-MIN_FEATURE_CH:]

sel = np.sort(sel).astype(np.int32)

print(f"Selected feature channels: {sel.size}")
print(f"Top RMS = {float(np.max(rms)):.2f}")

# -------------------------
# EXTRACT FEATURE SNIPPETS
# -------------------------
sn, valid_times = extract_snippets_fast_ram(
    raw_mod,
    times_feat,
    window=win_use,
    selected_channels=sel
)  # [Ksel, L, N]
valid_times = np.asarray(valid_times, dtype=np.int64)

Ksel, L, N = sn.shape
if N < 50:
    raise RuntimeError(f"Too few valid snippets after bounds check: N={N}")

X = sn.transpose(2, 0, 1).reshape(N, Ksel * L).astype(np.float32)

n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
pca = PCA(n_components=n_pc, svd_solver="randomized", random_state=0)
Xpc = pca.fit_transform(X)
vr = pca.explained_variance_ratio_

print(f"Feature matrix: X={X.shape}, PCA dims={n_pc}")
print("Explained variance ratio:", np.round(vr[:min(6, vr.size)], 3))

n_left_feat_final = np.intersect1d(valid_times, left_times_all).size
n_right_feat_final = np.intersect1d(valid_times, right5k_times_all).size
print(f"Final valid feature spikes: total={valid_times.size}, left={n_left_feat_final}, right={n_right_feat_final}")

# -------------------------
# REFERENCE MAIN CHANNEL
# -------------------------
main_ch = int(out["step4"].get("main_ch", 0)) if (out.get("step4") is not None) else int(sel[np.argmax(rms[sel])])

# -------------------------
# K-SWEEP
# -------------------------
n_left_feat_final = np.intersect1d(valid_times, left_times_all).size
n_right_feat_final = np.intersect1d(valid_times, right5k_times_all).size

print(f"Final valid feature spikes: total={valid_times.size}, left={n_left_feat_final}, right={n_right_feat_final}")

k_sweep_out = {
    "proposal_source": PROPOSAL_SOURCE,
    "balance_lr_for_kmeans": BALANCE_LR_FOR_KMEANS,
    "pool_times_all": pool_times_all.copy(),
    "pool_for_features": pool_for_features.copy(),
    "times_feat": valid_times.copy(),
    "n_left_feat": int(n_left_feat_final),
    "n_right_feat": int(n_right_feat_final),
    "sel_channels": sel.copy(),
    "Xpc": Xpc.copy(),
    "pca_explained_variance_ratio": vr.copy(),
    "main_ch": main_ch,
    "win": tuple(win_use),
    "core_fracs": list(CORE_FRACS),
    "plot_core_frac": float(PLOT_CORE_FRAC),
    "results_by_k": {},
}

for KVAL in K_LIST:
    print("\n" + "=" * 80)
    print(f"K = {KVAL}")
    print("=" * 80)

    km = KMeans(n_clusters=KVAL, n_init=50, random_state=0)
    labels = km.fit_predict(Xpc)
    centroids = km.cluster_centers_

    # distances to own centroid in PCA space
    dist = np.empty(N, dtype=np.float32)
    for kk in range(KVAL):
        m = labels == kk
        if np.any(m):
            d = Xpc[m] - centroids[kk][None, :]
            dist[m] = np.sqrt(np.sum(d * d, axis=1)).astype(np.float32)

    sizes = np.array([int(np.sum(labels == kk)) for kk in range(KVAL)], dtype=int)
    order = np.argsort(sizes)[::-1]   # biggest first for consistent plotting

    # reindex labels by descending cluster size
    label_map = {int(old): int(new) for new, old in enumerate(order)}
    labels_re = np.array([label_map[int(x)] for x in labels], dtype=np.int32)
    sizes_re = np.array([sizes[old] for old in order], dtype=int)
    centroids_re = np.stack([centroids[old] for old in order], axis=0)

    # scatter
    colors = cm.get_cmap("tab10", max(10, KVAL))
    fig, ax = plt.subplots(figsize=(6.0, 5.5))
    for kk in range(KVAL):
        m = labels_re == kk
        ax.scatter(
            Xpc[m, 0],
            Xpc[m, 1] if Xpc.shape[1] >= 2 else np.zeros(np.sum(m)),
            s=SCATTER_SIZE,
            alpha=SCATTER_ALPHA,
            color=colors(kk),
            label=f"cl{kk} (n={sizes_re[kk]})"
        )
        ctr = centroids_re[kk]
        ax.scatter([ctr[0]], [ctr[1] if Xpc.shape[1] >= 2 else 0.0],
                   marker="X", s=160, color=colors(kk), edgecolor="k", linewidths=1.0)
    ax.set_xlabel(f"PC1 ({vr[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({vr[1]*100:.1f}% var)" if Xpc.shape[1] >= 2 else "PC2")
    ax.set_title(f"K={KVAL} proposal scatter")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()

    result_k = {
        "labels": labels_re.copy(),
        "sizes": sizes_re.copy(),
        "centroids": centroids_re.copy(),
        "clusters": [],
    }

    # build central-core templates for each cluster
    chosen_eis = []
    chosen_labels = []

    for kk in range(KVAL):
        m = labels_re == kk
        times_cluster = np.asarray(valid_times[m], dtype=np.int64)
        dist_cluster = np.asarray(dist[m], dtype=np.float32)

        print(f"\n  cluster {kk}: n={times_cluster.size}")

        cluster_info = {
            "cluster_id": int(kk),
            "size": int(times_cluster.size),
            "times_cluster": times_cluster.copy(),
            "core_by_frac": {},
        }

        frac_eis = {}
        frac_counts = {}
        frac_times = {}

        for frac in CORE_FRACS:
            t_core, n_keep = _core_times_by_fraction(times_cluster, dist_cluster, frac)
            ei_core, times_used = _extract_full_ei(
                raw_mod,
                t_core,
                win_use,
                reducer=EI_REDUCER,
                max_spikes=EI_CAP
            )

            frac_key = f"{frac:.2f}"
            frac_eis[frac_key] = ei_core
            frac_counts[frac_key] = int(t_core.size)
            frac_times[frac_key] = t_core.copy()

            cluster_info["core_by_frac"][frac_key] = {
                "n_core": int(t_core.size),
                "times_core": t_core.copy(),
                "ei": None if ei_core is None else ei_core.copy(),
            }

            print(f"    frac {frac:.2f}: core_n={t_core.size}")

        # stability across fractions
        stab_pairs = []
        frac_keys = [f"{x:.2f}" for x in CORE_FRACS]
        for i in range(len(frac_keys)):
            for j in range(i + 1, len(frac_keys)):
                fi = frac_keys[i]
                fj = frac_keys[j]
                ei_i = frac_eis[fi]
                ei_j = frac_eis[fj]
                if (ei_i is None) or (ei_j is None):
                    c = np.nan
                else:
                    c, nch, lag, olen = cosine_two_eis(
                        ei_i, ei_j,
                        rms_gate=EI_RMS_GATE,
                        use_abs=True,
                        best_align_lag=EI_ALIGN_LAG
                    )
                stab_pairs.append((fi, fj, float(c) if np.isfinite(c) else np.nan))

        if len(stab_pairs):
            stab_vals = [x[2] for x in stab_pairs if np.isfinite(x[2])]
            min_stab = float(np.min(stab_vals)) if len(stab_vals) else np.nan
            med_stab = float(np.median(stab_vals)) if len(stab_vals) else np.nan
        else:
            min_stab = np.nan
            med_stab = np.nan

        cluster_info["stability_pairs"] = stab_pairs
        cluster_info["stability_min"] = min_stab
        cluster_info["stability_median"] = med_stab

        print(f"    stability min={min_stab:.3f} | median={med_stab:.3f}")
        for fi, fj, cc in stab_pairs:
            print(f"      {fi} vs {fj}: {cc:.3f}")

        # plot main-channel waveform across fractions
        fig, ax = plt.subplots(figsize=(6.2, 3.8))
        t_ms = (np.arange(L) + win_use[0]) / sample_rate_hz_use * 1000.0
        for ii, frac in enumerate(CORE_FRACS):
            frac_key = f"{frac:.2f}"
            ei = frac_eis[frac_key]
            if ei is None:
                continue
            ax.plot(
                t_ms,
                ei[main_ch, :],
                lw=MAIN_WF_LW,
                color=colors(kk),
                alpha=0.35 + 0.25 * ii,
                label=f"frac {frac_key} (n={frac_counts[frac_key]})"
            )
        ax.axvline(0.0, color="k", lw=0.7, alpha=0.3)
        ax.axhline(0.0, color="k", lw=0.7, alpha=0.2)
        ax.set_title(f"K={KVAL} cl{kk} | main-ch waveform stability")
        ax.set_xlabel("time (ms)")
        ax.set_ylabel("ADC")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False, fontsize=8)
        plt.tight_layout()
        plt.show()

        # chosen EI for downstream comparison = chosen core fraction
        chosen_key = f"{PLOT_CORE_FRAC:.2f}"
        chosen_ei = frac_eis.get(chosen_key, None)
        chosen_eis.append(chosen_ei)
        chosen_labels.append(f"cl{kk}")

        result_k["clusters"].append(cluster_info)

    # overlay chosen core EIs for this K
    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111)
    eis_to_plot = [x for x in chosen_eis if x is not None]
    cols_to_plot = [colors(i) for i in range(len(chosen_eis)) if chosen_eis[i] is not None]
    labs_to_plot = [chosen_labels[i] for i in range(len(chosen_eis)) if chosen_eis[i] is not None]

    if len(eis_to_plot) > 0:
        pew.plot_ei_waveforms(
            eis_to_plot,
            positions=ei_positions,
            ref_channel=main_ch,
            scale=70.0,
            box_height=1.0,
            box_width=50.0,
            ax=ax,
            colors=cols_to_plot,
            alpha=[1.0] * len(eis_to_plot),
            linewidth=[0.8] * len(eis_to_plot),
        )
        ax.set_title(f"K={KVAL} | chosen core EIs (frac={PLOT_CORE_FRAC:.2f})")
        plt.tight_layout()
        plt.show()
    else:
        plt.close(fig)
        print(f"No chosen EIs to plot for K={KVAL}")

    # pairwise EI cosine matrix for chosen core fraction
    cosM = _pairwise_cosine_matrix(chosen_eis, rms_gate=EI_RMS_GATE, best_align_lag=EI_ALIGN_LAG)
    result_k["chosen_core_cosine_matrix"] = cosM.copy()
    _plot_cosine_matrix(cosM, chosen_labels, title=f"K={KVAL} | pairwise EI cosine (frac={PLOT_CORE_FRAC:.2f})")

    # print matrix nicely
    print(f"\n  Pairwise chosen-core EI cosine matrix (frac={PLOT_CORE_FRAC:.2f}):")
    hdr = " " * 10 + "".join([f"{lab:>10s}" for lab in chosen_labels])
    print(hdr)
    for i, lab in enumerate(chosen_labels):
        row = f"{lab:>10s}"
        for j in range(len(chosen_labels)):
            val = cosM[i, j]
            row += f"{val:10.3f}" if np.isfinite(val) else f"{'nan':>10s}"
        print(row)

    # ------------------------------------------------------------
    # AB-shard screening for small extra clusters
    # ------------------------------------------------------------
    largest_n = int(np.max(sizes_re))
    small_ids = [kk for kk in range(KVAL) if sizes_re[kk] <= SMALL_CLUSTER_FRAC * largest_n]
    large_ids = [kk for kk in range(KVAL) if sizes_re[kk] >  SMALL_CLUSTER_FRAC * largest_n]

    result_k["ab_checks"] = []
    result_k["ab_ignore_clusters"] = []

    print(f"\n  AB-shard screening (small <= {SMALL_CLUSTER_FRAC:.2f} * largest = {SMALL_CLUSTER_FRAC * largest_n:.1f} spikes)")
    print(f"    large clusters: {large_ids}")
    print(f"    small clusters: {small_ids}")

    # chosen EIs for this K already live in `chosen_eis`
    # sizes already live in `sizes_re`
    for ks in small_ids:
        ei_small = chosen_eis[ks]
        if ei_small is None:
            print(f"    cl{ks}: missing EI -> skip")
            continue

        small_flag_ignore = False

        for kl in large_ids:
            if kl == ks:
                continue

            ei_large = chosen_eis[kl]
            if ei_large is None:
                continue

            ab = _ab_pair_verdict(
                ei_large, ei_small,   # large as putative core, small as possible additive shard
                max_lag=AB_MAX_LAG,
                support_rel=AB_SUPPORT_REL,
                support_abs=AB_SUPPORT_ABS,
                time_keep_rel=AB_TIME_KEEP_REL,
                frac_in_thr=AB_FRAC_IN_THR,
                out_in_ratio_thr=AB_OUT_IN_RATIO_THR,
                resid_frac_min=AB_RESID_FRAC_MIN,
                shared_cos_thr=AB_SHARED_COS_THR,
                shared_alpha_thr=AB_SHARED_ALPHA_THR,
            )

            rec = {
                "small_cluster": int(ks),
                "large_cluster": int(kl),
                "small_n": int(sizes_re[ks]),
                "large_n": int(sizes_re[kl]),
                "verdict": ab["verdict"],
                "shared_core": bool(ab["shared_core"]),
                "shared_dirs": ab["shared_dirs"],
                "contained_dirs": {
                    "large_in_small": bool(ab["contained_dirs"]["a_in_b"]),
                    "small_in_large": bool(ab["contained_dirs"]["b_in_a"]),
                },
                "m_large_in_small": ab["m_ab"],   # "large contained in small?"
                "m_small_in_large": ab["m_ba"],   # "small contained in large?"
            }
            result_k["ab_checks"].append(rec)

            print(f"    cl{ks} (n={sizes_re[ks]}) vs cl{kl} (n={sizes_re[kl]}): {ab['verdict']}")
            print(
                f"      large->small: cos={ab['m_ab']['cos_on_support']:.3f} "
                f"alpha={ab['m_ab']['alpha']:.3f} frac_in={ab['m_ab']['frac_in']:.3f} "
                f"out/in={ab['m_ab']['out_in']:.2f}"
            )
            print(
                f"      small->large: cos={ab['m_ba']['cos_on_support']:.3f} "
                f"alpha={ab['m_ba']['alpha']:.3f} frac_in={ab['m_ba']['frac_in']:.3f} "
                f"out/in={ab['m_ba']['out_in']:.2f}"
            )

            if ab["verdict"] == "AB_SHARD":
                small_flag_ignore = True

            if DO_AB_PLOTS:
                # choose direction that looked AB-like if asymmetric containment exists
                c_ls = rec["contained_dirs"]["large_in_small"]
                c_sl = rec["contained_dirs"]["small_in_large"]

                if c_ls and not c_sl:
                    X = ei_large
                    Y = ab["m_ab"]["Y_aligned"]
                    R = ab["m_ab"]["resid"]
                    ttl = f"K={KVAL} | cl{ks} small vs cl{kl} large | AB check (large in small)"
                elif c_sl and not c_ls:
                    X = ei_small
                    Y = ab["m_ba"]["Y_aligned"]
                    R = ab["m_ba"]["resid"]
                    ttl = f"K={KVAL} | cl{ks} small vs cl{kl} large | reverse containment"
                else:
                    X = ei_large
                    Y = ab["m_ab"]["Y_aligned"]
                    R = ab["m_ab"]["resid"]
                    ttl = f"K={KVAL} | cl{ks} small vs cl{kl} large | AB check"

                fig = plt.figure(figsize=(20, 10))
                ax = fig.add_subplot(111)
                pew.plot_ei_waveforms(
                    [X, Y, R],
                    positions=ei_positions,
                    ref_channel=main_ch,
                    scale=70.0,
                    box_height=1.0,
                    box_width=50.0,
                    ax=ax,
                    colors=["tab:green", "tab:red", "tab:blue"],
                    alpha=[1.0, 1.0, 1.0],
                    linewidth=[0.9, 0.9, 0.9],
                )
                ax.set_title(ttl)
                plt.tight_layout()
                plt.show()

        if small_flag_ignore:
            result_k["ab_ignore_clusters"].append(int(ks))

    result_k["ab_ignore_clusters"] = sorted(list(set(result_k["ab_ignore_clusters"])))

    if len(result_k["ab_ignore_clusters"]) > 0:
        print(f"  Ignore as AB-shard later: {result_k['ab_ignore_clusters']}")
    else:
        print("  No small clusters flagged as AB-shards.")

    k_sweep_out["results_by_k"][int(KVAL)] = result_k

print("\nDone. Result dict saved as `k_sweep_out`.")

In [ ]:
# ============================================================
# Proposal assessment by harm maps
# What it does:
#   - does NOT choose one K
#   - for every large proposal EI from every tested K,
#     computes a locked-channel harm map on one common spike pool
#   - plots one harm map per proposal
#   - stores outputs in proposal_harm_out for later inspection
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER KNOBS
# -------------------------
MIN_LARGE_N = 100
CHOSEN_K = 3

# scoring source for proposal assessment
#   "left_only"         -> score only LEFT spikes
#   "left_plus_right5k" -> score LEFT plus returned 5k strongest RIGHT spikes
SCORING_SOURCE = "left_plus_right5k"

# channel selection for each proposal EI before harm-map computation
EI_SIG_P2P_THR = 25.0
MIN_CH_PER_TEMPLATE = 10
MAX_CH_PER_TEMPLATE = 40

# harm-map params
HARM_LAG_RADIUS = 3
HARM_WEIGHT_BETA = 0.7
PLOT_FIELD = "harm_matrix"     # "harm_matrix" or "harm_matrix_r2"
SORT_SPIKES_BY = "mean_delta_weighted"   # or "mean_r2_weighted", or None

# for memory safety; set to None to use all available spikes
# total number of spikes to assess with harm maps
# if SCORING_SOURCE == "left_plus_right5k", this will be split roughly 50/50
# between LEFT and RIGHT pools, with spillover filled from the richer side
ASSESS_N_SPIKES_TOTAL = 10000

# for odd totals in mixed mode:
#   "left"  -> extra spike goes to LEFT
#   "right" -> extra spike goes to RIGHT
ODD_EXTRA_SIDE = "right"

# EI overlay plotting
DO_OVERLAY_PER_K = True

# -------------------------
# checks
# -------------------------
assert "k_sweep_out" in globals(), "Need k_sweep_out from the K-sweep cell"
assert "out" in globals() and out is not None and "step1" in out, "Need out['step1']"
assert "raw_mod" in globals(), "Need raw_mod"
assert "ei_positions" in globals(), "Need ei_positions"
assert "pew" in globals(), "Need plot_ei_waveforms as pew"
assert hasattr(collision_utils, "compute_harm_map_noamp"), "Need collision_utils.compute_harm_map_noamp"
assert hasattr(collision_utils, "plot_harm_heatmap"), "Need collision_utils.plot_harm_heatmap"
assert hasattr(collision_utils, "select_template_channels"), "Need collision_utils.select_template_channels"

rng = np.random.RandomState(0)
win_use = tuple(k_sweep_out["win"])
main_ch = int(k_sweep_out["main_ch"])
plot_core_frac = float(k_sweep_out["plot_core_frac"])
plot_core_key = f"{plot_core_frac:.2f}"

# -------------------------
# helpers
# -------------------------
def _safe_unique_sorted(x):
    x = np.asarray(x, dtype=np.int64).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.array([], dtype=np.int64)
    return np.sort(np.unique(x.astype(np.int64)))

def _subsample_times(times_abs, max_n, rng):
    times_abs = _safe_unique_sorted(times_abs)
    if (max_n is None) or (times_abs.size <= int(max_n)):
        return times_abs
    idx = np.sort(rng.choice(times_abs.size, size=int(max_n), replace=False))
    return times_abs[idx]

def harm_map_locked(E, SN, *, lag_radius=0, weight_beta=0.7):
    """
    E  : [M, L]     proposal EI on preselected channels only
    SN : [M, L, N]  snippets on those same channels
    """
    E = np.asarray(E, dtype=np.float32)
    SN = np.asarray(SN, dtype=np.float32)
    assert E.ndim == 2, "E must be [M, L]"
    assert SN.ndim == 3, "SN must be [M, L, N]"
    assert SN.shape[:2] == E.shape, "SN and E shape mismatch on [M, L]"

    M = int(E.shape[0])

    res = collision_utils.compute_harm_map_noamp(
        E,
        SN,
        p2p_thr=-1e9,             # force keeping everything we passed in
        max_channels=M,
        min_channels=M,
        lag_radius=int(lag_radius),
        weight_by_p2p=True,
        weight_beta=float(weight_beta),
        force_include_main=False
    )

    H = res["harm_matrix"]
    if H.shape[0] != M:
        print(f"[warn] harm_map_locked: expected {M} channels, got {H.shape[0]}")

    return res

# -------------------------
# common scoring pool
# -------------------------
left_times_all = np.asarray(
    out.get("all_left_times", out["step1"].get("left_times", [])),
    dtype=np.int64
)
right5k_times_all = np.asarray(
    out.get("right5000_times", out["step1"].get("rightk_times_by_amp", [])),
    dtype=np.int64
)

left_times_all = _safe_unique_sorted(left_times_all)
right5k_times_all = _safe_unique_sorted(right5k_times_all)

def _choose_balanced_left_right(left_times, right_times, n_total, rng, odd_extra_side="right"):
    """
    Choose up to n_total spikes, split as evenly as possible between left and right.
    If one side cannot meet its target, fill from the richer side.
    """
    left_times = _safe_unique_sorted(left_times)
    right_times = _safe_unique_sorted(right_times)

    n_total = int(n_total)
    if n_total <= 0:
        return (
            np.array([], dtype=np.int64),
            np.array([], dtype=np.int64),
            np.array([], dtype=np.int64)
        )

    if odd_extra_side not in ("left", "right"):
        raise ValueError("ODD_EXTRA_SIDE must be 'left' or 'right'")

    half = n_total // 2
    if n_total % 2 == 0:
        left_target = half
        right_target = half
    else:
        if odd_extra_side == "left":
            left_target = half + 1
            right_target = half
        else:
            left_target = half
            right_target = half + 1

    # initial allocation
    n_left = min(left_times.size, left_target)
    n_right = min(right_times.size, right_target)

    # fill remainder from whichever side still has capacity
    remaining = n_total - (n_left + n_right)
    left_extra_cap = left_times.size - n_left
    right_extra_cap = right_times.size - n_right

    while remaining > 0 and (left_extra_cap > 0 or right_extra_cap > 0):
        if left_extra_cap >= right_extra_cap and left_extra_cap > 0:
            take = min(remaining, left_extra_cap)
            n_left += take
        elif right_extra_cap > 0:
            take = min(remaining, right_extra_cap)
            n_right += take
        else:
            break

        remaining = n_total - (n_left + n_right)
        left_extra_cap = left_times.size - n_left
        right_extra_cap = right_times.size - n_right

    # sample each side independently, then merge/sort
    if n_left > 0:
        idx_left = np.sort(rng.choice(left_times.size, size=n_left, replace=False))
        left_sel = left_times[idx_left]
    else:
        left_sel = np.array([], dtype=np.int64)

    if n_right > 0:
        idx_right = np.sort(rng.choice(right_times.size, size=n_right, replace=False))
        right_sel = right_times[idx_right]
    else:
        right_sel = np.array([], dtype=np.int64)

    both = _safe_unique_sorted(np.r_[left_sel, right_sel])
    return both, left_sel, right_sel

if SCORING_SOURCE == "left_only":
    score_times_left = _subsample_times(left_times_all, ASSESS_N_SPIKES_TOTAL, rng)
    score_times_right = np.array([], dtype=np.int64)
    score_times_use = score_times_left.copy()
    score_label = "LEFT only"

elif SCORING_SOURCE == "left_plus_right5k":
    score_times_use, score_times_left, score_times_right = _choose_balanced_left_right(
        left_times_all,
        right5k_times_all,
        ASSESS_N_SPIKES_TOTAL,
        rng,
        odd_extra_side=ODD_EXTRA_SIDE
    )
    score_label = "LEFT + RIGHT5k"

else:
    raise ValueError("SCORING_SOURCE must be 'left_only' or 'left_plus_right5k'")

print(f"Assessment source: {score_label}")
print(f"Available LEFT spikes: {left_times_all.size}")
print(f"Available RIGHT spikes: {right5k_times_all.size}")
print(f"Requested total spikes: {ASSESS_N_SPIKES_TOTAL}")
print(f"Selected LEFT spikes: {score_times_left.size}")
print(f"Selected RIGHT spikes: {score_times_right.size}")
print(f"Total spikes used for harm maps: {score_times_use.size}")

# -------------------------
# collect proposal metadata first
# -------------------------
proposal_entries = []

for KVAL in sorted(list(k_sweep_out["results_by_k"].keys())):
    res = k_sweep_out["results_by_k"][KVAL]
    sizes = np.asarray(res["sizes"], dtype=int)

    kept_ids = [kk for kk, s in enumerate(sizes) if s >= MIN_LARGE_N]
    if len(kept_ids) == 0:
        print(f"K={KVAL}: no clusters >= {MIN_LARGE_N}; skip")
        continue

    print(f"K={KVAL}: assessing large clusters {kept_ids}")

    eis_this_k = []
    names_this_k = []

    for kk in kept_ids:
        cl = res["clusters"][kk]
        core = cl["core_by_frac"][plot_core_key]
        ei = core["ei"]
        if ei is None:
            print(f"  cl{kk}: missing EI at frac={plot_core_key}; skip")
            continue

        ei = np.asarray(ei, dtype=np.float32)
        ch_sel, ptp_all = collision_utils.select_template_channels(
            ei,
            p2p_thr=float(EI_SIG_P2P_THR),
            max_n=int(MAX_CH_PER_TEMPLATE),
            min_n=int(MIN_CH_PER_TEMPLATE),
            force_include_main=True
        )

        entry = {
            "K": int(KVAL),
            "cluster_id": int(kk),
            "size": int(sizes[kk]),
            "n_core": int(core["n_core"]),
            "times_core": np.asarray(core["times_core"], dtype=np.int64).copy(),
            "ei": ei.copy(),
            "channels_global": np.asarray(ch_sel, dtype=np.int32).copy(),
            "channel_ptp_global": np.asarray(ptp_all[ch_sel], dtype=np.float32).copy(),
        }
        proposal_entries.append(entry)

        eis_this_k.append(ei)
        names_this_k.append(f"cl{kk} (n={sizes[kk]})")

    if DO_OVERLAY_PER_K and len(eis_this_k) > 0:
        cmap = plt.cm.get_cmap("tab10", max(10, len(eis_this_k)))
        overlay_colors = [cmap(i) for i in range(len(eis_this_k))]

        fig = plt.figure(figsize=(20, 12))
        ax = fig.add_subplot(111)
        pew.plot_ei_waveforms(
            eis_this_k,
            positions=ei_positions,
            ref_channel=main_ch,
            scale=70.0,
            box_height=1.0,
            box_width=50.0,
            ax=ax,
            colors=overlay_colors,
            alpha=[1.0] * len(eis_this_k),
            linewidth=[0.9] * len(eis_this_k),
        )
        ax.set_title(
            f"K={KVAL} large proposal EIs at core frac {plot_core_key} | "
            f"assessment pool: {score_label} | spikes used: {score_times_use.size}"
        )
        plt.tight_layout()
        plt.show()

if len(proposal_entries) == 0:
    raise RuntimeError(f"No proposal EIs found with cluster size >= {MIN_LARGE_N}.")

# -------------------------
# extract snippets ONCE on union of proposal channels
# -------------------------
union_ch = np.array([], dtype=np.int32)
for entry in proposal_entries:
    union_ch = np.union1d(union_ch, entry["channels_global"]).astype(np.int32)

sn_all, valid_times = extract_snippets_fast_ram(
    raw_mod,
    score_times_use,
    window=win_use,
    selected_channels=union_ch
)  # [n_union, L, Nvalid]

valid_times = np.asarray(valid_times, dtype=np.int64)
print(f"\nUnion channels across all assessed proposals: {union_ch.size}")
print(f"Valid snippets returned: {valid_times.size}")

# map global channel -> local row in sn_all
union_map = {int(ch): int(i) for i, ch in enumerate(union_ch)}

# -------------------------
# compute and plot harm map per proposal
# -------------------------
proposal_harm_out = {}

for entry in proposal_entries:
    KVAL = int(entry["K"])
    kk = int(entry["cluster_id"])
    ei = entry["ei"]
    ch_global = np.asarray(entry["channels_global"], dtype=np.int32)

    local_rows = np.array([union_map[int(ch)] for ch in ch_global], dtype=np.int32)
    ei_sel = ei[ch_global, :].astype(np.float32)
    sn_sel = sn_all[local_rows, :, :].astype(np.float32)

    res_h = harm_map_locked(
        ei_sel,
        sn_sel,
        lag_radius=HARM_LAG_RADIUS,
        weight_beta=HARM_WEIGHT_BETA
    )

    # map local selected channels back to global channel ids
    res_h["selected_channels_global"] = ch_global[np.asarray(res_h["selected_channels"], dtype=int)]
    res_h["score_times_valid"] = valid_times.copy()
    res_h["proposal_meta"] = {
        "K": KVAL,
        "cluster_id": kk,
        "size": int(entry["size"]),
        "n_core": int(entry["n_core"]),
        "plot_core_frac": float(plot_core_frac),
        "score_source": score_label,
    }

    if SORT_SPIKES_BY is None:
        spike_order = None
    else:
        spike_order = np.argsort(np.asarray(res_h[SORT_SPIKES_BY], dtype=np.float32))

    proposal_harm_out[(KVAL, kk)] = {
        "meta": entry,
        "harm": res_h,
        "spike_order": None if spike_order is None else spike_order.copy(),
    }

    mean_delta = np.asarray(res_h["mean_delta_weighted"], dtype=np.float32)
    neg_frac = float(np.mean(mean_delta < 0.0))
    q05, q50, q95 = np.percentile(mean_delta, [5, 50, 95])

    print(
        f"K={KVAL} cl{kk}: "
        f"size={entry['size']} | core_n={entry['n_core']} | "
        f"nch={ch_global.size} | neg_mean_delta_frac={neg_frac:.3f} | "
        f"q05/q50/q95={q05:.2f}/{q50:.2f}/{q95:.2f}"
    )

    title = (
        f"K={KVAL} cl{kk} | size={entry['size']} | core_n={entry['n_core']} | "
        f"nch={ch_global.size} | pool={score_label}"
    )

    collision_utils.plot_harm_heatmap(
        res_h,
        field=PLOT_FIELD,
        sort_by_ptp=True,
        spike_order=spike_order,
        title=title
    )

print("\nSaved outputs in proposal_harm_out[(K, cluster_id)]")

In [ ]:
# ============================================================
# PCA on concatenated harm-map features across templates
#
# Add as a NEW cell after proposal_harm_out
#
# Each spike gets a feature vector:
#   [harm(template1, all its channels),
#    harm(template2, all its channels),
#    ...]
#
# Then PCA on spikes x features, and plot PC1 vs PC2
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# -------------------------
# USER SETTINGS
# -------------------------
K_FOR_PCA = CHOSEN_K
GROUP_CLUSTER_IDS = [0, 1, 2]   # the templates/proposals to concatenate

# feature preprocessing
ZSCORE_FEATURES = False
DROP_ZERO_VAR_FEATURES = True

# plotting
POINT_SIZE = 10
POINT_ALPHA = 0.45
USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

# optional coloring by soft winner from summed harm
COLOR_BY_SOFT_WINNER = True

# -------------------------
# checks
# -------------------------
assert "proposal_harm_out" in globals(), "Need proposal_harm_out from the harm-map cell"
assert len(GROUP_CLUSTER_IDS) >= 2, "Need at least 2 groups/templates for PCA"

# -------------------------
# helper
# -------------------------
def _lims_xy(x, y, use_pct=True, lo=0.5, hi=99.5, pad_frac=0.03):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]

    if x.size == 0:
        return (-1.0, 1.0, -1.0, 1.0)

    if use_pct:
        xmin, xmax = np.percentile(x, [lo, hi])
        ymin, ymax = np.percentile(y, [lo, hi])
    else:
        xmin, xmax = np.min(x), np.max(x)
        ymin, ymax = np.min(y), np.max(y)

    if xmax <= xmin:
        xmax = xmin + 1.0
    if ymax <= ymin:
        ymax = ymin + 1.0

    xspan = xmax - xmin
    yspan = ymax - ymin
    xmin -= pad_frac * xspan
    xmax += pad_frac * xspan
    ymin -= pad_frac * yspan
    ymax += pad_frac * yspan
    return xmin, xmax, ymin, ymax

# -------------------------
# gather harm matrices
# -------------------------
harm_blocks = []
score_times_list = []
nch_list = []
feature_slices = {}

col0 = 0
for kk in GROUP_CLUSTER_IDS:
    key = (K_FOR_PCA, kk)
    if key not in proposal_harm_out:
        raise RuntimeError(f"proposal_harm_out missing key {key}")

    harm_res = proposal_harm_out[key]["harm"]

    H = np.asarray(harm_res["harm_matrix"], dtype=np.float32)   # [nch, N]
    t = np.asarray(harm_res["score_times_valid"], dtype=np.int64)

    if H.ndim != 2:
        raise RuntimeError(f"K={K_FOR_PCA} cl{kk}: expected harm_matrix [nch, N], got shape {H.shape}")

    harm_blocks.append(H)
    score_times_list.append(t)
    nch_list.append(int(H.shape[0]))

    col1 = col0 + int(H.shape[0])
    feature_slices[int(kk)] = slice(col0, col1)
    col0 = col1

# -------------------------
# make sure all templates used the same spike pool
# -------------------------
base_times = np.asarray(score_times_list[0], dtype=np.int64)
for j in range(1, len(score_times_list)):
    tt = np.asarray(score_times_list[j], dtype=np.int64)
    if tt.size != base_times.size or np.any(tt != base_times):
        raise RuntimeError("Selected templates do not share the same score_times_valid pool")

score_times_valid = base_times.copy()
N = score_times_valid.size

print(f"Spikes: {N}")
for kk, nch in zip(GROUP_CLUSTER_IDS, nch_list):
    print(f"  cl{kk}: {nch} harm channels")
print(f"Total concatenated features: {sum(nch_list)}")

# -------------------------
# build feature matrix X = [N, total_features]
# -------------------------
X = np.concatenate([H.T for H in harm_blocks], axis=1).astype(np.float32)

if not np.all(np.isfinite(X)):
    raise RuntimeError("Feature matrix X contains non-finite values")

print(f"X shape before filtering: {X.shape}")

# -------------------------
# optional soft-winner labels from summed harm
# -------------------------
score_mat_sum = np.column_stack([
    H.sum(axis=0).astype(np.float32) for H in harm_blocks
])   # [N, n_groups], more negative = better

soft_winner = np.argmin(score_mat_sum, axis=1).astype(np.int32)

# -------------------------
# preprocess features
# -------------------------
feature_keep = np.ones(X.shape[1], dtype=bool)

if DROP_ZERO_VAR_FEATURES:
    feat_std = np.std(X, axis=0)
    feature_keep = feat_std > 0
    if not np.any(feature_keep):
        raise RuntimeError("All PCA features have zero variance")
    X = X[:, feature_keep]
    print(f"Kept {int(feature_keep.sum())}/{feature_keep.size} nonzero-variance features")

if ZSCORE_FEATURES:
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True)
    sd = np.maximum(sd, 1e-8)
    Xp = (X - mu) / sd
else:
    Xp = X.copy()

# -------------------------
# PCA
# -------------------------
pca = PCA(n_components=min(10, Xp.shape[1]), svd_solver="auto", random_state=0)
PC = pca.fit_transform(Xp).astype(np.float32)

expl = pca.explained_variance_ratio_
print("\nExplained variance ratio:")
for i in range(min(6, expl.size)):
    print(f"  PC{i+1}: {100.0 * expl[i]:.2f}%")

# -------------------------
# plot 1: PC1 vs PC2, plain
# -------------------------
x = PC[:, 0]
y = PC[:, 1]
xmin, xmax, ymin, ymax = _lims_xy(
    x, y,
    use_pct=USE_PERCENTILE_LIMITS,
    lo=PCTL_LOW,
    hi=PCTL_HIGH
)

plt.figure(figsize=(6.5, 6.5))
plt.scatter(x, y, s=POINT_SIZE, alpha=POINT_ALPHA)
plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel(f"PC1 ({100.0 * expl[0]:.1f}% var)")
plt.ylabel(f"PC2 ({100.0 * expl[1]:.1f}% var)")
plt.title("PCA of concatenated harm-map features")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# -------------------------
# plot 2: PC1 vs PC2, colored by soft winner
# -------------------------
if COLOR_BY_SOFT_WINNER:
    cmap = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:green",
        3: "tab:red",
        4: "tab:purple",
        5: "tab:brown",
    }

    plt.figure(figsize=(6.5, 6.5))
    for gi, kk in enumerate(GROUP_CLUSTER_IDS):
        m = (soft_winner == gi)
        if np.any(m):
            plt.scatter(
                x[m], y[m],
                s=POINT_SIZE,
                alpha=POINT_ALPHA,
                c=cmap.get(gi, "gray"),
                label=f"G{gi+1} / cl{kk}"
            )

    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel(f"PC1 ({100.0 * expl[0]:.1f}% var)")
    plt.ylabel(f"PC2 ({100.0 * expl[1]:.1f}% var)")
    plt.title("PCA of concatenated harm-map features\ncolored by soft winner")
    plt.grid(alpha=0.25)
    plt.legend(loc="best", frameon=False)
    plt.tight_layout()
    plt.show()

# -------------------------
# plot 3: PC2 vs PC3
# sometimes separation hides there
# -------------------------
if PC.shape[1] >= 3:
    x23 = PC[:, 1]
    y23 = PC[:, 2]
    xmin23, xmax23, ymin23, ymax23 = _lims_xy(
        x23, y23,
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW,
        hi=PCTL_HIGH
    )

    plt.figure(figsize=(6.5, 6.5))
    if COLOR_BY_SOFT_WINNER:
        for gi, kk in enumerate(GROUP_CLUSTER_IDS):
            m = (soft_winner == gi)
            if np.any(m):
                plt.scatter(
                    x23[m], y23[m],
                    s=POINT_SIZE,
                    alpha=POINT_ALPHA,
                    c=cmap.get(gi, "gray"),
                    label=f"G{gi+1} / cl{kk}"
                )
        plt.legend(loc="best", frameon=False)
    else:
        plt.scatter(x23, y23, s=POINT_SIZE, alpha=POINT_ALPHA)

    plt.xlim(xmin23, xmax23)
    plt.ylim(ymin23, ymax23)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel(f"PC2 ({100.0 * expl[1]:.1f}% var)")
    plt.ylabel(f"PC3 ({100.0 * expl[2]:.1f}% var)")
    plt.title("PCA of concatenated harm-map features")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

# -------------------------
# save outputs
# -------------------------
proposal_harm_pca_out = {
    "K_for_pca": int(K_FOR_PCA),
    "group_cluster_ids": [int(x) for x in GROUP_CLUSTER_IDS],
    "score_times_valid": score_times_valid.copy(),
    "nch_per_group": np.asarray(nch_list, dtype=np.int32),
    "feature_slices": feature_slices,
    "X_raw": X.copy(),
    "X_pca_input": Xp.copy(),
    "soft_winner": soft_winner.copy(),
    "PC": PC.copy(),
    "explained_variance_ratio": expl.copy(),
    "pca_components": pca.components_.copy(),
    "zscore_features": bool(ZSCORE_FEATURES),
}

print("\nSaved outputs in proposal_harm_pca_out")

In [ ]:
# ============================================================
# K-means on existing PCA space
#
# Add as a NEW cell after proposal_harm_pca_out
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# -------------------------
# USER SETTINGS
# -------------------------
KMEANS_K = 5             # number of clusters to find
N_PCS_FOR_KMEANS = 4     # use PC1..PCN as k-means input
STANDARDIZE_PCS = False  # leave False first; True only if you want equal PC weighting

POINT_SIZE = 10
POINT_ALPHA = 0.45

USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

RANDOM_STATE = 0
N_INIT = 20

# -------------------------
# checks
# -------------------------
assert "proposal_harm_pca_out" in globals(), "Need proposal_harm_pca_out"

PC = np.asarray(proposal_harm_pca_out["PC"], dtype=np.float32)
score_times_valid = np.asarray(proposal_harm_pca_out["score_times_valid"], dtype=np.int64)
expl = np.asarray(proposal_harm_pca_out["explained_variance_ratio"], dtype=np.float32)

if PC.ndim != 2:
    raise RuntimeError(f"Expected PC to be 2D, got shape {PC.shape}")

if N_PCS_FOR_KMEANS < 1 or N_PCS_FOR_KMEANS > PC.shape[1]:
    raise RuntimeError(
        f"N_PCS_FOR_KMEANS must be between 1 and {PC.shape[1]}, got {N_PCS_FOR_KMEANS}"
    )

# -------------------------
# helper
# -------------------------
def _lims_xy(x, y, use_pct=True, lo=0.5, hi=99.5, pad_frac=0.03):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]

    if x.size == 0:
        return (-1.0, 1.0, -1.0, 1.0)

    if use_pct:
        xmin, xmax = np.percentile(x, [lo, hi])
        ymin, ymax = np.percentile(y, [lo, hi])
    else:
        xmin, xmax = np.min(x), np.max(x)
        ymin, ymax = np.min(y), np.max(y)

    if xmax <= xmin:
        xmax = xmin + 1.0
    if ymax <= ymin:
        ymax = ymin + 1.0

    xspan = xmax - xmin
    yspan = ymax - ymin
    xmin -= pad_frac * xspan
    xmax += pad_frac * xspan
    ymin -= pad_frac * yspan
    ymax += pad_frac * yspan
    return xmin, xmax, ymin, ymax

# -------------------------
# k-means input
# -------------------------
Xk = PC[:, :N_PCS_FOR_KMEANS].copy()

if STANDARDIZE_PCS:
    mu = np.mean(Xk, axis=0, keepdims=True)
    sd = np.std(Xk, axis=0, keepdims=True)
    sd = np.maximum(sd, 1e-8)
    Xk = (Xk - mu) / sd

# -------------------------
# run k-means
# -------------------------
km = KMeans(
    n_clusters=int(KMEANS_K),
    random_state=int(RANDOM_STATE),
    n_init=int(N_INIT)
)
cluster_id = km.fit_predict(Xk).astype(np.int32)

print(f"KMEANS_K = {KMEANS_K}")
print(f"N_PCS_FOR_KMEANS = {N_PCS_FOR_KMEANS}")
print(f"STANDARDIZE_PCS = {STANDARDIZE_PCS}\n")

for kk in range(KMEANS_K):
    n = int(np.sum(cluster_id == kk))
    print(f"cluster {kk}: {n} spikes")

# -------------------------
# plot: PC1 vs PC2 colored by k-means cluster
# -------------------------
x = PC[:, 0]
y = PC[:, 1]

xmin, xmax, ymin, ymax = _lims_xy(
    x, y,
    use_pct=USE_PERCENTILE_LIMITS,
    lo=PCTL_LOW,
    hi=PCTL_HIGH
)

cmap = plt.cm.get_cmap("tab10", max(10, KMEANS_K))

plt.figure(figsize=(6.5, 6.5))
for kk in range(KMEANS_K):
    m = (cluster_id == kk)
    if np.any(m):
        plt.scatter(
            x[m], y[m],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=cmap(kk),
            label=f"C{kk} (n={int(m.sum())})"
        )

plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel(f"PC1 ({100.0 * expl[0]:.1f}% var)")
plt.ylabel(f"PC2 ({100.0 * expl[1]:.1f}% var)")
plt.title(f"K-means on PCA space (K={KMEANS_K}, PCs 1-{N_PCS_FOR_KMEANS})")
plt.grid(alpha=0.25)
plt.legend(loc="best", frameon=False)
plt.tight_layout()
plt.show()

# -------------------------
# optional: PC2 vs PC3
# -------------------------
if PC.shape[1] >= 3:
    x23 = PC[:, 1]
    y23 = PC[:, 2]

    xmin23, xmax23, ymin23, ymax23 = _lims_xy(
        x23, y23,
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW,
        hi=PCTL_HIGH
    )

    plt.figure(figsize=(6.5, 6.5))
    for kk in range(KMEANS_K):
        m = (cluster_id == kk)
        if np.any(m):
            plt.scatter(
                x23[m], y23[m],
                s=POINT_SIZE,
                alpha=POINT_ALPHA,
                color=cmap(kk),
                label=f"C{kk}" if kk == 0 else None
            )

    plt.xlim(xmin23, xmax23)
    plt.ylim(ymin23, ymax23)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel(f"PC2 ({100.0 * expl[1]:.1f}% var)")
    plt.ylabel(f"PC3 ({100.0 * expl[2]:.1f}% var)")
    plt.title(f"K-means on PCA space (K={KMEANS_K}, PCs 1-{N_PCS_FOR_KMEANS})")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

# -------------------------
# save outputs
# -------------------------
proposal_harm_pca_kmeans_out = {
    "KMEANS_K": int(KMEANS_K),
    "N_PCS_FOR_KMEANS": int(N_PCS_FOR_KMEANS),
    "STANDARDIZE_PCS": bool(STANDARDIZE_PCS),
    "score_times_valid": score_times_valid.copy(),
    "cluster_id": cluster_id.copy(),
    "pc_used": Xk.copy(),
    "pc_all": PC.copy(),
    "cluster_centers_pc_used": km.cluster_centers_.copy(),
    "inertia": float(km.inertia_),
}

print("\nSaved outputs in proposal_harm_pca_kmeans_out")

In [ ]:
# ============================================================
# EI for each k-means cluster in one 2 x 5 grid
#
# Add as a NEW cell after proposal_harm_pca_kmeans_out
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER SETTINGS
# -------------------------
EI_REDUCER = "median"   # "median" or "mean"
EI_CAP_PER_CLUSTER = 1000   # max spikes used per cluster to build EI
NROWS = 3
NCOLS = 2
# -------------------------
# checks
# -------------------------
assert "proposal_harm_pca_kmeans_out" in globals(), "Need proposal_harm_pca_kmeans_out"
assert "raw_mod" in globals(), "Need raw_mod"
assert "ei_positions" in globals(), "Need ei_positions"
assert "extract_snippets_fast_ram" in globals(), "Need extract_snippets_fast_ram"
assert "pew" in globals(), "Need plot_ei_waveforms as pew"
assert "k_sweep_out" in globals(), "Need k_sweep_out"

cluster_id = np.asarray(proposal_harm_pca_kmeans_out["cluster_id"], dtype=np.int32)
score_times_valid = np.asarray(proposal_harm_pca_kmeans_out["score_times_valid"], dtype=np.int64)



win_use = tuple(k_sweep_out["win"])
rng = np.random.RandomState(0)

# -------------------------
# helpers
# -------------------------
def _safe_unique_sorted(x):
    x = np.asarray(x, dtype=np.int64).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.array([], dtype=np.int64)
    return np.sort(np.unique(x.astype(np.int64)))

def _subsample_times(times_abs, max_n, rng):
    times_abs = _safe_unique_sorted(times_abs)
    if (max_n is None) or (times_abs.size <= int(max_n)):
        return times_abs
    idx = np.sort(rng.choice(times_abs.size, size=int(max_n), replace=False))
    return times_abs[idx]

def _build_ei_from_times(times_abs, raw, window, reducer="median", max_spikes=1000):
    times_abs = _safe_unique_sorted(times_abs)
    if times_abs.size == 0:
        return None, np.array([], dtype=np.int64)

    times_use = _subsample_times(times_abs, max_spikes, rng)

    sn, tv = extract_snippets_fast_ram(
        raw,
        times_use,
        window=window,
        selected_channels=np.arange(raw.shape[1], dtype=np.int32)
    )
    tv = np.asarray(tv, dtype=np.int64)

    if sn.shape[2] == 0:
        return None, tv

    if reducer == "median":
        ei = np.median(sn, axis=2).astype(np.float32)
    elif reducer == "mean":
        ei = np.mean(sn, axis=2).astype(np.float32)
    else:
        raise ValueError("EI_REDUCER must be 'median' or 'mean'")

    return ei, tv

def _peak_channel_from_ei(ei):
    # pick channel with largest peak-to-peak amplitude
    ptp = np.ptp(ei, axis=1)
    return int(np.argmax(ptp))

# -------------------------
# build EIs
# -------------------------
cluster_ei_out = {}

for kk in range(KMEANS_K):
    times_k = score_times_valid[cluster_id == kk]
    ei_k, tv_k = _build_ei_from_times(
        times_k,
        raw_mod,
        window=win_use,
        reducer=EI_REDUCER,
        max_spikes=EI_CAP_PER_CLUSTER
    )

    cluster_ei_out[kk] = {
        "times_all": _safe_unique_sorted(times_k),
        "ei": None if ei_k is None else ei_k.copy(),
        "n_total": int(times_k.size),
        "n_valid_for_ei": int(tv_k.size),
    }

# -------------------------
# plot grid
# -------------------------
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(20, 28))
axes = np.asarray(axes).reshape(-1)

for kk in range(KMEANS_K):
    ax = axes[kk]
    out = cluster_ei_out[kk]
    ei_k = out["ei"]

    if ei_k is None:
        ax.text(0.5, 0.5, f"C{kk}\nno valid EI", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        continue

    ref_ch = _peak_channel_from_ei(ei_k)

    pew.plot_ei_waveforms(
        ei_k,
        positions=ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax,
    )
    ax.set_title(
        f"C{kk} | n={out['n_total']} | EI_n={out['n_valid_for_ei']} | ref_ch={ref_ch}",
        fontsize=11
    )

for ax in axes[KMEANS_K:]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

print("Built EIs for k-means clusters in cluster_ei_out")

In [ ]:
# ============================================================
# Plot per-spike harm-map competition across 3 proposals
#
# For each spike:
#   score_g = sum over channels of harm_matrix for proposal g
#
# More negative is better.
# Then:
#   best   = most negative score among the 3 proposals
#   second = second most negative score among the 3 proposals
#
# This cell plots:
#   1) best vs second
#   2) pairwise proposal-score scatters
#   3) histogram of separation: second - best
#
# Requires:
#   proposal_harm_out from the proposal-harm-map cell
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER SETTINGS
# -------------------------
K_FOR_PLOT = CHOSEN_K
GROUP_CLUSTER_IDS = [0, 1, 2]   # the 3 proposal clusters you want to compare

# optional clipping for prettier scatter if there are huge outliers
USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

# -------------------------
# checks
# -------------------------
assert "proposal_harm_out" in globals(), "Need proposal_harm_out from the proposal-harm cell"
assert len(GROUP_CLUSTER_IDS) == 3, "This plotting cell expects exactly 3 groups"

# -------------------------
# collect proposal harm outputs
# -------------------------
harm_list = []
score_times_list = []
nch_list = []

for kk in GROUP_CLUSTER_IDS:
    key = (K_FOR_PLOT, kk)
    if key not in proposal_harm_out:
        raise RuntimeError(f"proposal_harm_out missing key {key}")

    harm_res = proposal_harm_out[key]["harm"]
    H = np.asarray(harm_res["harm_matrix"], dtype=np.float32)   # [nch, N]
    t = np.asarray(harm_res["score_times_valid"], dtype=np.int64)

    harm_list.append(H)
    score_times_list.append(t)
    nch_list.append(int(H.shape[0]))

# make sure all 3 proposals were scored on the same spike pool
base_times = np.asarray(score_times_list[0], dtype=np.int64)
for j in range(1, 3):
    tt = np.asarray(score_times_list[j], dtype=np.int64)
    if tt.size != base_times.size or np.any(tt != base_times):
        raise RuntimeError("Selected proposals do not share the same score_times_valid pool")

score_times_valid = base_times.copy()
N = score_times_valid.size

print(f"Scored spikes: {N}")
for gi, kk in enumerate(GROUP_CLUSTER_IDS):
    print(f"  G{gi+1} = cl{kk}: nch={nch_list[gi]}")

# -------------------------
# summed harm scores
# -------------------------
# score_mat[:, g] = summed harm for proposal g on each spike
# more negative = better
score_mat = np.column_stack([
    H.sum(axis=0).astype(np.float32) for H in harm_list
])   # [N, 3]

# also keep mean-harm version as a sanity check if channel counts differ
score_mat_mean = np.column_stack([
    H.mean(axis=0).astype(np.float32) for H in harm_list
])   # [N, 3]

# sort per spike: ascending because more negative = better
order = np.argsort(score_mat, axis=1)     # [N, 3]
winner = order[:, 0]                      # 0/1/2
second_idx = order[:, 1]

best_score = score_mat[np.arange(N), winner]
second_score = score_mat[np.arange(N), second_idx]

# positive => winner beats runner-up by this much
separation = second_score - best_score

winner_text = np.array([f"G{i+1}" for i in winner], dtype=object)

print("\nSummed-harm score summary:")
for gi in range(3):
    m = (winner == gi)
    print(
        f"  G{gi+1} wins: {int(m.sum())} spikes | "
        f"median best={np.median(best_score[m]):.2f} | "
        f"median sep={np.median(separation[m]):.2f}"
        if np.any(m) else
        f"  G{gi+1} wins: 0 spikes"
    )

# -------------------------
# axis limits
# -------------------------
def _lims_xy(x, y, use_pct=True, lo=0.5, hi=99.5):
    if not use_pct:
        xmin = np.nanmin(x)
        xmax = np.nanmax(x)
        ymin = np.nanmin(y)
        ymax = np.nanmax(y)
        return xmin, xmax, ymin, ymax

    xmin, xmax = np.nanpercentile(x, [lo, hi])
    ymin, ymax = np.nanpercentile(y, [lo, hi])
    return xmin, xmax, ymin, ymax

# -------------------------
# plot 1: best vs second-best summed harm
# -------------------------
color_map = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:green",
}

plt.figure(figsize=(8, 7))
for gi in range(3):
    m = (winner == gi)
    if np.any(m):
        plt.scatter(
            best_score[m],
            second_score[m],
            s=12,
            alpha=0.45,
            c=color_map[gi],
            label=f"G{gi+1} winner (cl{GROUP_CLUSTER_IDS[gi]})"
        )

# diagonal: second = best
xx = np.linspace(np.nanmin(best_score), np.nanmax(second_score), 200)
plt.plot(xx, xx, "k--", lw=1.2, alpha=0.8)

if USE_PERCENTILE_LIMITS:
    xmin, xmax, ymin, ymax = _lims_xy(
        best_score, second_score,
        use_pct=True, lo=PCTL_LOW, hi=PCTL_HIGH
    )
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)

plt.xlabel("Best summed harm (most negative)")
plt.ylabel("Second-best summed harm")
plt.title("Per-spike harm competition: best vs second-best")
plt.grid(alpha=0.25)
plt.legend(loc="best", frameon=False)
plt.tight_layout()
plt.show()

# -------------------------
# plot 2: separation histogram
# -------------------------
plt.figure(figsize=(8, 4.5))
plt.hist(separation, bins=80, color="lightgray", edgecolor="none")
plt.xlabel("Second-best minus best summed harm")
plt.ylabel("Count")
plt.title("Per-spike separation in summed harm")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# -------------------------
# plot 3: pairwise proposal-score scatters
# -------------------------
pairs = [(0, 1), (0, 2), (1, 2)]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False, sharey=False)

for ax, (a, b) in zip(axes, pairs):
    for gi in range(3):
        m = (winner == gi)
        if np.any(m):
            ax.scatter(
                score_mat[m, a],
                score_mat[m, b],
                s=10,
                alpha=0.4,
                c=color_map[gi],
                label=f"G{gi+1}" if (a, b) == (0, 1) else None
            )

    lims = _lims_xy(
        score_mat[:, a], score_mat[:, b],
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW, hi=PCTL_HIGH
    )
    ax.set_xlim(lims[0], lims[1])
    ax.set_ylim(lims[2], lims[3])

    ax.plot(
        [min(lims[0], lims[2]), max(lims[1], lims[3])],
        [min(lims[0], lims[2]), max(lims[1], lims[3])],
        "k--", lw=1.0, alpha=0.7
    )

    ax.set_xlabel(f"G{a+1} summed harm (cl{GROUP_CLUSTER_IDS[a]})")
    ax.set_ylabel(f"G{b+1} summed harm (cl{GROUP_CLUSTER_IDS[b]})")
    ax.set_title(f"G{a+1} vs G{b+1}")
    ax.grid(alpha=0.25)

axes[0].legend(loc="best", frameon=False)
plt.tight_layout()
plt.show()

# -------------------------
# plot 4: same thing but MEAN harm (sanity check)
# -------------------------
order_mean = np.argsort(score_mat_mean, axis=1)
winner_mean = order_mean[:, 0]
best_mean = score_mat_mean[np.arange(N), winner_mean]
second_mean = score_mat_mean[np.arange(N), order_mean[:, 1]]
separation_mean = second_mean - best_mean

plt.figure(figsize=(8, 7))
for gi in range(3):
    m = (winner_mean == gi)
    if np.any(m):
        plt.scatter(
            best_mean[m],
            second_mean[m],
            s=12,
            alpha=0.45,
            c=color_map[gi],
            label=f"G{gi+1} winner"
        )

xx = np.linspace(np.nanmin(best_mean), np.nanmax(second_mean), 200)
plt.plot(xx, xx, "k--", lw=1.2, alpha=0.8)

if USE_PERCENTILE_LIMITS:
    xmin, xmax, ymin, ymax = _lims_xy(
        best_mean, second_mean,
        use_pct=True, lo=PCTL_LOW, hi=PCTL_HIGH
    )
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)

plt.xlabel("Best mean harm (most negative)")
plt.ylabel("Second-best mean harm")
plt.title("Sanity check: best vs second-best using mean harm")
plt.grid(alpha=0.25)
plt.legend(loc="best", frameon=False)
plt.tight_layout()
plt.show()

# -------------------------
# save outputs
# -------------------------
proposal_harm_competition = {
    "K_for_plot": int(K_FOR_PLOT),
    "group_cluster_ids": [int(x) for x in GROUP_CLUSTER_IDS],
    "score_times_valid": score_times_valid.copy(),
    "score_mat_sum": score_mat.copy(),          # [N, 3], more negative = better
    "score_mat_mean": score_mat_mean.copy(),    # [N, 3], sanity-check version
    "winner_sum": winner.copy(),
    "best_sum": best_score.copy(),
    "second_sum": second_score.copy(),
    "separation_sum": separation.copy(),
    "winner_mean": winner_mean.copy(),
    "best_mean": best_mean.copy(),
    "second_mean": second_mean.copy(),
    "separation_mean": separation_mean.copy(),
    "nch_per_group": np.asarray(nch_list, dtype=np.int32),
}

print("\nSaved outputs in proposal_harm_competition")

In [ ]:
# ============================================================
# Split spikes into 3 main groups by pairwise harm thresholds
#
# Add as a NEW cell after proposal_harm_competition
#
# Logic:
#   d12 = score_G2 - score_G1   # positive => G1 better than G2
#   d13 = score_G3 - score_G1   # positive => G1 better than G3
#   d23 = score_G3 - score_G2   # positive => G2 better than G3
#
# Assign:
#   G1 if d12 > THR12 and d13 > THR13
#   G2 if d12 < -THR12 and d23 > THR23
#   G3 if d13 < -THR13 and d23 < -THR23
#   else unresolved
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER SETTINGS
# -------------------------
K_FOR_ASSIGN = CHOSEN_K
GROUP_CLUSTER_IDS = [0, 1, 2]

# use "sum" unless you specifically want to sanity-check channel-count effects
SCORE_KIND = "sum"   # "sum" or "mean"

# pairwise thresholds
# THR12 separates G1 vs G2
# THR13 separates G1 vs G3
# THR23 separates G2 vs G3
THR12 = 40.0
THR13 = 40.0
THR23 = 40.0

# plotting
USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

# EI summaries
EI_CAP = 1000
EI_REDUCER = "median"   # "median" or "mean"

# -------------------------
# checks
# -------------------------
assert "proposal_harm_competition" in globals(), "Need proposal_harm_competition"
assert "k_sweep_out" in globals(), "Need k_sweep_out"
assert "raw_mod" in globals(), "Need raw_mod"
assert "ei_positions" in globals(), "Need ei_positions"
assert "extract_snippets_fast_ram" in globals(), "Need extract_snippets_fast_ram"
assert "pew" in globals(), "Need plot_ei_waveforms as pew"

if proposal_harm_competition["K_for_plot"] != K_FOR_ASSIGN:
    raise RuntimeError(
        f"proposal_harm_competition was built for K={proposal_harm_competition['K_for_plot']}, "
        f"but K_FOR_ASSIGN={K_FOR_ASSIGN}"
    )

if list(proposal_harm_competition["group_cluster_ids"]) != list(GROUP_CLUSTER_IDS):
    raise RuntimeError(
        f"proposal_harm_competition uses groups {proposal_harm_competition['group_cluster_ids']}, "
        f"but GROUP_CLUSTER_IDS={GROUP_CLUSTER_IDS}"
    )

win_use = tuple(k_sweep_out["win"])
main_ch = int(k_sweep_out["main_ch"])
plot_core_frac = float(k_sweep_out["plot_core_frac"])
plot_core_key = f"{plot_core_frac:.2f}"
rng = np.random.RandomState(0)

# -------------------------
# helpers
# -------------------------
def _safe_unique_sorted(x):
    x = np.asarray(x, dtype=np.int64).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.array([], dtype=np.int64)
    return np.sort(np.unique(x.astype(np.int64)))

def _subsample_times(times_abs, max_n, rng):
    times_abs = _safe_unique_sorted(times_abs)
    if (max_n is None) or (times_abs.size <= int(max_n)):
        return times_abs
    idx = np.sort(rng.choice(times_abs.size, size=int(max_n), replace=False))
    return times_abs[idx]

def _extract_full_ei(raw, times_abs, window, reducer="median", max_spikes=1000):
    times_abs = _safe_unique_sorted(times_abs)
    if times_abs.size == 0:
        return None, np.array([], dtype=np.int64)

    times_use = _subsample_times(times_abs, max_spikes, rng)

    sn, tv = extract_snippets_fast_ram(
        raw,
        times_use,
        window=window,
        selected_channels=np.arange(raw.shape[1], dtype=np.int32)
    )
    tv = np.asarray(tv, dtype=np.int64)

    if sn.shape[2] == 0:
        return None, tv

    if reducer == "median":
        ei = np.median(sn, axis=2).astype(np.float32)
    elif reducer == "mean":
        ei = np.mean(sn, axis=2).astype(np.float32)
    else:
        raise ValueError("EI_REDUCER must be 'median' or 'mean'")

    return ei, tv

def _lims_xy(x, y, use_pct=True, lo=0.5, hi=99.5):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    if not use_pct:
        return np.nanmin(x), np.nanmax(x), np.nanmin(y), np.nanmax(y)

    xmin, xmax = np.nanpercentile(x, [lo, hi])
    ymin, ymax = np.nanpercentile(y, [lo, hi])
    return xmin, xmax, ymin, ymax

# -------------------------
# choose score matrix
# -------------------------
score_times_valid = np.asarray(proposal_harm_competition["score_times_valid"], dtype=np.int64)

if SCORE_KIND == "sum":
    score_mat = np.asarray(proposal_harm_competition["score_mat_sum"], dtype=np.float32)
elif SCORE_KIND == "mean":
    score_mat = np.asarray(proposal_harm_competition["score_mat_mean"], dtype=np.float32)
else:
    raise ValueError("SCORE_KIND must be 'sum' or 'mean'")

if score_mat.shape[1] != 3:
    raise RuntimeError(f"Expected score_mat with 3 columns, got shape {score_mat.shape}")

s1 = score_mat[:, 0]   # G1
s2 = score_mat[:, 1]   # G2
s3 = score_mat[:, 2]   # G3

# -------------------------
# pairwise margins
# positive means the first group wins
# -------------------------
d12 = s2 - s1   # positive => G1 better than G2
d13 = s3 - s1   # positive => G1 better than G3
d23 = s3 - s2   # positive => G2 better than G3

# hard winner just for reference
hard_winner = np.argmin(score_mat, axis=1).astype(np.int32)   # 0,1,2

# -------------------------
# symmetric threshold-based split
# -------------------------
is_g1 = (d12 > float(THR12)) & (d13 > float(THR13))
is_g2 = (d12 < -float(THR12)) & (d23 > float(THR23))
is_g3 = (d13 < -float(THR13)) & (d23 < -float(THR23))

# these should not overlap if logic is consistent
overlap = (is_g1.astype(int) + is_g2.astype(int) + is_g3.astype(int)) > 1
if np.any(overlap):
    raise RuntimeError("Threshold regions overlap; something is wrong")

main_group = np.full(score_times_valid.size, -1, dtype=np.int32)
main_group[is_g1] = 0
main_group[is_g2] = 1
main_group[is_g3] = 2

is_unresolved = (main_group < 0)

label_text = np.empty(score_times_valid.size, dtype=object)
label_text[main_group == 0] = "G1"
label_text[main_group == 1] = "G2"
label_text[main_group == 2] = "G3"
label_text[is_unresolved] = "unresolved"

print(f"SCORE_KIND = {SCORE_KIND}")
print(f"Thresholds: THR12={THR12:.2f}, THR13={THR13:.2f}, THR23={THR23:.2f}\n")

print("Main-group counts:")
for lab in ["G1", "G2", "G3", "unresolved"]:
    n = int(np.sum(label_text == lab))
    print(f"  {lab:12s} {n}")

print("\nHard-winner counts (reference only):")
for gi, lab in enumerate(["G1", "G2", "G3"]):
    n = int(np.sum(hard_winner == gi))
    print(f"  {lab:12s} {n}")

# -------------------------
# plot 1: pairwise scatters with threshold bands
# -------------------------
color_map = {
    "G1": "tab:blue",
    "G2": "tab:orange",
    "G3": "tab:green",
    "unresolved": "red",
}

pair_info = [
    (0, 1, s1, s2, float(THR12), "G1 summed harm", "G2 summed harm", "G1 vs G2"),
    (0, 2, s1, s3, float(THR13), "G1 summed harm", "G3 summed harm", "G1 vs G3"),
    (1, 2, s2, s3, float(THR23), "G2 summed harm", "G3 summed harm", "G2 vs G3"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False, sharey=False)

for ax, (_, _, xa, ya, thr, xlabel, ylabel, title) in zip(axes, pair_info):
    for lab in ["G1", "G2", "G3", "unresolved"]:
        m = (label_text == lab)
        if np.any(m):
            ax.scatter(
                xa[m], ya[m],
                s=10, alpha=0.45, c=color_map[lab],
                label=lab if title == "G1 vs G2" else None
            )

    xmin, xmax, ymin, ymax = _lims_xy(
        xa, ya,
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW, hi=PCTL_HIGH
    )
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    xx = np.linspace(xmin, xmax, 200)
    ax.plot(xx, xx, "k--", lw=1.0, alpha=0.7)
    ax.plot(xx, xx + thr, color="k", lw=1.2, alpha=0.9)
    ax.plot(xx, xx - thr, color="k", lw=1.2, alpha=0.9)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)

axes[0].legend(loc="best", frameon=False)
plt.tight_layout()
plt.show()

# -------------------------
# plot 2: margin histograms
# -------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

hist_specs = [
    (d12, THR12, "d12 = G2 - G1  (positive => G1 better)"),
    (d13, THR13, "d13 = G3 - G1  (positive => G1 better)"),
    (d23, THR23, "d23 = G3 - G2  (positive => G2 better)"),
]

for ax, (dd, thr, title) in zip(axes, hist_specs):
    ax.hist(dd, bins=80, color="lightgray", edgecolor="none")
    ax.axvline(+thr, color="k", lw=1.5)
    ax.axvline(-thr, color="k", lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("margin")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

# -------------------------
# proposal EIs for overlay
# -------------------------
res_pick = k_sweep_out["results_by_k"][K_FOR_ASSIGN]
proposal_eis = []
for kk in GROUP_CLUSTER_IDS:
    cl = res_pick["clusters"][kk]
    ei_prop = cl["core_by_frac"][plot_core_key]["ei"]
    if ei_prop is None:
        raise RuntimeError(f"K={K_FOR_ASSIGN} cl{kk}: missing proposal EI at frac {plot_core_key}")
    proposal_eis.append(np.asarray(ei_prop, dtype=np.float32))

# -------------------------
# EI summaries of the threshold-split groups
# -------------------------
group_ei_summary = {}

for lab in ["G1", "G2", "G3", "unresolved"]:
    tt = score_times_valid[label_text == lab]
    ei_lab, tv_lab = _extract_full_ei(
        raw_mod,
        tt,
        window=win_use,
        reducer=EI_REDUCER,
        max_spikes=EI_CAP
    )
    group_ei_summary[lab] = {
        "times": tt.copy(),
        "ei": None if ei_lab is None else ei_lab.copy(),
        "n_valid_for_ei": int(tv_lab.size),
    }

# -------------------------
# plot 3: proposal EI vs split-group EI
# -------------------------
for gi, lab in enumerate(["G1", "G2", "G3"]):
    ei_prop = proposal_eis[gi]
    ei_grp = group_ei_summary[lab]["ei"]

    if ei_grp is None:
        print(f"{lab}: no valid EI for overlay plot")
        continue

    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111)
    pew.plot_ei_waveforms(
        [ei_prop, ei_grp],
        positions=ei_positions,
        ref_channel=main_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax,
        colors=["black", "tab:red"],
        alpha=[0.9, 0.9],
        linewidth=[1.2, 1.0],
    )
    ax.set_title(
        f"{lab} | proposal EI (black) vs threshold-split EI (red)\n"
        f"cluster cl{GROUP_CLUSTER_IDS[gi]} | spikes={group_ei_summary[lab]['times'].size}"
    )
    plt.tight_layout()
    plt.show()

# unresolved EI
if group_ei_summary["unresolved"]["ei"] is not None:
    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111)
    pew.plot_ei_waveforms(
        group_ei_summary["unresolved"]["ei"],
        positions=ei_positions,
        ref_channel=main_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax,
    )
    ax.set_title(
        f"unresolved | threshold-split EI | spikes={group_ei_summary['unresolved']['times'].size}"
    )
    plt.tight_layout()
    plt.show()
else:
    print("unresolved: no valid EI plot")

# -------------------------
# save outputs
# -------------------------
proposal_pair_threshold_out = {
    "K_for_assign": int(K_FOR_ASSIGN),
    "group_cluster_ids": [int(x) for x in GROUP_CLUSTER_IDS],
    "score_kind": SCORE_KIND,
    "score_times_valid": score_times_valid.copy(),
    "score_mat": score_mat.copy(),     # columns = G1,G2,G3
    "s1": s1.copy(),
    "s2": s2.copy(),
    "s3": s3.copy(),
    "d12": d12.copy(),
    "d13": d13.copy(),
    "d23": d23.copy(),
    "thresholds": {
        "THR12": float(THR12),
        "THR13": float(THR13),
        "THR23": float(THR23),
    },
    "hard_winner": hard_winner.copy(),   # 0,1,2 for reference only
    "main_group": main_group.copy(),     # 0,1,2 or -1
    "label_text": label_text.copy(),
    "group_ei_summary": group_ei_summary,
}

print("\nSaved outputs in proposal_pair_threshold_out")

In [ ]:
# ============================================================
# Soft-assigned group plots:
# self (best) score vs second-best score
#
# Add as a NEW cell after proposal_harm_competition
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER SETTINGS
# -------------------------
K_FOR_PLOT = CHOSEN_K
GROUP_CLUSTER_IDS = [0, 1, 2]

# use "sum" unless you explicitly want the mean-harm version
SCORE_KIND = "sum"   # "sum" or "mean"

USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

POINT_SIZE = 10
POINT_ALPHA = 0.45

# -------------------------
# checks
# -------------------------
assert "proposal_harm_competition" in globals(), "Need proposal_harm_competition"

if proposal_harm_competition["K_for_plot"] != K_FOR_PLOT:
    raise RuntimeError(
        f"proposal_harm_competition was built for K={proposal_harm_competition['K_for_plot']}, "
        f"but K_FOR_PLOT={K_FOR_PLOT}"
    )

if list(proposal_harm_competition["group_cluster_ids"]) != list(GROUP_CLUSTER_IDS):
    raise RuntimeError(
        f"proposal_harm_competition uses groups {proposal_harm_competition['group_cluster_ids']}, "
        f"but GROUP_CLUSTER_IDS={GROUP_CLUSTER_IDS}"
    )

# -------------------------
# choose score matrix
# -------------------------
if SCORE_KIND == "sum":
    score_mat = np.asarray(proposal_harm_competition["score_mat_sum"], dtype=np.float32)
elif SCORE_KIND == "mean":
    score_mat = np.asarray(proposal_harm_competition["score_mat_mean"], dtype=np.float32)
else:
    raise ValueError("SCORE_KIND must be 'sum' or 'mean'")

# columns are G1, G2, G3
if score_mat.shape[1] != 3:
    raise RuntimeError(f"Expected score_mat with 3 columns, got shape {score_mat.shape}")

score_times_valid = np.asarray(proposal_harm_competition["score_times_valid"], dtype=np.int64)

# -------------------------
# soft assignment and second-best
# more negative = better
# -------------------------
soft_group = np.argmin(score_mat, axis=1).astype(np.int32)   # 0,1,2
order = np.argsort(score_mat, axis=1)                        # ascending: most negative first

best_score = score_mat[np.arange(score_mat.shape[0]), order[:, 0]]
second_score = score_mat[np.arange(score_mat.shape[0]), order[:, 1]]
second_group = order[:, 1].astype(np.int32)

# sanity: best_score should match soft_group column
best_score_from_soft = score_mat[np.arange(score_mat.shape[0]), soft_group]
if not np.allclose(best_score, best_score_from_soft, equal_nan=True):
    raise RuntimeError("best_score mismatch with soft_group")

# -------------------------
# helper for square limits
# -------------------------
def _square_limits(x, y, use_pct=True, lo=0.5, hi=99.5, pad_frac=0.03):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]

    if x.size == 0:
        return (-1, 1)

    if use_pct:
        xmin, xmax = np.percentile(x, [lo, hi])
        ymin, ymax = np.percentile(y, [lo, hi])
    else:
        xmin, xmax = np.min(x), np.max(x)
        ymin, ymax = np.min(y), np.max(y)

    lo_all = min(xmin, ymin)
    hi_all = max(xmax, ymax)

    if not np.isfinite(lo_all) or not np.isfinite(hi_all):
        return (-1, 1)

    if hi_all <= lo_all:
        hi_all = lo_all + 1.0

    span = hi_all - lo_all
    pad = pad_frac * span
    return (lo_all - pad, hi_all + pad)

# -------------------------
# colors
# -------------------------
color_map = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:green",
}

# -------------------------
# counts
# -------------------------
print(f"SCORE_KIND = {SCORE_KIND}")
for gi in range(3):
    n = int(np.sum(soft_group == gi))
    print(f"G{gi+1} soft-assigned: {n}")

# -------------------------
# three stand-alone square scatters
# -------------------------
for gi in range(3):
    m = (soft_group == gi)

    x = best_score[m]      # self / best
    y = second_score[m]    # second-best, whichever it is

    second_counts = {
        f"G{sj+1}": int(np.sum(second_group[m] == sj))
        for sj in range(3) if sj != gi
    }

    lim0, lim1 = _square_limits(
        x, y,
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW,
        hi=PCTL_HIGH
    )

    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(
        x, y,
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        c=color_map[gi]
    )
    plt.plot([lim0, lim1], [lim0, lim1], "k--", lw=1.2, alpha=0.8)

    plt.xlim(lim0, lim1)
    plt.ylim(lim0, lim1)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.xlabel(f"G{gi+1} harm score (best / self)")
    plt.ylabel("Second-best harm score")
    plt.title(
        f"G{gi+1} soft-assigned spikes\n"
        f"cl{GROUP_CLUSTER_IDS[gi]} | N={x.size} | second-best: {second_counts}"
    )
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

# -------------------------
# save outputs
# -------------------------
proposal_soft_assign_plots = {
    "K_for_plot": int(K_FOR_PLOT),
    "group_cluster_ids": [int(x) for x in GROUP_CLUSTER_IDS],
    "score_kind": SCORE_KIND,
    "score_times_valid": score_times_valid.copy(),
    "score_mat": score_mat.copy(),
    "soft_group": soft_group.copy(),
    "best_score": best_score.copy(),
    "second_score": second_score.copy(),
    "second_group": second_group.copy(),
}

print("\nSaved outputs in proposal_soft_assign_plots")

In [ ]:
# ============================================================
# Uncertainty flags v2:
#   1) bad-fit tail within each soft group
#   2) competitor class:
#         second-best unusually negative for that group
#         AND close to diagonal (small gap) for that group
#
# Add as a NEW cell after proposal_soft_assign_plots
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# USER SETTINGS
# -------------------------
BADFIT_PERCENTILE = 95.0      # worst self-score tail within each soft group

# competitor / ambiguity class:
# more negative second-best = stronger runner-up
SECOND_BEST_PERCENTILE = 10.0   # lower tail of second-best within each soft group
GAP_PERCENTILE = 10.0           # lower tail of gap=(second-best-self), i.e. closer to diagonal

POINT_SIZE = 10
POINT_ALPHA = 0.50

USE_PERCENTILE_LIMITS = True
PCTL_LOW  = 0.5
PCTL_HIGH = 99.5

# -------------------------
# checks
# -------------------------
assert "proposal_soft_assign_plots" in globals(), "Need proposal_soft_assign_plots"

soft_group   = np.asarray(proposal_soft_assign_plots["soft_group"], dtype=np.int32)       # 0,1,2
best_score   = np.asarray(proposal_soft_assign_plots["best_score"], dtype=np.float32)      # self / winner
second_score = np.asarray(proposal_soft_assign_plots["second_score"], dtype=np.float32)    # runner-up
second_group = np.asarray(proposal_soft_assign_plots["second_group"], dtype=np.int32)
score_times_valid = np.asarray(proposal_soft_assign_plots["score_times_valid"], dtype=np.int64)

K_FOR_PLOT = int(proposal_soft_assign_plots["K_for_plot"])
GROUP_CLUSTER_IDS = list(proposal_soft_assign_plots["group_cluster_ids"])
SCORE_KIND = proposal_soft_assign_plots["score_kind"]

gap = (second_score - best_score).astype(np.float32)   # small = close to diagonal

# -------------------------
# helpers
# -------------------------
def _square_limits(x, y, use_pct=True, lo=0.5, hi=99.5, pad_frac=0.03):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]

    if x.size == 0:
        return (-1.0, 1.0)

    if use_pct:
        xmin, xmax = np.percentile(x, [lo, hi])
        ymin, ymax = np.percentile(y, [lo, hi])
    else:
        xmin, xmax = np.min(x), np.max(x)
        ymin, ymax = np.min(y), np.max(y)

    lo_all = min(xmin, ymin)
    hi_all = max(xmax, ymax)
    if hi_all <= lo_all:
        hi_all = lo_all + 1.0

    span = hi_all - lo_all
    pad = pad_frac * span
    return (lo_all - pad, hi_all + pad)

# -------------------------
# thresholds per soft group
# -------------------------
badfit_thresh = np.full(3, np.nan, dtype=np.float32)
second_thresh = np.full(3, np.nan, dtype=np.float32)
gap_thresh    = np.full(3, np.nan, dtype=np.float32)

uncertain_badfit = np.zeros(best_score.shape[0], dtype=bool)
uncertain_competitor = np.zeros(best_score.shape[0], dtype=bool)

for gi in range(3):
    m = (soft_group == gi)
    if not np.any(m):
        continue

    # bad-fit: winner itself is in the worst (most positive) tail
    badfit_thresh[gi] = np.percentile(best_score[m], BADFIT_PERCENTILE)
    uncertain_badfit[m] = best_score[m] >= badfit_thresh[gi]

    # competitor: runner-up unusually good (more negative than usual)
    second_thresh[gi] = np.percentile(second_score[m], SECOND_BEST_PERCENTILE)

    # close to diagonal: unusually small gap
    gap_thresh[gi] = np.percentile(gap[m], GAP_PERCENTILE)

    uncertain_competitor[m] = (
        (second_score[m] <= second_thresh[gi]) &
        (gap[m] <= gap_thresh[gi])
    )

# -------------------------
# combined labels
# -------------------------
is_clean = (~uncertain_badfit) & (~uncertain_competitor)
is_badfit_only = uncertain_badfit & (~uncertain_competitor)
is_comp_only = (~uncertain_badfit) & uncertain_competitor
is_both = uncertain_badfit & uncertain_competitor

label_text = np.full(best_score.shape[0], "", dtype=object)
label_text[is_clean] = "clean"
label_text[is_badfit_only] = "badfit_only"
label_text[is_comp_only] = "competitor_only"
label_text[is_both] = "both"

print(f"SCORE_KIND = {SCORE_KIND}")
print(f"BADFIT_PERCENTILE = {BADFIT_PERCENTILE}")
print(f"SECOND_BEST_PERCENTILE = {SECOND_BEST_PERCENTILE}")
print(f"GAP_PERCENTILE = {GAP_PERCENTILE}\n")

for gi in range(3):
    m = (soft_group == gi)
    print(
        f"G{gi+1} soft-assigned (cl{GROUP_CLUSTER_IDS[gi]}): "
        f"N={int(m.sum())} | "
        f"badfit_thr={badfit_thresh[gi]:.3f} | "
        f"second_thr={second_thresh[gi]:.3f} | "
        f"gap_thr={gap_thresh[gi]:.3f}"
    )
    if np.any(m):
        print(f"   clean            : {int(np.sum(m & is_clean))}")
        print(f"   badfit_only      : {int(np.sum(m & is_badfit_only))}")
        print(f"   competitor_only  : {int(np.sum(m & is_comp_only))}")
        print(f"   both             : {int(np.sum(m & is_both))}")

# -------------------------
# plots per soft group
# x = self / winner
# y = second-best
#
# show:
#   - unity line
#   - badfit vertical line
#   - second-best horizontal line
#   - close-to-diagonal line: y = x + gap_thresh
# -------------------------
color_map = {
    "clean": "tab:blue",
    "badfit_only": "tab:red",
    "competitor_only": "tab:orange",
    "both": "purple",
}

for gi in range(3):
    m = (soft_group == gi)

    x = best_score[m]
    y = second_score[m]
    g = gap[m]
    lab = label_text[m]

    lim0, lim1 = _square_limits(
        x, y,
        use_pct=USE_PERCENTILE_LIMITS,
        lo=PCTL_LOW,
        hi=PCTL_HIGH
    )

    plt.figure(figsize=(6.5, 6.5))

    for lab_name in ["clean", "badfit_only", "competitor_only", "both"]:
        mm = (lab == lab_name)
        if np.any(mm):
            plt.scatter(
                x[mm], y[mm],
                s=POINT_SIZE,
                alpha=POINT_ALPHA,
                c=color_map[lab_name],
                label=lab_name
            )

    xx = np.linspace(lim0, lim1, 300)

    # unity line
    plt.plot(xx, xx, "k--", lw=1.2, alpha=0.8)

    # badfit line
    plt.axvline(float(badfit_thresh[gi]), color="k", lw=1.2, alpha=0.8)

    # runner-up unusually good line
    plt.axhline(float(second_thresh[gi]), color="gray", lw=1.2, alpha=0.8)

    # close-to-diagonal line
    plt.plot(xx, xx + float(gap_thresh[gi]), color="gray", lw=1.2, alpha=0.8)

    plt.xlim(lim0, lim1)
    plt.ylim(lim0, lim1)
    plt.gca().set_aspect("equal", adjustable="box")

    second_counts = {
        f"G{sj+1}": int(np.sum(second_group[m] == sj))
        for sj in range(3) if sj != gi
    }

    plt.xlabel(f"G{gi+1} harm score (self / winner)")
    plt.ylabel("Second-best harm score")
    plt.title(
        f"G{gi+1} soft-assigned spikes\n"
        f"cl{GROUP_CLUSTER_IDS[gi]} | N={x.size} | second-best: {second_counts}"
    )
    plt.grid(alpha=0.25)
    plt.legend(loc="best", frameon=False)
    plt.tight_layout()
    plt.show()

# -------------------------
# histograms
# -------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharey='row')

for gi in range(3):
    m = (soft_group == gi)

    # self-score histogram
    axes[0, gi].hist(best_score[m], bins=70, color="lightgray", edgecolor="none")
    axes[0, gi].axvline(float(badfit_thresh[gi]), color="k", lw=1.5)
    axes[0, gi].set_title(f"G{gi+1} self-score | cl{GROUP_CLUSTER_IDS[gi]}")
    axes[0, gi].set_xlabel("self / winner harm score")
    axes[0, gi].grid(alpha=0.25)

    # gap histogram
    axes[1, gi].hist(gap[m], bins=70, color="lightgray", edgecolor="none")
    axes[1, gi].axvline(float(gap_thresh[gi]), color="k", lw=1.5)
    axes[1, gi].set_title(f"G{gi+1} gap = second-best - self")
    axes[1, gi].set_xlabel("gap")
    axes[1, gi].grid(alpha=0.25)

axes[0, 0].set_ylabel("count")
axes[1, 0].set_ylabel("count")
plt.tight_layout()
plt.show()

# -------------------------
# save outputs
# -------------------------
proposal_uncertainty_flags_v2 = {
    "K_for_plot": K_FOR_PLOT,
    "group_cluster_ids": GROUP_CLUSTER_IDS,
    "score_kind": SCORE_KIND,
    "score_times_valid": score_times_valid.copy(),
    "soft_group": soft_group.copy(),
    "best_score": best_score.copy(),
    "second_score": second_score.copy(),
    "second_group": second_group.copy(),
    "gap": gap.copy(),
    "badfit_percentile": float(BADFIT_PERCENTILE),
    "second_best_percentile": float(SECOND_BEST_PERCENTILE),
    "gap_percentile": float(GAP_PERCENTILE),
    "badfit_thresh_per_group": badfit_thresh.copy(),
    "second_thresh_per_group": second_thresh.copy(),
    "gap_thresh_per_group": gap_thresh.copy(),
    "uncertain_badfit": uncertain_badfit.copy(),
    "uncertain_competitor": uncertain_competitor.copy(),
    "label_text": label_text.copy(),
}

print("\nSaved outputs in proposal_uncertainty_flags_v2")

### inspect individual spikes

In [ ]:
def plot_ranked_bl_tr_spikes(
    out,
    bl_range=(0, 5),
    tr_range=(0, 5),
):
    """
    Plot suspicious BL spikes and promising TR spikes using the stored gallery data.

    Ranking:
      BL spikes are ordered by aggregate BL-cloud score, lowest first.
      TR spikes are ordered by aggregate BL-cloud score, highest first.

    bl_range, tr_range:
      tuples like (0, 5), (5, 10), etc.
    """
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None. Re-run inspect_lighthouse_channel after patching.")

    top12 = np.asarray(gallery["top12"], dtype=np.int32)
    discrim_channels = np.asarray(gallery["discrim_channels"], dtype=np.int32)
    discrim_idx = np.asarray(gallery["discrim_idx"], dtype=np.int32)
    discrim_weights = np.asarray(gallery["discrim_weights"], dtype=np.float32)

    t_ms = np.asarray(gallery["t_ms"], dtype=np.float32)

    med_tl = np.asarray(gallery["med_top_left"], dtype=np.float32)      # [12, L]
    med_bl = np.asarray(gallery["med_bottom_left"], dtype=np.float32)   # [12, L]
    med_tr = gallery["med_top_right"]
    med_tr = None if med_tr is None else np.asarray(med_tr, dtype=np.float32)

    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)     # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    sn_tr = None if sn_tr is None else np.asarray(sn_tr, dtype=np.float32)

    cloud_scores_bl = np.asarray(gallery["cloud_scores_bl"], dtype=np.float32)   # [n_good, Nbl]
    cloud_scores_tr = np.asarray(gallery["cloud_scores_tr"], dtype=np.float32)   # [n_good, Ntr]
    agg_score_bl = np.asarray(gallery["agg_score_bl"], dtype=np.float32)         # [Nbl]
    agg_score_tr = np.asarray(gallery["agg_score_tr"], dtype=np.float32)         # [Ntr]

    order_bl_low = np.asarray(gallery["order_bl_low"], dtype=np.int64)
    order_tr_high = np.asarray(gallery["order_tr_high"], dtype=np.int64)

    source_ch = int(gallery.get("source_ch", -1))
    main_ch = int(gallery.get("main_ch", -1))

    discrim_set = set(int(x) for x in discrim_channels.tolist())
    local_good_lookup = {int(discrim_channels[j]): int(j) for j in range(discrim_channels.size)}

    def _plot_one(group_name, spike_idx, rank_pos, raw_snip_12, agg_score, per_ch_scores):
        fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
        axes = axes.ravel()

        print(f"\n[{group_name}] rank {rank_pos} | spike idx {spike_idx} | aggregate BL-cloud score = {agg_score:.4f}")
        for j_good, ch_good in enumerate(discrim_channels):
            print(
                f"    ch {int(ch_good):3d}: score = {float(per_ch_scores[j_good]):.4f} "
                f"(weight {float(discrim_weights[j_good]):.3f})"
            )

        for i in range(min(12, top12.size)):
            ax = axes[i]
            ch_i = int(top12[i])

            # current spike
            ax.plot(
                t_ms, raw_snip_12[i],
                color="black", lw=1.8,
                label="current spike" if i == 0 else None,
                zorder=4
            )

            # TL median
            ax.plot(
                t_ms, med_tl[i],
                color="tab:blue", lw=1.3,
                label="TL median" if i == 0 else None,
                zorder=2
            )

            # BL median
            ax.plot(
                t_ms, med_bl[i],
                color="tab:red", lw=1.3,
                label="BL median" if i == 0 else None,
                zorder=2
            )

            # TR median
            if med_tr is not None:
                ax.plot(
                    t_ms, med_tr[i],
                    color="purple", lw=1.3,
                    label="TR median" if i == 0 else None,
                    zorder=2
                )

            ax.axvline(0.0, color="k", lw=0.5, alpha=0.25)
            ax.axhline(0.0, color="k", lw=0.5, alpha=0.15)
            ax.grid(alpha=0.20)

            if ch_i in discrim_set:
                local_score = float(per_ch_scores[local_good_lookup[ch_i]])
                title = f"ch {ch_i} * | score={local_score:.3f}"
            else:
                title = f"ch {ch_i}"
            ax.set_title(title, fontsize=10)

        for j in range(12, len(axes)):
            axes[j].axis("off")

        axes[0].legend(loc="best", frameon=False)

        fig.suptitle(
            f"source ch {source_ch} | main ch {main_ch} | {group_name} spike idx {spike_idx} "
            f"| rank {rank_pos} | aggregate score {agg_score:.4f}",
            y=0.98
        )
        plt.tight_layout()
        plt.show()

    # ---- BL spikes: lowest-scoring first ----
    bl_start, bl_stop = int(bl_range[0]), int(bl_range[1])
    bl_sel = order_bl_low[bl_start:bl_stop]

    for k, spike_idx in enumerate(bl_sel, start=bl_start):
        raw_snip = sn_bl[:, :, int(spike_idx)]              # [12, L]
        agg = float(agg_score_bl[int(spike_idx)])
        per_ch = cloud_scores_bl[:, int(spike_idx)]         # [n_good]
        _plot_one("BL", int(spike_idx), k, raw_snip, agg, per_ch)

    # ---- TR spikes: highest-scoring first ----
    if sn_tr is None:
        print("No TR snippets stored in spike_gallery_data.")
        return

    tr_start, tr_stop = int(tr_range[0]), int(tr_range[1])
    tr_sel = order_tr_high[tr_start:tr_stop]

    for k, spike_idx in enumerate(tr_sel, start=tr_start):
        raw_snip = sn_tr[:, :, int(spike_idx)]              # [12, L]
        agg = float(agg_score_tr[int(spike_idx)])
        per_ch = cloud_scores_tr[:, int(spike_idx)]         # [n_good]
        _plot_one("TR", int(spike_idx), k, raw_snip, agg, per_ch)

# choose ranges
BL_RANGE = (0, 3)   # 5 lowest-scoring BL spikes
TR_RANGE = (0, 3)   # 5 highest-scoring TR spikes

plot_ranked_bl_tr_spikes(out, bl_range=BL_RANGE, tr_range=TR_RANGE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# =========================
# USER SETTINGS
# =========================
COS_MASK_ADC = 30.0

BL_RANGE = (0, 10)   # lowest-amplitude BL spikes on main channel
TR_RANGE = (0, 10)   # highest-amplitude TR spikes on main channel

N_COLS = 5
HIST_BINS = np.linspace(-1.0, 1.0, 61)


def _flatten_masked_snips(snips_12_l_n, mask_12_l):
    """
    snips_12_l_n : [12, L, N]
    mask_12_l    : [12, L] bool

    returns X : [N, M] where M = number of True entries in mask
    """
    snips_12_l_n = np.asarray(snips_12_l_n, dtype=np.float32)
    mask_12_l = np.asarray(mask_12_l, dtype=bool)

    assert snips_12_l_n.ndim == 3
    assert mask_12_l.shape == snips_12_l_n.shape[:2]

    X = snips_12_l_n.transpose(2, 0, 1).reshape(snips_12_l_n.shape[2], -1)
    return X[:, mask_12_l.ravel()].astype(np.float32)


def _row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    nrm = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(nrm, eps)


def plot_bl_tr_similarity_histograms(out,
                                     cos_mask_adc=30.0,
                                     bl_range=(0, 20),
                                     tr_range=(0, 20),
                                     n_cols=5,
                                     hist_bins=None):
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None.")

    top12 = np.asarray(gallery["top12"], dtype=np.int32)
    main_ch = int(gallery.get("main_ch", -1))
    source_ch = int(gallery.get("source_ch", -1))

    med_bl = np.asarray(gallery["med_bottom_left"], dtype=np.float32)   # [12, L]
    med_tr = gallery["med_top_right"]
    if med_tr is None:
        raise ValueError("No TR median stored in spike_gallery_data.")
    med_tr = np.asarray(med_tr, dtype=np.float32)                       # [12, L]

    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)     # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    if sn_tr is None:
        raise ValueError("No TR snippets stored in spike_gallery_data.")
    sn_tr = np.asarray(sn_tr, dtype=np.float32)                         # [12, L, Ntr]

    if hist_bins is None:
        hist_bins = np.linspace(-1.0, 1.0, 61)

    # ---- mask on concatenated 12-channel waveform ----
    mask = (np.abs(med_bl) >= float(cos_mask_adc)) | (np.abs(med_tr) >= float(cos_mask_adc))
    n_mask = int(mask.sum())
    if n_mask == 0:
        raise ValueError(f"Mask is empty for COS_MASK_ADC={cos_mask_adc}.")
    print(f"Using {n_mask} masked samples across the 12 channels.")

    # ---- figure out which stored channel is the main channel ----
    main_matches = np.flatnonzero(top12 == main_ch)
    if main_matches.size == 0:
        # fallback to source channel if main_ch somehow not in stored top12
        src_matches = np.flatnonzero(top12 == source_ch)
        if src_matches.size == 0:
            raise ValueError(
                f"Neither main_ch={main_ch} nor source_ch={source_ch} is present in top12={top12.tolist()}."
            )
        i_main = int(src_matches[0])
        print(f"main_ch not in top12; using source_ch={source_ch} for amplitude ranking.")
    else:
        i_main = int(main_matches[0])

    # ---- rank BL / TR by main-channel amplitude ----
    # same definition as before: peak abs value within the snippet on the chosen main channel
    amp_bl = np.max(np.abs(sn_bl[i_main, :, :]), axis=0).astype(np.float32)
    amp_tr = np.max(np.abs(sn_tr[i_main, :, :]), axis=0).astype(np.float32)

    order_bl_low = np.argsort(amp_bl)         # lowest BL amplitude first
    order_tr_high = np.argsort(amp_tr)[::-1]  # highest TR amplitude first

    bl_sel = order_bl_low[int(bl_range[0]):int(bl_range[1])]
    tr_sel = order_tr_high[int(tr_range[0]):int(tr_range[1])]

    print(f"BL selected ranks: {bl_range} -> {len(bl_sel)} spikes")
    print(f"TR selected ranks: {tr_range} -> {len(tr_sel)} spikes")

    # ---- flattened, masked, normalized vectors for all BL / TR spikes ----
    X_bl = _flatten_masked_snips(sn_bl, mask)   # [Nbl, M]
    X_tr = _flatten_masked_snips(sn_tr, mask)   # [Ntr, M]

    X_bl_n = _row_normalize(X_bl)
    X_tr_n = _row_normalize(X_tr)

    def _plot_one_group(candidate_group_name, selected_indices, X_self_n, X_other_n, amps_self,
                        color_self="red", color_other="purple"):
        n_plots = len(selected_indices)
        if n_plots == 0:
            print(f"No {candidate_group_name} spikes selected.")
            return

        n_cols_use = int(n_cols)
        n_rows_use = int(np.ceil(n_plots / float(n_cols_use)))

        fig, axes = plt.subplots(
            n_rows_use, n_cols_use,
            figsize=(4.4 * n_cols_use, 3.6 * n_rows_use),
            sharex=True, sharey=True,
            squeeze=False
        )
        axes = axes.ravel()

        for kk, spike_idx in enumerate(selected_indices):
            spike_idx = int(spike_idx)
            ax = axes[kk]

            v = X_self_n[spike_idx]  # normalized candidate vector

            cos_self = X_self_n @ v
            cos_other = X_other_n @ v

            # remove self-match on same-side histogram
            cos_self = cos_self.copy()
            cos_self[spike_idx] = np.nan

            ax.hist(
                cos_self[np.isfinite(cos_self)],
                bins=hist_bins,
                color=color_self,
                alpha=0.45,
                label="to BL" if candidate_group_name == "BL" else "to TR"
            )
            ax.hist(
                cos_other[np.isfinite(cos_other)],
                bins=hist_bins,
                color=color_other,
                alpha=0.45,
                label="to TR" if candidate_group_name == "BL" else "to BL"
            )

            med_self = float(np.nanmedian(cos_self))
            med_other = float(np.nanmedian(cos_other))

            ax.axvline(med_self, color=color_self, lw=1.2, alpha=0.9)
            ax.axvline(med_other, color=color_other, lw=1.2, alpha=0.9)
            ax.grid(alpha=0.25)

            rank_pos = int(kk + (bl_range[0] if candidate_group_name == "BL" else tr_range[0]))

            ax.set_title(
                f"{candidate_group_name} rank {rank_pos} | idx {spike_idx}\n"
                f"amp={amps_self[spike_idx]:.1f} | "
                f"med self={med_self:.3f}, other={med_other:.3f}",
                fontsize=9
            )
            ax.set_xlim(-1.0, 1.0)

            if kk == 0:
                ax.legend(frameon=False, fontsize=8)

        for jj in range(n_plots, len(axes)):
            axes[jj].axis("off")

        fig.suptitle(
            f"{candidate_group_name} candidate spikes | cosine to BL vs TR clouds | "
            f"source ch {source_ch}, main ch {main_ch}, mask >= {cos_mask_adc} ADC",
            y=0.995
        )
        plt.tight_layout()
        plt.show()

    # For BL candidates: self cloud = BL, other cloud = TR
    _plot_one_group(
        candidate_group_name="BL",
        selected_indices=bl_sel,
        X_self_n=X_bl_n,
        X_other_n=X_tr_n,
        amps_self=amp_bl,
        color_self="red",
        color_other="purple"
    )

    # For TR candidates: self cloud = TR, other cloud = BL
    _plot_one_group(
        candidate_group_name="TR",
        selected_indices=tr_sel,
        X_self_n=X_tr_n,
        X_other_n=X_bl_n,
        amps_self=amp_tr,
        color_self="purple",
        color_other="red"
    )


plot_bl_tr_similarity_histograms(
    out,
    cos_mask_adc=COS_MASK_ADC,
    bl_range=BL_RANGE,
    tr_range=TR_RANGE,
    n_cols=N_COLS,
    hist_bins=HIST_BINS,
)

In [ ]:
decision_out = compute_bl_tr_support_decisions(
    out,
    cos_mask_adc=COS_MASK_ADC,
    k_peak=K_PEAK,
    k_bulk=K_BULK,
    min_bl_bulk=MIN_BL_BULK,
    min_tr_bulk=MIN_TR_BULK,
    bulk_margin=BULK_MARGIN
)

In [ ]:
bl_unc_idx = np.flatnonzero(decision_out["bl_labels"] == "uncertain_lowBL") # uncertain_boundary,uncertain_lowBL
print(bl_unc_idx)
print("n uncertain in BL:", bl_unc_idx.size)

tr_unc_idx = np.flatnonzero(decision_out["tr_labels"] == "uncertain_lowBL") # uncertain_boundary,uncertain_lowBL
print(tr_unc_idx)
print("n uncertain in TR:", tr_unc_idx.size)

In [ ]:
# Scatter: each spike by (BL_bulk, TR_bulk), colored by source group

import numpy as np
import matplotlib.pyplot as plt

bl_metrics = decision_out["bl_metrics"]
tr_metrics = decision_out["tr_metrics"]

bl_x = np.array([m["BL_bulk"] for m in bl_metrics], dtype=float)
bl_y = np.array([m["TR_bulk"] for m in bl_metrics], dtype=float)

tr_x = np.array([m["BL_bulk"] for m in tr_metrics], dtype=float)
tr_y = np.array([m["TR_bulk"] for m in tr_metrics], dtype=float)

p = decision_out["params"]
min_bl_bulk = p["MIN_BL_BULK"]
min_tr_bulk = p["MIN_TR_BULK"]
bulk_margin = p["BULK_MARGIN"]

plt.figure(figsize=(7, 7))

plt.scatter(bl_x, bl_y, s=20, alpha=0.7, label="BL spikes")
plt.scatter(tr_x, tr_y, s=20, alpha=0.7, label="TR spikes")

# equal-support diagonal
xx = np.linspace(
    min(np.min(bl_x), np.min(tr_x), np.min(bl_y), np.min(tr_y)) - 0.02,
    max(np.max(bl_x), np.max(tr_x), np.max(bl_y), np.max(tr_y)) + 0.02,
    200
)
plt.plot(xx, xx, "--", linewidth=1, alpha=0.8, label="BL_bulk = TR_bulk")

# bulk thresholds
plt.axvline(min_bl_bulk, linestyle=":", linewidth=1, alpha=0.8)
plt.axhline(min_tr_bulk, linestyle=":", linewidth=1, alpha=0.8)

plt.xlabel("BL bulk support")
plt.ylabel("TR bulk support")
plt.title("BL vs TR bulk support for all BL and TR spikes")
plt.legend(frameon=False)
plt.grid(alpha=0.25)
plt.axis("equal")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Histograms of delta = BL_bulk - TR_bulk
# Positive delta: below diagonal (BL wins)
# Negative delta: above diagonal (TR wins)
# --------------------------------------------------

DIAG_EPS = 0.05   # tentative "near diagonal" band; adjust later
NBINS = 60

bl_metrics = decision_out["bl_metrics"]
tr_metrics = decision_out["tr_metrics"]

bl_delta = np.array([m["BL_bulk"] - m["TR_bulk"] for m in bl_metrics], dtype=float)
tr_delta = np.array([m["BL_bulk"] - m["TR_bulk"] for m in tr_metrics], dtype=float)

all_delta = np.r_[bl_delta, tr_delta]
xmin = np.min(all_delta) - 0.02
xmax = np.max(all_delta) + 0.02
bins = np.linspace(xmin, xmax, NBINS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)

# BL-source spikes
axes[0].hist(bl_delta, bins=bins, alpha=0.8)
axes[0].axvline(0.0, linestyle="--", linewidth=1)
axes[0].axvspan(-DIAG_EPS, DIAG_EPS, alpha=0.15)
axes[0].set_title(f"BL-source spikes\nn={bl_delta.size}")
axes[0].set_xlabel("delta = BL_bulk - TR_bulk")
axes[0].set_ylabel("count")

# TR-source spikes
axes[1].hist(tr_delta, bins=bins, alpha=0.8)
axes[1].axvline(0.0, linestyle="--", linewidth=1)
axes[1].axvspan(-DIAG_EPS, DIAG_EPS, alpha=0.15)
axes[1].set_title(f"TR-source spikes\nn={tr_delta.size}")
axes[1].set_xlabel("delta = BL_bulk - TR_bulk")

plt.suptitle("Histogram of BL-vs-TR bulk support difference")
plt.tight_layout()
plt.show()

# Simple counts for the tentative near-diagonal band
n_bl_near = np.sum(np.abs(bl_delta) <= DIAG_EPS)
n_tr_near = np.sum(np.abs(tr_delta) <= DIAG_EPS)

print(f"DIAG_EPS = {DIAG_EPS:.3f}")
print(f"BL near-diagonal: {n_bl_near}/{bl_delta.size} = {n_bl_near / max(1, bl_delta.size):.1%}")
print(f"TR near-diagonal: {n_tr_near}/{tr_delta.size} = {n_tr_near / max(1, tr_delta.size):.1%}")

In [ ]:
idx = 75

m = query_support_decision(decision_out, "TR", idx, plot=QUERY_PLOT)
plot_one_bl_tr_spike_overlay(out, "TR", idx)

# plot_spike_support_profiles(out, "TR", 1018)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Build full-array EIs for labeled BL / TR subsets
# --------------------------------------------------

gallery = out["spike_gallery_data"]

labels_bl = np.asarray(decision_out["bl_labels"], dtype=object)
labels_tr = np.asarray(decision_out["tr_labels"], dtype=object)

bl_times = np.asarray(gallery["times_bottom_left"], dtype=np.int64)
tr_times = np.asarray(gallery["times_top_right"], dtype=np.int64)

if bl_times.size != labels_bl.size:
    raise ValueError(
        f"BL times/labels mismatch: {bl_times.size} times vs {labels_bl.size} labels"
    )
if tr_times.size != labels_tr.size:
    raise ValueError(
        f"TR times/labels mismatch: {tr_times.size} times vs {labels_tr.size} labels"
    )

# Prefer original raw if available; otherwise fall back to raw_mod
raw_src = raw_orig if "raw_orig" in globals() else raw_mod
win_use = win if "win" in globals() else (-40, 80)
all_ch = np.arange(raw_src.shape[1], dtype=np.int32)

ref_ch = int(gallery.get("main_ch", out["step4"].get("main_ch", 0)))

def build_ei_from_times(times_abs):
    times_abs = np.asarray(times_abs, dtype=np.int64)
    if times_abs.size == 0:
        return None, 0

    snips, valid_times = extract_snippets_fast_ram(
        raw_src,
        times_abs,
        window=win_use,
        selected_channels=all_ch
    )

    if snips.shape[2] == 0:
        return None, 0

    # use median for robustness
    ei = np.median(snips, axis=2).astype(np.float32)
    return ei, int(snips.shape[2])

# labeled groups
bl_lh_times   = bl_times[labels_bl == "LH"]
tr_lh_times   = tr_times[labels_tr == "LH"]
bl_unc_times  = bl_times[labels_bl == "uncertain_boundary"] # uncertain_boundary,uncertain_lowBL
tr_unc_times  = tr_times[labels_tr == "uncertain_boundary"]  # uncertain_boundary,uncertain_lowBL

# build EIs
ei_bl_lh,  n_bl_lh  = build_ei_from_times(bl_lh_times)
ei_tr_lh,  n_tr_lh  = build_ei_from_times(tr_lh_times)
ei_bl_unc, n_bl_unc = build_ei_from_times(bl_unc_times)
ei_tr_unc, n_tr_unc = build_ei_from_times(tr_unc_times)

# --------------------------------------------------
# Plot: 3 EI panels
#   top    : LH overlay (BL-LH vs TR-LH)
#   bottom : BL-uncertain and TR-uncertain
# --------------------------------------------------

fig = plt.figure(figsize=(20, 12))

ax1 = plt.subplot(2, 2, (1, 2))
if (ei_bl_lh is not None) and (ei_tr_lh is not None):
    pew.plot_ei_waveforms(
        [ei_bl_lh, ei_tr_lh],
        ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax1,
        colors=["black", "purple"],
        alpha=[1.0, 1.0],
        linewidth=[0.8, 0.8],
    )
    ax1.set_title(f"LH-labeled EI overlay | BL n={n_bl_lh} vs TR n={n_tr_lh}")
elif ei_bl_lh is not None:
    pew.plot_ei_waveforms(
        ei_bl_lh,
        ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax1,
        colors="black",
        alpha=1.0,
        linewidth=0.8,
    )
    ax1.set_title(f"LH-labeled EI | BL only, n={n_bl_lh}")
elif ei_tr_lh is not None:
    pew.plot_ei_waveforms(
        ei_tr_lh,
        ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax1,
        colors="purple",
        alpha=1.0,
        linewidth=0.8,
    )
    ax1.set_title(f"LH-labeled EI | TR only, n={n_tr_lh}")
else:
    ax1.text(0.5, 0.5, "No LH-labeled spikes in BL or TR", ha="center", va="center")
    ax1.axis("off")

ax2 = plt.subplot(2, 2, 3)
if ei_bl_unc is not None:
    pew.plot_ei_waveforms(
        ei_bl_unc,
        ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax2,
        colors="red",
        alpha=1.0,
        linewidth=0.8,
    )
    ax2.set_title(f"BL uncertain EI | n={n_bl_unc}")
else:
    ax2.text(0.5, 0.5, "No BL uncertain spikes", ha="center", va="center")
    ax2.axis("off")

ax3 = plt.subplot(2, 2, 4)
if ei_tr_unc is not None:
    pew.plot_ei_waveforms(
        ei_tr_unc,
        ei_positions,
        ref_channel=ref_ch,
        scale=70.0,
        box_height=1.0,
        box_width=50.0,
        ax=ax3,
        colors="purple",
        alpha=1.0,
        linewidth=0.8,
    )
    ax3.set_title(f"TR uncertain EI | n={n_tr_unc}")
else:
    ax3.text(0.5, 0.5, "No TR uncertain spikes", ha="center", va="center")
    ax3.axis("off")

plt.tight_layout()
plt.show()

print("Counts:")
print(f"  BL LH        : {n_bl_lh}")
print(f"  TR LH        : {n_tr_lh}")
print(f"  BL uncertain : {n_bl_unc}")
print(f"  TR uncertain : {n_tr_unc}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# USER-EDITABLE PARAMETERS
# ============================================================

# Mask for concatenated 12-channel waveform:
# include points where either BL or TR median has meaningful signal
COS_MASK_ADC = 30.0

# Neighborhood sizes used to summarize support curves
K_PEAK = [5, 10, 20]
K_BULK = [50, 100, 200]

# Decision thresholds
MIN_BL_BULK = 0.70        # BL must reach this to be called LH
DIAG_EPS = 0.05           # |BL_bulk - TR_bulk| <= this -> boundary uncertain

# Keep these two names alive so existing scatter-plot cells still work.
# They are plotting guides / compatibility aliases, not separate decision thresholds.
MIN_TR_BULK = 0.70
BULK_MARGIN = DIAG_EPS

# Optional plotting when querying one spike
QUERY_PLOT = True



# ============================================================
# HELPERS
# ============================================================

def _flatten_masked_snips(snips_12_l_n, mask_12_l):
    """
    snips_12_l_n : [12, L, N]
    mask_12_l    : [12, L] bool
    returns      : [N, M]
    """
    snips_12_l_n = np.asarray(snips_12_l_n, dtype=np.float32)
    mask_12_l = np.asarray(mask_12_l, dtype=bool)
    assert snips_12_l_n.ndim == 3
    assert mask_12_l.shape == snips_12_l_n.shape[:2]

    X = snips_12_l_n.transpose(2, 0, 1).reshape(snips_12_l_n.shape[2], -1)
    return X[:, mask_12_l.ravel()].astype(np.float32)


def _row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    nrm = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(nrm, eps)


def _resolve_k_list(k_list, n_available):
    return [int(k) for k in k_list if int(k) >= 1 and int(k) <= int(n_available)]


def _topk_mean_curve(sorted_desc):
    """
    sorted_desc : descending sorted similarities, shape [N]
    returns cumulative mean of top-k similarities for k=1..N
    """
    sorted_desc = np.asarray(sorted_desc, dtype=np.float32)
    return np.cumsum(sorted_desc) / np.arange(1, sorted_desc.size + 1, dtype=np.float32)


def _support_metrics_from_curves(bl_curve, tr_curve, k_peak, k_bulk):
    """
    bl_curve, tr_curve : cumulative top-k mean curves
    """
    kmax = min(bl_curve.size, tr_curve.size)
    if kmax < 1:
        raise ValueError("Need at least one neighbor on each side.")

    k_peak_use = _resolve_k_list(k_peak, kmax)
    k_bulk_use = _resolve_k_list(k_bulk, kmax)
    if len(k_peak_use) == 0 or len(k_bulk_use) == 0:
        raise ValueError(f"k lists invalid for kmax={kmax}")

    bl_peak = float(np.mean([bl_curve[k - 1] for k in k_peak_use]))
    tr_peak = float(np.mean([tr_curve[k - 1] for k in k_peak_use]))
    bl_bulk = float(np.mean([bl_curve[k - 1] for k in k_bulk_use]))
    tr_bulk = float(np.mean([tr_curve[k - 1] for k in k_bulk_use]))

    d_peak = bl_peak - tr_peak
    d_bulk = bl_bulk - tr_bulk

    return {
        "kmax": int(kmax),
        "k_peak_used": k_peak_use,
        "k_bulk_used": k_bulk_use,
        "BL_peak": bl_peak,
        "TR_peak": tr_peak,
        "BL_bulk": bl_bulk,
        "TR_bulk": tr_bulk,
        "D_peak": d_peak,
        "D_bulk": d_bulk,
    }


def _assign_label(metrics, min_bl_bulk, diag_eps):
    """
    New decision rule:
      1) near diagonal -> uncertain_boundary
      2) BL wins strongly enough and absolute BL bulk is high -> LH
      3) BL wins but absolute BL bulk is too low -> uncertain_lowBL
      4) otherwise TR wins -> soup
    """
    bl_bulk = float(metrics["BL_bulk"])
    d_bulk = float(metrics["D_bulk"])   # BL_bulk - TR_bulk

    # boundary / genuine ambiguity gets priority
    if abs(d_bulk) <= float(diag_eps):
        return "uncertain_boundary"

    # BL side wins
    if d_bulk > float(diag_eps):
        if bl_bulk >= float(min_bl_bulk):
            return "LH"
        return "uncertain_lowBL"

    # otherwise TR side wins clearly enough
    return "soup"


def _compute_one_spike_metrics(v, X_bl_n, X_tr_n, side, idx, k_peak, k_bulk):
    """
    v      : normalized candidate vector [M]
    X_bl_n : normalized BL matrix [Nbl, M]
    X_tr_n : normalized TR matrix [Ntr, M]
    side   : "BL" or "TR"
    idx    : index on that side (for leave-one-out)
    """
    cos_bl = X_bl_n @ v
    cos_tr = X_tr_n @ v

    if side == "BL":
        cos_bl = cos_bl.copy()
        cos_bl[int(idx)] = np.nan
    elif side == "TR":
        cos_tr = cos_tr.copy()
        cos_tr[int(idx)] = np.nan
    else:
        raise ValueError("side must be BL or TR")

    bl_valid = cos_bl[np.isfinite(cos_bl)]
    tr_valid = cos_tr[np.isfinite(cos_tr)]

    bl_sorted = np.sort(bl_valid)[::-1]
    tr_sorted = np.sort(tr_valid)[::-1]

    bl_curve = _topk_mean_curve(bl_sorted)
    tr_curve = _topk_mean_curve(tr_sorted)

    metrics = _support_metrics_from_curves(bl_curve, tr_curve, k_peak, k_bulk)
    metrics.update({
        "side": side,
        "idx": int(idx),
        "cos_to_BL_sorted": bl_sorted,
        "cos_to_TR_sorted": tr_sorted,
        "BL_curve": bl_curve,
        "TR_curve": tr_curve,
        "diff_curve": bl_curve[:metrics["kmax"]] - tr_curve[:metrics["kmax"]],
    })
    return metrics


def _plot_query_curves(metrics, title_prefix=""):
    bl_sorted = np.asarray(metrics["cos_to_BL_sorted"], dtype=np.float32)
    tr_sorted = np.asarray(metrics["cos_to_TR_sorted"], dtype=np.float32)
    bl_curve = np.asarray(metrics["BL_curve"], dtype=np.float32)
    tr_curve = np.asarray(metrics["TR_curve"], dtype=np.float32)
    diff = np.asarray(metrics["diff_curve"], dtype=np.float32)

    kmax = int(metrics["kmax"])
    ks = np.arange(1, kmax + 1)
    ref_ks = sorted(set(metrics["k_peak_used"] + metrics["k_bulk_used"]))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

    # sorted cosine support
    ax = axes[0]
    ax.plot(np.arange(1, bl_sorted.size + 1), bl_sorted, color="red", lw=1.5, label="to BL")
    ax.plot(np.arange(1, tr_sorted.size + 1), tr_sorted, color="purple", lw=1.5, label="to TR")
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("Sorted cosine support")
    ax.set_xlabel("Neighbor rank")
    ax.set_ylabel("Cosine")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    # top-k mean support
    ax = axes[1]
    ax.plot(ks, bl_curve[:kmax], color="red", lw=1.8, label="BL top-k mean")
    ax.plot(ks, tr_curve[:kmax], color="purple", lw=1.8, label="TR top-k mean")
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("Top-k mean support")
    ax.set_xlabel("k")
    ax.set_ylabel("Mean cosine of top-k neighbors")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    # difference
    ax = axes[2]
    ax.plot(ks, diff[:kmax], color="black", lw=1.8)
    ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("BL minus TR support")
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k mean difference")
    ax.grid(alpha=0.25)

    fig.suptitle(title_prefix, y=0.98)
    plt.tight_layout()
    plt.show()


# ============================================================
# MAIN COMPUTATION
# ============================================================

def compute_bl_tr_support_decisions(
    out,
    cos_mask_adc=30.0,
    k_peak=(5, 10, 20),
    k_bulk=(50, 100, 200),
    min_bl_bulk=0.70,
    min_tr_bulk=0.70,
    bulk_margin=0.03,
    peak_tol=0.02,
    both_high_margin=0.015,
    weak_support_floor=0.60,
):
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None.")

    top12 = np.asarray(gallery["top12"], dtype=np.int32)
    source_ch = int(gallery.get("source_ch", -1))
    main_ch = int(gallery.get("main_ch", -1))

    med_bl = np.asarray(gallery["med_bottom_left"], dtype=np.float32)   # [12, L]
    med_tr = gallery["med_top_right"]
    if med_tr is None:
        raise ValueError("No TR median stored in spike_gallery_data.")
    med_tr = np.asarray(med_tr, dtype=np.float32)

    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)     # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    if sn_tr is None:
        raise ValueError("No TR snippets stored in spike_gallery_data.")
    sn_tr = np.asarray(sn_tr, dtype=np.float32)

    # Union mask over BL/TR medians
    mask = (np.abs(med_bl) >= float(cos_mask_adc)) | (np.abs(med_tr) >= float(cos_mask_adc))
    n_mask = int(mask.sum())
    if n_mask == 0:
        raise ValueError(f"Mask is empty for cos_mask_adc={cos_mask_adc}.")

    # Flatten + normalize
    X_bl = _flatten_masked_snips(sn_bl, mask)
    X_tr = _flatten_masked_snips(sn_tr, mask)
    X_bl_n = _row_normalize(X_bl)
    X_tr_n = _row_normalize(X_tr)

    diag_eps = float(bulk_margin)

    # BL spikes
    bl_metrics = []
    bl_labels = []
    for idx in range(X_bl_n.shape[0]):
        m = _compute_one_spike_metrics(
            v=X_bl_n[idx],
            X_bl_n=X_bl_n,
            X_tr_n=X_tr_n,
            side="BL",
            idx=idx,
            k_peak=k_peak,
            k_bulk=k_bulk,
        )
        label = _assign_label(
            m,
            min_bl_bulk=min_bl_bulk,
            diag_eps=diag_eps,
        )
        m["label"] = label
        bl_metrics.append(m)
        bl_labels.append(label)

    # TR spikes
    tr_metrics = []
    tr_labels = []
    for idx in range(X_tr_n.shape[0]):
        m = _compute_one_spike_metrics(
            v=X_tr_n[idx],
            X_bl_n=X_bl_n,
            X_tr_n=X_tr_n,
            side="TR",
            idx=idx,
            k_peak=k_peak,
            k_bulk=k_bulk,
        )
        label = _assign_label(
            m,
            min_bl_bulk=min_bl_bulk,
            diag_eps=diag_eps,
        )
        m["label"] = label
        tr_metrics.append(m)
        tr_labels.append(label)

    result = {
        "params": {
            "COS_MASK_ADC": float(cos_mask_adc),
            "K_PEAK": list(k_peak),
            "K_BULK": list(k_bulk),
            "MIN_BL_BULK": float(min_bl_bulk),
            "DIAG_EPS": float(diag_eps),

            # compatibility / plotting guides
            "MIN_TR_BULK": float(min_tr_bulk),
            "BULK_MARGIN": float(diag_eps),
        },
        "source_ch": source_ch,
        "main_ch": main_ch,
        "top12": top12.copy(),
        "mask": mask.copy(),
        "n_mask": n_mask,
        "bl_metrics": bl_metrics,
        "tr_metrics": tr_metrics,
        "bl_labels": np.asarray(bl_labels, dtype=object),
        "tr_labels": np.asarray(tr_labels, dtype=object),
    }

    def _count_labels(lbls):
        return {
            "LH": int(np.sum(lbls == "LH")),
            "soup": int(np.sum(lbls == "soup")),
            "uncertain_boundary": int(np.sum(lbls == "uncertain_boundary")),
            "uncertain_lowBL": int(np.sum(lbls == "uncertain_lowBL")),
            "total": int(lbls.size),
        }

    bl_counts = _count_labels(result["bl_labels"])
    tr_counts = _count_labels(result["tr_labels"])
    result["bl_counts"] = bl_counts
    result["tr_counts"] = tr_counts

    print(f"source ch {source_ch} | main ch {main_ch}")
    print(f"masked samples used = {n_mask}")
    print("\nDecision parameters:")
    for k, v in result["params"].items():
        print(f"  {k}: {v}")

    print("\nBL assignments:")
    print(f"  LH                 : {bl_counts['LH']} / {bl_counts['total']}")
    print(f"  soup               : {bl_counts['soup']} / {bl_counts['total']}")
    print(f"  uncertain_boundary : {bl_counts['uncertain_boundary']} / {bl_counts['total']}")
    print(f"  uncertain_lowBL    : {bl_counts['uncertain_lowBL']} / {bl_counts['total']}")

    print("\nTR assignments:")
    print(f"  LH                 : {tr_counts['LH']} / {tr_counts['total']}")
    print(f"  soup               : {tr_counts['soup']} / {tr_counts['total']}")
    print(f"  uncertain_boundary : {tr_counts['uncertain_boundary']} / {tr_counts['total']}")
    print(f"  uncertain_lowBL    : {tr_counts['uncertain_lowBL']} / {tr_counts['total']}")

    return result


def query_support_decision(decision_out, group_label, spike_idx, plot=True):
    """
    Print all decision numbers for one spike and optionally plot support curves.
    group_label : "BL" or "TR"
    """
    group_label = str(group_label).upper()
    if group_label not in ("BL", "TR"):
        raise ValueError("group_label must be 'BL' or 'TR'.")

    metrics_list = decision_out["bl_metrics"] if group_label == "BL" else decision_out["tr_metrics"]
    if spike_idx < 0 or spike_idx >= len(metrics_list):
        raise IndexError(f"{group_label} spike_idx {spike_idx} out of range 0..{len(metrics_list)-1}")

    m = metrics_list[int(spike_idx)]

    print(f"{group_label} spike idx {int(spike_idx)}")
    print(f"Assigned label: {m['label']}")
    print(f"k_peak used: {m['k_peak_used']}")
    print(f"k_bulk used: {m['k_bulk_used']}")
    print(f"BL_peak : {m['BL_peak']:.4f}")
    print(f"TR_peak : {m['TR_peak']:.4f}")
    print(f"BL_bulk : {m['BL_bulk']:.4f}")
    print(f"TR_bulk : {m['TR_bulk']:.4f}")
    print(f"D_peak  : {m['D_peak']:+.4f}")
    print(f"D_bulk  : {m['D_bulk']:+.4f}")
    print(f"kmax    : {m['kmax']}")
    p = decision_out["params"]
    diag_eps = float(p["DIAG_EPS"])
    min_bl_bulk = float(p["MIN_BL_BULK"])

    print(f"DIAG_EPS: {diag_eps:.4f}")
    print(f"near diagonal? {abs(m['D_bulk']) <= diag_eps}")
    print(f"BL strong enough for LH? {m['BL_bulk'] >= min_bl_bulk}")

    if plot:
        title = f"{group_label} spike idx {int(spike_idx)} | assigned {m['label']}"
        _plot_query_curves(m, title_prefix=title)

    return m


# ============================================================
# RUN
# ============================================================

decision_out = compute_bl_tr_support_decisions(
    out,
    cos_mask_adc=COS_MASK_ADC,
    k_peak=K_PEAK,
    k_bulk=K_BULK,
    min_bl_bulk=MIN_BL_BULK,
    min_tr_bulk=MIN_TR_BULK,
    bulk_margin=BULK_MARGIN
)

# Example query:
# m = query_support_decision(decision_out, "BL", 17, plot=QUERY_PLOT)
# m = query_support_decision(decision_out, "TR", 42, plot=QUERY_PLOT)

In [ ]:
m = query_support_decision(decision_out, "TR", 412, plot=QUERY_PLOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_spike_support_profiles(out, group_label, spike_idx, cos_mask_adc=30.0):
    """
    For one spike (BL or TR), compare support from BL cloud vs TR cloud.

    Uses:
      - all 12 stored channels concatenated
      - mask includes points where |median BL| >= cos_mask_adc OR |median TR| >= cos_mask_adc

    Plots:
      1) sorted cosine curves to BL and TR sets
      2) top-k mean support curves
      3) BL minus TR top-k mean difference
    """
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None.")

    group_label = str(group_label).upper()
    if group_label not in ("BL", "TR"):
        raise ValueError("group_label must be 'BL' or 'TR'.")

    top12 = np.asarray(gallery["top12"], dtype=np.int32)
    source_ch = int(gallery.get("source_ch", -1))
    main_ch = int(gallery.get("main_ch", -1))

    med_bl = np.asarray(gallery["med_bottom_left"], dtype=np.float32)   # [12, L]
    med_tr = gallery["med_top_right"]
    if med_tr is None:
        raise ValueError("No TR median stored in spike_gallery_data.")
    med_tr = np.asarray(med_tr, dtype=np.float32)                       # [12, L]

    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)     # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    if sn_tr is None:
        raise ValueError("No TR snippets stored in spike_gallery_data.")
    sn_tr = np.asarray(sn_tr, dtype=np.float32)                         # [12, L, Ntr]

    def _flatten_masked_snips(snips_12_l_n, mask_12_l):
        X = snips_12_l_n.transpose(2, 0, 1).reshape(snips_12_l_n.shape[2], -1)
        return X[:, mask_12_l.ravel()].astype(np.float32)

    def _row_normalize(X, eps=1e-12):
        nrm = np.linalg.norm(X, axis=1, keepdims=True)
        return X / np.maximum(nrm, eps)

    # union mask over BL/TR medians
    mask = (np.abs(med_bl) >= float(cos_mask_adc)) | (np.abs(med_tr) >= float(cos_mask_adc))
    n_mask = int(mask.sum())
    if n_mask == 0:
        raise ValueError(f"Mask is empty for cos_mask_adc={cos_mask_adc}.")

    # flatten + normalize all spikes in both groups
    X_bl = _flatten_masked_snips(sn_bl, mask)
    X_tr = _flatten_masked_snips(sn_tr, mask)

    X_bl_n = _row_normalize(X_bl)
    X_tr_n = _row_normalize(X_tr)

    # choose candidate spike
    if group_label == "BL":
        if spike_idx < 0 or spike_idx >= X_bl_n.shape[0]:
            raise IndexError(f"BL spike_idx {spike_idx} out of range 0..{X_bl_n.shape[0]-1}")
        v = X_bl_n[int(spike_idx)]
        cos_bl = X_bl_n @ v
        cos_tr = X_tr_n @ v
        cos_bl = cos_bl.copy()
        cos_bl[int(spike_idx)] = np.nan   # remove self-match
        amp_self = float(np.max(np.abs(sn_bl[np.where(top12 == main_ch)[0][0] if np.any(top12 == main_ch) else 0, :, int(spike_idx)])))
    else:
        if spike_idx < 0 or spike_idx >= X_tr_n.shape[0]:
            raise IndexError(f"TR spike_idx {spike_idx} out of range 0..{X_tr_n.shape[0]-1}")
        v = X_tr_n[int(spike_idx)]
        cos_bl = X_bl_n @ v
        cos_tr = X_tr_n @ v
        cos_tr = cos_tr.copy()
        cos_tr[int(spike_idx)] = np.nan   # remove self-match
        amp_self = float(np.max(np.abs(sn_tr[np.where(top12 == main_ch)[0][0] if np.any(top12 == main_ch) else 0, :, int(spike_idx)])))

    # drop nan self-match and sort descending
    cos_bl_valid = cos_bl[np.isfinite(cos_bl)]
    cos_tr_valid = cos_tr[np.isfinite(cos_tr)]

    bl_sorted = np.sort(cos_bl_valid)[::-1]
    tr_sorted = np.sort(cos_tr_valid)[::-1]

    # top-k mean curves
    bl_cummean = np.cumsum(bl_sorted) / np.arange(1, bl_sorted.size + 1)
    tr_cummean = np.cumsum(tr_sorted) / np.arange(1, tr_sorted.size + 1)

    kmax = min(bl_cummean.size, tr_cummean.size)
    ks = np.arange(1, kmax + 1)
    diff = bl_cummean[:kmax] - tr_cummean[:kmax]

    # some reference K values for eyeballing
    ref_ks = [5, 10, 20, 50, 100, 200, 500, 1000]
    ref_ks = [k for k in ref_ks if k <= kmax]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

    # 1) sorted cosine curves
    ax = axes[0]
    ax.plot(np.arange(1, bl_sorted.size + 1), bl_sorted, color="red", lw=1.5, label="to BL")
    ax.plot(np.arange(1, tr_sorted.size + 1), tr_sorted, color="purple", lw=1.5, label="to TR")
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("Sorted cosine support")
    ax.set_xlabel("Neighbor rank")
    ax.set_ylabel("Cosine")
    ax.set_xlim(1, max(bl_sorted.size, tr_sorted.size))
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    # 2) top-k mean curves
    ax = axes[1]
    ax.plot(ks, bl_cummean[:kmax], color="red", lw=1.8, label="BL top-k mean")
    ax.plot(ks, tr_cummean[:kmax], color="purple", lw=1.8, label="TR top-k mean")
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("Top-k mean support")
    ax.set_xlabel("k")
    ax.set_ylabel("Mean cosine of top-k neighbors")
    ax.set_xlim(1, kmax)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    # 3) difference curve
    ax = axes[2]
    ax.plot(ks, diff, color="black", lw=1.8)
    ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
    for k in ref_ks:
        ax.axvline(k, color="k", lw=0.6, alpha=0.15)
    ax.set_title("BL minus TR support")
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k mean difference")
    ax.set_xlim(1, kmax)
    ax.grid(alpha=0.25)

    # print a few reference values
    print(f"{group_label} spike idx {int(spike_idx)} | source ch {source_ch} | main ch {main_ch}")
    print(f"main-channel snippet amplitude = {amp_self:.2f}")
    print(f"masked samples used = {n_mask}")
    print("Reference top-k means:")
    for k in ref_ks:
        print(
            f"  k={k:4d}: "
            f"BL={bl_cummean[k-1]:.4f}, "
            f"TR={tr_cummean[k-1]:.4f}, "
            f"BL-TR={diff[k-1]:+.4f}"
        )

    fig.suptitle(
        f"{group_label} spike idx {int(spike_idx)} | support profiles in concatenated 12-channel space",
        y=0.98
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_one_bl_tr_spike_overlay(out, group_label, spike_idx):
    """
    Plot one spike from BL or TR with 12 channel subplots.

    Overlays per subplot:
      - current spike: black
      - BL median: red
      - TR median: purple

    Parameters
    ----------
    out : dict
        Output from inspect_lighthouse_channel(...)
    group_label : str
        "BL" or "TR"
    spike_idx : int
        Index inside the BL or TR set
    """
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None.")

    group_label = str(group_label).upper()
    if group_label not in ("BL", "TR"):
        raise ValueError("group_label must be 'BL' or 'TR'.")

    top12 = np.asarray(gallery["top12"], dtype=np.int32)
    discrim_channels = np.asarray(gallery["discrim_channels"], dtype=np.int32)
    discrim_weights = np.asarray(gallery["discrim_weights"], dtype=np.float32)
    t_ms = np.asarray(gallery["t_ms"], dtype=np.float32)

    med_bl = np.asarray(gallery["med_bottom_left"], dtype=np.float32)   # [12, L]
    med_tr = gallery["med_top_right"]
    if med_tr is None:
        raise ValueError("No TR median stored in spike_gallery_data.")
    med_tr = np.asarray(med_tr, dtype=np.float32)                       # [12, L]

    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)     # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    if sn_tr is None:
        raise ValueError("No TR snippets stored in spike_gallery_data.")
    sn_tr = np.asarray(sn_tr, dtype=np.float32)                         # [12, L, Ntr]

    source_ch = int(gallery.get("source_ch", -1))
    main_ch = int(gallery.get("main_ch", -1))

    discrim_set = set(int(x) for x in discrim_channels.tolist())
    discrim_weight_lookup = {
        int(discrim_channels[j]): float(discrim_weights[j])
        for j in range(discrim_channels.size)
    }

    if group_label == "BL":
        if spike_idx < 0 or spike_idx >= sn_bl.shape[2]:
            raise IndexError(f"BL spike_idx {spike_idx} out of range 0..{sn_bl.shape[2]-1}")
        raw_snip = sn_bl[:, :, int(spike_idx)]   # [12, L]
    else:
        if spike_idx < 0 or spike_idx >= sn_tr.shape[2]:
            raise IndexError(f"TR spike_idx {spike_idx} out of range 0..{sn_tr.shape[2]-1}")
        raw_snip = sn_tr[:, :, int(spike_idx)]   # [12, L]

    fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
    axes = axes.ravel()

    for i in range(min(12, top12.size)):
        ax = axes[i]
        ch_i = int(top12[i])

        # current spike
        ax.plot(
            t_ms, raw_snip[i],
            color="black", lw=1.8,
            label="current spike" if i == 0 else None,
            zorder=4
        )

        # BL median
        ax.plot(
            t_ms, med_bl[i],
            color="tab:red", lw=1.3,
            label="BL median" if i == 0 else None,
            zorder=2
        )

        # TR median
        ax.plot(
            t_ms, med_tr[i],
            color="purple", lw=1.3,
            label="TR median" if i == 0 else None,
            zorder=2
        )

        ax.axvline(0.0, color="k", lw=0.5, alpha=0.25)
        ax.axhline(0.0, color="k", lw=0.5, alpha=0.15)
        ax.grid(alpha=0.20)

        if ch_i in discrim_set:
            title = f"ch {ch_i} * | w={discrim_weight_lookup[ch_i]:.3f}"
        else:
            title = f"ch {ch_i}"
        ax.set_title(title, fontsize=10)

    for j in range(12, len(axes)):
        axes[j].axis("off")

    axes[0].legend(loc="best", frameon=False)

    fig.suptitle(
        f"source ch {source_ch} | main ch {main_ch} | {group_label} spike idx {int(spike_idx)}",
        y=0.98
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_cloudscore_vs_tlpeak_amplitude(out):
    gallery = out.get("spike_gallery_data", None)
    if gallery is None:
        raise ValueError("out['spike_gallery_data'] is missing or None.")

    discrim_channels = np.asarray(gallery["discrim_channels"], dtype=np.int32)   # [n_good]
    discrim_idx = np.asarray(gallery["discrim_idx"], dtype=np.int32)             # indices into top12

    med_tl = np.asarray(gallery["med_top_left"], dtype=np.float32)               # [12, L]
    sn_bl = np.asarray(gallery["sn_bottom_left"], dtype=np.float32)              # [12, L, Nbl]
    sn_tr = gallery["sn_top_right"]
    if sn_tr is None:
        raise ValueError("No TR snippets stored in spike_gallery_data.")
    sn_tr = np.asarray(sn_tr, dtype=np.float32)                                  # [12, L, Ntr]

    cloud_scores_bl = np.asarray(gallery["cloud_scores_bl"], dtype=np.float32)   # [n_good, Nbl]
    cloud_scores_tr = np.asarray(gallery["cloud_scores_tr"], dtype=np.float32)   # [n_good, Ntr]

    n_good = discrim_channels.size
    if n_good == 0:
        raise ValueError("No discriminative channels stored.")

    fig, axes = plt.subplots(
        1, n_good,
        figsize=(4.8 * n_good, 4.8),
        sharey=True,
        squeeze=False
    )
    axes = axes.ravel()

    for jj in range(n_good):
        ch = int(discrim_channels[jj])
        i12 = int(discrim_idx[jj])   # corresponding index inside the 12-channel arrays

        # time point where TL median has largest absolute amplitude on this channel
        t_peak = int(np.argmax(np.abs(med_tl[i12])))

        # signed amplitude at that exact sample for every spike
        amp_bl = sn_bl[i12, t_peak, :].astype(np.float32)   # [Nbl]
        amp_tr = sn_tr[i12, t_peak, :].astype(np.float32)   # [Ntr]

        cos_bl = cloud_scores_bl[jj]   # [Nbl]
        cos_tr = cloud_scores_tr[jj]   # [Ntr]

        ax = axes[jj]
        ax.scatter(amp_bl, cos_bl, c="red", s=10, alpha=0.40, label="BL")
        ax.scatter(amp_tr, cos_tr, c="purple", s=10, alpha=0.40, label="TR")

        ax.axhline(0.0, color="k", lw=0.6, alpha=0.25)
        ax.axvline(float(med_tl[i12, t_peak]), color="tab:blue", lw=1.0, alpha=0.8)

        ax.grid(alpha=0.25)
        ax.set_title(
            f"ch {ch} | TL peak sample {t_peak}\nTL median @peak = {med_tl[i12, t_peak]:.1f}"
        )
        ax.set_xlabel("Amplitude at TL-peak sample")
        if jj == 0:
            ax.set_ylabel("BL-cloud score")

    axes[0].legend(loc="best", frameon=False)
    fig.suptitle(
        f"BL and TR overlaid: cloud score vs amplitude at TL-median peak sample | source ch {gallery.get('source_ch', '?')}",
        y=0.98
    )
    plt.tight_layout()
    plt.show()


plot_cloudscore_vs_tlpeak_amplitude(out)

### Check k-means for AB shard vs 2 cells

In [ ]:
# ==========================
# Containment test on KMeans EIs (AB shard detector)
# ==========================
# Interprets two clusters as potentially:
#   (A) vs (A+B)  -> asymmetric containment
#   amplitude split/drift -> symmetric containment + tiny residual
#   two distinct units -> neither contains the other

assert out is not None and out.get("kmeans") is not None, "Need out['kmeans'] with stored EIs (re-run inspect cell after patch)."
km = out["kmeans"]
assert "ei_c0" in km and "ei_c1" in km, "kmeans EIs not stored yet."

ei0 = np.asarray(km["ei_c0"], dtype=np.float32)
ei1 = np.asarray(km["ei_c1"], dtype=np.float32)
n0  = int(km.get("n0", np.sum(km.get("labels", np.zeros(1)) == 0)))
n1  = int(km.get("n1", np.sum(km.get("labels", np.zeros(1)) == 1)))

# ---- knobs ----
MAX_LAG = 3
SUPPORT_REL = 0.1      # support channels = p2p >= SUPPORT_REL * max_p2p
SUPPORT_ABS = 30.0      # ... and >= SUPPORT_ABS (ADC)
TIME_KEEP_REL = 0.10    # keep timepoints where max|X| on support >= TIME_KEEP_REL * peak
FRAC_IN_THR = 0.20      # "contained" requires low mismatch on support
OUT_IN_RATIO_THR = 2.0  # "extra mostly outside" requires E_out / E_in >= this
RESID_FRAC_MIN = 0.08   # to call AB (not drift), require residual not tiny
# ---- shared core flag (keep/track regardless of subtype) ----
SHARED_COS_THR   = 0.95
SHARED_ALPHA_THR = 0.95
DO_PLOT = True

# ---- helpers ----
def shift_ei(ei, lag):
    """Shift along time axis by 'lag' samples. lag>0 moves waveform later (to the right)."""
    if lag == 0:
        return ei
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :-lag]
    else:
        out[:, :lag] = ei[:, -lag:]
    return out

def support_from_ei(ei):
    p2p = np.ptp(ei, axis=1)
    thr = max(float(SUPPORT_ABS), float(SUPPORT_REL) * float(p2p.max()))
    S = p2p >= thr
    return S, p2p, thr

def best_lag_on_support(X, Y, S):
    # time mask from X on support
    Xs = X[S, :]
    env = np.max(np.abs(Xs), axis=0) if Xs.size else np.max(np.abs(X), axis=0)
    tthr = float(TIME_KEEP_REL) * float(env.max() + 1e-12)
    T = env >= tthr
    if not np.any(T):
        T = np.ones(X.shape[1], dtype=bool)

    # brute-force lag search
    best = dict(lag=0, cos=-np.inf)
    A = X[S, :][:, T].ravel()
    nA = np.linalg.norm(A) + 1e-12
    for lag in range(-MAX_LAG, MAX_LAG + 1):
        Ys = shift_ei(Y, lag)
        B = Ys[S, :][:, T].ravel()
        nB = np.linalg.norm(B) + 1e-12
        cos = float((A @ B) / (nA * nB))
        if cos > best["cos"]:
            best = dict(lag=int(lag), cos=cos, T=T, tthr=tthr)
    return best

def containment_metrics(X, Y):
    # define X support
    S, p2pX, thr = support_from_ei(X)

    # align Y to X by maximizing cosine on X-support
    best = best_lag_on_support(X, Y, S)
    lag = best["lag"]
    Yal = shift_ei(Y, lag)

    # time mask used for fitting (same mask used for lag search)
    T = best["T"]

    # fit scale alpha on support+time mask
    A = X[S, :][:, T].ravel()
    B = Yal[S, :][:, T].ravel()
    denom = float(A @ A) + 1e-12
    alpha = float((A @ B) / denom)

    # residual
    R = Yal - alpha * X

    # energies
    Yin = Yal[S, :]
    Rin = R[S, :]
    Yout = Yal[~S, :]
    Rout = R[~S, :]

    Ein = float(np.linalg.norm(Rin))
    Eout = float(np.linalg.norm(Rout))
    frac_in = float(np.linalg.norm(Rin) / (np.linalg.norm(Yin) + 1e-12))
    frac_all = float(np.linalg.norm(R) / (np.linalg.norm(Yal) + 1e-12))
    out_in = float(Eout / (Ein + 1e-12))

    # residual support footprint
    SR, p2pR, thrR = support_from_ei(R)
    jacc = float((np.sum(S & SR)) / (np.sum(S | SR) + 1e-12))

    return dict(
        lag=lag, cos_on_support=float(best["cos"]),
        alpha=alpha,
        support_thr=float(thr), support_n=int(np.sum(S)),
        frac_in=frac_in, frac_all=frac_all, out_in=out_in,
        resid_support_thr=float(thrR), resid_support_n=int(np.sum(SR)),
        support_resid_jacc=jacc,
        X_support=S, resid_support=SR, Y_aligned=Yal, resid=R
    )

# ---- run both directions ----
m01 = containment_metrics(ei0, ei1)  # "ei0 contained in ei1?"
m10 = containment_metrics(ei1, ei0)  # "ei1 contained in ei0?"



shared01 = (m01["cos_on_support"] >= SHARED_COS_THR) and (m01["alpha"] >= SHARED_ALPHA_THR)
shared10 = (m10["cos_on_support"] >= SHARED_COS_THR) and (m10["alpha"] >= SHARED_ALPHA_THR)
shared_core = bool(shared01 or shared10)

out["kmeans"]["shared_core"] = shared_core
out["kmeans"]["shared_core_dirs"] = dict(ei0_to_ei1=bool(shared01), ei1_to_ei0=bool(shared10))
out["kmeans"]["shared_core_criteria"] = dict(cos_thr=float(SHARED_COS_THR), alpha_thr=float(SHARED_ALPHA_THR))

print("\nSHARED_CORE:", shared_core, "| dirs:", out["kmeans"]["shared_core_dirs"])

def pretty(label, m):
    print(f"\n{label}")
    print(f"  lag={m['lag']:+d}  cos_on_support={m['cos_on_support']:.3f}  alpha={m['alpha']:.3f}")
    print(f"  support_n={m['support_n']}  support_thr={m['support_thr']:.1f} ADC")
    print(f"  frac_in={m['frac_in']:.3f}  frac_all={m['frac_all']:.3f}  out/in={m['out_in']:.2f}")
    print(f"  resid_support_n={m['resid_support_n']}  resid_thr={m['resid_support_thr']:.1f}  Jacc(support,resid)={m['support_resid_jacc']:.3f}")

print(f"KMeans sizes: n0={n0}, n1={n1}")
pretty("Containment test: ei0 -> ei1", m01)
pretty("Containment test: ei1 -> ei0", m10)

# ---- crude verdict ----
def is_contained(m):
    return (m["frac_in"] <= FRAC_IN_THR) and (m["out_in"] >= OUT_IN_RATIO_THR)

c01 = is_contained(m01)
c10 = is_contained(m10)

# ---- verdict logic (your desired policy) ----
if shared_core and c01 and c10 and (m01["frac_all"] < RESID_FRAC_MIN) and (m10["frac_all"] < RESID_FRAC_MIN):
    verdict = "SAME UNIT split (amplitude/drift): symmetric containment + tiny residual"
elif shared_core and ((c01 and not c10) or (c10 and not c01)):
    verdict = "AB-SHARD-like: asymmetric containment (shared core)"
elif shared_core:
    # This bucket includes: AA/doublets, heavy-overlap AB, and other shared-core-but-not-contained cases.
    # Per your policy: we still proceed and keep all spikes.
    verdict = "SHARED-CORE (overlap/AA/complex): proceed, keep all spikes"
elif (not c01) and (not c10):
    verdict = "TWO-UNITS-like: no shared core and no containment -> reject channel"
else:
    verdict = "AMBIGUOUS: no shared core but partial containment -> reject channel"

proceed = verdict.startswith(("SAME UNIT", "AB-SHARD", "SHARED-CORE"))

out["kmeans"]["verdict"] = verdict
out["kmeans"]["proceed"] = bool(proceed)
out["kmeans"]["keep_all_spikes"] = bool(proceed)   # matches your stated policy

print("\nSHARED_CORE:", shared_core, "| dirs:", out["kmeans"]["shared_core_dirs"])
print("VERDICT:", verdict)
print("PROCEED:", proceed, "| keep_all_spikes:", out["kmeans"]["keep_all_spikes"])

# ---- optional plots ----
if DO_PLOT:
    try:
        fig = plt.figure(figsize=(20, 8))
        ax = fig.add_subplot(111)
        # plot larger cluster in black, smaller in red, and residual (if AB) in blue
        # choose direction that looked most AB-like (one-way containment)
        if (c01 and not c10):
            X = ei0; Y = m01["Y_aligned"]; R = m01["resid"]
            ttl = "Plot: ei0 (black) vs aligned ei1 (red) and residual (blue) | direction 0->1"
        elif (c10 and not c01):
            X = ei1; Y = m10["Y_aligned"]; R = m10["resid"]
            ttl = "Plot: ei1 (black) vs aligned ei0 (red) and residual (blue) | direction 1->0"
        else:
            X = ei0; Y = m01["Y_aligned"]; R = m01["resid"]
            ttl = "Plot: ei0 (black) vs aligned ei1 (red) and residual (blue)"

        pew.plot_ei_waveforms(
            [X, Y, R], positions=ei_positions, scale=70.0, ax=ax,
            colors=["black", "red", "blue"], alpha=1.0, linewidth=1.0,
            box_height=1.0, box_width=50.0, aspect=0.5,
        )
        ax.set_title(ttl)
        plt.show()
    except Exception as e:
        print("Plot failed:", type(e).__name__, e)

### fork: examine 2-cell LH

In [ ]:
# ---- settings ----
BIN_MS = 0.1
MAX_MS = 10.0

# ---- inputs ----
left_times = np.asarray(step1.get("left_times", []), dtype=np.int64)
left_times = np.sort(left_times)

sr = float(globals().get("sample_rate_hz", 20_000))

if left_times.size < 2:
    print("Not enough LEFT times to compute ISI histogram.")
else:
    isi_ms = np.diff(left_times) / sr * 1000.0

    edges = np.arange(0.0, MAX_MS + BIN_MS, BIN_MS)
    counts, _ = np.histogram(isi_ms, bins=edges)
    centers = edges[:-1] + 0.5 * BIN_MS

    plt.figure(figsize=(12, 3))
    plt.plot(centers, counts, lw=1.5)
    plt.xlabel("ISI (ms)")
    plt.ylabel("Count")
    plt.title(f"LEFT ISI histogram | bin={BIN_MS} ms | 0–{MAX_MS} ms | Nspikes={left_times.size} | Nisi={isi_ms.size}")
    plt.xlim(0, MAX_MS)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
ch = 53
trace = raw_mod
win = (-40, 80)
pre, post = win
center = -pre  # index of t within the snippet

# IMPORTANT: recompute step1 for THIS channel on THIS trace
step1 = find_valley_and_times(
    trace, ch,
    window=win, start=0, stop=None,
    bin_width=10.0, valley_bins=5,
    min_valid_count=500,
    ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10
)

left_times = np.asarray(step1["left_times"], dtype=np.int64)
left_vals  = np.asarray(step1["left_vals"],  dtype=np.float32)

print(f"[CH {ch}] step1 LEFT count: {left_times.size}")
print(f"[CH {ch}] valley_low={step1.get('valley_low')} valley_high={step1.get('valley_high')} accepted={step1.get('accepted')}")

# sample a subset
rng = np.random.RandomState(0)
if left_times.size < 10:
    raise RuntimeError("Too few LEFT spikes to sanity-check.")

pick = left_times if left_times.size <= 200 else np.sort(rng.choice(left_times, 200, replace=False))

# extract snippets on THIS channel only
sn, tv = extract_snippets_fast_ram(trace, pick, window=win, selected_channels=np.asarray([ch], dtype=np.int32))
sn = sn[0]  # (L, N)
tv = np.asarray(tv, dtype=np.int64)

# build left_val lookup by time
lv = {int(t): float(v) for t, v in zip(left_times, left_vals)}
vals_step1 = np.asarray([lv[int(t)] for t in tv], dtype=np.float32)

vals_trace = trace[tv, ch].astype(np.float32)
vals_snip_center = sn[center, :].astype(np.float32)

d1 = vals_snip_center - vals_trace
d2 = vals_snip_center - vals_step1

print(f"[CH {ch}] center sample index in snippet: {center}")
print(f"[CH {ch}] |center - trace[t,ch]|: max={np.max(np.abs(d1)):.6f}  mean={np.mean(np.abs(d1)):.6f}")
print(f"[CH {ch}] |center - step1_left_vals|: max={np.max(np.abs(d2)):.6f}  mean={np.mean(np.abs(d2)):.6f}")

# where is the minimum inside the snippet?
argmin = np.argmin(sn, axis=0)
print(f"[CH {ch}] argmin location (samples): median={int(np.median(argmin))}  "
      f"% at center={100*np.mean(argmin==center):.1f}%  "
      f"% within ±2 of center={100*np.mean(np.abs(argmin-center)<=2):.1f}%")

# plot a handful of waveforms for eyeballing
plt.figure(figsize=(10,3))
for i in range(min(50, sn.shape[1])):
    plt.plot(sn[:, i], alpha=0.25, lw=1)
plt.axvline(center, ls="--", lw=1)
plt.title(f"CH {ch} snippets (LEFT) | n={sn.shape[1]} | dashed = extraction center")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# --------------------
# SETTINGS (match your notebook defaults)
# --------------------
ch = 53
trace = raw_mod          # or raw_orig
win = (-40, 80)

SUBSAMPLE_MAX = 5000
RMS_THRESH = 5.0
MIN_CH = 30
MAX_CH = 80
N_PC = 10
RNG = np.random.RandomState(123)

SUPPORT_REL = 0.20
SUPPORT_ABS = 50.0

# amplitude band (in percentiles) to take for k-means. Cutting MAX at 30% ensures no colliding waveforms?
CUTOFF_MAX = 30
CUTOFF_MIN = 90

assert "step1" in globals(), "Need step1 for THIS channel already computed."
assert "extract_snippets_fast_ram" in globals()
assert "pew" in globals()
assert "ei_positions" in globals()
assert "cosine_two_eis" in globals(), "Need cosine_two_eis from joint_utils (already in your notebook imports)."

T, C = trace.shape

def safe_times(times, win, T):
    times = np.asarray(times, dtype=np.int64)
    times = np.sort(np.unique(times))
    pre, post = win
    ok = (times + pre >= 0) & (times + post < T)
    return times[ok]

def mean_ei_channel_rms_blockwise(trace, times, win, ch_block=128):
    times = safe_times(times, win, trace.shape[0])
    rms = np.zeros(512, dtype=np.float32)
    for c0 in range(0, 512, ch_block):
        chs = np.arange(c0, min(512, c0 + ch_block), dtype=np.int32)
        sn, tv = extract_snippets_fast_ram(trace, times, window=win, selected_channels=chs)  # (K,L,N)
        if sn.shape[2] == 0:
            continue
        m = sn.astype(np.float32).mean(axis=2)  # (K,L)
        rms[chs] = np.sqrt(np.mean(m**2, axis=1)).astype(np.float32)
    return rms

def build_mean_ei_full_blockwise(trace, times, win, ch_block=128):
    pre, post = win
    L = post - pre + 1
    times = safe_times(times, win, trace.shape[0])
    EI = np.zeros((512, L), dtype=np.float32)
    for c0 in range(0, 512, ch_block):
        chs = np.arange(c0, min(512, c0 + ch_block), dtype=np.int32)
        sn, tv = extract_snippets_fast_ram(trace, times, window=win, selected_channels=chs)
        if sn.shape[2] == 0:
            continue
        EI[chs, :] = sn.astype(np.float32).mean(axis=2)
    return EI

def support_mask(ei, rel=0.2, abs_thr=50.0):
    p2p = np.ptp(ei, axis=1)
    thr = max(float(abs_thr), float(rel) * float(p2p.max() + 1e-12))
    return p2p >= thr, thr

def cosine_on_support(eiA, eiB, mask):
    A = eiA[mask, :].ravel()
    B = eiB[mask, :].ravel()
    return float((A @ B) / ((np.linalg.norm(A) + 1e-12) * (np.linalg.norm(B) + 1e-12)))

# --------------------
# Step 0: LEFT times/vals (aligned) + bounds-safe
# --------------------
lt = np.asarray(step1.get("left_times", []), dtype=np.int64)
lv = np.asarray(step1.get("left_vals",  []), dtype=np.float32)
if lt.size == 0:
    raise RuntimeError("No LEFT spikes in step1.")

# ensure sorted and aligned
ordr = np.argsort(lt)
lt = lt[ordr]
lv = lv[ordr]

pre, post = win
ok = (lt + pre >= 0) & (lt + post < T)
lt = lt[ok]
lv = lv[ok]

print(f"[CH {ch}] LEFT total (bounds-safe) = {lt.size}")

# --------------------
# Step 1A: amplitude filter (20–80 percentile on trough magnitude)
# --------------------
amps = -lv  # positive magnitude
lo, hi = np.percentile(amps, [CUTOFF_MAX, CUTOFF_MIN])
mask_amp = (amps >= lo) & (amps <= hi)

# --------------------
# Step 1B: LEFT-only isolation filter: no other LEFT spike within ±40 samples
# (computed against ALL LEFT spikes, not just the amp-filtered set)
# --------------------
ISO = 40
dt_prev = np.full(lt.size, np.iinfo(np.int64).max, dtype=np.int64)
dt_next = np.full(lt.size, np.iinfo(np.int64).max, dtype=np.int64)
if lt.size >= 2:
    d = np.diff(lt)
    dt_prev[1:] = d
    dt_next[:-1] = d

mask_iso = (dt_prev >= ISO) & (dt_next >= ISO)

seed_times = lt[mask_amp & mask_iso]

print(f"[CH {ch}] amp band 20–80%: [{lo:.1f}, {hi:.1f}] -> {int(mask_amp.sum())}")
print(f"[CH {ch}] iso(±{ISO} samples) on LEFT -> {int(mask_iso.sum())}")
print(f"[CH {ch}] seeds (amp & iso) -> {seed_times.size}")

# --------------------
# times used for channel selection (keep ORIGINAL behavior: use broad LEFT set)
# --------------------
times_rms = lt
if times_rms.size > SUBSAMPLE_MAX:
    times_rms = np.sort(RNG.choice(times_rms, SUBSAMPLE_MAX, replace=False))
print(f"[CH {ch}] RMS channel selection using N={times_rms.size} LEFT spikes")

# --------------------
# times used for KMeans (filtered seeds)
# --------------------
times_km = seed_times
if times_km.size > SUBSAMPLE_MAX:
    times_km = np.sort(RNG.choice(times_km, SUBSAMPLE_MAX, replace=False))
print(f"[CH {ch}] KMeans using N={times_km.size} filtered LEFT seeds")

if times_km.size < 200:
    raise RuntimeError(f"Too few seeds for KMeans after filters: {times_km.size} (try widening amp band or lowering ISO)")

# --------------------
# Step 1: channel selection by RMS(mean EI) threshold (like your notebook)
# --------------------
rms = mean_ei_channel_rms_blockwise(trace, times_km, win, ch_block=128)
sel = np.flatnonzero(rms > RMS_THRESH)

# enforce min/max number of channels for stable features
if sel.size > MAX_CH:
    sel = np.argsort(rms)[-MAX_CH:]
elif sel.size < MIN_CH:
    sel = np.argsort(rms)[-MIN_CH:]
sel = np.sort(sel).astype(np.int32)

print(f"[CH {ch}] feature channels: {sel.size} (RMS>{RMS_THRESH} with min/max clamp). "
      f"Top RMS={float(np.max(rms)):.2f}, RMS@ch={float(rms[ch]):.2f}")

# --------------------
# Step 2: extract feature snippets and run PCA+KMeans (NO per-snippet mean subtraction)
# --------------------
sn, tv = extract_snippets_fast_ram(trace, times_km, window=win, selected_channels=sel)  # (Ksel,L,N)

tv = np.asarray(tv, dtype=np.int64)
Ksel, L, N = sn.shape
print(f"[CH {ch}] feature snippets: {sn.shape} valid={N}/{times_km.size}")

if N < 50:
    raise RuntimeError("Too few valid snippets after bounds check.")

X = sn.transpose(2,0,1).reshape(N, Ksel * L).astype(np.float32)

del sn
gc.collect()

n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
pca = PCA(n_components=n_pc, svd_solver="randomized", random_state=0)
Xpc = pca.fit_transform(X)
vr = pca.explained_variance_ratio_

del X
gc.collect()

km = KMeans(n_clusters=2, n_init=50, random_state=0)
lab = km.fit_predict(Xpc)

n0, n1 = int(np.sum(lab==0)), int(np.sum(lab==1))
print(f"[CH {ch}] KMeans sizes: n0={n0}, n1={n1} | PC1/2 EV%={np.round(vr[:2]*100,2)}")

plt.figure(figsize=(5,5))
plt.scatter(Xpc[lab==0,0], Xpc[lab==0,1], s=6, alpha=0.6)
plt.scatter(Xpc[lab==1,0], Xpc[lab==1,1], s=6, alpha=0.6)
plt.xlabel(f"PC1 ({vr[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({vr[1]*100:.1f}%)" if n_pc >= 2 else "PC2")
plt.title(f"CH {ch} KMeans on LEFT (no prefilter)")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

del Xpc, pca
gc.collect()

# --------------------
# Step 3: build two full EIs (mean) from cluster times (blockwise)
# --------------------
t0 = np.sort(np.unique(tv[lab==0]))
t1 = np.sort(np.unique(tv[lab==1]))

CAP_EI = 5000
if t0.size > CAP_EI: t0 = np.sort(RNG.choice(t0, CAP_EI, replace=False))
if t1.size > CAP_EI: t1 = np.sort(RNG.choice(t1, CAP_EI, replace=False))

ei0 = build_mean_ei_full_blockwise(trace, t0, win, ch_block=128)
ei1 = build_mean_ei_full_blockwise(trace, t1, win, ch_block=128)

base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
    ei0, ei1, rms_gate=5.0, use_abs=True, best_align_lag=3
)

S0, thr0 = support_mask(ei0, rel=SUPPORT_REL, abs_thr=SUPPORT_ABS)
S1, thr1 = support_mask(ei1, rel=SUPPORT_REL, abs_thr=SUPPORT_ABS)
cos_0on0 = cosine_on_support(ei0, ei1, S0)
cos_1on1 = cosine_on_support(ei1, ei0, S1)

print(f"[CH {ch}] EI cosine_two_eis={base_cos:.3f} (lag={base_lag}, nch={base_nch})")
print(f"[CH {ch}] cos(ei0 vs ei1 on ei0-support)={cos_0on0:.3f} | support0 n={int(S0.sum())} thr={thr0:.1f}")
print(f"[CH {ch}] cos(ei1 vs ei0 on ei1-support)={cos_1on1:.3f} | support1 n={int(S1.sum())} thr={thr1:.1f}")

# overlay plot
fig = plt.figure(figsize=(20, 10))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms([ei0, ei1], positions=ei_positions, ref_channel=ch,
                      scale=70.0, box_height=1.0, box_width=50.0,
                      ax=ax, colors=["black","red"], alpha=1.0, linewidth=1.0)
ax.set_title(f"CH {ch} KMeans cluster EIs | n0={t0.size} n1={t1.size} | cos={base_cos:.3f}")
plt.show()

### Build templates from LH

In [ ]:
# ---- knobs ----
N_BINS      = 5
SPIKES_PER  = 5000
EI_P2P_THR  = 50.0          # ADC p2p threshold for channel inclusion
RNG_SEED    = 123           # only used if you switch to random sampling (currently deterministic)

# expects: out, raw_mod, extract_snippets_fast_ram
# optional: sample_rate_hz, win

# ---- grab basics ----
assert out is not None and "step4" in out and out["step4"] is not None, "Need out['step4'] from inspect_lighthouse_channel"
step4 = out["step4"]

sr = float(globals().get("sample_rate_hz", 20_000))
win = globals().get("win", (-40, 80))

main_ch = int(step4.get("main_ch", 0))
final_ei = np.asarray(step4["final_ei"], dtype=np.float32)  # (C, L)
ei_p2p = np.ptp(final_ei, axis=1)

# channels to template/plot: EI p2p >= threshold
chan_sel = np.flatnonzero(ei_p2p >= EI_P2P_THR).astype(np.int32)
if chan_sel.size == 0:
    raise RuntimeError(f"No channels pass EI p2p >= {EI_P2P_THR} ADC. (max EI p2p={ei_p2p.max():.1f})")

# sort selected channels by descending EI p2p (so the important ones come first)
chan_sel = chan_sel[np.argsort(ei_p2p[chan_sel])[::-1]]

# spike times for this LH unit
times_all = np.sort(np.asarray(step4.get("final_times", step4.get("final_times_used", [])), dtype=np.int64))
if times_all.size == 0:
    raise RuntimeError("No spike times found in step4['final_times'] / ['final_times_used'].")

print(f"Template bank: main_ch={main_ch}, spikes={times_all.size}, chan_sel={chan_sel.size} (EI p2p >= {EI_P2P_THR})")

# ---- per-spike amplitude on main channel (p2p within snippet window) ----
sn_main, valid_times = extract_snippets_fast_ram(
    raw_mod, times_all, window=win, selected_channels=np.asarray([main_ch], dtype=np.int32)
)  # (1, L, Nvalid)
valid_times = np.asarray(valid_times, dtype=np.int64)
amps = np.ptp(sn_main[0, :, :], axis=0).astype(np.float32)  # (Nvalid,)

N = amps.size
if N < N_BINS * 10:
    raise RuntimeError(f"Too few valid snippets for binning: Nvalid={N}")

# ---- build templates as contiguous blocks in amplitude rank order (your scheme) ----
order_desc = np.argsort(amps)[::-1]  # biggest amp first
N = int(order_desc.size)
B = int(N_BINS)
M = int(SPIKES_PER)

bin_times = []
bin_amp_ranges = []

if N >= B * M:
    # spaced contiguous windows of length M, evenly distributed across the sorted list
    if B == 1:
        starts = np.array([0], dtype=int)
    else:
        starts = np.round(np.linspace(0, N - M, B)).astype(int)

    mode = f"spaced windows (B={B}, M={M}, using {B*M} of {N} spikes)"
    for b, s in enumerate(starts):
        idx = order_desc[s:s+M]  # contiguous block in rank-order
        tbin = valid_times[idx]
        abin = amps[idx]
        bin_times.append(tbin)
        bin_amp_ranges.append((float(np.min(abin)), float(np.max(abin))))
else:
    # not enough spikes for B*M: split into B even bins and assign every spike to one template
    mode = f"even bins (B={B}, using ALL {N} spikes; ~{N/B:.1f} per bin)"
    groups = np.array_split(order_desc, B)  # disjoint, covers all spikes
    for g in groups:
        idx = g
        tbin = valid_times[idx]
        abin = amps[idx]
        bin_times.append(tbin)
        if abin.size:
            bin_amp_ranges.append((float(np.min(abin)), float(np.max(abin))))
        else:
            bin_amp_ranges.append((np.nan, np.nan))

print(f"Template selection mode: {mode}")
print("Bin amp ranges (main-ch p2p ADC), bin 0 = highest-amp block:")
for b, (a0, a1) in enumerate(bin_amp_ranges):
    print(f"  bin {b}: {a0:.1f} .. {a1:.1f}  (n={bin_times[b].size})")

# Safety: if any bin is empty, stop early (happens only if B > N)
if any(t.size == 0 for t in bin_times):
    raise RuntimeError(f"At least one bin is empty (B={B} > N={N}). Reduce N_BINS or get more spikes.")

# ---- compute median templates for each bin on selected channels ----
templates = []  # list of (K, L) float32
for b in range(N_BINS):
    tbin = bin_times[b]
    if tbin.size == 0:
        templates.append(None)
        continue

    sn, _ = extract_snippets_fast_ram(raw_mod, tbin, window=win, selected_channels=chan_sel)
    # sn: (K, L, n)
    med = np.median(sn, axis=2).astype(np.float32)
    templates.append(med)

# stack -> (B, K, L)
if any(t is None for t in templates):
    raise RuntimeError("At least one bin ended up empty; cannot stack templates cleanly.")
templates = np.stack(templates, axis=0)

B, K, L = templates.shape
t_ms = (np.arange(L) + win[0]) / sr * 1000.0

# ---- plotting: overlay 10 bins per channel, consistent colors ----
cmap = plt.get_cmap("tab10", B)  # 10 distinct, consistent colors

ncols = 6
nrows = int(np.ceil(K / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 1.9 * nrows), sharex=True, sharey=False)
axes = np.array(axes).ravel()

for i in range(K):
    ax = axes[i]
    for b in range(B):
        ax.plot(t_ms, templates[b, i, :], lw=1.2, color=cmap(b), alpha=0.95)
    ax.axvline(0, color="k", lw=0.6, alpha=0.25)
    ax.axhline(0, color="k", lw=0.5, alpha=0.12)
    ch_i = int(chan_sel[i])
    ax.set_title(f"ch {ch_i} (EI p2p {ei_p2p[ch_i]:.0f})", fontsize=8)

for j in range(K, len(axes)):
    axes[j].axis("off")

# legend once (bin meaning = amplitude bin on main channel)
legend_lines = [plt.Line2D([0], [0], color=cmap(b), lw=2) for b in range(B)]
legend_labels = [f"bin {b} [{bin_amp_ranges[b][0]:.0f}-{bin_amp_ranges[b][1]:.0f}]" for b in range(B)]
axes[0].legend(legend_lines, legend_labels, fontsize=8, frameon=False, ncol=2, loc="upper right")

fig.suptitle(f"LH templates by main-ch amplitude bins | main_ch={main_ch} | {B} bins × ~{SPIKES_PER} spikes | chans: EI p2p≥{EI_P2P_THR}",
             y=0.995, fontsize=12)
plt.tight_layout()
plt.show()

# ---- keep the bank for later subtraction experiments ----
lh_template_bank = dict(
    main_ch=main_ch,
    win=win,
    sample_rate_hz=sr,
    channels=chan_sel,
    ei_p2p=ei_p2p[chan_sel],
    bin_amp_ranges=bin_amp_ranges,
    templates=templates,  # (10, K, L)
)

### Plot example templates subtraction

In [ ]:
# ----------------------------
# Settings
# ----------------------------
JMAX = 3                     # jitter search +/- samples
N_SHOW_PER_LEVEL = 1         # 2 high, 2 mid, 2 low
OFFSET = 200                 # spacing within each level (so you don't pick adjacent ranks)

# ----------------------------
# Grab bank + basics
# ----------------------------
assert out is not None and "step4" in out and out["step4"] is not None, "Need out['step4']"
assert "lh_template_bank" in globals(), "Need lh_template_bank dict from your template bank cell"

bank = lh_template_bank
templates = np.asarray(bank["templates"], dtype=np.float32)     # (B, K, L)
chs       = np.asarray(bank["channels"], dtype=np.int32)        # (K,)
win       = tuple(bank["win"])                                  # (start, end), inclusive
sr        = float(bank.get("sample_rate_hz", 20_000))
main_ch   = int(bank["main_ch"])

B, K, L = templates.shape
print(f"Bank: B={B} templates, K={K} channels, L={L} samples, main_ch={main_ch}, jitter=±{JMAX}")

# ----------------------------
# Choose test spikes by main-channel amplitude (p2p)
# ----------------------------
times_all = np.sort(np.asarray(out["step4"]["final_times"], dtype=np.int64))
if times_all.size < 50:
    raise RuntimeError("Too few spikes in out['step4']['final_times'].")

# Compute per-spike p2p on main channel for ALL spikes (fast: 1 channel)
sn_main, valid_times = extract_snippets_fast_ram(
    raw_mod, times_all, window=win,
    selected_channels=np.asarray([main_ch], dtype=np.int32)
)  # (1, L, Nvalid)
valid_times = np.asarray(valid_times, dtype=np.int64)

amps = np.ptp(sn_main[0, :, :], axis=0).astype(np.float32)  # (Nvalid,)
order_desc = np.argsort(amps)[::-1]  # high -> low
N = order_desc.size

print(f"Valid spikes for ranking: {N}")
print(f"Main-ch p2p amps: min={amps.min():.1f}, med={np.median(amps):.1f}, max={amps.max():.1f} ADC")

def pick_indices(N, n_each=2, offset=200):
    # 2 high, 2 mid, 2 low (or more if you change n_each)
    idx = []
    # high
    for i in range(n_each):
        idx.append(min(i * offset, N-1))
    # mid
    mid0 = N // 2
    for i in range(n_each):
        idx.append(min(mid0 + i * offset, N-1))
    # low
    for i in range(n_each):
        idx.append(max(N-1 - i * offset, 0))
    # preserve order, unique
    out_idx = []
    seen = set()
    for x in idx:
        if x not in seen:
            out_idx.append(x); seen.add(x)
    return out_idx

rank_idx = pick_indices(N, n_each=N_SHOW_PER_LEVEL, offset=OFFSET)
test_times = valid_times[order_desc[rank_idx]]
test_amps  = amps[order_desc[rank_idx]]

print("\nTest spikes (time samples, main-ch p2p amp):")
for t, a in zip(test_times, test_amps):
    print(f"  t={int(t)}  amp={a:.1f}")

# ----------------------------
# Precompute flattened template norms for cosine
# ----------------------------
tmpl_flat = templates.reshape(B, -1)  # (B, K*L)
tmpl_norm = np.linalg.norm(tmpl_flat, axis=1) + 1e-12

# Extract EXTENDED snippets once (so we can slide a window for jitter)
win_ext = (win[0] - JMAX, win[1] + JMAX)
sn_ext, valid_times2 = extract_snippets_fast_ram(
    raw_mod, test_times, window=win_ext, selected_channels=chs
)  # (K, Lext, Nvalid2)
valid_times2 = np.asarray(valid_times2, dtype=np.int64)

Lext = sn_ext.shape[1]
anchor0 = JMAX  # anchor window start inside ext snippet (corresponds to original win)
if Lext != L + 2 * JMAX:
    print(f"NOTE: Lext={Lext} but expected {L+2*JMAX} (window semantics mismatch?)")

# Map time->amp for prints (only for those test spikes)
amp_by_time = {int(t): float(a) for t, a in zip(test_times, test_amps)}

# ----------------------------
# Matching + plotting
# ----------------------------
def best_template_and_jitter(snip_ext):
    """
    snip_ext: (K, Lext) float/int
    returns: best_b, best_j, best_cos
    """
    best_cos = -np.inf
    best_b = None
    best_j = None

    # Precompute norms for each jitter segment
    for j in range(-JMAX, JMAX + 1):
        start = JMAX + j
        seg = snip_ext[:, start:start+L].astype(np.float32)  # (K, L)
        seg_flat = seg.reshape(-1)
        seg_norm = np.linalg.norm(seg_flat) + 1e-12

        # cosine vs all templates
        # dot = tmpl_flat @ seg_flat
        dots = tmpl_flat @ seg_flat
        cos  = dots / (tmpl_norm * seg_norm)

        b = int(np.argmax(cos))
        c = float(cos[b])
        if c > best_cos:
            best_cos = c
            best_b = b
            best_j = j

    return best_b, best_j, best_cos

def plot_before_after(snip_ext, b, j, spike_t):
    """
    Plot original anchor window and residual (after subtracting template b at jitter j),
    for ALL selected channels.
    """
    # build residual in EXT space, then slice anchor window
    snip_ext = snip_ext.astype(np.float32)
    res_ext = snip_ext.copy()
    start = JMAX + j
    res_ext[:, start:start+L] -= templates[b, :, :]

    orig = snip_ext[:, anchor0:anchor0+L]
    resid = res_ext[:, anchor0:anchor0+L]

    # aligned template (in the anchor window coordinates) for overlay
    fit = np.zeros((K, L), dtype=np.float32)
    if j >= 0:
        fit[:, j:] = templates[b, :, :L-j]
    else:
        fit[:, :L+j] = templates[b, :, -j:]

    t_ms = (np.arange(L) + win[0]) / sr * 1000.0

    # plot grid
    ncols = 6
    nrows = int(np.ceil(K / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 1.8*nrows), sharex=True)
    axes = np.array(axes).ravel()

    # find main_ch index in chs (for marking)
    main_idx = int(np.where(chs == main_ch)[0][0]) if np.any(chs == main_ch) else None

    for i in range(K):
        ax = axes[i]
        ax.plot(t_ms, orig[i, :],  color="black", lw=1.0,  label="orig" if i == 0 else None)
        ax.plot(t_ms, fit[i, :],   color="green", lw=0.8, alpha=0.45, label="template" if i == 0 else None)
        ax.plot(t_ms, resid[i, :], color="red",   lw=1.0,  label="resid" if i == 0 else None)
        ax.axvline(0, color="k", lw=0.5, alpha=0.25)
        ax.axhline(0, color="k", lw=0.5, alpha=0.12)

        ch_i = int(chs[i])
        if main_idx is not None and i == main_idx:
            ax.set_title(f"ch {ch_i} *MAIN*", fontsize=8)
        else:
            ax.set_title(f"ch {ch_i}", fontsize=8)

    for k in range(K, len(axes)):
        axes[k].axis("off")

    amp = amp_by_time.get(int(spike_t), np.nan)
    fig.suptitle(f"Spike t={int(spike_t)} (main p2p={amp:.1f}) | best template={b} | jitter={j} samples | black=orig red=resid",
                 y=0.995, fontsize=12)
    plt.tight_layout()
    plt.show()

print("\nMatching results:")
for n in range(sn_ext.shape[2]):
    t = int(valid_times2[n])
    snipE = sn_ext[:, :, n]
    b, j, c = best_template_and_jitter(snipE)
    print(f"  t={t}  amp={amp_by_time.get(t, np.nan):.1f}  ->  template={b}  jitter={j}  cosine={c:.4f}")
    plot_before_after(snipE, b, j, t)

### Check right spikes for cosine match

In [ ]:
# ===== knobs =====
JMAX = 3
COS_THR = 0.95     # with your masked cosine ~0.99, you can crank this up
STOP_RUN = 100
BATCH = 512

TIME_KEEP_FRAC = 0.08
WEIGHT_POWER   = 1.0
USE_CH_WEIGHT  = False

assert out is not None and out.get("step1") is not None
assert "lh_template_bank" in globals()
assert "raw_mod" in globals()

bank = lh_template_bank
templates = np.asarray(bank["templates"], dtype=np.float32)   # (B,K,L)
chs       = np.asarray(bank["channels"], dtype=np.int32)
win       = tuple(bank["win"])
B, K, L   = templates.shape

# --- stable unique that preserves order (crucial for "100 consecutive fails") ---
def stable_unique_int64(x):
    seen = set()
    out_ = []
    for v in x:
        vv = int(v)
        if vv not in seen:
            out_.append(vv)
            seen.add(vv)
    return np.asarray(out_, dtype=np.int64)

# ===== 1) candidate right list, KEEP ORDER =====
right_raw = np.asarray(out["step1"].get("rightk_times_by_amp", []), dtype=np.int64).ravel()
right_times = right_raw[np.isfinite(right_raw)]
right_times = stable_unique_int64(right_times)

print("Right candidates (ordered):", right_times.size)

# ===== 2) build masked scorer from CURRENT bank =====
ref = np.median(templates, axis=0)   # (K,L)
absref = np.abs(ref)

time_env = absref.max(axis=0)
t_thr = TIME_KEEP_FRAC * time_env.max()
keep_t = time_env >= t_thr

if USE_CH_WEIGHT:
    ch_env = np.ptp(ref, axis=1)
    ch_w = ch_env / (ch_env.max() + 1e-12)
else:
    ch_w = np.ones(K, dtype=np.float32)

W = (absref ** WEIGHT_POWER) * ch_w[:, None]
W[:, ~keep_t] = 0.0

mask_idx = np.flatnonzero(W.ravel() > 0)
w_mask   = W.ravel()[mask_idx].astype(np.float32)

tmpl_flat  = templates.reshape(B, -1).astype(np.float32)           # (B, K*L)
tmplW      = tmpl_flat[:, mask_idx] * w_mask[None, :]              # (B, Dmask)
tmplW_norm = np.linalg.norm(tmplW, axis=1) + 1e-12

print(f"Masked scoring dims: Dmask={mask_idx.size} out of D={K*L} (kept {mask_idx.size/(K*L):.2%}), time_thr={t_thr:.2f}")

# ===== 3) score right candidates in order (ON raw_mod), stop after STOP_RUN consecutive fails =====
win_ext = (win[0] - JMAX, win[1] + JMAX)

accepted_times = []
accepted_info  = []  # (t, cos, template, jitter)
consec_fail = 0
stop_hit = False

for s0 in range(0, right_times.size, BATCH):
    if stop_hit:
        break

    tb = right_times[s0:s0+BATCH]
    sn_ext, t_valid = extract_snippets_fast_ram(raw_mod, tb, window=win_ext, selected_channels=chs)
    t_valid = np.asarray(t_valid, dtype=np.int64)
    N = sn_ext.shape[2]
    if N == 0:
        continue

    Lext = sn_ext.shape[1]
    anchor = (Lext - L) // 2

    best_cos = np.full(N, -np.inf, dtype=np.float32)
    best_b   = np.zeros(N, dtype=np.int16)
    best_j   = np.zeros(N, dtype=np.int8)

    for j in range(-JMAX, JMAX+1):
        start = anchor + j
        seg = sn_ext[:, start:start+L, :].astype(np.float32)          # (K,L,N)
        if seg.shape[1] != L:
            continue

        Xfull = seg.transpose(2,0,1).reshape(N, -1)                   # (N, K*L)
        if mask_idx.max() >= Xfull.shape[1]:
            raise RuntimeError(f"mask_idx mismatch: max={mask_idx.max()} vs D={Xfull.shape[1]} (rebuild bank/scorer)")

        Xw = Xfull[:, mask_idx] * w_mask[None, :]                     # (N,Dmask)
        Xn = np.linalg.norm(Xw, axis=1) + 1e-12
        dots = Xw @ tmplW.T                                           # (N,B)
        cos  = dots / (Xn[:, None] * tmplW_norm[None, :])             # (N,B)

        b = np.argmax(cos, axis=1).astype(np.int16)
        c = cos[np.arange(N), b].astype(np.float32)

        upd = c > best_cos
        best_cos[upd] = c[upd]
        best_b[upd]   = b[upd]
        best_j[upd]   = np.int8(j)

    # walk in order, apply stop rule
    for i in range(N):
        t = int(t_valid[i])
        c = float(best_cos[i])
        if c >= COS_THR:
            consec_fail = 0
            accepted_times.append(t)
            accepted_info.append((t, c, int(best_b[i]), int(best_j[i])))
        else:
            consec_fail += 1
            if consec_fail >= STOP_RUN:
                stop_hit = True
                break

print(f"Accepted right spikes: {len(accepted_times)} (COS_THR={COS_THR}); stopped_by_{STOP_RUN}fails={stop_hit}")

# ===== 4) merge with core spikes for convenience (NOTE: we will recompute again right before subtraction) =====
core_times = np.sort(np.unique(np.asarray(out["step4"]["final_times"], dtype=np.int64)))
lh_spike_times_extended = np.sort(np.unique(np.concatenate([core_times, np.asarray(accepted_times, dtype=np.int64)])))

print("Core:", core_times.size, "Extended:", lh_spike_times_extended.size)

if len(accepted_info):
    print("Last 5 accepted (t, cos, b, j):", accepted_info[-5:])

### plot EIs - optional

In [ ]:
# ---- knobs ----
MIN_RIGHT_SPIKES = 20     # only bother plotting if at least this many right spikes
MAX_SPIKES_EI    = 1000    # cap for EI construction (median)
EI_WINDOW        = None    # if None, uses lh_template_bank["win"]

# ---- requirements ----
assert out is not None and out.get("step4") is not None, "Need out from inspect_lighthouse_channel"
assert "raw_orig" in globals(), "Need raw_orig"
assert "ei_positions" in globals(), "Need ei_positions"
assert "lh_template_bank" in globals(), "Need lh_template_bank (for window + main_ch)"

import plot_ei_waveforms as pew

def subsample_even(times, max_n):
    times = np.asarray(times, dtype=np.int64)
    times = np.sort(np.unique(times))
    if times.size <= max_n:
        return times
    idx = np.round(np.linspace(0, times.size - 1, max_n)).astype(int)
    return times[idx]

def build_median_ei(raw, times, window, C=512, max_spikes=5000):
    times = subsample_even(times, max_spikes)
    sn, t_valid = extract_snippets_fast_ram(raw, times, window=window,
                                           selected_channels=np.arange(C, dtype=np.int32))
    if sn.shape[2] == 0:
        return None, 0
    ei = np.median(sn, axis=2).astype(np.float32)  # (C, L)
    return ei, int(sn.shape[2])

# ---- identify left/core spikes and added-right spikes ----
core_times = np.asarray(out["step4"]["final_times"], dtype=np.int64)
core_times = np.sort(np.unique(core_times))

if "accepted_times" in globals():
    right_added = np.asarray(accepted_times, dtype=np.int64)
elif "right_keep_times" in globals():
    right_added = np.asarray(right_keep_times, dtype=np.int64)
elif "lh_spike_times_extended" in globals():
    right_added = np.setdiff1d(np.asarray(lh_spike_times_extended, dtype=np.int64), core_times)
else:
    raise RuntimeError("Can't find right spikes. Need accepted_times/right_keep_times or lh_spike_times_extended.")

right_added = np.sort(np.unique(right_added))

print(f"Core spikes: {core_times.size}")
print(f"Added right spikes: {right_added.size}")

if right_added.size < MIN_RIGHT_SPIKES:
    print(f"Not plotting EI overlay (right_added < {MIN_RIGHT_SPIKES}).")
else:
    win = EI_WINDOW if EI_WINDOW is not None else tuple(lh_template_bank["win"])
    main_ch = int(lh_template_bank.get("main_ch", out["step4"].get("main_ch", 0)))

    # Build EIs on raw_orig (pre-subtraction)
    ei_core, n_core = build_median_ei(raw_orig, core_times, window=win, max_spikes=MAX_SPIKES_EI)
    ei_right, n_right = build_median_ei(raw_orig, right_added, window=win, max_spikes=MAX_SPIKES_EI)

    if ei_core is None or ei_right is None:
        print("EI build failed (no valid snippets). Check bounds/window.")
    else:
        # Optional quick scalar similarity on selected channels (not required, but helpful)
        # (uses only channels already selected in lh_template_bank)
        chs = np.asarray(lh_template_bank["channels"], dtype=np.int32)
        A = ei_core[chs].reshape(-1)
        B = ei_right[chs].reshape(-1)
        cos = float((A @ B) / ((np.linalg.norm(A) + 1e-12) * (np.linalg.norm(B) + 1e-12)))
        print(f"Core vs Right EI cosine (on selected chans): {cos:.4f}")
        print(f"EI spikes used: core={n_core}, right={n_right} (cap={MAX_SPIKES_EI})")

        # Plot overlay
        fig = plt.figure(figsize=(25, 18))
        ax = fig.add_subplot(111)
        pew.plot_ei_waveforms(
            [ei_core, ei_right],
            positions=ei_positions,
            ref_channel=main_ch,
            scale=70.0,
            ax=ax,
            colors=["black", "green"],
            alpha=[1.0, 0.6],
            linewidth=[0.5, 0.5],
            box_height=1.0,
            box_width=50.0,
            aspect=0.5,
        )
        ax.set_title("EI overlay: core/left spikes (black) vs added-right spikes (green)")
        plt.show()

### check accepted right EI for similarity with core left EI

In [ ]:
# ---- knobs ----
EI_PRES_P2P_THR = 50.0     # "presence" threshold (ADC p2p)
EI_COS_THR      = 0.90
MAX_SPIKES_EI   = 5000     # cap for EI building (median)
MIN_RIGHT_SPIKES_FOR_TEST = 20  # if fewer, EI_right is too noisy to trust

assert out is not None and out.get("step4") is not None
assert "raw_mod" in globals()
assert "lh_template_bank" in globals()
assert "accepted_times" in globals(), "Need accepted_times from your right-extension step"

win = tuple(lh_template_bank["win"])

# EI_core: reuse from inspection (no recompute)
ei_core = np.asarray(out["step4"]["final_ei"], dtype=np.float32)   # (512, L)
L = ei_core.shape[1]

right_times = np.asarray(accepted_times, dtype=np.int64)
right_times = np.sort(np.unique(right_times))

print("Right spikes proposed:", right_times.size)

def subsample_even(times, max_n):
    times = np.asarray(times, dtype=np.int64)
    times = np.sort(np.unique(times))
    if times.size <= max_n:
        return times
    idx = np.round(np.linspace(0, times.size - 1, max_n)).astype(int)
    return times[idx]

def build_median_ei(raw, times, window, n_channels=512, max_spikes=5000):
    times = subsample_even(times, max_spikes)
    sn, t_valid = extract_snippets_fast_ram(
        raw, times, window=window,
        selected_channels=np.arange(n_channels, dtype=np.int32)
    )  # (C, L, N)
    if sn.shape[2] == 0:
        return None, 0
    return np.median(sn, axis=2).astype(np.float32), int(sn.shape[2])

# If too few right spikes, do NOT add them (or you can override)
if right_times.size < MIN_RIGHT_SPIKES_FOR_TEST:
    print(f"Too few right spikes for a reliable EI_right test (<{MIN_RIGHT_SPIKES_FOR_TEST}).")
    accepted_times = []   # nuke them by default
else:
    ei_right, n_used = build_median_ei(raw_mod, right_times, window=win, max_spikes=MAX_SPIKES_EI)
    if ei_right is None:
        print("EI_right build failed (bounds?) -> rejecting right spikes.")
        accepted_times = []
    else:
        p2p_core  = np.ptp(ei_core, axis=1)
        p2p_right = np.ptp(ei_right, axis=1)

        chan_union = np.flatnonzero((p2p_core >= EI_PRES_P2P_THR) | (p2p_right >= EI_PRES_P2P_THR))
        if chan_union.size == 0:
            print("No channels pass presence threshold in either EI -> rejecting right spikes.")
            accepted_times = []
        else:
            A = ei_core[chan_union, :].reshape(-1)
            B = ei_right[chan_union, :].reshape(-1)
            cos = float((A @ B) / ((np.linalg.norm(A) + 1e-12) * (np.linalg.norm(B) + 1e-12)))

            print(f"EI_right vs EI_core cosine = {cos:.4f} on {chan_union.size} union channels (p2p≥{EI_PRES_P2P_THR})")
            print(f"EI_right spikes used (cap {MAX_SPIKES_EI}): {n_used}")

            if cos < EI_COS_THR:
                print(f"Rejecting right spikes: cos<{EI_COS_THR}")
                accepted_times = []
            else:
                print(f"Keeping right spikes: cos≥{EI_COS_THR}")

### SUBTRACT TEMPLATES FROM RAW DATA (raw_mod)

In [ ]:
# =========================
# 0) knobs you can touch
# =========================
JMAX = 3
BATCH = 256

# Mask/weight settings
TIME_KEEP_FRAC = 0.08   # keep timepoints where max(|template|) >= 8% of peak (across selected channels)
WEIGHT_POWER   = 1.0    # 1.0 = abs(template), 2.0 = abs(template)^2
USE_CH_WEIGHT  = True   # weight channels by their p2p in ref template
MIN_SEP_MS = 1.0        # if two candidate spikes are closer than this, treat as duplicates and keep only best-fit one

# Rejection gate (start conservative, then tighten once you see distribution)
COS_THR = 0.9           # skip subtraction if best masked cosine < this

VERBOSE_EVERY = 2000

# =========================
# 1) inputs
# =========================
assert "lh_template_bank" in globals()
assert out is not None and out.get("step4") is not None
assert "raw_mod" in globals()

bank = lh_template_bank
templates = np.asarray(bank["templates"], dtype=np.float32)   # (B, K, L)
chs       = np.asarray(bank["channels"], dtype=np.int32)      # (K,)
win       = tuple(bank["win"])
B, K, L   = templates.shape

sample_rate_hz = float(globals().get("sample_rate_hz", bank.get("sample_rate_hz", 20_000)))

# spikes to subtract = core + (accepted_times, if present *after* EI check)
core_times = np.sort(np.unique(np.asarray(out["step4"]["final_times"], dtype=np.int64)))

if "accepted_times" in globals():
    acc = np.asarray(accepted_times, dtype=np.int64).ravel()
    acc = acc[np.isfinite(acc)]
    acc = np.sort(np.unique(acc))
    spike_times = np.sort(np.unique(np.concatenate([core_times, acc])))
    print(f"Spikes to subtract: core={core_times.size} + accepted_right={acc.size} -> total={spike_times.size}")
else:
    spike_times = core_times
    print(f"Spikes to subtract: core only={spike_times.size} (no accepted_times variable)")

# int16 templates for subtraction
templates_i16 = np.rint(templates).astype(np.int16)

# =========================
# 2) build ONE shared mask + weights (from ref template)
#    AND define main-channel reference shift (dt_main)
# =========================
ref = np.median(templates, axis=0)        # (K, L), robust average shape across bins
absref = np.abs(ref)

# time support: keep timepoints where the template is "active" somewhere
time_env = absref.max(axis=0)             # (L,)
t_thr = TIME_KEEP_FRAC * time_env.max()
keep_t = time_env >= t_thr               # (L,)

# channel weights: emphasize big channels within selected set
if USE_CH_WEIGHT:
    ch_env = np.ptp(ref, axis=1)          # (K,)
    ch_w = ch_env / (ch_env.max() + 1e-12)
else:
    ch_w = np.ones(K, dtype=np.float32)

# weights matrix W (K,L)
W = (absref ** WEIGHT_POWER) * ch_w[:, None]
W[:, ~keep_t] = 0.0

mask_idx = np.flatnonzero(W.ravel() > 0)
w_mask = W.ravel()[mask_idx].astype(np.float32)   # (Dmask,)
Dmask = mask_idx.size
print(f"Masked scoring dims: Dmask={Dmask} out of D={K*L} (kept {Dmask/(K*L):.2%} of elements), time_thr={t_thr:.2f}")

# precompute weighted flattened templates and norms
tmpl_flat_full = templates.reshape(B, -1).astype(np.float32)  # (B, D)
tmplW = tmpl_flat_full[:, mask_idx] * w_mask[None, :]         # (B, Dmask)
tmplW_norm = np.linalg.norm(tmplW, axis=1) + 1e-12

# ---- main-channel reference shift (dt_main) ----
main_ch = int(bank.get("main_ch", out["step4"].get("main_ch", 0)))
if not np.any(chs == main_ch):
    raise RuntimeError(f"main_ch={main_ch} not found in template channel set (chs).")

main_ci = int(np.where(chs == main_ch)[0][0])

center_idx = int(-win[0])   # sample index in template window that corresponds to time=0 for the input spike_times
if not (0 <= center_idx < L):
    raise RuntimeError(f"Bad center_idx={center_idx} for win={win} with L={L}")

tmin_main = int(np.argmin(ref[main_ci, :]))
dt_main = int(tmin_main - center_idx)

print(f"Main-channel timebase: main_ch={main_ch} | center_idx={center_idx} | tmin_main={tmin_main} | dt_main={dt_main} samples")

# =========================
# 3) subtraction with masked cosine + rejection gate
#    (dedup uses jitter-corrected centers t_center = t + j)
# =========================
win_ext = (win[0] - JMAX, win[1] + JMAX)

def subtract_lh_masked(raw_match, raw_mod, spike_times, min_sep_ms=MIN_SEP_MS):
    """
    2-pass version:
      Pass A: compute best template/jitter/cosine for every spike time (NO subtraction)
      Pass B: dedup spikes that are too close (<min_sep_ms) USING t_center = t + jitter
      Pass C: subtract only the kept spikes from raw_mod

    Returns:
      spike_times_center / spike_times_main are the FINAL unit spike train:
        - center: jitter-corrected time (t + j)
        - main:   main-channel trough time (t + j + dt_main)
    """
    Ntot = spike_times.size

    best_b_list = []
    best_j_list = []
    best_c_list = []
    time_list   = []

    # One-time anchor sanity
    tmp_sn, _ = extract_snippets_fast_ram(raw_match, spike_times[:1], window=win_ext, selected_channels=chs)
    if tmp_sn.shape[2] == 0:
        return dict(times=np.array([], dtype=np.int64),
                    best_template=np.array([], dtype=np.int16),
                    best_jitter=np.array([], dtype=np.int8),
                    best_cosine=np.array([], dtype=np.float32),
                    did_subtract=np.array([], dtype=bool),
                    times_center=np.array([], dtype=np.int64),
                    times_main=np.array([], dtype=np.int64),
                    spike_times_center=np.array([], dtype=np.int64),
                    spike_times_main=np.array([], dtype=np.int64),
                    dt_main=dt_main, main_ch=main_ch,
                    n_sub=0, n_skip=0,
                    n_dedup_dropped=0, n_close_clusters=0)

    Lext0 = tmp_sn.shape[1]
    anchor0 = (Lext0 - L) // 2

    # ---------- Pass A: match everything ----------
    for s0 in range(0, Ntot, BATCH):
        t_batch = spike_times[s0:s0+BATCH]

        sn_ext, t_valid = extract_snippets_fast_ram(
            raw_match, t_batch, window=win_ext, selected_channels=chs
        )  # (K, Lext, Nvalid)
        t_valid = np.asarray(t_valid, dtype=np.int64)
        N = sn_ext.shape[2]
        if N == 0:
            continue

        Lext = sn_ext.shape[1]
        anchor = (Lext - L) // 2  # should equal anchor0, but keep it robust

        best_cos = np.full(N, -np.inf, dtype=np.float32)
        best_b   = np.zeros(N, dtype=np.int16)
        best_j   = np.zeros(N, dtype=np.int8)

        for j in range(-JMAX, JMAX + 1):
            start = anchor + j
            seg = sn_ext[:, start:start+L, :].astype(np.float32)           # (K,L,N)
            Xfull = seg.transpose(2, 0, 1).reshape(N, -1)                  # (N,D)
            Xw = Xfull[:, mask_idx] * w_mask[None, :]                      # (N,Dmask)
            Xn = np.linalg.norm(Xw, axis=1) + 1e-12
            dots = Xw @ tmplW.T                                            # (N,B)
            cos = dots / (Xn[:, None] * tmplW_norm[None, :])               # (N,B)

            b = np.argmax(cos, axis=1).astype(np.int16)
            c = cos[np.arange(N), b].astype(np.float32)

            upd = c > best_cos
            best_cos[upd] = c[upd]
            best_b[upd]   = b[upd]
            best_j[upd]   = np.int8(j)

        time_list.append(t_valid)
        best_b_list.append(best_b)
        best_j_list.append(best_j)
        best_c_list.append(best_cos)

        if VERBOSE_EVERY and (s0 % VERBOSE_EVERY == 0):
            medc = float(np.median(best_cos))
            q05  = float(np.quantile(best_cos, 0.05))
            print(f"matched {min(s0+BATCH, Ntot)}/{Ntot} | median cos={medc:.3f} | 5%={q05:.3f}")

    if len(time_list) == 0:
        return dict(times=np.array([], dtype=np.int64),
                    best_template=np.array([], dtype=np.int16),
                    best_jitter=np.array([], dtype=np.int8),
                    best_cosine=np.array([], dtype=np.float32),
                    did_subtract=np.array([], dtype=bool),
                    times_center=np.array([], dtype=np.int64),
                    times_main=np.array([], dtype=np.int64),
                    spike_times_center=np.array([], dtype=np.int64),
                    spike_times_main=np.array([], dtype=np.int64),
                    dt_main=dt_main, main_ch=main_ch,
                    n_sub=0, n_skip=0,
                    n_dedup_dropped=0, n_close_clusters=0)

    used_t_all = np.concatenate(time_list).astype(np.int64)
    best_b_all = np.concatenate(best_b_list).astype(np.int16)
    best_j_all = np.concatenate(best_j_list).astype(np.int8)
    best_c_all = np.concatenate(best_c_list).astype(np.float32)

    # jitter-corrected center times (this is the timebase used for dedup + final spike train)
    t_center_all = used_t_all + best_j_all.astype(np.int64)

    did_sub = best_c_all >= COS_THR

    # ---------- Pass B: dedup too-close accepted spikes (USING t_center = t + j) ----------
    min_sep_samp = int(round((min_sep_ms / 1000.0) * sample_rate_hz))
    accept_idx = np.flatnonzero(did_sub)

    n_dedup_dropped = 0
    n_close_clusters = 0

    if accept_idx.size > 1 and min_sep_samp > 0:
        accept_sorted = accept_idx[np.argsort(t_center_all[accept_idx])]
        tacc = t_center_all[accept_sorted]

        i = 0
        while i < accept_sorted.size:
            j = i
            while (j + 1 < accept_sorted.size) and ((tacc[j + 1] - tacc[j]) < min_sep_samp):
                j += 1

            if j > i:
                n_close_clusters += 1
                cluster = accept_sorted[i:j+1]
                keep = cluster[np.argmax(best_c_all[cluster])]
                drop = cluster[cluster != keep]
                did_sub[drop] = False
                n_dedup_dropped += int(drop.size)

            i = j + 1

    if n_dedup_dropped:
        print(f"Dedup: dropped {n_dedup_dropped} accepted spikes in {n_close_clusters} close clusters (<{min_sep_ms} ms), using t_center=t+j")

    # ---------- Pass C: subtract only kept spikes ----------
    keep_idx = np.flatnonzero(did_sub)
    keep_idx = keep_idx[np.argsort(t_center_all[keep_idx])]  # chronological in t_center

    for idx in keep_idx:
        t = int(used_t_all[idx])
        b = int(best_b_all[idx])
        j = int(best_j_all[idx])

        t0 = t + win_ext[0] + (anchor0 + j)
        t1 = t0 + L

        tmpl = templates_i16[b]  # (K,L)
        for ci, ch in enumerate(chs):
            raw_mod[t0:t1, int(ch)] -= tmpl[ci, :]

    # main-channel trough timebase (canonical)
    t_main_all = t_center_all + int(dt_main)

    spike_times_center = np.sort(t_center_all[did_sub])
    spike_times_main   = np.sort(t_main_all[did_sub])

    n_sub = int(did_sub.sum())
    n_skip = int(did_sub.size - n_sub)

    return dict(times=used_t_all,
                best_template=best_b_all,
                best_jitter=best_j_all,
                best_cosine=best_c_all,
                did_subtract=did_sub,
                times_center=t_center_all,
                times_main=t_main_all,
                spike_times_center=spike_times_center,
                spike_times_main=spike_times_main,
                dt_main=int(dt_main),
                main_ch=int(main_ch),
                n_sub=n_sub,
                n_skip=n_skip,
                n_dedup_dropped=n_dedup_dropped,
                n_close_clusters=n_close_clusters)

# IMPORTANT: score against current residual (raw_mod), then subtract into raw_mod
match = subtract_lh_masked(raw_match=raw_mod, raw_mod=raw_mod, spike_times=spike_times)

print("\nDone.")
print("Valid spikes processed:", match["times"].size)
print("Subtracted:", match["n_sub"], "Skipped:", match["n_skip"], f"(COS_THR={COS_THR})")

c = match["best_cosine"]
print("Cosine stats:", float(c.min()), float(np.median(c)), float(c.max()))
print("Jitter counts:", {int(j): int(np.sum(match["best_jitter"]==j)) for j in range(-JMAX, JMAX+1)})

print("Final unit spikes (center, jitter-corrected):", match["spike_times_center"].size)
print("Final unit spikes (main-ch reference):", match["spike_times_main"].size)

# stash the canonical spike train for the unit
lh_spike_times_main = match["spike_times_main"]
out["lh_spike_times_main"] = lh_spike_times_main
out["lh_subtraction_match"] = match

# ---- counts by template (among subtracted spikes) ----
bt  = match["best_template"]
sub = match["did_subtract"]

counts_sub = np.bincount(bt[sub], minlength=B)
counts_all = np.bincount(bt, minlength=B)

print("\nTemplate assignment counts:")
den = max(int(sub.sum()), 1)
for b in range(B):
    a0, a1 = bank.get("bin_amp_ranges", [(np.nan, np.nan)]*B)[b]
    print(f"  template {b:2d}: {counts_sub[b]:5d} spikes  ({counts_sub[b]/den*100:5.1f}%)"
          f"   main-ch p2p bin ~[{a0:.0f}, {a1:.0f}]")

print("Dedup dropped:", match["n_dedup_dropped"], "in clusters:", match["n_close_clusters"])

# ISI histogram for the FINAL spikes (main-ch reference == center reference up to constant shift)
times = np.asarray(match["spike_times_main"], dtype=np.int64)
times = np.sort(times)

if times.size < 2:
    print("Not enough spikes for ISI plot.")
else:
    isi_ms = np.diff(times) / float(sample_rate_hz) * 1000.0

    bin_ms = 0.5
    max_ms = 20.0
    edges = np.arange(0.0, max_ms + bin_ms, bin_ms)
    counts, _ = np.histogram(isi_ms, bins=edges)
    centers = edges[:-1] + 0.5 * bin_ms

    plt.figure(figsize=(8, 3))
    plt.bar(centers, counts, width=bin_ms, align="center")
    plt.xlim(0, max_ms)
    plt.xlabel("ISI (ms)")
    plt.ylabel("Count")
    plt.title(f"ISI (bin={bin_ms} ms, 0–{max_ms} ms) | Nspikes={times.size} | Nisi={isi_ms.size}")
    plt.tight_layout()
    plt.show()

### visualize dropped dups

In [ ]:
# ---- settings ----
N_EXAMPLES = 3
PAD_SAMPLES = 80             # extra samples around the template windows for plotting
MIN_SEP_MS = 1.0             # same criterion as dedup
assert "match" in globals()
assert "lh_template_bank" in globals()
assert "raw_orig" in globals()

bank = lh_template_bank
templates = np.asarray(bank["templates"], dtype=np.float32)   # (B,K,L)
chs = np.asarray(bank["channels"], dtype=np.int32)
win = tuple(bank["win"])
B, K, L = templates.shape

sr = float(globals().get("sample_rate_hz", 20_000))
main_ch = int(bank.get("main_ch", out["step4"].get("main_ch", 0)))

# index of main channel within the selected channels used by templates
if np.any(chs == main_ch):
    main_ci = int(np.where(chs == main_ch)[0][0])
else:
    raise RuntimeError(f"main_ch={main_ch} not found in template channel set.")

times = np.asarray(match["times"], dtype=np.int64)
best_c = np.asarray(match["best_cosine"], dtype=np.float32)
best_b = np.asarray(match["best_template"], dtype=np.int16)
best_j = np.asarray(match["best_jitter"], dtype=np.int8)
did_sub = np.asarray(match["did_subtract"], dtype=bool)

# dedup-dropped = would have passed COS_THR but was turned off by dedup
COS_THR = float(globals().get("COS_THR", 0.9))
dropped_idx = np.flatnonzero((best_c >= COS_THR) & (~did_sub))

print(f"Dedup-dropped candidates (cos≥{COS_THR} but not subtracted): {dropped_idx.size}")

if dropped_idx.size == 0:
    print("No dropped duplicates to plot.")
else:
    # Build a lookup from time->index in match arrays (times are unique)
    t2i = {int(t): i for i, t in enumerate(times)}

    min_sep_samp = int(round((MIN_SEP_MS / 1000.0) * sr))

    # For each dropped spike, find the nearest kept spike within 1 ms (the winner)
    pairs = []
    kept_times = times[did_sub]
    kept_times = np.sort(kept_times)

    for di in dropped_idx:
        td = int(times[di])
        # nearest kept time via searchsorted
        kpos = np.searchsorted(kept_times, td)
        cand = []
        if kpos > 0: cand.append(int(kept_times[kpos-1]))
        if kpos < kept_times.size: cand.append(int(kept_times[kpos]))
        # choose closest within min_sep_samp
        best = None
        best_dt = None
        for tk in cand:
            dt = abs(tk - td)
            if dt < min_sep_samp and (best is None or dt < best_dt):
                best = tk
                best_dt = dt
        if best is not None:
            pairs.append((best, td))  # (winner_time, dropped_time)

    if len(pairs) == 0:
        print("No dropped spikes had a kept neighbor within 1 ms (unexpected).")
    else:
        # de-duplicate by winner_time (one plot per winner cluster)
        # pick first N_EXAMPLES unique winners
        seen = set()
        clusters = []
        for wk, td in pairs:
            if wk in seen: 
                continue
            seen.add(wk)
            # collect all dropped times linked to this winner
            drops = [d for w, d in pairs if w == wk]
            clusters.append((wk, sorted(drops)))
            if len(clusters) >= N_EXAMPLES:
                break

        print(f"Plotting {len(clusters)} example clusters.")

        for (t_keep, drop_list) in clusters:
            i_keep = t2i[int(t_keep)]
            b = int(best_b[i_keep])
            j = int(best_j[i_keep])
            c_keep = float(best_c[i_keep])

            # choose one dropped to highlight (closest)
            drop_list = [int(x) for x in drop_list]
            t_drop = min(drop_list, key=lambda x: abs(x - t_keep))
            i_drop = t2i[int(t_drop)]
            c_drop = float(best_c[i_drop])

            # Plot window: cover both events + template window + padding
            t_min = min(t_keep, t_drop)
            t_max = max(t_keep, t_drop)

            # winner template placement start (general form simplifies nicely):
            # start = t_keep + win[0] + j
            t0 = int(t_keep + win[0] + j)
            t1 = t0 + L

            seg_start = min(t_min + win[0], t0) - PAD_SAMPLES
            seg_end   = max(t_max + win[1], t1) + PAD_SAMPLES

            seg_start = max(seg_start, 0)
            seg_end   = min(seg_end, raw_orig.shape[0])
            y = raw_orig[seg_start:seg_end, main_ch].astype(np.float32)

            # x-axis in ms relative to the earlier spike time
            x_ms = (np.arange(y.size) + seg_start - t_min) / sr * 1000.0

            # template overlay vector (NaNs outside template)
            fit = np.full_like(y, np.nan, dtype=np.float32)
            ins0 = t0 - seg_start
            ins1 = ins0 + L
            if ins0 >= 0 and ins1 <= fit.size:
                fit[ins0:ins1] = templates[b, main_ci, :]

            plt.figure(figsize=(10, 3))
            plt.plot(x_ms, y, color="black", lw=1.0)

            # overlay the winner template
            plt.plot(x_ms, fit, color="green", lw=0.9, alpha=0.5)

            # dashed lines at the two candidate spike times
            x_keep = (t_keep - t_min) / sr * 1000.0
            x_drop = (t_drop - t_min) / sr * 1000.0
            plt.axvline(x_keep, color="green", ls="--", lw=1.2, alpha=0.8)
            plt.axvline(x_drop, color="red",   ls="--", lw=1.2, alpha=0.8)

            dt_ms = abs(t_keep - t_drop) / sr * 1000.0
            plt.title(f"Dedup example | Δt={dt_ms:.3f} ms | keep t={t_keep} cos={c_keep:.4f} (tmpl {b}, j={j}) | drop t={t_drop} cos={c_drop:.4f}")
            plt.xlabel("time (ms) relative to earlier candidate")
            plt.ylabel("ADC (main channel)")
            plt.tight_layout()
            plt.show()

# LOOP

In [ ]:
# ============================
# LH LOOP OVER ALL CHANNELS (minimal plots, strict valley, skip 2-unit/ambiguous)
# ============================

import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import math
import numpy as np

# Make sure EI plotter exists
try:
    pew
except NameError:
    import plot_ei_waveforms as pew

# ----------------------------
# REQUIRED globals (assumed)
# ----------------------------
assert "raw_mod" in globals(), "Need raw_mod (baseline-subtracted int16) loaded."
assert "ei_positions" in globals(), "Need ei_positions (512x2) loaded."
assert "extract_snippets_fast_ram" in globals(), "Need extract_snippets_fast_ram imported."
assert "find_valley_and_times" in globals(), "Need lighthouse_utils.find_valley_and_times."
assert "build_left_low_high_eis" in globals(), "Need build_left_low_high_eis."
assert "filter_valley_spikes_by_batches" in globals(), "Need filter_valley_spikes_by_batches."
assert "finalize_ei_with_cap" in globals(), "Need finalize_ei_with_cap."

T, C = raw_mod.shape
sr = float(globals().get("sample_rate_hz", 20_000))

# Prefer truncated unique onsets if present
if "on_u30" in globals():
    on_u_use = np.asarray(on_u30, dtype=np.float64)
elif "on_u" in globals():
    on_u_use = np.asarray(on_u, dtype=np.float64)
else:
    on_u_use = None
    print("WARNING: on_u30/on_u not found; unique PSTH will be skipped.")

# ----------------------------
# KNOBS
# ----------------------------
win = (-40, 80)

# Step1 params
BIN_WIDTH = 10.0
VALLEY_BINS = 5
MIN_VALID_COUNT = 500
RATIO_BASE, RATIO_STEP, RATIO_FLOOR, RATIO_CAP = 3, 500, 2, 10

# Your new strict valley rule
MAX_VALLEY_COUNT = 500

# Histogram rejection rules
ISI_10_30_MAX = 5
R2_MIN = 0.0

# Step2 cosine gate
MIN_COSINE = 0.65

# KMeans settings (no plots)
DO_KMEANS = True
KMEANS_ISO_SEP = 80          # samples, used to reduce AA-doublet contamination
SUBSAMPLE_MAX = 5_000
RMS_THRESH = 5.0
N_PC = 10
RNG = np.random.RandomState(123)

# Containment / verdict settings
MAX_LAG = 3
SUPPORT_REL = 0.10
SUPPORT_ABS = 30.0
TIME_KEEP_REL = 0.10
FRAC_IN_THR = 0.20
OUT_IN_RATIO_THR = 2.0
RESID_FRAC_MIN = 0.08
SHARED_COS_THR = 0.95
SHARED_ALPHA_THR = 0.95

# Template bank settings
N_BINS = 5
SPIKES_PER = 5000
EI_P2P_THR = 50.0

# Right-extension settings
JMAX_EXT = 3
COS_THR_EXT = 0.95
STOP_RUN = 100
BATCH_EXT = 512
TIME_KEEP_FRAC = 0.08
WEIGHT_POWER = 1.0
USE_CH_WEIGHT = False

# EI-right similarity gate (keeps accepted right spikes sane)
EI_PRES_P2P_THR = 50.0
EI_COS_THR = 0.90
MAX_SPIKES_EI = 5000
MIN_RIGHT_SPIKES_FOR_TEST = 20

# Subtraction settings
JMAX_SUB = 3
BATCH_SUB = 256
USE_CH_WEIGHT_SUB = True
MIN_SEP_MS = 1.0
COS_THR_SUB = 0.90
MIN_FINAL_SPIKES_TO_CALL_SUCCESS = 200  # plots only if ≥ this

# PSTH settings (unique only)
PSTH_WIN_SEC = 0.5
PSTH_BIN_MS = 1.0

# ISI plot settings (your request)
ISI_BIN_MS = 0.5
ISI_MAX_MS = 100.0

# ----------------------------
# Helpers
# ----------------------------
def reject(ch, reason, **kw):
    msg = f"[CH {ch}] REJECT: {reason}"
    if kw:
        # compact key=val formatting
        parts = []
        for k, v in kw.items():
            if isinstance(v, float):
                parts.append(f"{k}={v:.3f}")
            else:
                parts.append(f"{k}={v}")
        msg += " | " + " ".join(parts)
    print(msg)

def _norm_cdf(x, mu, sig):
    z = (x - mu) / (sig * math.sqrt(2.0))
    return 0.5 * (1.0 + math.erf(z))

def compute_left_isi_pairs_10_30(step1):
    lt = np.asarray(step1.get("left_times", []), dtype=np.int64)
    lt = np.sort(lt)
    if lt.size < 2:
        return 0
    d = np.diff(lt)
    return int(np.sum((d >= 10) & (d <= 30)))

def compute_left_rightside_gauss_r2(step1, min_right_samples=200, min_bins=6):
    """
    Fits a Gaussian to the RIGHT side of the LEFT bump:
      region = [mu_mode, valley_low)
    Returns R2 (float). If can't compute robustly, returns np.nan.
    """
    left_vals = np.asarray(step1.get("left_vals", []), dtype=np.float64)
    left_vals = left_vals[np.isfinite(left_vals)]
    if left_vals.size < 200:
        return np.nan

    edges = np.asarray(step1.get("amp_hist_edges", []), dtype=np.float64)
    if edges.size < 3:
        return np.nan

    valley_low = float(step1.get("valley_low", np.nan))
    if not np.isfinite(valley_low):
        return np.nan

    # LEFT-only histogram using same edges
    hL, _ = np.histogram(left_vals, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])

    # mode (peak bin center) as mu estimate
    imode = int(np.argmax(hL))
    mu = float(centers[imode])

    # right-side samples only (mu .. valley_low)
    xR = left_vals[(left_vals >= mu) & (left_vals < valley_low)]
    if xR.size < min_right_samples:
        return np.nan

    # sigma estimate from right half: E[(X-mu)^2 | X>=mu] = sigma^2 for a symmetric normal
    sig = float(np.sqrt(np.mean((xR - mu) ** 2) + 1e-12))
    if not np.isfinite(sig) or sig <= 1e-6:
        return np.nan

    # bins on the right side [mu, valley_low)
    use = (centers >= mu) & (centers < valley_low)
    if int(np.sum(use)) < min_bins:
        return np.nan

    obs = hL[use].astype(np.float64)
    e0 = edges[:-1][use]
    e1 = edges[1:][use]

    exp = np.zeros_like(obs)
    for i in range(obs.size):
        p = _norm_cdf(e1[i], mu, sig) - _norm_cdf(e0[i], mu, sig)
        exp[i] = xR.size * max(p, 0.0)

    ss_res = float(np.sum((obs - exp) ** 2))
    ss_tot = float(np.sum((obs - np.mean(obs)) ** 2) + 1e-12)
    r2 = 1.0 - ss_res / ss_tot
    return float(r2)
def plot_amp_hist(step1, ch):
    # hard cap so the noise peak doesn't dwarf the LH bump
    HIST_YMAX = 5000  # <-- tune: 300, 500, 800, 1500 etc.

    edges = np.asarray(step1["amp_hist_edges"], dtype=np.float64)
    counts = np.asarray(step1["amp_hist_counts"], dtype=np.float64)

    plt.figure(figsize=(12, 3))
    plt.bar(edges[:-1],
            counts,
            width=np.diff(edges),
            align="edge")

    plt.axvspan(step1["valley_low"], step1["valley_high"], color="r", alpha=0.25)

    plt.ylim(0, HIST_YMAX)
    plt.title(f"CH {ch}: amplitude histogram (valley shaded) | left={step1['left_count']} valley={step1['valley_count']} | ycap={HIST_YMAX}")
    plt.xlabel("minima amplitude (ADC)")
    plt.ylabel("count")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

def plot_final_ei(ei, main_ch, title):
    fig = plt.figure(figsize=(20, 8))
    ax = fig.add_subplot(111)
    pew.plot_ei_waveforms(
        ei, positions=ei_positions, ref_channel=int(main_ch),
        scale=70.0, ax=ax,
        colors="black", alpha=1.0, linewidth=1.0,
        box_height=1.0, box_width=50.0, aspect=0.5
    )
    ax.set_title(title)
    plt.show()

def simple_psth(spikes_sec, onsets_sec, win_sec=PSTH_WIN_SEC, bin_ms=PSTH_BIN_MS):
    if onsets_sec is None:
        return None, None, 0
    onsets_sec = np.asarray(onsets_sec, dtype=np.float64)
    onsets_sec = onsets_sec[np.isfinite(onsets_sec)]
    if onsets_sec.size == 0:
        return None, None, 0

    bin_s = bin_ms / 1000.0
    edges = np.arange(0.0, win_sec + bin_s, bin_s)
    centers = edges[:-1] + 0.5 * bin_s
    nbins = centers.size

    spikes_sec = np.asarray(spikes_sec, dtype=np.float64)
    spikes_sec = np.sort(spikes_sec)

    counts = np.zeros(nbins, dtype=np.float64)
    for onset in onsets_sec:
        i0 = np.searchsorted(spikes_sec, onset, side="left")
        i1 = np.searchsorted(spikes_sec, onset + win_sec, side="left")
        rel = spikes_sec[i0:i1] - onset
        if rel.size == 0:
            continue
        idx = (rel / bin_s).astype(np.int64)
        idx = idx[(idx >= 0) & (idx < nbins)]
        np.add.at(counts, idx, 1)

    ntr = int(onsets_sec.size)
    rate_hz = counts / (ntr * bin_s)
    return centers, rate_hz, ntr

def plot_unique_psth(spike_times_samples):
    if on_u_use is None:
        return
    spikes_sec = np.asarray(spike_times_samples, dtype=np.int64) / sr
    centers, rate, ntr = simple_psth(spikes_sec, on_u_use)
    if centers is None:
        print("No unique onsets available for PSTH.")
        return
    plt.figure(figsize=(12, 3))
    plt.plot(centers * 1000.0, rate, lw=1.5)
    plt.xlabel("time from onset (ms)")
    plt.ylabel("Hz")
    plt.title(f"Unique PSTH (0–{PSTH_WIN_SEC*1000:.0f} ms, {PSTH_BIN_MS:.1f} ms bin) | Ntrials={ntr}")
    plt.xlim(0, PSTH_WIN_SEC * 1000.0)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def plot_isi_line(spike_times_samples):
    st = np.sort(np.asarray(spike_times_samples, dtype=np.int64))
    if st.size < 2:
        print("Not enough spikes for ISI.")
        return
    isi_ms = np.diff(st) / sr * 1000.0
    edges = np.arange(0.0, ISI_MAX_MS + ISI_BIN_MS, ISI_BIN_MS)
    counts, edges = np.histogram(isi_ms, bins=edges)
    centers = edges[:-1] + 0.5 * ISI_BIN_MS
    plt.figure(figsize=(12, 3))
    plt.plot(centers, counts, lw=1.5)
    plt.xlabel("ISI (ms)")
    plt.ylabel("count")
    plt.title(f"Final ISI (0–{ISI_MAX_MS:.0f} ms, {ISI_BIN_MS:.1f} ms bin) | Nspikes={st.size}")
    plt.xlim(0, ISI_MAX_MS)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def stable_unique_int64(x):
    seen = set()
    out_ = []
    for v in x:
        vv = int(v)
        if vv not in seen:
            out_.append(vv)
            seen.add(vv)
    return np.asarray(out_, dtype=np.int64)

# --- Containment helpers (no plots) ---
def shift_ei(ei, lag):
    if lag == 0:
        return ei
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :-lag]
    else:
        out[:, :lag] = ei[:, -lag:]
    return out

def support_from_ei(ei):
    p2p = np.ptp(ei, axis=1)
    thr = max(float(SUPPORT_ABS), float(SUPPORT_REL) * float(p2p.max()))
    S = p2p >= thr
    return S, p2p, thr

def best_lag_on_support(X, Y, S):
    Xs = X[S, :]
    env = np.max(np.abs(Xs), axis=0) if Xs.size else np.max(np.abs(X), axis=0)
    tthr = float(TIME_KEEP_REL) * float(env.max() + 1e-12)
    T = env >= tthr
    if not np.any(T):
        T = np.ones(X.shape[1], dtype=bool)

    best = dict(lag=0, cos=-np.inf, T=T)
    A = X[S, :][:, T].ravel()
    nA = np.linalg.norm(A) + 1e-12
    for lag in range(-MAX_LAG, MAX_LAG + 1):
        Ys = shift_ei(Y, lag)
        B = Ys[S, :][:, T].ravel()
        nB = np.linalg.norm(B) + 1e-12
        cos = float((A @ B) / (nA * nB))
        if cos > best["cos"]:
            best = dict(lag=int(lag), cos=cos, T=T)
    return best

def containment_metrics(X, Y):
    S, p2pX, thr = support_from_ei(X)
    best = best_lag_on_support(X, Y, S)
    lag = best["lag"]
    Yal = shift_ei(Y, lag)
    T = best["T"]

    A = X[S, :][:, T].ravel()
    B = Yal[S, :][:, T].ravel()
    alpha = float((A @ B) / ((A @ A) + 1e-12))

    R = Yal - alpha * X

    Yin  = Yal[S, :]
    Rin  = R[S, :]
    Rout = R[~S, :]

    Ein = float(np.linalg.norm(Rin))
    Eout = float(np.linalg.norm(Rout))

    frac_in  = float(np.linalg.norm(Rin) / (np.linalg.norm(Yin) + 1e-12))
    frac_all = float(np.linalg.norm(R) / (np.linalg.norm(Yal) + 1e-12))
    out_in   = float(Eout / (Ein + 1e-12))

    return dict(lag=lag, cos_on_support=float(best["cos"]), alpha=alpha,
                support_n=int(np.sum(S)), support_thr=float(thr),
                frac_in=frac_in, frac_all=frac_all, out_in=out_in)

def verdict_from_kmeans(ei0, ei1):
    m01 = containment_metrics(ei0, ei1)
    m10 = containment_metrics(ei1, ei0)

    shared01 = (m01["cos_on_support"] >= SHARED_COS_THR) and (m01["alpha"] >= SHARED_ALPHA_THR)
    shared10 = (m10["cos_on_support"] >= SHARED_COS_THR) and (m10["alpha"] >= SHARED_ALPHA_THR)
    shared_core = bool(shared01 or shared10)

    def is_contained(m):
        return (m["frac_in"] <= FRAC_IN_THR) and (m["out_in"] >= OUT_IN_RATIO_THR)

    c01 = is_contained(m01)
    c10 = is_contained(m10)

    if shared_core and c01 and c10 and (m01["frac_all"] < RESID_FRAC_MIN) and (m10["frac_all"] < RESID_FRAC_MIN):
        verdict = "SAME UNIT split (amplitude/drift)"
        proceed = True
    elif shared_core and ((c01 and not c10) or (c10 and not c01)):
        verdict = "AB-SHARD-like (shared core)"
        proceed = True
    elif shared_core:
        verdict = "SHARED-CORE (overlap/AA/complex)"
        proceed = True
    elif (not c01) and (not c10):
        verdict = "TWO-UNITS-like (reject)"
        proceed = False
    else:
        verdict = "AMBIGUOUS (reject)"
        proceed = False

    return dict(
        verdict=verdict, proceed=bool(proceed),
        shared_core=shared_core,
        shared_dirs=dict(ei0_to_ei1=bool(shared01), ei1_to_ei0=bool(shared10)),
        m01=m01, m10=m10
    )

# --- Template bank ---
def build_template_bank(step4):
    main_ch = int(step4.get("main_ch", 0))
    final_ei = np.asarray(step4["final_ei"], dtype=np.float32)
    ei_p2p = np.ptp(final_ei, axis=1)

    chan_sel = np.flatnonzero(ei_p2p >= EI_P2P_THR).astype(np.int32)
    if chan_sel.size == 0:
        raise RuntimeError(f"No channels pass EI p2p >= {EI_P2P_THR} (max={ei_p2p.max():.1f})")
    chan_sel = chan_sel[np.argsort(ei_p2p[chan_sel])[::-1]]
    if not np.any(chan_sel == main_ch):
        chan_sel = np.concatenate([np.asarray([main_ch], dtype=np.int32), chan_sel])
        chan_sel = np.unique(chan_sel)

    times_all = np.sort(np.asarray(step4.get("final_times", step4.get("final_times_used", [])), dtype=np.int64))
    if times_all.size == 0:
        raise RuntimeError("No spike times in step4.")

    sn_main, valid_times = extract_snippets_fast_ram(
        raw_mod, times_all, window=win, selected_channels=np.asarray([main_ch], dtype=np.int32)
    )
    valid_times = np.asarray(valid_times, dtype=np.int64)
    amps = np.ptp(sn_main[0, :, :], axis=0).astype(np.float32)

    if amps.size < N_BINS * 10:
        raise RuntimeError(f"Too few valid snippets for binning: Nvalid={amps.size}")

    order_desc = np.argsort(amps)[::-1]
    N = int(order_desc.size)
    B = int(N_BINS)
    M = int(SPIKES_PER)

    bin_times = []
    bin_amp_ranges = []

    if N >= B * M:
        starts = np.round(np.linspace(0, N - M, B)).astype(int) if B > 1 else np.array([0], dtype=int)
        for s in starts:
            idx = order_desc[s:s+M]
            tbin = valid_times[idx]
            abin = amps[idx]
            bin_times.append(tbin)
            bin_amp_ranges.append((float(np.min(abin)), float(np.max(abin))))
    else:
        groups = np.array_split(order_desc, B)
        for g in groups:
            tbin = valid_times[g]
            abin = amps[g]
            bin_times.append(tbin)
            bin_amp_ranges.append((float(np.min(abin)), float(np.max(abin))) if abin.size else (np.nan, np.nan))

    if any(t.size == 0 for t in bin_times):
        raise RuntimeError("Empty template bin; reduce N_BINS or get more spikes.")

    templates = []
    for tbin in bin_times:
        sn, _ = extract_snippets_fast_ram(raw_mod, tbin, window=win, selected_channels=chan_sel)
        templates.append(np.median(sn, axis=2).astype(np.float32))
    templates = np.stack(templates, axis=0)  # (B,K,L)

    return dict(
        main_ch=main_ch, win=win, sample_rate_hz=sr,
        channels=chan_sel,
        ei_p2p=ei_p2p[chan_sel],
        bin_amp_ranges=bin_amp_ranges,
        templates=templates
    )

# --- Right extension (masked cosine) ---
def extend_right_spikes(step1, bank):
    templates = np.asarray(bank["templates"], dtype=np.float32)  # (B,K,L)
    chs = np.asarray(bank["channels"], dtype=np.int32)
    win_ = tuple(bank["win"])
    B, Kk, L = templates.shape

    ref = np.median(templates, axis=0)  # (K,L)
    absref = np.abs(ref)
    time_env = absref.max(axis=0)
    t_thr = TIME_KEEP_FRAC * time_env.max()
    keep_t = time_env >= t_thr

    if USE_CH_WEIGHT:
        ch_env = np.ptp(ref, axis=1)
        ch_w = ch_env / (ch_env.max() + 1e-12)
    else:
        ch_w = np.ones(Kk, dtype=np.float32)

    W = (absref ** WEIGHT_POWER) * ch_w[:, None]
    W[:, ~keep_t] = 0.0

    mask_idx = np.flatnonzero(W.ravel() > 0)
    w_mask = W.ravel()[mask_idx].astype(np.float32)

    tmpl_flat = templates.reshape(B, -1).astype(np.float32)
    tmplW = tmpl_flat[:, mask_idx] * w_mask[None, :]
    tmplW_norm = np.linalg.norm(tmplW, axis=1) + 1e-12

    right_raw = np.asarray(step1.get("rightk_times_by_amp", []), dtype=np.int64).ravel()
    right_times = right_raw[np.isfinite(right_raw)]
    right_times = stable_unique_int64(right_times)

    if right_times.size == 0:
        return []

    win_ext = (win_[0] - JMAX_EXT, win_[1] + JMAX_EXT)
    accepted = []
    consec_fail = 0
    stop_hit = False

    for s0 in range(0, right_times.size, BATCH_EXT):
        if stop_hit:
            break
        tb = right_times[s0:s0+BATCH_EXT]
        sn_ext, t_valid = extract_snippets_fast_ram(raw_mod, tb, window=win_ext, selected_channels=chs)
        t_valid = np.asarray(t_valid, dtype=np.int64)
        N = sn_ext.shape[2]
        if N == 0:
            continue

        Lext = sn_ext.shape[1]
        anchor = (Lext - L) // 2

        best_cos = np.full(N, -np.inf, dtype=np.float32)
        best_b = np.zeros(N, dtype=np.int16)
        best_j = np.zeros(N, dtype=np.int8)

        for j in range(-JMAX_EXT, JMAX_EXT + 1):
            start = anchor + j
            seg = sn_ext[:, start:start+L, :].astype(np.float32)  # (K,L,N)
            if seg.shape[1] != L:
                continue
            Xfull = seg.transpose(2,0,1).reshape(N, -1)
            Xw = Xfull[:, mask_idx] * w_mask[None, :]
            Xn = np.linalg.norm(Xw, axis=1) + 1e-12
            dots = Xw @ tmplW.T
            cos = dots / (Xn[:, None] * tmplW_norm[None, :])

            b = np.argmax(cos, axis=1).astype(np.int16)
            c = cos[np.arange(N), b].astype(np.float32)

            upd = c > best_cos
            best_cos[upd] = c[upd]
            best_b[upd] = b[upd]
            best_j[upd] = np.int8(j)

        for i in range(N):
            t = int(t_valid[i])
            c = float(best_cos[i])
            if c >= COS_THR_EXT:
                consec_fail = 0
                accepted.append(t)
            else:
                consec_fail += 1
                if consec_fail >= STOP_RUN:
                    stop_hit = True
                    break

    return np.sort(np.unique(np.asarray(accepted, dtype=np.int64))).tolist()

def build_median_ei_from_times(raw, times, window):
    times = np.asarray(times, dtype=np.int64)
    times = np.sort(np.unique(times))
    if times.size == 0:
        return None, 0
    if times.size > MAX_SPIKES_EI:
        idx = np.round(np.linspace(0, times.size - 1, MAX_SPIKES_EI)).astype(int)
        times = times[idx]
    sn, _ = extract_snippets_fast_ram(raw, times, window=window, selected_channels=np.arange(C, dtype=np.int32))
    if sn.shape[2] == 0:
        return None, 0
    return np.median(sn, axis=2).astype(np.float32), int(sn.shape[2])

def filter_right_by_ei_similarity(step4, accepted_times, bank):
    right_times = np.asarray(accepted_times, dtype=np.int64)
    right_times = np.sort(np.unique(right_times))
    if right_times.size < MIN_RIGHT_SPIKES_FOR_TEST:
        return []

    ei_core = np.asarray(step4["final_ei"], dtype=np.float32)
    win_ = tuple(bank["win"])

    ei_right, n_used = build_median_ei_from_times(raw_mod, right_times, window=win_)
    if ei_right is None:
        return []

    p2p_core = np.ptp(ei_core, axis=1)
    p2p_right = np.ptp(ei_right, axis=1)
    chan_union = np.flatnonzero((p2p_core >= EI_PRES_P2P_THR) | (p2p_right >= EI_PRES_P2P_THR))
    if chan_union.size == 0:
        return []

    A = ei_core[chan_union, :].reshape(-1)
    B = ei_right[chan_union, :].reshape(-1)
    cos = float((A @ B) / ((np.linalg.norm(A) + 1e-12) * (np.linalg.norm(B) + 1e-12)))

    if cos < EI_COS_THR:
        return []
    return right_times.tolist()

# --- Subtraction (jitter-corrected, main-channel reference), adapted from your cell ---
def subtract_lh_masked(raw_mod, spike_times, bank):
    templates = np.asarray(bank["templates"], dtype=np.float32)  # (B,K,L)
    chs = np.asarray(bank["channels"], dtype=np.int32)
    win_ = tuple(bank["win"])
    B, Kk, L = templates.shape

    templates_i16 = np.rint(templates).astype(np.int16)

    ref = np.median(templates, axis=0)  # (K,L)
    absref = np.abs(ref)

    time_env = absref.max(axis=0)
    t_thr = TIME_KEEP_FRAC * time_env.max()
    keep_t = time_env >= t_thr

    if USE_CH_WEIGHT_SUB:
        ch_env = np.ptp(ref, axis=1)
        ch_w = ch_env / (ch_env.max() + 1e-12)
    else:
        ch_w = np.ones(Kk, dtype=np.float32)

    W = (absref ** WEIGHT_POWER) * ch_w[:, None]
    W[:, ~keep_t] = 0.0

    mask_idx = np.flatnonzero(W.ravel() > 0)
    w_mask = W.ravel()[mask_idx].astype(np.float32)

    tmpl_flat_full = templates.reshape(B, -1).astype(np.float32)
    tmplW = tmpl_flat_full[:, mask_idx] * w_mask[None, :]
    tmplW_norm = np.linalg.norm(tmplW, axis=1) + 1e-12

    main_ch = int(bank.get("main_ch", 0))
    if not np.any(chs == main_ch):
        raise RuntimeError(f"main_ch={main_ch} not found in bank channels.")

    main_ci = int(np.where(chs == main_ch)[0][0])
    center_idx = int(-win_[0])
    tmin_main = int(np.argmin(ref[main_ci, :]))
    dt_main = int(tmin_main - center_idx)

    win_ext = (win_[0] - JMAX_SUB, win_[1] + JMAX_SUB)
    spike_times = np.asarray(spike_times, dtype=np.int64)
    spike_times = np.sort(np.unique(spike_times))
    if spike_times.size == 0:
        return None

    # Pass A: match
    used_t_all = []
    best_b_all = []
    best_j_all = []
    best_c_all = []

    # anchor
    tmp_sn, _ = extract_snippets_fast_ram(raw_mod, spike_times[:1], window=win_ext, selected_channels=chs)
    if tmp_sn.shape[2] == 0:
        return None
    Lext0 = tmp_sn.shape[1]
    anchor0 = (Lext0 - L) // 2

    for s0 in range(0, spike_times.size, BATCH_SUB):
        t_batch = spike_times[s0:s0+BATCH_SUB]
        sn_ext, t_valid = extract_snippets_fast_ram(raw_mod, t_batch, window=win_ext, selected_channels=chs)
        t_valid = np.asarray(t_valid, dtype=np.int64)
        N = sn_ext.shape[2]
        if N == 0:
            continue

        Lext = sn_ext.shape[1]
        anchor = (Lext - L) // 2

        best_cos = np.full(N, -np.inf, dtype=np.float32)
        best_b = np.zeros(N, dtype=np.int16)
        best_j = np.zeros(N, dtype=np.int8)

        for j in range(-JMAX_SUB, JMAX_SUB + 1):
            start = anchor + j
            seg = sn_ext[:, start:start+L, :].astype(np.float32)
            Xfull = seg.transpose(2,0,1).reshape(N, -1)
            Xw = Xfull[:, mask_idx] * w_mask[None, :]
            Xn = np.linalg.norm(Xw, axis=1) + 1e-12
            dots = Xw @ tmplW.T
            cos = dots / (Xn[:, None] * tmplW_norm[None, :])

            b = np.argmax(cos, axis=1).astype(np.int16)
            c = cos[np.arange(N), b].astype(np.float32)

            upd = c > best_cos
            best_cos[upd] = c[upd]
            best_b[upd] = b[upd]
            best_j[upd] = np.int8(j)

        used_t_all.append(t_valid)
        best_b_all.append(best_b)
        best_j_all.append(best_j)
        best_c_all.append(best_cos)

    if len(used_t_all) == 0:
        return None

    used_t_all = np.concatenate(used_t_all).astype(np.int64)
    best_b_all = np.concatenate(best_b_all).astype(np.int16)
    best_j_all = np.concatenate(best_j_all).astype(np.int8)
    best_c_all = np.concatenate(best_c_all).astype(np.float32)

    t_center_all = used_t_all + best_j_all.astype(np.int64)
    did_sub = best_c_all >= COS_THR_SUB

    # Pass B: dedup on t_center
    min_sep_samp = int(round((MIN_SEP_MS / 1000.0) * sr))
    accept_idx = np.flatnonzero(did_sub)
    if accept_idx.size > 1 and min_sep_samp > 0:
        accept_sorted = accept_idx[np.argsort(t_center_all[accept_idx])]
        tacc = t_center_all[accept_sorted]
        i = 0
        while i < accept_sorted.size:
            j = i
            while (j + 1 < accept_sorted.size) and ((tacc[j + 1] - tacc[j]) < min_sep_samp):
                j += 1
            if j > i:
                cluster = accept_sorted[i:j+1]
                keep = cluster[np.argmax(best_c_all[cluster])]
                drop = cluster[cluster != keep]
                did_sub[drop] = False
            i = j + 1

    keep_idx = np.flatnonzero(did_sub)
    keep_idx = keep_idx[np.argsort(t_center_all[keep_idx])]

    # Pass C: subtract
    for idx in keep_idx:
        t = int(used_t_all[idx])
        b = int(best_b_all[idx])
        j = int(best_j_all[idx])
        t0 = t + win_ext[0] + (anchor0 + j)
        t1 = t0 + L
        tmpl = templates_i16[b]
        for ci, ch in enumerate(chs):
            raw_mod[t0:t1, int(ch)] -= tmpl[ci, :]

    spike_times_center = np.sort(t_center_all[did_sub])
    spike_times_main = np.sort((t_center_all + dt_main)[did_sub])

    return dict(
        main_ch=int(main_ch),
        dt_main=int(dt_main),
        spike_times_center=spike_times_center,
        spike_times_main=spike_times_main,
        n_sub=int(did_sub.sum()),
        best_cosine=best_c_all,
    )

# ----------------------------
# MAIN LOOP
# ----------------------------
# Keep prior rounds if they exist
# lh_units = []
if "lh_units" not in globals():
    lh_units = []
skip_counts = defaultdict(int)

t_loop0 = time.time()
print(f"Starting LH loop over {C} channels...")

for ch in range(C):
    try:
        # ---------- STEP 1 ----------
        step1 = find_valley_and_times(
            raw_mod, ch,
            window=win, start=0, stop=None,
            bin_width=BIN_WIDTH, valley_bins=VALLEY_BINS,
            min_valid_count=MIN_VALID_COUNT,
            ratio_base=RATIO_BASE, ratio_step=RATIO_STEP, ratio_floor=RATIO_FLOOR, ratio_cap=RATIO_CAP
        )

        # Your strict valley rule
        if int(step1.get("valley_count", 0)) > int(MAX_VALLEY_COUNT):
            skip_counts["valley_count>max"] += 1
            reject(ch, "valley_count>max", valley_count=int(step1.get("valley_count",0)), max=int(MAX_VALLEY_COUNT))
            continue

        if not step1.get("accepted", False):
            skip_counts["valley_not_accepted"] += 1
            reject(ch, "valley_not_accepted",
                left=int(step1.get("left_count",-1)),
                valley=int(step1.get("valley_count",-1)),
                Rstar=float(step1.get("required_ratio", np.nan)))
            continue

        # ---------- NEW: abort checks (R2 + short-ISI) ----------
        r2 = compute_left_rightside_gauss_r2(step1)
        isi_pairs_10_30 = compute_left_isi_pairs_10_30(step1)

        print(f"[CH {ch}] LEFT diagnostics: R2={r2 if np.isfinite(r2) else np.nan:.3f} | ISI_pairs_10_30={isi_pairs_10_30}")

        r2 = compute_left_rightside_gauss_r2(step1)
        isi_pairs_10_30 = compute_left_isi_pairs_10_30(step1)

        if (not np.isfinite(r2)) or (r2 < R2_MIN):
            skip_counts["abort_R2<0.3"] += 1
            reject(ch, "abort_R2<0.3", R2=float(r2) if np.isfinite(r2) else np.nan)
            continue

        if isi_pairs_10_30 > ISI_10_30_MAX:
            skip_counts["abort_ISI10_30>10"] += 1
            reject(ch, "abort_ISI10_30>10", isi_pairs=int(isi_pairs_10_30))
            continue

        # ---------- STEP 2 ----------
        step2 = build_left_low_high_eis(
            raw_mod,
            step1["left_times"], step1["left_vals"],
            window=win, left_cap=500, reducer="median", rms_gate=5.0
        )
        if float(step2.get("base_cosine", 0.0)) < float(MIN_COSINE):
            skip_counts["low_high_cos_fail"] += 1
            reject(ch, "low_high_cos_fail", base_cos=float(step2.get("base_cosine", 0.0)), thr=float(MIN_COSINE))
            continue

        # ---------- OPTIONAL: KMEANS + VERDICT (no plots) ----------
        km_info = None
        proceed = True

        if DO_KMEANS:
            left_times = np.asarray(step1["left_times"], dtype=np.int64)
            if left_times.size >= 50:
                # Build "all_times" for isolation. Prefer true all_times if present; else approximate from available sets.
                if "all_times" in step1:
                    all_times = np.asarray(step1["all_times"], dtype=np.int64)
                else:
                    parts = []
                    for k in ["left_times", "valley_times", "rightk_times_sorted", "rightk_times_by_amp"]:
                        if k in step1:
                            parts.append(np.asarray(step1[k], dtype=np.int64).ravel())
                    all_times = np.sort(np.unique(np.concatenate(parts))) if len(parts) else np.array([], dtype=np.int64)

                left_for_km = left_times
                if all_times.size:
                    idx = np.searchsorted(all_times, left_times)
                    ok_map = (idx < all_times.size) & (all_times[idx] == left_times)
                    idx_ok = idx[ok_map]
                    lt_ok  = left_times[ok_map]

                    dt_prev = np.full(lt_ok.size, np.iinfo(np.int64).max, dtype=np.int64)
                    dt_next = np.full(lt_ok.size, np.iinfo(np.int64).max, dtype=np.int64)

                    m = idx_ok > 0
                    dt_prev[m] = lt_ok[m] - all_times[idx_ok[m] - 1]
                    m = (idx_ok + 1) < all_times.size
                    dt_next[m] = all_times[idx_ok[m] + 1] - lt_ok[m]

                    iso = (dt_prev >= KMEANS_ISO_SEP) & (dt_next >= KMEANS_ISO_SEP)
                    left_iso = lt_ok[iso]
                    if left_iso.size >= 50:
                        left_for_km = left_iso

                pick = left_for_km if left_for_km.size <= SUBSAMPLE_MAX else left_for_km[RNG.choice(left_for_km.size, SUBSAMPLE_MAX, replace=False)]

                snips, valid_times = extract_snippets_fast_ram(
                    raw_mod, pick, window=win,
                    selected_channels=np.arange(C, dtype=np.int32)
                )
                Kk, L, N = snips.shape
                if N >= 50:
                    ei_mean = snips.mean(axis=2).astype(np.float32)
                    rms = np.sqrt(np.mean(ei_mean**2, axis=1))
                    sel = np.flatnonzero(rms > RMS_THRESH)
                    if sel.size == 0:
                        sel = np.argsort(rms)[-16:]
                        sel.sort()

                    X = snips[sel, :, :].transpose(2,0,1).reshape(N, sel.size * L).astype(np.float32)
                    n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
                    pca = PCA(n_components=n_pc, svd_solver="randomized", random_state=0)
                    Xpc = pca.fit_transform(X)

                    km = KMeans(n_clusters=2, n_init=50, random_state=0)
                    lab = km.fit_predict(Xpc)

                    ei_c0 = snips[:, :, lab == 0].mean(axis=2).astype(np.float32)
                    ei_c1 = snips[:, :, lab == 1].mean(axis=2).astype(np.float32)

                    verdict_info = verdict_from_kmeans(ei_c0, ei_c1)
                    proceed = bool(verdict_info["proceed"])

                    km_info = dict(
                        n0=int(np.sum(lab == 0)),
                        n1=int(np.sum(lab == 1)),
                        ei_c0=ei_c0, ei_c1=ei_c1,
                        verdict=verdict_info["verdict"],
                        proceed=proceed,
                        shared_core=verdict_info["shared_core"],
                        shared_dirs=verdict_info["shared_dirs"],
                        m01=verdict_info["m01"], m10=verdict_info["m10"],
                    )

                    if not proceed:
                        skip_counts["kmeans_reject"] += 1
                        reject(ch, "kmeans_reject", verdict=str(km_info.get("verdict","")))
                        continue

        # ---------- STEP 3 ----------
        step3 = filter_valley_spikes_by_batches(
            raw_mod,
            step1["rightk_times_by_amp"],
            step1["left_times"], step1["left_vals"],
            window=win,
            batch_size=100,
            reducer="median",
            tail_n=100,
            abs_cos_min=0.85,
            base_cos_cap=0.97,
            cos_slack=0.02,
            rms_gate=5.0,
            best_align_lag=3,
            use_abs=True
        )

        # ---------- STEP 4 ----------
        step4 = finalize_ei_with_cap(
            raw_mod,
            step1["left_times"],
            step3["accepted_right_times"],
            window=win,
            final_max_spikes=5_000,
            reducer="median",
            rng_seed=123,
            ref_ch=ch
        )

        # ---------- TEMPLATE BANK ----------
        bank = build_template_bank(step4)

        # ---------- RIGHT EXTENSION ----------
        accepted_times = extend_right_spikes(step1, bank)
        accepted_times = filter_right_by_ei_similarity(step4, accepted_times, bank)

        # ---------- SUBTRACT (final spike list = what got subtracted) ----------
        core_times = np.sort(np.unique(np.asarray(step4["final_times"], dtype=np.int64)))
        if len(accepted_times):
            spike_times = np.sort(np.unique(np.concatenate([core_times, np.asarray(accepted_times, dtype=np.int64)])))
        else:
            spike_times = core_times

        match = subtract_lh_masked(raw_mod, spike_times, bank)
        if match is None or int(match.get("n_sub", 0)) == 0:
            skip_counts["subtraction_empty"] += 1
            reject(ch, "subtraction_empty")
            continue

        final_spikes_main = match["spike_times_main"]
        if final_spikes_main.size < MIN_FINAL_SPIKES_TO_CALL_SUCCESS:
            skip_counts["too_few_final_spikes"] += 1
            reject(ch, "too_few_final_spikes", n=int(final_spikes_main.size), thr=int(MIN_FINAL_SPIKES_TO_CALL_SUCCESS))
            continue

        # ---------- SUCCESS: store unit + plots ----------
        unit = dict(
            detect_ch=int(ch),
            step1=step1, step2=step2, step3=step3, step4=step4,
            bank=bank,
            accepted_times=np.asarray(accepted_times, dtype=np.int64),
            match=match,
            final_spikes_main=final_spikes_main,
            kmeans=km_info
        )

        unit["diag_r2"] = float(r2)
        unit["diag_isi_pairs_10_30"] = int(isi_pairs_10_30)

        lh_units.append(unit)
        skip_counts["SUCCESS"] += 1

        print(f"\n✅ SUCCESS unit#{len(lh_units):d} | detect_ch={ch} | main_ch={match['main_ch']} | Nspikes={final_spikes_main.size}"
              + (f" | kmeans={km_info['verdict']}" if km_info is not None else ""))

        # core plots only
        plot_amp_hist(step1, ch)
        plot_final_ei(step4["final_ei"], bank["main_ch"], title=f"Final EI | detect_ch={ch} main_ch={bank['main_ch']} | N={final_spikes_main.size}")
        plot_unique_psth(final_spikes_main)
        plot_isi_line(final_spikes_main)

    except Exception as e:
        skip_counts["exception"] += 1
        print(f"[CH {ch}] Exception: {type(e).__name__}: {e}")
        continue

t_loop1 = time.time()
print("\n====================")
print(f"Done. Found {len(lh_units)} successful units.")
print(f"Elapsed: {t_loop1 - t_loop0:.1f} s")
print("Skip summary:")
for k in sorted(skip_counts.keys()):
    print(f"  {k:>22s} : {skip_counts[k]}")

In [ ]:
import os, time, json
import numpy as np
import h5py

LH_SAVE_PATH = "/Volumes/Lab/Users/alexth/axolotl/201711290/lighthouse_NS_round5.h5"

assert "lh_units" in globals() and len(lh_units) > 0, "lh_units is empty; nothing to save."
sr = float(globals().get("sample_rate_hz", 20_000))
win = tuple(globals().get("win", (-40, 80)))

# ---- choose where to save ----
# put it wherever you keep other outputs; this default is just local directory
out_path = globals().get("LH_SAVE_PATH", "lighthouse_ns_results.h5")

def _as_float32(x):
    return np.asarray(x, dtype=np.float32)

def _as_int32(x):
    return np.asarray(x, dtype=np.int32)

def _as_int64(x):
    return np.asarray(x, dtype=np.int64)

# Build a simple summary array for quick inspection
summary = []
for i, u in enumerate(lh_units):
    main_ch = int(u["match"]["main_ch"])
    det_ch  = int(u.get("detect_ch", u.get("detect_channel", -1)))
    nsp = int(u["final_spikes_main"].size)
    kmi = u.get("kmeans", None)
    verdict = ""
    shared_core = False
    if kmi is not None:
        verdict = str(kmi.get("verdict", ""))
        shared_core = bool(kmi.get("shared_core", False))
    summary.append((i, det_ch, main_ch, nsp, int(shared_core), verdict))

# HDF5: store strings as fixed-length or variable-length UTF8
dt_str = h5py.string_dtype(encoding="utf-8")

with h5py.File(out_path, "w") as h5:
    # global metadata
    h5.attrs["created_unix"] = int(time.time())
    h5.attrs["sample_rate_hz"] = sr
    h5.attrs["win_pre"] = int(win[0])
    h5.attrs["win_post"] = int(win[1])
    h5.attrs["n_units"] = int(len(lh_units))

    # summary table
    gsum = h5.create_group("summary")
    gsum.create_dataset("unit_index", data=np.asarray([s[0] for s in summary], dtype=np.int32))
    gsum.create_dataset("detect_ch",  data=np.asarray([s[1] for s in summary], dtype=np.int32))
    gsum.create_dataset("main_ch",    data=np.asarray([s[2] for s in summary], dtype=np.int32))
    gsum.create_dataset("n_spikes",   data=np.asarray([s[3] for s in summary], dtype=np.int32))
    gsum.create_dataset("shared_core",data=np.asarray([s[4] for s in summary], dtype=np.int8))
    gsum.create_dataset("verdict",    data=np.asarray([s[5] for s in summary], dtype=dt_str))

    # per-unit payloads
    guns = h5.create_group("units")
    for i, u in enumerate(lh_units):
        gu = guns.create_group(f"unit_{i:03d}")

        # attrs (small stuff)
        gu.attrs["unit_index"] = int(i)
        gu.attrs["detect_ch"]  = int(u.get("detect_ch", u.get("detect_channel", -1)))
        gu.attrs["main_ch"]    = int(u["match"]["main_ch"])
        gu.attrs["dt_main"]    = int(u["match"]["dt_main"])
        gu.attrs["n_sub"]      = int(u["match"]["n_sub"])

        # valley stats
        step1 = u["step1"]
        gu.attrs["valley_low"]   = float(step1["valley_low"])
        gu.attrs["valley_high"]  = float(step1["valley_high"])
        gu.attrs["left_count"]   = int(step1["left_count"])
        gu.attrs["valley_count"] = int(step1["valley_count"])
        gu.attrs["required_ratio"]= float(step1.get("required_ratio", np.nan))

        # kmeans (if present)
        kmi = u.get("kmeans", None)
        if kmi is not None:
            gu.attrs["kmeans_verdict"] = str(kmi.get("verdict",""))
            gu.attrs["kmeans_proceed"] = bool(kmi.get("proceed", True))
            gu.attrs["kmeans_shared_core"] = bool(kmi.get("shared_core", False))
            if "n0" in kmi: gu.attrs["kmeans_n0"] = int(kmi["n0"])
            if "n1" in kmi: gu.attrs["kmeans_n1"] = int(kmi["n1"])

        # --- spike trains ---
        gu.create_dataset("spike_times_main",   data=_as_int64(u["final_spikes_main"]), compression="gzip")
        gu.create_dataset("spike_times_center", data=_as_int64(u["match"]["spike_times_center"]), compression="gzip")

        # --- EI ---
        gu.create_dataset("final_ei", data=_as_float32(u["step4"]["final_ei"]), compression="gzip")

        # --- template bank ---
        bank = u["bank"]
        gu.create_dataset("bank_channels", data=_as_int32(bank["channels"]), compression="gzip")
        gu.create_dataset("bank_templates", data=_as_float32(bank["templates"]), compression="gzip")
        gu.create_dataset("bin_amp_ranges", data=_as_float32(np.asarray(bank["bin_amp_ranges"])), compression="gzip")

        # --- amplitude histogram (for QC) ---
        gu.create_dataset("amp_hist_edges",  data=_as_float32(step1["amp_hist_edges"]), compression="gzip")
        gu.create_dataset("amp_hist_counts", data=_as_float32(step1["amp_hist_counts"]), compression="gzip")

print(f"✅ Saved {len(lh_units)} units to: {os.path.abspath(out_path)}")

In [ ]:
import h5py
with h5py.File(out_path, "r") as h5:
    print("n_units:", h5.attrs["n_units"])
    print("summary keys:", list(h5["summary"].keys()))
    print("first unit keys:", list(h5["units"]["unit_000"].keys()))
    print("first unit n_spikes:", h5["units"]["unit_000"]["spike_times_main"].shape[0])

### Save raw data (modified)

In [ ]:
import os, json, time
import numpy as np

assert "raw_mod" in globals()
assert raw_mod.dtype == np.int16, f"raw_mod dtype is {raw_mod.dtype}, expected int16"
T, C = raw_mod.shape
sr = float(globals().get("sample_rate_hz", 20_000))

OUTDIR = "/Volumes/Lab/Users/alexth/axolotl/201711290"
ROUND_LABEL = "round5"   # <-- change to round3 etc
base = f"data004_lighthouse_residual_{ROUND_LABEL}"

out_dat  = os.path.join(OUTDIR, base + ".dat")
out_json = os.path.join(OUTDIR, base + ".json")

print("Writing:", out_dat)
# Ensure row-major contiguous for fast streaming write
raw_to_write = np.ascontiguousarray(raw_mod)
raw_to_write.tofile(out_dat)

meta = dict(
    created_unix=int(time.time()),
    sample_rate_hz=sr,
    dtype="int16",
    n_channels=int(C),
    n_samples=int(T),
    shape=[int(T), int(C)],
    order="time x channels (row-major), interleaved channels per sample",
    source="raw_mod after lighthouse subtraction",
    win=list(globals().get("win", (-40, 80))),
    notes="Residual trace after lighthouse peeling. Use as new input to downstream Axolotl stages."
)

# If you have lh_units, store a pointer-ish summary
if "lh_units" in globals():
    meta["n_units_in_memory"] = int(len(lh_units))
    # Optional: record which unit file you saved (if you tracked it)
    if "LH_SAVE_PATH" in globals():
        meta["units_h5"] = str(LH_SAVE_PATH)

with open(out_json, "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Done.")
print("DAT bytes:", os.path.getsize(out_dat), "JSON:", out_json)

### load and plot a short segment of mod data

In [ ]:
# ---- point to the saved residual file ----
# Use the same path you wrote earlier; edit ROUND_LABEL if needed
OUTDIR = "/Volumes/Lab/Users/alexth/axolotl/201711290"
ROUND_LABEL = "round5"   # round3 etc if you changed it
dat_path = f"{OUTDIR}/data004_lighthouse_residual_{ROUND_LABEL}.dat"

n_channels = 512
dtype = np.int16

# ---- choose the window to load ----
t0_load = 2137047-200
t1_load = t0_load+1000          # loads 1000 samples
t0_plot = 2137047-20
n_plot  = 100              # plot 200 continuous samples
assert t0_plot >= t0_load and (t0_plot + n_plot) <= t1_load

# ---- memmap the file as (T, C) without loading everything ----
mm = np.memmap(dat_path, dtype=dtype, mode="r")
total_samples = mm.size // n_channels
assert mm.size % n_channels == 0, "File size not divisible by n_channels; wrong n_channels or corrupt file?"

# reshape view (no copy)
mm2 = mm.reshape(total_samples, n_channels)

# load the chunk into RAM (1000 x 512)
chunk = np.array(mm2[t0_load:t1_load, :], copy=True)  # int16 -> in RAM
print("Loaded chunk:", chunk.shape, chunk.dtype, f"from samples [{t0_load}, {t1_load}) out of T={total_samples}")

# pull the 200-sample segment for plotting
seg = chunk[(t0_plot - t0_load):(t0_plot - t0_load + n_plot), :]   # (200, 512)
ei_like = seg.T.astype(np.float32)                                 # (512, 200)  <-- plot_ei_waveforms wants C x T

# choose a ref channel = biggest p2p in this segment
p2p = np.ptp(ei_like, axis=1)
ref_ch = int(np.argmax(p2p))
print("Ref channel (max p2p):", ref_ch, "p2p:", float(p2p[ref_ch]))

# ---- plot like an EI ----
fig = plt.figure(figsize=(25, 18))
ax = fig.add_subplot(111)

pew.plot_ei_waveforms(
    ei_like,
    positions=ei_positions,
    ref_channel=387,
    scale=70.0,
    box_height=1.0,
    box_width=50.0,
    ax=ax
)

ax.set_title(f"Residual trace snippet as EI | samples {t0_plot}..{t0_plot+n_plot-1} ({n_plot} samples)")
plt.show()

### Check a unit in detail (from orig data)

In [ ]:
import glob, os

folder = "/Volumes/Lab/Users/alexth/axolotl/201711290"
cands = glob.glob(os.path.join(folder, "lighthouse_NS_round*.h5"))

assert len(cands) > 0, f"No lighthouse_NS_round*.h5 files found in {folder}"

H5_PATH = max(cands, key=os.path.getmtime)
print("Using:", H5_PATH)

with h5py.File(H5_PATH, "r") as h5:
    print("n_units:", int(h5.attrs["n_units"]))
    print("summary keys:", list(h5["summary"].keys()))
    print("first unit keys:", list(h5["units"]["unit_000"].keys()))
    # quick peek at first 5
    n = int(h5.attrs["n_units"])
    det = h5["summary"]["detect_ch"][:min(5,n)]
    mch = h5["summary"]["main_ch"][:min(5,n)]
    nsp = h5["summary"]["n_spikes"][:min(5,n)]
    print("first rows (detect, main, nsp):")
    for i in range(len(det)):
        print(i, int(det[i]), int(mch[i]), int(nsp[i]))

def load_lh_units(h5_path, load_ei=True, load_bank=True):
    units = []
    with h5py.File(h5_path, "r") as h5:
        n_units = int(h5.attrs["n_units"])
        for i in range(n_units):
            g = h5["units"][f"unit_{i:03d}"]

            u = {
                "unit_index": i,
                "detect_ch": int(g.attrs.get("detect_ch", -1)),
                "main_ch": int(g.attrs.get("main_ch", -1)),
                "dt_main": int(g.attrs.get("dt_main", 0)),
                "n_sub": int(g.attrs.get("n_sub", 0)),
                "valley_low": float(g.attrs.get("valley_low", np.nan)),
                "valley_high": float(g.attrs.get("valley_high", np.nan)),
                "left_count": int(g.attrs.get("left_count", -1)),
                "valley_count": int(g.attrs.get("valley_count", -1)),
                "kmeans_verdict": str(g.attrs.get("kmeans_verdict", "")),
                "kmeans_shared_core": bool(g.attrs.get("kmeans_shared_core", False)),
                "spike_times_main": g["spike_times_main"][...].astype(np.int64),
                "spike_times_center": g["spike_times_center"][...].astype(np.int64),
            }

            if load_ei:
                u["final_ei"] = g["final_ei"][...].astype(np.float32)

            if load_bank:
                u["bank_channels"] = g["bank_channels"][...].astype(np.int32)
                u["bank_templates"] = g["bank_templates"][...].astype(np.float32)
                u["bin_amp_ranges"] = g["bin_amp_ranges"][...].astype(np.float32)

            units.append(u)
    return units

lh_units_loaded = load_lh_units(H5_PATH, load_ei=True, load_bank=True)
print("Loaded units:", len(lh_units_loaded))
print("Example:", {k: lh_units_loaded[0][k] for k in ["unit_index","detect_ch","main_ch","kmeans_verdict"]})

### Do targeted subtraction from raw data to obtain mod data

In [ ]:
import numpy as np
import time, gc

assert "raw_mod" in globals(), "Need raw_mod in memory."
assert "raw_orig" in globals(), "Need raw_orig (original baseline-subtracted) in memory."
assert "lh_units_loaded" in globals(), "Need lh_units_loaded loaded from the last saved round."
assert "extract_snippets_fast_ram" in globals(), "Need extract_snippets_fast_ram."

# --------------------
# CONFIG
# --------------------
STOP_BEFORE = 82        # subtract units 0..81; leave unit 82 untouched
WIN = (-40, 80)
N_BINS = 5
SPIKES_PER_BIN = 5000    # template build spikes per bin
BATCH = 256             # scoring batch size
COS_THR = 0.80          # fit gate (lower -> more aggressive)
USE_MEDIAN = True
RNG = np.random.RandomState(0)

# --------------------
# 0) RESET raw_mod in-place (NO new array)
# --------------------
# This ensures you're not double-subtracting on top of a previously-peeled residual.
print("Resetting raw_mod[:] = raw_orig (in-place, no allocation)...")
raw_mod[:] = raw_orig
gc.collect()

T, C = raw_mod.shape
sr = float(globals().get("sample_rate_hz", 20_000))
print("raw_mod shape:", raw_mod.shape, "dtype:", raw_mod.dtype)

def safe_times(times, win, T):
    times = np.asarray(times, dtype=np.int64)
    times = np.sort(np.unique(times))
    pre, post = win
    ok = (times + pre >= 0) & (times + post < T)
    return times[ok]

def build_amp_binned_templates(raw, times_center, bank_channels, main_ch, win, n_bins=5, spikes_per=800):
    times_center = safe_times(times_center, win, raw.shape[0])
    if times_center.size < max(200, n_bins * 20):
        return None

    bank_channels = np.asarray(bank_channels, dtype=np.int32)
    if bank_channels.size == 0:
        return None

    # amplitude proxy from main channel
    sn_main, vt = extract_snippets_fast_ram(raw, times_center, window=win,
                                           selected_channels=np.asarray([int(main_ch)], dtype=np.int32))
    vt = np.asarray(vt, dtype=np.int64)
    if sn_main.shape[2] == 0:
        return None

    amps = np.ptp(sn_main[0, :, :], axis=0).astype(np.float32)
    order = np.argsort(amps)[::-1]
    vt = vt[order]

    N = vt.size
    B = int(n_bins)
    M = int(spikes_per)

    bin_times = []
    if N >= B * M:
        starts = np.round(np.linspace(0, N - M, B)).astype(int) if B > 1 else np.array([0], dtype=int)
        for s in starts:
            bin_times.append(vt[s:s+M])
    else:
        groups = np.array_split(np.arange(N), B)
        for g in groups:
            bin_times.append(vt[g])

    templates = []
    for bt in bin_times:
        bt = safe_times(bt, win, raw.shape[0])
        if bt.size == 0:
            return None
        sn, _ = extract_snippets_fast_ram(raw, bt, window=win, selected_channels=bank_channels)  # (K,L,N)
        if sn.shape[2] == 0:
            return None
        tmpl = np.median(sn, axis=2).astype(np.float32) if USE_MEDIAN else sn.astype(np.float32).mean(axis=2)
        templates.append(tmpl)

    templates = np.stack(templates, axis=0)  # (B,K,L)
    templates_i16 = np.rint(templates).astype(np.int16)
    return templates, templates_i16

def score_best_template(raw, times, bank_channels, win, templates,
                        time_keep_frac=0.08, weight_power=1.0, use_ch_weight=True):
    """
    Returns times_valid (N,), best_b (N,), best_cos (N,)
    Masked/weighted cosine (like LH) but used ONLY to choose best bin + diagnostics.
    """
    bank_channels = np.asarray(bank_channels, dtype=np.int32)
    templates = np.asarray(templates, dtype=np.float32)  # (B,K,L)
    B, Kk, L = templates.shape

    ref = np.median(templates, axis=0)          # (K,L)
    absref = np.abs(ref)

    time_env = absref.max(axis=0)
    t_thr = float(time_keep_frac) * float(time_env.max() + 1e-12)
    keep_t = time_env >= t_thr

    if use_ch_weight:
        ch_env = np.ptp(ref, axis=1)
        ch_w = ch_env / (float(ch_env.max()) + 1e-12)
    else:
        ch_w = np.ones(Kk, dtype=np.float32)

    W = (absref ** float(weight_power)) * ch_w[:, None]
    W[:, ~keep_t] = 0.0

    mask_idx = np.flatnonzero(W.ravel() > 0)
    if mask_idx.size == 0:
        mask_idx = np.arange(Kk * L, dtype=np.int64)
        w_mask = np.ones(mask_idx.size, dtype=np.float32)
    else:
        w_mask = W.ravel()[mask_idx].astype(np.float32)

    Tf = templates.reshape(B, -1).astype(np.float32)      # (B,D)
    Tw = Tf[:, mask_idx] * w_mask[None, :]                # (B,Dm)
    Tn = np.linalg.norm(Tw, axis=1) + 1e-12

    times = safe_times(times, win, raw.shape[0])
    if times.size == 0:
        return None, None, None

    out_t, out_b, out_c = [], [], []

    for s0 in range(0, times.size, BATCH):
        tb = times[s0:s0+BATCH]
        sn, tv = extract_snippets_fast_ram(raw, tb, window=win, selected_channels=bank_channels)  # (K,L,N)
        tv = np.asarray(tv, dtype=np.int64)
        N = sn.shape[2]
        if N == 0:
            continue

        X = sn.transpose(2,0,1).reshape(N, -1).astype(np.float32)   # (N,D)
        Xw = X[:, mask_idx] * w_mask[None, :]
        Xn = np.linalg.norm(Xw, axis=1) + 1e-12

        dots = Xw @ Tw.T
        cos = dots / (Xn[:, None] * Tn[None, :])

        b = np.argmax(cos, axis=1).astype(np.int16)
        c = cos[np.arange(N), b].astype(np.float32)

        out_t.append(tv); out_b.append(b); out_c.append(c)

    if len(out_t) == 0:
        return None, None, None

    return (np.concatenate(out_t).astype(np.int64),
            np.concatenate(out_b).astype(np.int16),
            np.concatenate(out_c).astype(np.float32))

def subtract_inplace(raw, times, bank_channels, win, templates_i16, best_b, best_cos=None):
    """
    Subtract EVERY spike (control mode). No cos threshold.
    Only skips out-of-bounds.
    Returns (n_subtracted, n_oob_skipped, lowcos_count_if_provided).
    """
    bank_channels = np.asarray(bank_channels, dtype=np.int32)
    templates_i16 = np.asarray(templates_i16, dtype=np.int16)  # (B,K,L)
    pre, post = win
    L = templates_i16.shape[2]   # IMPORTANT: 121 for (-40,80)

    order = np.argsort(times)
    times = times[order]
    best_b = best_b[order]
    if best_cos is not None:
        best_cos = best_cos[order]

    n_sub = 0
    n_oob = 0
    lowcos = 0

    for i, (t, b) in enumerate(zip(times, best_b)):
        t0 = int(t + pre)
        t1 = int(t0 + L)
        if t0 < 0 or t1 > raw.shape[0]:
            n_oob += 1
            continue

        if best_cos is not None and float(best_cos[i]) < 0.90:
            lowcos += 1

        tmpl = templates_i16[int(b)]  # (K,L)
        for ci, ch in enumerate(bank_channels):
            raw[t0:t1, int(ch)] -= tmpl[ci, :]

        n_sub += 1

    return n_sub, n_oob, lowcos

# --------------------
# 1) TARGETED SUBTRACTION LOOP (units 0..STOP_BEFORE-1)
# --------------------
t0 = time.time()
print(f"Targeted peel on raw_mod: subtract units 0..{STOP_BEFORE-1}, leave unit {STOP_BEFORE} untouched.")

for ui in range(STOP_BEFORE):
    u = lh_units_loaded[ui]

    main_ch = int(u["main_ch"])

    # Use centered times if available; otherwise convert main->center by subtracting dt_main
    if "spike_times_center" in u and u["spike_times_center"] is not None:
        times = np.asarray(u["spike_times_center"], dtype=np.int64)
    else:
        times = np.asarray(u["spike_times_main"], dtype=np.int64) - int(u.get("dt_main", 0))

    times = safe_times(times, WIN, T)
    if times.size == 0:
        print(f"[unit {ui:03d}] no valid times; skip")
        continue

    # channels to subtract on: prefer bank_channels from file, else main_ch only
    chs = np.asarray(u.get("bank_channels", [main_ch]), dtype=np.int32)
    if chs.size == 0:
        chs = np.asarray([main_ch], dtype=np.int32)

    out = build_amp_binned_templates(raw_mod, times, chs, main_ch, WIN, n_bins=N_BINS, spikes_per=SPIKES_PER_BIN)
    if out is None:
        print(f"[unit {ui:03d}] template build failed; skip")
        continue
    templates, templates_i16 = out

    t_valid, best_b, best_cos = score_best_template(raw_mod, times, chs, WIN, templates)
    if t_valid is None:
        print(f"[unit {ui:03d}] scoring failed; skip")
        continue

    n_sub, n_oob, n_lowcos = subtract_inplace(raw_mod, t_valid, chs, WIN, templates_i16, best_b, best_cos)
    print(f"[unit {ui:03d}] main_ch={main_ch:3d} | spikes={times.size:5d} | sub={n_sub:5d} | oob={n_oob:4d} | lowcos(<0.90)={n_lowcos:4d}")

    gc.collect()

t1 = time.time()
print(f"✅ Done. Elapsed {t1 - t0:.1f} s")
print(f"raw_mod now has units 0..{STOP_BEFORE-1} peeled. Unit {STOP_BEFORE} is untouched and ready to inspect.")